# Fused multiply-add kernel

Your first Triton kernel. Each **program** (Triton's unit of parallel work, roughly one CUDA thread block) handles one contiguous block of `BLOCK` elements, and every load and store is masked so the last, partial block never touches memory past the end.

Write `fma(x, y, z)` that returns `x * y + z` elementwise, computed by a `@triton.jit` kernel.

## Inputs
- `x`, `y`, `z`: `float32` CUDA tensors of the **same shape**, contiguous. Any shape and any number of elements, including 0 and sizes that are not a multiple of the block size.

## Output
- A new `float32` CUDA tensor with the shape of `x`.
- If `x` has no elements, return an empty tensor without launching the kernel (a grid of 0 programs is an error).

## Requirements
- Treat the tensors as flat arrays of `n = x.numel()` elements: program `pid` covers offsets `pid * BLOCK + [0, BLOCK)`.
- `BLOCK` must be a `tl.constexpr` power of two; launch `triton.cdiv(n, BLOCK)` programs.

This problem runs in **Colab with a GPU runtime** (Runtime → Change runtime type → GPU). Expected outputs were computed with an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def fma(x, y, z):
    """x, y, z: float32 CUDA tensors of one shape. Return x * y + z from a Triton kernel."""
    # TODO: allocate the output, pick BLOCK, compute the grid, launch _fma_kernel
    raise NotImplementedError


@triton.jit
def _fma_kernel(x_ptr, y_ptr, z_ptr, out_ptr, n, BLOCK: tl.constexpr):
    # TODO: offsets for this program, a mask for the tail, masked loads, compute, masked store
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0], "dtype": "float32"}, {"__nd__": [4.0, 5.0, 6.0], "dtype": "float32"}, {"__nd__": [0.5, -1.0, 0.0], "dtype": "float32"}], "e": {"__nd__": [4.5, 9.0, 18.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, -2.0], [0.0, 3.0]], "dtype": "float32"}, {"__nd__": [[2.0, 2.0], [2.0, 2.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0], [1.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[3.0, -3.0], [1.0, 7.0]], "dtype": "float32"}}, {"a": [{"__nd__": [7.0], "dtype": "float32"}, {"__nd__": [-3.0], "dtype": "float32"}, {"__nd__": [1.0], "dtype": "float32"}], "e": {"__nd__": [-20.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [2.040919065475464, -2.5556650161743164, 0.4180988371372223, -0.5677695870399475, -0.4526492953300476, -0.21559716761112213, -2.019986152648926, -0.2319323718547821, -0.8652130961418152, 3.3229994773864746, 0.22578661143779755, -0.3526307940483093, -0.28128743171691895, -0.6680463552474976, -1.0551505088806152, -0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 0.9577587246894836, -0.19980213046073914, 0.02425956539809704, 1.5458208322525024, 0.545105516910553, -0.5052287578582764, -0.18283897638320923, 0.5405251383781433, 1.935088038444519, -0.2696203291416168, -0.24355867505073547, 1.0023136138916016, -0.8864599466323853, -0.2917202413082123, 0.8825389742851257, 0.5803500413894653, 0.09151670336723328, 0.6701043844223022, -2.82816219329834, 1.0213068723678589, -0.9596447348594666, -1.6686198711395264, 0.27644574642181396, 0.7005448937416077, -0.44476744532585144, -1.0764058828353882, 0.026124833151698112, -0.05274730920791626, 1.4055981636047363, 0.7474079728126526, 0.19381564855575562, 1.1116331815719604, -0.2055230438709259, -0.9258995652198792, 0.5840582847595215, 0.5825384259223938, -0.21482890844345093, -0.7828086018562317, 0.22915390133857727, -2.493894338607788, 0.6901247501373291, 0.49136826395988464, -1.638857126235962, 0.06135350838303566, -0.964099645614624, 0.7572210431098938, -2.0341672897338867, -0.914494514465332, 0.7095800042152405, 1.1564010381698608, -2.158005475997925, -0.4980398416519165, 0.3280200958251953, -0.6092161536216736, 1.5906401872634888, -1.1912267208099365, 0.3545319437980652, -1.0484055280685425, 1.4059629440307617, -0.02165122888982296, -0.3722505569458008, -1.7181849479675293, 1.6818255186080933, 0.7527785897254944, 0.7535638213157654, 1.1378812789916992, 0.34922656416893005, -0.6392465829849243, -0.800241231918335, -0.8001999855041504, 1.370072364807129, -1.4603811502456665, -0.5963695049285889, -0.32124391198158264, 0.2246190309524536, 0.5753493905067444, -1.2490969896316528, -1.7300134897232056, -0.004414232447743416, 1.2135637998580933, 0.7570580840110779, 0.2156507819890976, -0.3171556293964386, 0.2932336926460266, -0.2433350831270218, 0.8172065615653992, -0.7944473624229431, 0.13423994183540344, -0.11078013479709625, 0.5433593988418579, 0.22463852167129517, 2.550034523010254, 1.4986547231674194, 1.4967371225357056, -2.039503812789917, -0.34031662344932556, -0.6086106300354004, 0.5327215790748596, -2.279026508331299, 1.1744986772537231, 1.0669833421707153, -1.3020708560943604, -0.978548526763916, -0.8011720180511475, 0.04329590126872063, 0.6409710645675659, 2.0478861331939697, -0.19744543731212616, 0.7675025463104248, 0.1554178148508072, 1.7599263191223145, 0.7421578764915466, 1.368550419807434, -1.077675223350525, -0.19224071502685547, -0.813772439956665, 1.5049474239349365, 0.6576399207115173, -0.30514442920684814, -0.4524678885936737, 0.4846648871898651, -0.7014955282211304, -0.9305889010429382, 0.48127448558807373, 2.46313214302063, -0.24613355100154877, -0.5558657646179199, -1.171156883239746, -1.3350110054016113, 0.5249832272529602, 0.850803017616272, 0.00917472131550312, 0.3325759768486023, 0.11591657996177673, 0.13865484297275543, -1.5261590480804443, -0.4581182599067688, 0.11147930473089218, -0.783166766166687, -0.4764297306537628, -0.8191201686859131, -0.33349665999412537, 0.853108286857605, -0.4065803587436676, -0.15387068688869476, 0.8137183785438538, 0.6447702050209045, 1.6952075958251953, -2.090485095977783, 0.8568589687347412, -0.4822840988636017, 0.13469181954860687, 0.8377193808555603, 1.0832531452178955, 1.0393507480621338, 0.15510669350624084, 1.609662652015686, -0.2829742133617401, -0.14098191261291504, 0.7993512153625488, -0.5513724088668823, 2.1609063148498535, 1.0192065238952637, 2.1755752563476562, -0.02658918686211109, -0.3830885589122772, 0.16704800724983215, 0.7345765233039856, -0.5874276757240295, 0.37970173358917236, -0.016804279759526253, 1.615681767463684, -0.6627017855644226, 1.0461617708206177, -0.6438635587692261, -0.960634171962738, -0.7102994322776794, -1.1901859045028687, 0.14635035395622253, 1.0312608480453491, 0.1643148809671402, 0.6243250966072083, 1.6321742534637451, 0.27002644538879395, 0.1951623558998108, -0.2750614583492279, -1.6082422733306885, 0.7597389221191406, -1.756460428237915, 0.6526764631271362, -0.01432033535093069, 1.1267814636230469, -0.06777025759220123, -0.8232001662254333, 0.35791027545928955, -0.5602361559867859, -0.18095682561397552, 0.04187078773975372, -0.13454970717430115, -0.18880227208137512, -0.832444965839386, -0.18913377821445465, -2.138340711593628, -0.15733209252357483, -1.1980559825897217, 1.120263695716858, 1.2699267864227295, -1.9510257244110107, 0.14491678774356842, -0.12636391818523407, -1.0467156171798706, 0.5316531658172607, -0.4616895318031311, -1.767599105834961, -0.266676664352417, -0.14825311303138733, 0.1064310148358345, -1.23123300075531, 0.6156809329986572, 0.7354570627212524, -1.145827293395996, -0.6588555574417114, -0.08033731579780579, -0.5659253001213074, 1.7437546253204346, 0.2083721160888672, -1.0105820894241333, -0.7881089448928833, -0.057470258325338364, 2.295778751373291, -0.178264319896698, 0.12747642397880554, 0.5140453577041626, -0.04012593626976013, 2.28066086769104, -0.5315249562263489, 0.7442031502723694, 0.16043797135353088, 0.6211963891983032, -1.1766430139541626, 1.7681126594543457, -0.12152750790119171, 0.03416326269507408, -0.9349762797355652, -0.7296952605247498, 0.5642394423484802, 0.9864999055862427, 0.7532547116279602, 1.208970308303833, 0.7144889235496521, 0.02846183255314827, 0.8365259766578674, 0.593603253364563, -0.1005752682685852, 0.7260727286338806, 1.285697340965271, 0.23457132279872894, -0.3562051057815552, 0.7187113761901855, 1.900721788406372, -0.21056205034255981, -0.0923096239566803, -0.13639752566814423, 1.2230174541473389, -1.837198257446289, 0.36584538221359253, 1.1922639608383179, -0.813383162021637, 1.4889774322509766, 0.5353090167045593, -0.549035906791687, 0.2041398137807846, -1.5344921350479126, -0.5655710101127625, 1.8269038200378418, -0.8938040733337402, 1.8445717096328735, -0.08157359063625336, 0.994662880897522, 0.04389702156186104, -2.2414815425872803, -0.5726513266563416, 0.18985359370708466, -1.1509497165679932, -1.3474106788635254, 0.36618584394454956, -0.6494150161743164, -1.6958272457122803, -0.6841151714324951, 0.8607375621795654, -0.47374606132507324, 0.9347725510597229, 1.5272728204727173, 0.032515399158000946, -1.1296833753585815, -1.3015650510787964, -0.026253854855895042, 0.8810556530952454, 0.2690153419971466, 0.5472571849822998, -0.44334179162979126, 0.3004227578639984, -0.47130876779556274, -0.32560989260673523, -1.2922464609146118, -1.572213053703308, -0.48195669054985046, -1.1075993776321411, -0.9587457180023193, 0.6710474491119385, -0.10708092153072357, 2.914125919342041, 0.9263420701026917, -0.6331875920295715, 0.7780073285102844, 0.35680389404296875, -0.6856811046600342, 1.1025949716567993, -0.7753286361694336, -3.0583813190460205, 0.7940647602081299, -0.61725914478302, 1.4860374927520752, -0.6496722102165222, -1.181354284286499, 1.548461675643921, -1.075498342514038, 0.20339369773864746, 1.449727177619934, 0.1144973412156105, -0.08058491349220276, 0.10242343693971634, 0.8390536308288574, -0.8068121671676636, 0.6106410026550293, 0.5770754218101501, 2.332711935043335, -0.3365255296230316, -0.9148298501968384, 0.7356870174407959, -0.3971572518348694, 0.28911617398262024, 0.13257652521133423, 1.1683939695358276, -0.6568374633789062, -0.09227953106164932, -2.552112579345703, -0.890113890171051, 1.5363388061523438, -0.6815059185028076, -0.7014768123626709, -1.217907190322876, 0.5798342823982239, 0.1779901683330536, -1.1868668794631958, 0.6674503684043884, 0.2601058781147003, 0.7540333867073059, -0.6414794921875, 1.738008737564087, 0.40986761450767517, -0.11688951402902603, -0.5188342928886414, -0.24482737481594086, 0.8587920069694519, -0.2689667046070099, 1.078143835067749, 0.5806761980056763, 1.3464951515197754, 0.8141041994094849, -0.06293926388025284, -1.9093399047851562, 0.29857057332992554, 0.789076030254364, 0.16289228200912476, 0.5564317107200623, -0.471620112657547, -0.38047757744789124, -0.9423890709877014, 1.3619004487991333, 0.15839426219463348, -0.4924860894680023, -1.3318525552749634, -0.15330727398395538, 1.4426312446594238, 0.4619216322898865, -0.7245215773582458, 1.4294949769973755, 1.4285101890563965, 0.9807761907577515, 0.7888168692588806, 1.1556518077850342, 0.5432590246200562, 0.775000810623169, -0.7800253629684448, 1.4995578527450562, 0.23118019104003906, 2.024211883544922, -2.042248010635376, 0.7054088711738586, 0.9814615249633789, -0.3429326117038727, 0.45244431495666504, 0.7483264207839966, 0.5773814916610718, -0.5396592617034912, -0.14193783700466156, 1.019039511680603, 1.2773200273513794, 0.09897684305906296, 0.07703247666358948, 0.2004963606595993, 1.0376652479171753, -1.0540385246276855, -1.3290657997131348, 0.12431496381759644, -1.1065287590026855, -0.5871094465255737, 0.08629573881626129, 0.4825049340724945, -1.072091817855835, 0.8016054630279541, -1.7737146615982056, 0.5454160571098328, -1.577731728553772, -1.2749234437942505, 0.6422219276428223, -0.40636223554611206, -0.7955582737922668, -0.4447767734527588, 0.4650326073169708, -0.05404679477214813, -0.029382774606347084, 1.0547764301300049, -0.7163906097412109, -0.4719336926937103, -0.5224300026893616, 1.1689221858978271, 0.9914056658744812, 1.2870765924453735, -0.03375060856342316, -0.15680254995822906, 0.8527730107307434, -0.39441168308258057, 0.8306663632392883, 0.7242927551269531, -0.6983180046081543, -0.08319466561079025, -0.11925099045038223, 1.7333403825759888, 2.5878233909606934, -0.09992724657058716, 0.01406467892229557, -2.2815845012664795, 0.21153229475021362, -0.007032972294837236, -1.0728437900543213, 0.7855942249298096, -1.5657538175582886, -0.2148529589176178, 0.13885676860809326, -1.2489999532699585, 0.8567407131195068, 1.0555617809295654, -1.3864208459854126, -1.4078385829925537, 0.6022594571113586, -0.030497003346681595, -1.2114031314849854, -0.29522672295570374, -0.10751668363809586, -0.14303892850875854, 0.8274886608123779, 0.38419824838638306, 1.7694637775421143, 1.37436044216156, 0.6784737706184387, 1.07707941532135, 1.323819637298584, 0.7059948444366455, -1.225742220878601, 0.8594171404838562, 0.07227258384227753, 1.618811011314392, 0.09347418695688248, 0.5259924530982971, 0.32963991165161133, 1.6812255382537842, 0.17718231678009033, -0.26953989267349243, 0.30909180641174316, -0.43950825929641724, 0.16203923523426056, -0.007472030818462372, 0.27162256836891174, 1.9232840538024902, -0.40569058060646057, 0.03697320818901062, -0.19949175417423248, 0.6726115345954895, -1.1815745830535889, -0.6927294731140137, -0.03948960080742836, 0.5461478233337402, -0.04628121107816696, -0.5002862811088562, -0.8668274283409119, 0.5117526054382324, 1.442198634147644, 0.3228754997253418, 0.188135027885437, 0.44387301802635193, -0.9426529407501221, -0.31167489290237427, -0.267926424741745, 0.29524046182632446, -0.032025691121816635, -1.2164000272750854, 1.125229001045227, 1.6924084424972534, -0.263197124004364, 0.39884087443351746, 0.2671615779399872, 0.7846636176109314, -1.479783058166504, 1.6130096912384033, -0.46785232424736023, 1.2642261981964111, 0.08067114651203156, -0.3895556330680847, -0.14045269787311554, 0.8971729278564453, -0.8632761240005493, -0.37517791986465454, -0.48218023777008057, -1.7154483795166016, 1.3004252910614014, 0.5788888335227966, -0.021315021440386772, 1.1636626720428467, -0.9564298391342163, -0.07344981282949448, 1.72540283203125, 0.9796664118766785, -2.2254505157470703, -3.3320813179016113, -1.037903904914856, 1.0612833499908447, -0.4724457859992981, -0.21774406731128693, 0.6082714200019836, -0.8852022886276245, 0.21287789940834045, 0.7920982241630554, 0.4670132100582123, 0.3959454596042633, -0.6020461320877075, 0.5142871737480164, -0.42763006687164307, 1.3049468994140625, 2.5700926780700684, -0.7399058938026428, -2.17450213432312, -0.42543989419937134, 0.376845121383667, 1.0501313209533691, 1.0790154933929443, -0.7839513421058655, -0.7932353615760803, 1.2673009634017944, -1.0923876762390137, 1.0902312994003296, -0.22688792645931244, 0.7664036154747009, 0.7061647176742554, 0.776652455329895, 0.8204267024993896, -0.9893909692764282, 0.29752111434936523, 0.013982411473989487, 0.3084080219268799, 1.2332483530044556, -0.2711082398891449, -0.7119704484939575, -0.7281154990196228, -0.3108183741569519, 0.7526105642318726, -0.8148156404495239, 1.1592578887939453, 0.2904514670372009, -0.22611092031002045, -1.331650972366333, 0.24493318796157837, 0.6914693117141724, -1.7657657861709595, 1.1919596195220947, 1.6477470397949219, 0.33850228786468506, 0.4917809069156647, 2.4136950969696045, 1.2318906784057617, -0.8977571725845337, -0.01664518564939499, -0.156931072473526, -0.15087679028511047, -2.3642678260803223, 0.7570048570632935, -0.6367360949516296, 0.1636514663696289, -0.13613934814929962, 0.8710953593254089, -0.29580870270729065, -0.7954280376434326, 0.5324224829673767, 1.4048856496810913, -0.319810688495636, 1.3699480295181274, 0.26218315958976746, -0.5327295064926147, 1.5251578092575073, -0.818626344203949, 1.0140482187271118, -0.4981859624385834, 0.45021864771842957, -1.2341482639312744, -0.837592601776123, 1.0650522708892822, 0.18811684846878052, 0.0382930152118206, -0.4398082494735718, 0.4464806616306305, -1.3972692489624023, -1.1199352741241455, -0.15779036283493042, -0.8342810273170471, -0.7005937695503235, -0.07150325924158096, -0.13752132654190063, -0.22250670194625854, 1.6652755737304688, 1.0110758543014526, 1.2297345399856567, -0.028162000700831413, -0.5046842694282532, -1.4538991451263428, -0.34813740849494934, -2.0859556198120117, -0.583741307258606, 0.7642389535903931, 1.621530294418335, 0.9656069278717041, -0.8616004586219788, 0.49261853098869324, -0.7891640067100525, 0.8229737877845764, -0.11136118322610855, 0.8582001328468323, 2.5728955268859863, 0.9225243330001831, 2.3324499130249023, 0.17744231224060059, -0.3297177255153656, 1.3337043523788452, 1.7243565320968628, -0.5550146102905273, 0.07663196325302124, -1.4345295429229736, -0.2000458836555481, 0.8883322477340698, -0.0863848403096199, -0.5692201852798462, -1.4072399139404297, 0.36825451254844666, 0.7975406646728516, 0.5965006947517395, 0.2553156912326813, -1.274065375328064, 1.7618086338043213, -0.056536365300416946, -0.1447964459657669, 0.7329285740852356, -2.1631863117218018, -0.06446723639965057, -0.13678960502147675, 0.37228313088417053, 1.3471598625183105, 0.4722001254558563, -1.832098364830017, -0.3390767574310303, 0.20553170144557953, 0.14716723561286926, -1.2812628746032715, -1.6637585163116455, 0.5490108728408813, 0.34353509545326233, -0.021301068365573883, -1.7254220247268677, 1.7740503549575806, -2.900233745574951, -0.3640972971916199, 2.091151475906372, -0.1574258655309677, 0.2320702224969864, -1.179146409034729, 1.3586400747299194, -1.2926229238510132, 0.2544191777706146, 0.1349056512117386, -1.0507245063781738, 0.8869794011116028, -1.415958046913147, -0.6834845542907715, -0.028702203184366226, -0.41249769926071167, 0.4678066670894623, 0.9240548610687256, -0.2937737703323364, -0.5414659380912781, -0.5169782042503357, -0.36559394001960754, 0.809648871421814, -0.2175850123167038, 0.4076395332813263, -0.09115277975797653, -0.8444651961326599, 0.8637197613716125, -1.0380412340164185, 0.006197796668857336, 1.4870238304138184, -1.093106985092163, -0.05518852174282074, -1.5847810506820679, -0.5094162821769714, -1.49459969997406, -0.36158716678619385, -0.5382974743843079, 0.13197161257266998, 1.5843008756637573, 0.28298452496528625, 0.658174455165863, -0.050701938569545746, -0.24452361464500427, -0.8339818120002747, -1.8008568286895752, 0.06862036883831024, 1.629123568534851, 0.47629988193511963, -0.5339889526367188, 1.6494061946868896, -0.4326235055923462, -0.6188397407531738, -1.2126944065093994, -0.9620509147644043, -0.2463628649711609, 1.9990206956863403, -0.20617620646953583, -1.0315163135528564, -0.6054186224937439, -0.22286176681518555, -1.711751937866211, 0.12662842869758606, 0.7006685137748718, 0.32081225514411926, 0.6007326245307922, -1.289547085762024, -1.698591947555542, 1.2855745553970337, 1.0943313837051392, -1.5402644872665405, 0.2063816636800766, 0.025527870282530785, 0.5448862910270691, 0.2085862159729004, -0.5021318793296814, 0.19525663554668427, 0.15704688429832458, -0.04862762987613678, -1.8524186611175537, -2.9952011108398438, -0.7901887893676758, -1.634771466255188, 0.2525620460510254, 0.7160966992378235, 0.26311543583869934, -0.3695279061794281, 0.0704808458685875, 1.1673409938812256, -0.7288080453872681, 0.9222947359085083, 0.7076163291931152, 0.26168426871299744, 1.4519301652908325, -0.717818558216095, 0.9425060153007507, -0.9289515614509583, -0.7670315504074097, -1.0597110986709595, 0.4458315968513489, -0.9397600293159485, 0.05215989798307419, -1.1367119550704956, 1.0662575960159302, 1.8361735343933105, 0.2260766476392746, 1.6433653831481934, 0.560663640499115, 2.1693100929260254, 0.0997142642736435, -1.5704236030578613, 1.4905840158462524, -0.12636713683605194, -1.0358450412750244, -0.6562658548355103, -1.6263316869735718, -0.1677018105983734, -1.0712637901306152, 1.4841805696487427, 0.6644920110702515, -1.6604059934616089, -1.509372353553772, -0.5669757723808289, -1.9250410795211792, -0.10980580747127533, -0.04344815015792847, -0.7728063464164734, -0.5464943647384644, 0.49885284900665283, 1.4216846227645874, 1.4956731796264648, 0.03430883213877678, 0.8558490872383118, -0.7383553981781006, 0.41195377707481384, -0.9596719741821289, 0.7295733690261841, -1.238401174545288, -0.9650830626487732, -0.00834464468061924, 0.5569891929626465, -1.4381744861602783, -0.5366385579109192, 0.13515105843544006, 0.2290463000535965, -0.838799238204956, 0.09688255935907364, -0.2672370374202728, 1.4038292169570923, 0.556898295879364, -1.324095368385315, -2.201557159423828, 0.09988179802894592, 1.182084560394287, 0.17374005913734436, 0.882066011428833, -0.484673410654068, -1.3639205694198608, 0.7485937476158142, 0.09894227236509323, 0.683682918548584, -1.3815659284591675, 0.1815534383058548, -0.6255009770393372, 0.8065888285636902, -0.5806873440742493, -0.6073948740959167, -0.5857203602790833, 1.2637170553207397, 0.028864195570349693, -0.7599971294403076, -1.0225841999053955, -1.0441224575042725, 1.4626275300979614, -1.814774513244629, -0.8503575325012207, 0.6378903388977051, 1.1784957647323608, -0.1482774019241333, 0.785516083240509, 0.759617269039154, 0.9861871004104614, 0.04641503095626831, 1.6549681425094604, 0.21612536907196045, 1.5482001304626465, -1.0827884674072266, 0.9606130719184875, -1.2350369691848755, -1.4755833148956299, 0.5254991054534912, -0.7997444868087769, 0.47699373960494995, -1.0392570495605469, 1.6403754949569702, 0.04541300609707832, -1.0942738056182861, 0.21565909683704376, 0.924079954624176, -0.7268860936164856, 0.5617496967315674, 2.0933477878570557, 0.9154857993125916, 0.03924795985221863, -0.5195255875587463, 0.6291380524635315, -1.4644056558609009, -2.986478090286255, -0.6133970022201538, -1.0284662246704102, 1.799839735031128, 0.1958865523338318, -0.9284030795097351, -1.1052078008651733, 0.5581414699554443, -0.7712110280990601, -0.5223010182380676, -0.030369741842150688, 0.15036295354366302, 0.16054552793502808, 0.17942208051681519, 1.3678315877914429, 1.768678903579712, -1.5849144458770752, 2.6462831497192383, 0.19913043081760406, 1.314860224723816, -0.6682167649269104, 0.3973124921321869, 0.28750643134117126, 0.18613076210021973, 0.9257142543792725, 0.7057353854179382, -0.9314151406288147, -0.043422214686870575, 0.8034627437591553, 0.7619158625602722, -0.26096925139427185, -0.20561031997203827, -0.07871096581220627, -1.0290894508361816, -1.2122299671173096, 1.6446222066879272, -0.31982892751693726, 0.17068354785442352, -0.672717809677124, 1.3727784156799316, 1.8276342153549194, -0.14458514750003815, 0.5897448658943176, -0.6796556115150452, 1.8403476476669312, 1.001141905784607, -0.7646186947822571, 1.3231734037399292, 0.4352547824382782, 0.48870596289634705, 0.43224677443504333, -0.5942724347114563, -0.4640330970287323, 0.6962088942527771, 0.023503541946411133, -0.8758109211921692, 2.103349208831787, -0.13811509311199188, -0.4884083867073059, -0.6271817088127136, 0.7604384422302246, 1.0446656942367554, 0.3153950572013855, 0.9384251236915588, -1.4301458597183228, -0.3031352162361145, -0.40237241983413696, 0.07267668098211288, 0.5553210973739624, -1.9734467267990112, 3.1912713050842285, -0.8026558756828308, 0.40858787298202515, 0.8906661868095398, 0.9132422804832458, 0.3015294671058655, -2.851038694381714, -1.0294182300567627, 0.815044641494751, -0.8672524094581604, -1.0034019947052002, -2.304955244064331, 1.266568899154663, 0.5565275549888611, -0.8489745259284973, -0.09093708544969559, -0.06102960556745529, -1.0791676044464111, -0.2390943467617035, 0.28724735975265503, 0.5837607383728027, 1.2956645488739014, 0.05704819783568382, -0.4357512593269348, 0.4279448390007019, -0.6054584980010986, -1.2284018993377686, 1.165817379951477, -0.3596625328063965, 0.1418885886669159, 0.6701045036315918], "dtype": "float32"}, {"__nd__": [-1.087502121925354, 0.6610705852508545, 0.4700078070163727, -0.07869519293308258, 0.413828581571579, -0.737575352191925, -2.1866278648376465, -0.26442113518714905, -0.8926159143447876, 0.8870077729225159, 0.2828402817249298, -0.5116117000579834, 0.0655950978398323, 1.3710854053497314, 1.1530663967132568, 0.38690221309661865, 0.44852638244628906, -0.26040804386138916, -0.11286324262619019, 0.6066693663597107, -0.10218464583158493, 0.3161347210407257, -1.0989238023757935, -1.6523231267929077, -1.682691216468811, -0.4212658107280731, 1.3442906141281128, 1.1995958089828491, -0.9814296364784241, -1.5408070087432861, 0.8088977932929993, 0.4789857864379883, -1.353267788887024, 1.0735628604888916, 0.19024643301963806, 1.3394519090652466, 0.3579726219177246, 0.9059391617774963, 0.33684447407722473, -0.9376466870307922, -0.2178904265165329, -0.6630188226699829, 0.8452523350715637, -0.981764554977417, -1.519432783126831, 0.6968106031417847, 1.1609312295913696, 1.4599090814590454, 1.1067994832992554, -1.4944950342178345, -0.6837617754936218, -0.5355051159858704, -0.010022813454270363, -2.1419713497161865, 0.002466826234012842, 0.2888009548187256, -0.9390360116958618, 0.12963184714317322, 1.2030428647994995, 0.05754097178578377, 1.068389892578125, 1.0254358053207397, 0.4216483533382416, 0.39493289589881897, -0.3604743480682373, 1.2524467706680298, -0.45974308252334595, -0.6860493421554565, 0.7173232436180115, -0.7147547006607056, 0.43736329674720764, 1.0686253309249878, 0.09235601127147675, 0.11751622706651688, 0.9307212233543396, 0.7662864923477173, -1.5274426937103271, -1.2976537942886353, 1.3449342250823975, 0.4041241705417633, -2.067204236984253, -1.3341705799102783, 0.04009436070919037, -0.9646182656288147, 0.48522254824638367, 0.09518151730298996, 0.5078786015510559, -1.1506434679031372, 0.6393343210220337, 1.978328824043274, -0.7758641839027405, 1.3338230848312378, 1.0021175146102905, 0.6649901270866394, -0.04457911476492882, -1.1405709981918335, -0.8561702966690063, 0.7861270904541016, 1.0016069412231445, 1.2456564903259277, 0.9835174679756165, 0.9265053272247314, -0.41126105189323425, -1.1844086647033691, 0.22776035964488983, 0.5240649580955505, 0.1097026914358139, -1.4964087009429932, -0.4962444603443146, -1.4530391693115234, 0.16998548805713654, -1.0258519649505615, -0.142476424574852, 0.3472633957862854, 0.1621048003435135, 1.3936094045639038, -1.0797241926193237, -2.1580617427825928, 0.9708937406539917, 1.428524136543274, 1.4587852954864502, 0.3118498623371124, 0.5667908787727356, 0.15065422654151917, -1.266661286354065, -0.5423584580421448, 0.5257276892662048, -0.5477734804153442, 1.9372409582138062, -1.5109803676605225, -0.4037870466709137, 1.1405305862426758, -0.5772757530212402, -1.1019749641418457, -1.3286824226379395, -0.6961856484413147, 1.5409595966339111, 0.4400498569011688, 0.5478272438049316, -0.4877318739891052, 0.8760697245597839, 0.13884399831295013, -0.8888521194458008, -0.41992905735969543, 0.23508520424365997, 1.2034703493118286, -0.025284837931394577, 0.5330597758293152, 2.052851438522339, -0.7653564214706421, -1.2071739435195923, 1.0169624090194702, 0.4721352756023407, -0.06420072913169861, 0.298200786113739, -0.29524552822113037, 0.5504210591316223, -0.30122679471969604, 0.8346636891365051, -1.1357535123825073, -0.7816544771194458, 0.1515355259180069, -0.4345260262489319, 0.7310964465141296, -0.006476094946265221, -0.5063658952713013, 0.24596531689167023, -0.49146533012390137, -1.520829439163208, -1.0989676713943481, -0.764117956161499, -0.4321034252643585, 0.13610687851905823, 0.9499886631965637, 0.591343343257904, 1.8206182718276978, 1.3160899877548218, -0.1465705782175064, 1.6935900449752808, 1.2241992950439453, -0.40974128246307373, 0.35544049739837646, 1.1388827562332153, 0.28662607073783875, 1.06144380569458, 0.44097617268562317, 0.7493008375167847, -0.2722027003765106, 0.4252520203590393, -0.31735706329345703, 0.7226077914237976, -1.6420605182647705, 0.38173675537109375, 1.297328233718872, 0.6274482011795044, 0.021638665348291397, 0.5861490368843079, -0.798447847366333, -0.7843064069747925, -0.31168362498283386, -0.1006540060043335, 1.013187050819397, -0.8469995260238647, 1.0477770566940308, -1.5258344411849976, -0.7400428056716919, 0.010092291980981827, 1.2045551538467407, -0.5668904781341553, -2.0968539714813232, 0.46663737297058105, 0.017535120248794556, -1.0356242656707764, -0.9973533749580383, -0.649196982383728, 0.144624724984169, 0.709801971912384, -1.4321069717407227, 2.3350119590759277, 0.00820211973041296, -0.32695433497428894, 0.7186461091041565, 0.1545155793428421, 2.3817873001098633, 1.1414964199066162, -0.32327789068222046, -0.3454863727092743, -0.45306384563446045, -0.29686376452445984, 0.7174218893051147, -0.6014658212661743, 1.657326102256775, -0.037853263318538666, -0.1016811951994896, 0.992730975151062, -1.9356498718261719, -0.8924553990364075, 0.6421632766723633, 0.05458779260516167, 0.0309041365981102, 0.43976300954818726, -0.558366060256958, -0.1283116638660431, -1.3187379837036133, -1.833094835281372, 1.3067233562469482, -0.8911605477333069, -0.6746698021888733, 1.4376716613769531, 1.100531816482544, -0.48571476340293884, -0.490479975938797, -0.04172052443027496, -0.9150159358978271, 0.5880361199378967, 0.1845998913049698, 0.1689651757478714, 0.7926115393638611, 0.757807195186615, 1.8810261487960815, 1.5556578636169434, -1.6711822748184204, 0.9366335868835449, -1.0798875093460083, -1.3894935846328735, 0.7497140169143677, -0.663746178150177, -0.27746066451072693, 1.1084840297698975, -0.6249480247497559, -0.8568428754806519, 0.3652093708515167, 1.2615996599197388, 0.5671926736831665, -1.5152347087860107, -0.6126543879508972, -0.11370231956243515, 1.9686957597732544, 0.9317599534988403, 0.8898720145225525, 1.2413420677185059, -0.2706436216831207, 0.5181459188461304, 0.5287922620773315, -0.938443660736084, 0.02170277014374733, 0.6026281714439392, -1.0412603616714478, 0.8466441631317139, 0.7972095608711243, 1.0722075700759888, -0.7363685369491577, 0.14570656418800354, 1.4916578531265259, 0.8310011029243469, -0.7500245571136475, 0.263302743434906, 1.458528995513916, -0.038349635899066925, 0.4187130331993103, 2.606179714202881, 0.0004128905711695552, -0.0708533301949501, -0.9691065549850464, -0.6225593090057373, 1.5211036205291748, -0.10413961857557297, -1.15211820602417, -0.8976781368255615, 0.03908656910061836, 0.7977871894836426, -0.7123287916183472, 0.7443654537200928, -0.45831558108329773, 0.3848016858100891, 0.5471117496490479, 0.6337680816650391, -0.40673431754112244, -0.14548945426940918, 0.5283980965614319, -1.273240566253662, 0.19109147787094116, 0.5307928323745728, 1.0592479705810547, -0.023392777889966965, -0.17200535535812378, 0.1524926722049713, -0.7232627272605896, -0.20064637064933777, 0.4159854054450989, 0.28739240765571594, -1.0392892360687256, -0.32265856862068176, -2.111755132675171, 0.801702618598938, -0.532426655292511, 0.2373330146074295, -1.102174997329712, -0.08905167877674103, 0.4396975636482239, 0.9738321900367737, -1.3697545528411865, 0.8326238393783569, -0.1757824867963791, -0.7873982191085815, 0.46329113841056824, -0.3082805871963501, -0.3110032379627228, -0.7189329862594604, 0.20695945620536804, 0.01984628476202488, -0.19113457202911377, -0.9731241464614868, -0.5300232768058777, 0.3835427761077881, 1.2592462301254272, -0.04607333987951279, 1.3936539888381958, -0.030390454456210136, -1.1096733808517456, 0.6112676858901978, -0.9851086735725403, 0.38207483291625977, 1.0499378442764282, -0.3277072310447693, -0.10802119970321655, 2.4462101459503174, 0.21949394047260284, 0.24460020661354065, 0.761101245880127, -0.11728048324584961, -0.10719336569309235, 0.36194998025894165, -2.4084315299987793, 1.3955397605895996, -1.0846890211105347, -1.0954275131225586, 0.5201892256736755, 0.008816112764179707, -1.266526222229004, 0.19369088113307953, -1.156624436378479, -0.7133705019950867, -1.5188568830490112, -0.8435394167900085, 0.3790154755115509, 0.10514663904905319, 0.2910861670970917, 0.9027455449104309, -0.06060607358813286, -0.6768487095832825, -0.2972157895565033, 1.6273870468139648, 0.015504619106650352, 0.30083930492401123, -0.11406422406435013, 0.22672437131404877, 0.4124878942966461, -0.027173079550266266, -0.172787144780159, -0.7982817888259888, -0.8626048564910889, 0.38799381256103516, 0.7287136912345886, 0.9030340909957886, 0.03557128831744194, -0.841515839099884, 0.6105508804321289, 0.5961484909057617, -1.5178544521331787, 0.7828530669212341, 0.5630512833595276, 1.6445955038070679, -1.1425522565841675, -1.2950359582901, 0.7669473886489868, -1.8088688850402832, -0.7460106015205383, -1.1059008836746216, -1.0267740488052368, -1.917872428894043, -0.252991646528244, 0.9571146965026855, 0.4253012537956238, 0.42965278029441833, -0.3706599771976471, 1.4285941123962402, 0.513006865978241, 1.0352050065994263, -1.5575213432312012, 0.33247533440589905, -1.1505359411239624, 0.3378314971923828, 0.18840168416500092, -1.1319832801818848, 0.16810542345046997, 1.1498833894729614, 0.49358564615249634, 0.5839363932609558, -0.8643968105316162, -0.6808387637138367, -1.111741304397583, 0.12878969311714172, 1.1826921701431274, 0.43184301257133484, -0.0034248074516654015, -1.1165767908096313, 0.44173526763916016, -0.23774702847003937, -1.8976415395736694, -1.7217886447906494, -0.3377601206302643, -0.8907064199447632, 1.64602792263031, 0.03591098636388779, -1.0325337648391724, -0.4844639003276825, 1.5880379676818848, -0.9419392943382263, 0.7695557475090027, 0.43886658549308777, 1.465262770652771, 0.5318866968154907, -0.39721816778182983, 0.5285932421684265, 0.07031619548797607, 0.9742986559867859, -0.05680607259273529, 1.9751167297363281, -0.09490149468183517, 0.44469738006591797, -0.6460459232330322, 0.13374091684818268, 0.036246493458747864, 0.17917950451374054, 0.9070638418197632, -0.5239865779876709, -0.8315182328224182, 1.1150941848754883, -0.272482693195343, -0.6534436941146851, 1.1615248918533325, 0.7875040173530579, -2.6181998252868652, -0.36186227202415466, 0.7963047027587891, 0.8459317088127136, -0.4945407211780548, 1.5728285312652588, -0.7711960673332214, -0.21201692521572113, 0.14625568687915802, 0.014140341430902481, 0.8223953247070312, 1.0362454652786255, -0.0231077391654253, 1.4129376411437988, -0.6593137383460999, -0.1894417554140091, 1.7349375486373901, 0.12129327654838562, 1.08705735206604, -0.4028674066066742, -0.23735825717449188, -0.07818762958049774, 0.19849228858947754, -0.30805739760398865, 0.04580876976251602, 0.6959390044212341, 1.2295970916748047, 0.1392023265361786, 0.6767752766609192, 0.9390639066696167, -1.0389410257339478, -1.5325171947479248, 1.6407750844955444, 1.345535159111023, 0.09034295380115509, 0.38273125886917114, -0.9037477970123291, -1.3427045345306396, 0.34758979082107544, -1.445356845855713, 0.9417096376419067, -0.643819272518158, 0.3217112421989441, 1.9572620391845703, 1.910284161567688, -0.23866190016269684, 1.0937919616699219, -0.9944708347320557, -0.5517165064811707, 0.3992884159088135, 1.4106777906417847, -0.5413222908973694, 0.5688710808753967, 0.13061578571796417, -1.2283730506896973, -0.10485062748193741, 1.5210869312286377, -0.0637921467423439, 0.5277710556983948, 0.8529094457626343, -0.7059682607650757, 1.2532442808151245, 1.5241742134094238, 1.6178134679794312, 1.3345118761062622, -0.43692368268966675, -0.5579150915145874, -0.7572281956672668, -0.8790299296379089, -0.7681241631507874, 1.0123260021209717, 1.4809725284576416, -0.6153769493103027, 0.8306652903556824, -1.2391290664672852, -0.30372118949890137, -0.9720001816749573, -1.603305697441101, 0.07313946634531021, 0.09654352813959122, -0.3884628117084503, -0.5046857595443726, -0.9667872190475464, -0.11661641299724579, -0.584105372428894, 0.25431448221206665, -1.189256191253662, 0.23966781795024872, 0.522671103477478, 1.2523924112319946, 1.0603009462356567, 0.20144441723823547, -0.6709134578704834, -1.8168476819992065, 0.0026918591465801, 1.4207584857940674, 1.665061116218567, -2.529310941696167, -0.7472798824310303, -0.3420387804508209, -1.0319069623947144, 0.1937917172908783, 0.04599625617265701, -0.5092268586158752, -0.028704989701509476, -1.58358895778656, -0.9637572765350342, -0.8882927894592285, 0.9028779864311218, -1.6191641092300415, -0.20645304024219513, 0.783644437789917, -0.04793410748243332, 0.8168278932571411, -0.6461483836174011, 0.441234827041626, 0.9096816778182983, 2.2376174926757812, 0.7338247299194336, 1.4342801570892334, 0.01947159692645073, 0.1071014329791069, 1.5773988962173462, 0.5929288268089294, 1.428602933883667, 0.7207574248313904, 2.1328043937683105, 2.394803047180176, -0.8218938708305359, 1.8732020854949951, 0.4500902593135834, 0.49367761611938477, -0.09755853563547134, 0.7372644543647766, -1.0916911363601685, -0.7062584757804871, -0.7429202795028687, 0.3118625581264496, 0.9444763660430908, 0.7609955072402954, 0.39364364743232727, -0.5666345357894897, -1.733282208442688, 0.43093422055244446, 0.17362281680107117, 0.3855600953102112, 1.4560154676437378, -0.15326151251792908, -0.1728496253490448, -2.2867226600646973, 0.5885845422744751, -0.019332516938447952, -0.7095016241073608, 0.41531333327293396, -1.0146247148513794, -0.5123655796051025, 1.9919570684432983, 0.7810935378074646, 0.9548371434211731, 0.7776476740837097, 0.6709345579147339, 0.22161290049552917, 2.3129711151123047, -0.0466807559132576, 0.7619715332984924, -0.1926318258047104, -0.3258672058582306, -2.4131600856781006, -1.8994190692901611, 1.571799635887146, 1.511946439743042, -0.4144875705242157, -0.010676424950361252, 0.6338039040565491, -0.9740737080574036, 1.7892078161239624, -1.0210282802581787, -0.7019392848014832, -2.745591402053833, 1.5320641994476318, 0.0748327374458313, 1.6393996477127075, -0.9860650897026062, 0.35382410883903503, 0.09005791693925858, 0.21417666971683502, 2.0897886753082275, 2.2807819843292236, 1.3934330940246582, -0.38719886541366577, 0.9069615602493286, 1.3587636947631836, 0.5713238716125488, 0.5823290944099426, 0.4946015179157257, -0.8889856338500977, 0.2572779059410095, 1.6154931783676147, -0.9842339158058167, 0.09306641668081284, 0.846075713634491, -1.3539173603057861, -0.12730292975902557, -1.686989426612854, -0.5965573787689209, -1.3980454206466675, -0.8877118229866028, -0.4822198450565338, -0.26986026763916016, -0.1984267383813858, -2.4011573791503906, 1.593066930770874, -0.27143871784210205, -1.0933544635772705, -0.6267236471176147, 0.31149446964263916, -1.001158356666565, -0.41353851556777954, -0.4698505997657776, 0.5397399663925171, -0.748589813709259, 0.4534892439842224, 0.34906983375549316, -0.14843380451202393, 1.2411036491394043, 0.02043960615992546, -1.1776244640350342, 0.297817587852478, 0.1231890395283699, -0.04050282761454582, 0.23232640326023102, 2.349553108215332, 0.23351901769638062, 0.45779475569725037, -0.24645480513572693, 1.3103865385055542, -0.056560587137937546, -0.8021083474159241, -1.2226428985595703, 0.5270918607711792, 0.7442358732223511, -1.5885518789291382, -0.7237141132354736, 0.641386866569519, 1.027116060256958, -1.3212260007858276, -0.40871185064315796, 0.8317688703536987, -0.5348942875862122, -0.45173537731170654, -0.5417974591255188, -1.3082160949707031, 0.045409709215164185, 0.30292508006095886, -1.4035862684249878, 0.098485566675663, 1.106058120727539, 0.20567147433757782, 0.9453840255737305, -0.5513148307800293, 1.36720609664917, -0.7881364822387695, 1.8833832740783691, -0.45110389590263367, -0.7207785248756409, -0.8949810862541199, -0.4150449335575104, -0.49557724595069885, -0.6917725801467896, -0.33178502321243286, 0.6620391011238098, 0.8797226548194885, -0.34002912044525146, 1.2321466207504272, -1.0006740093231201, -1.031279444694519, -1.1338053941726685, 0.5986278653144836, -1.553841233253479, -1.3998346328735352, 1.2750935554504395, 1.7088872194290161, 1.3391127586364746, -1.4963747262954712, 0.006539255380630493, 0.8261597156524658, 0.09072638303041458, -1.6242270469665527, -0.34872177243232727, -1.7051392793655396, -1.239441156387329, 2.1451520919799805, 0.6719362735748291, -1.475069284439087, -1.5086182355880737, 0.4481253921985626, -0.6179530620574951, 0.5686865448951721, -0.6563408374786377, -0.20918555557727814, 0.22913804650306702, -1.0425843000411987, -1.9301940202713013, -1.2045025825500488, -0.7088424563407898, -0.21507838368415833, -0.07068876922130585, -0.07921269536018372, 0.5019532442092896, -0.9050357341766357, 1.382136344909668, 1.9883475303649902, 0.3520614206790924, 0.17394126951694489, 1.2332775592803955, 0.6337528824806213, 0.17293903231620789, -0.18279434740543365, -0.75407874584198, -1.184386968612671, 0.6499515175819397, 1.172234296798706, -1.0725984573364258, -0.08787476271390915, 0.20212575793266296, -0.4414781332015991, -0.7774019241333008, -0.408725768327713, -1.9865738153457642, -0.07726774364709854, -0.3106481730937958, 2.10026478767395, 0.9197710752487183, -0.36233624815940857, -0.024983713403344154, 1.6989367008209229, 1.5887950658798218, 0.16741353273391724, -0.14574536681175232, -1.4623169898986816, -0.3444855213165283, -0.18632164597511292, -0.7531993389129639, 1.7125872373580933, -1.8930187225341797, -0.15312954783439636, 0.38952407240867615, 0.4192516505718231, 1.6691080331802368, -1.4161486625671387, -0.7655752301216125, 0.07940022647380829, 0.11466497927904129, -0.4034154415130615, -0.7818980813026428, 1.0552581548690796, -1.0677045583724976, -0.5929406881332397, 1.4195395708084106, 1.7160123586654663, 0.8537982106208801, -0.8782702088356018, -0.8709902763366699, -0.9172168374061584, -1.1524791717529297, -0.4340750575065613, 1.2524681091308594, 0.20403172075748444, -0.7745705246925354, -0.797057569026947, -1.19520103931427, 1.1866185665130615, -2.0405044555664062, 0.2517307996749878, -1.4622981548309326, -1.082761526107788, -1.512913465499878, 0.9379881024360657, -0.10783299058675766, 2.7162272930145264, -1.064976453781128, -1.1233488321304321, 0.5610081553459167, 0.5322912931442261, -0.3127525746822357, -2.052834987640381, 1.4116051197052002, 0.5343497395515442, 0.36350691318511963, -0.6297994256019592, 0.6034522652626038, -0.5992394089698792, 0.0027823620475828648, -0.23031124472618103, 1.2153445482254028, -0.5142291784286499, -0.3904013931751251, 1.0593440532684326, 0.17986184358596802, -0.06755071878433228, -0.29001107811927795, 0.901050865650177, -0.7011904716491699, 0.4197467565536499, 1.9785337448120117, 0.34653183817863464, -0.8393389582633972, -1.2570782899856567, 0.0913928896188736, -1.2842551469802856, -1.651800274848938, -0.8178593516349792, -1.154883861541748, 0.815321683883667, -1.5129321813583374, 1.5206859111785889, -0.8151857852935791, 0.054662980139255524, 0.6076080203056335, 0.15800535678863525, 0.6552466750144958, -1.0543657541275024, -2.341078996658325, -0.2707422375679016, -1.752894639968872, -0.652518630027771, 0.8720541000366211, -0.27137988805770874, -0.6681801676750183, -1.205816626548767, -0.3686716854572296, -0.25081679224967957, -1.6610217094421387, 0.21573515236377716, -1.155592679977417, 0.1384168416261673, -0.5325832366943359, 0.9094871282577515, 0.7757476568222046, -0.07191889733076096, -0.1894349902868271, 0.3586345314979553, 1.3057222366333008, 0.9909160137176514, 0.08859570324420929, 0.24926498532295227, 1.5156350135803223, -1.695516586303711, 0.48489823937416077, -0.9567307829856873, 1.3039183616638184, -0.5436124205589294, -0.15811483561992645, -1.1306650638580322, -0.02417014352977276, 0.7569795250892639, 1.639538049697876, -1.2954630851745605, -0.4699377417564392, 3.8864552974700928, 0.9217358827590942, -0.025641659274697304, 0.5070818662643433, -2.681901454925537, 1.6035196781158447, 0.14751459658145905, -1.445626974105835, 1.1621214151382446, -0.6370449662208557, -1.0342037677764893, 0.5195441842079163, 0.39870986342430115, -0.4621754288673401, -0.0758126750588417, -1.346692681312561, -1.4483717679977417, 0.8991607427597046, 1.3172838687896729, 0.43128523230552673, 0.37363964319229126, -0.3960976004600525, -0.13019312918186188, -0.9373248815536499, 1.7582709789276123, -0.5680426955223083, 0.4106023907661438, 0.0001667467731749639, -1.4002809524536133, 1.9391934871673584, -0.37582993507385254, -0.7687258720397949, 0.11466401070356369, -0.898296594619751, 0.029630368575453758, -0.960284411907196, 0.5653350949287415, 0.055658962577581406, -1.368286371231079, 1.0376981496810913, 0.1943228840827942, -1.1029078960418701, 0.361402302980423, 0.43308210372924805, 1.8657306432724, -0.8602581024169922, 0.8611261248588562, -0.08875591307878494, 0.6232821345329285, -0.2556106746196747, -0.050496410578489304, 0.5057615041732788, -1.1841301918029785, -2.0985774993896484, -0.0001655638770898804, -0.4367785155773163, 0.09812106192111969, 0.6559041738510132, -0.20338432490825653, 1.2112683057785034, -0.024050770327448845, 0.13227415084838867, 0.8550949692726135, 0.6115972995758057, -0.0645970031619072, 0.15147794783115387, -0.7325077056884766, -0.588540256023407, 0.13132265210151672, 0.8462540507316589, -0.4059438109397888, 0.8735383152961731, 1.9794363975524902, 2.1877756118774414, 0.06357245147228241, 1.095914363861084, 0.6528558731079102, 0.830687940120697, 0.8841544985771179, 0.5700922608375549, 0.8754392266273499, 0.24068191647529602], "dtype": "float32"}, {"__nd__": [-0.20984597504138947, 0.30575820803642273, 0.5470753312110901, 1.2804758548736572, 0.4263070523738861, 0.7989928722381592, 1.4833457469940186, -0.879856288433075, -0.20736753940582275, 0.5968579649925232, 0.08149553090333939, -0.9605385661125183, 1.046521544456482, -0.015999235212802887, -0.9484673738479614, 1.9530872106552124, -1.2323038578033447, 0.03862753137946129, -0.6207221746444702, 0.16117168962955475, 0.6161870956420898, -0.7686342000961304, -0.05466770380735397, -0.08203095942735672, -0.9570086002349854, 1.2168262004852295, 0.22016793489456177, -0.24215134978294373, -0.6210333704948425, -0.2890438735485077, 0.1883605718612671, -0.41753193736076355, 1.296987771987915, -2.5053977966308594, -2.102890729904175, -0.2128668874502182, -1.8045495748519897, -0.024169601500034332, -0.6157132387161255, 0.6257994174957275, 0.45880621671676636, 1.3551604747772217, 0.96669602394104, 1.3010443449020386, 0.5754666328430176, 1.11487877368927, 0.5473601222038269, -0.5467557907104492, -1.2079026699066162, -1.1677119731903076, 0.528984785079956, -2.344456434249878, 0.2211552858352661, -0.7085970640182495, 0.7815117835998535, 0.2450992912054062, -0.7064484357833862, 0.059004005044698715, -2.2333641052246094, 0.31525757908821106, 0.5122212767601013, -1.7966276407241821, 0.9052960872650146, -0.07355114072561264, 0.21914319694042206, -1.3762871026992798, 0.6531754732131958, 0.6302028298377991, 0.03348367288708687, -0.9371272921562195, 0.8133684992790222, 1.2992558479309082, 0.4526406526565552, 1.05582857131958, -0.5307132601737976, 0.2122935950756073, 0.4359525442123413, 0.2067253589630127, 2.8865253925323486, 0.48701784014701843, -0.4927061200141907, 0.9004853367805481, -1.0137019157409668, 0.44453874230384827, 0.8487818837165833, -0.41249802708625793, -0.23828640580177307, -0.8553231954574585, 0.2178591787815094, -0.588180661201477, 1.8263884782791138, 0.7398642897605896, -0.12996773421764374, -1.322251796722412, 2.1339375972747803, 1.402572512626648, 0.5735793113708496, 0.299108624458313, 0.2745548188686371, -0.3655281066894531, 1.986129879951477, 0.3093262314796448, -1.335917592048645, -0.5531930327415466, -2.4793617725372314, 0.7640093564987183, 0.8690643310546875, 1.876631259918213, 2.2044386863708496, -0.6787261962890625, 0.5533410906791687, 1.1001068353652954, 0.8960789442062378, -0.4079291522502899, 0.9048473834991455, 0.5957071781158447, 1.4620180130004883, 0.06936802715063095, -0.14207717776298523, -0.41482821106910706, -0.2837981879711151, -0.6733856201171875, -0.24865064024925232, 0.5934916734695435, 0.5493717789649963, 1.4085243940353394, 1.1823714971542358, 0.7123072147369385, -1.0688021183013916, 0.6528002619743347, 1.2463040351867676, -1.2251017093658447, -1.0397617816925049, -0.8794495463371277, -0.9670960307121277, -0.21878889203071594, -0.8675790429115295, 0.4544622302055359, -1.1485435962677002, -0.06447743624448776, -0.4381052553653717, -0.9098559021949768, 1.094797968864441, 0.08675415813922882, 0.6002408266067505, -0.2757418751716614, 0.7611515522003174, 0.6721111536026001, -0.09537194669246674, -1.7179102897644043, 0.8777901530265808, 0.7497206330299377, 0.23805901408195496, 0.29991328716278076, 0.18103457987308502, 0.2500183880329132, -0.947805643081665, 0.0017189881764352322, 0.37575238943099976, -0.5547694563865662, -0.9129834175109863, 1.9173918962478638, 0.6651930212974548, -0.2487480342388153, -0.43437638878822327, -2.58504581451416, -1.3675827980041504, -0.1913093775510788, 1.3467440605163574, -1.9998259544372559, -0.6417015194892883, 1.9730949401855469, 0.7285888195037842, 0.4735696315765381, 1.2539916038513184, -0.06093714386224747, -0.11823772639036179, 0.774385392665863, 0.9110653400421143, -1.0072636604309082, 1.588135838508606, -1.683752417564392, -0.6347077488899231, 3.135681629180908, -0.4506058394908905, 0.7173622846603394, -0.9405189752578735, -0.2588171362876892, -0.6583632230758667, -0.3365699350833893, 0.7187452912330627, 1.8119304180145264, 0.9365940093994141, 1.6203142404556274, -0.4437192678451538, 1.529422402381897, 0.6273380517959595, 1.4351396560668945, -0.355573832988739, -0.17041146755218506, 1.428621530532837, 0.6709538102149963, 0.16643300652503967, -0.6267417669296265, -0.04386284947395325, 0.5398931503295898, 0.7751210331916809, -0.5037563443183899, -0.29154324531555176, 0.47608259320259094, -1.0634695291519165, 0.9797362685203552, 0.02994203194975853, -0.10251457989215851, -2.386324405670166, -1.573457956314087, -1.0216054916381836, -0.5714914202690125, -1.0890519618988037, 0.37822476029396057, 0.13628891110420227, -0.26628658175468445, -2.0343573093414307, 1.3978474140167236, -0.278790146112442, 0.5904706120491028, -0.6711690425872803, -0.0029405353125184774, 0.6943869590759277, 0.23325610160827637, 0.9592137336730957, -0.18441547453403473, 1.9883666038513184, -1.897481918334961, -0.5563602447509766, 0.9744526743888855, -0.25700584053993225, -0.4497135877609253, -1.7821519374847412, 1.190725326538086, -1.9066672325134277, 0.49793556332588196, 0.18430539965629578, 0.8637610673904419, 1.4469974040985107, -0.7158894538879395, -0.8238552212715149, -0.33265525102615356, -1.735034704208374, 0.8031315803527832, -0.4167933464050293, 0.1382836103439331, -1.1521764993667603, -0.014647300355136395, -0.6481513381004333, -0.10353849828243256, -1.1224991083145142, 0.9942165017127991, -0.68248051404953, -2.5877280235290527, 0.6099529266357422, -0.33607131242752075, -0.9213085770606995, 2.467170238494873, -0.09132832288742065, -1.3786239624023438, -0.6557888388633728, 0.338618665933609, -0.2680213153362274, 1.1991018056869507, 1.0216524600982666, -0.8339216709136963, -0.004582902882248163, 0.5990666151046753, -0.45530712604522705, -0.5063516497612, 0.8370848894119263, 0.4084610044956207, -1.4448473453521729, -0.5911694169044495, -0.45565178990364075, -0.1603986918926239, -0.7598997354507446, 1.3000696897506714, -1.0117992162704468, -1.1277618408203125, 0.16890501976013184, -2.2950961589813232, -1.0725997686386108, 0.5848870277404785, -1.365986704826355, -0.8850347399711609, 1.0463696718215942, 0.417703241109848, 0.018991127610206604, 0.20674896240234375, -0.7132614254951477, 0.2741943299770355, -0.23711468279361725, 2.4666926860809326, 0.5068399310112, -0.36892151832580566, -0.08286812901496887, 0.02450433000922203, -1.4086029529571533, -0.8518156409263611, 1.4998193979263306, -1.4307782649993896, -0.6069023013114929, 0.9984365701675415, 1.4870041608810425, -1.0472524166107178, 1.4401201009750366, -0.38452863693237305, 1.635938048362732, 0.4814639985561371, -0.3014393448829651, 0.7112938165664673, 0.54352867603302, -0.7892229557037354, 2.1329989433288574, -0.3593624234199524, 1.1382944583892822, -0.4997788369655609, -1.8414429426193237, -1.136546015739441, -0.1143496185541153, 0.6690422892570496, 1.7481712102890015, 0.06817593425512314, -0.2956811785697937, -1.243565320968628, -0.05959330499172211, 0.2867417633533478, -2.227774143218994, 1.2913390398025513, 1.123835802078247, 2.2635679244995117, -0.9252534508705139, 0.024259066209197044, 1.7177671907120384e-05, -1.3433594703674316, -0.8674712777137756, -1.0028724670410156, 0.4974759817123413, 0.41373687982559204, 1.3314073085784912, -0.42221546173095703, -1.4182991981506348, 0.41974276304244995, 0.3502131402492523, 0.6464753746986389, 1.2269155979156494, -0.6066330075263977, 0.7783809304237366, -1.0265306234359741, 0.13558101654052734, 0.6694432497024536, -0.13722003996372223, 0.1963340789079666, -1.4165822267532349, -0.6728836297988892, 0.12781967222690582, -1.2268056869506836, -0.8775528073310852, 2.412447214126587, 0.23366864025592804, -0.851930558681488, -1.0626219511032104, 1.3499202728271484, -2.8426356315612793, -0.4077172577381134, 0.5435798168182373, -0.6556543707847595, 0.4250890612602234, -1.153435230255127, -0.12242576479911804, -0.06943148374557495, -0.10122037678956985, 0.029246743768453598, -0.08028572797775269, 1.0012798309326172, -0.38539427518844604, -0.7129018902778625, -0.40711838006973267, -0.43082723021507263, -1.6526169776916504, -2.0419864654541016, -1.4839376211166382, 0.15324482321739197, -0.7994989156723022, -0.5444471836090088, 2.737698554992676, 0.9931285977363586, -0.34309127926826477, -0.7650974988937378, 1.2438132762908936, -0.9154293537139893, 1.1042662858963013, 0.4549177587032318, 1.2527658939361572, -0.8592309355735779, 0.7707862854003906, -1.948150396347046, -0.3085043132305145, 0.9452505111694336, 0.5608177185058594, -1.973645806312561, 0.5494218468666077, 0.3177606165409088, -0.20235425233840942, -0.8623090386390686, -0.42053475975990295, -0.7138211727142334, 0.8081549406051636, 0.32695645093917847, 0.9301929473876953, 0.5505679845809937, -1.422502040863037, -0.0036481020506471395, 2.0051002502441406, -1.251207709312439, 0.14084790647029877, -0.9215608239173889, -1.0304752588272095, 0.07115548104047775, 0.7788941860198975, 0.6279189586639404, 0.32190531492233276, 0.8101217746734619, -1.7717849016189575, 0.6765504479408264, 0.26267164945602417, -0.32582399249076843, -0.15281446278095245, -0.9964066743850708, 1.0230422019958496, -0.8969496488571167, -1.619987964630127, -0.14953769743442535, -0.7231910228729248, -0.21331830322742462, 1.3711321353912354, -0.28041642904281616, -0.018401550129055977, 2.127685308456421, -1.5467555522918701, 0.2276153862476349, 1.0300360918045044, 1.379024624824524, 2.1122491359710693, -0.7967726588249207, 0.4749739170074463, -0.230729341506958, -0.5896416306495667, -0.8614416718482971, -1.6004722118377686, 0.2020159810781479, 0.6228167414665222, -1.1983243227005005, 0.6320664882659912, -0.6868362426757812, -1.1491180658340454, -2.765415906906128, 0.9779204726219177, 0.5900130271911621, -0.44342759251594543, -0.5181060433387756, -1.4363276958465576, 0.39181700348854065, -0.19110986590385437, 0.417136013507843, 0.05362837761640549, -1.5397968292236328, -1.6890698671340942, 0.2998860478401184, -1.8213775157928467, 0.1572043001651764, 0.7969985008239746, -1.4054841995239258, 0.5694829225540161, -0.06115039065480232, 0.43666011095046997, -0.7351228594779968, -0.10857977718114853, 0.34052541851997375, 1.7564114332199097, 0.6585507988929749, 0.4240565001964569, -1.3804391622543335, 0.2639543116092682, -0.7550179958343506, -0.13604216277599335, 0.2071915864944458, 0.9158821105957031, 0.5392610430717468, 0.13434144854545593, -0.11184676736593246, 1.3137245178222656, -0.9425020813941956, -0.009123696945607662, 1.3387553691864014, -0.2414776235818863, 0.7907440662384033, 0.852977454662323, 1.9500311613082886, 1.0958369970321655, -1.7058370113372803, -0.04420303925871849, 0.24934324622154236, 0.7854777574539185, 0.3401229679584503, 1.8472790718078613, 1.05498468875885, -1.2480685710906982, -0.7006041407585144, -2.1460163593292236, 1.673776626586914, -0.2036363184452057, -0.22646598517894745, -1.6842881441116333, -0.5421620011329651, 0.32484492659568787, -0.5238729119300842, -0.5194655060768127, -0.7537512183189392, 1.8672086000442505, -0.6957301497459412, -1.6134731769561768, -0.6759340167045593, -0.5135419964790344, -0.260604590177536, -0.4185172915458679, -1.3075921535491943, -0.3830552399158478, 0.2070455700159073, -1.60598886013031, 1.8144290447235107, 1.5321985483169556, -2.030519723892212, -0.5649585127830505, 0.8303572535514832, 1.1683423519134521, 1.618346095085144, -2.0661847591400146, -1.3190441131591797, -0.7134859561920166, 0.36679452657699585, -1.2806649208068848, 0.25236639380455017, 0.2753777801990509, 0.5494953989982605, -1.4874353408813477, 0.02401883713901043, -0.09884186089038849, -0.5925266146659851, 1.1594327688217163, -0.45735248923301697, 1.3156324625015259, -0.7565046548843384, 1.5841286182403564, 1.53162682056427, -0.9746177196502686, 0.2079622745513916, 0.1882728785276413, -1.1192102432250977, 0.675058126449585, 0.48549655079841614, -1.000077247619629, -0.011015204712748528, 0.7970510721206665, -0.26605284214019775, -0.827451765537262, 1.5677791833877563, -1.0920722484588623, 0.2560620605945587, 1.0545178651809692, 0.7209298610687256, -1.9468703269958496, -0.4081268012523651, -0.20663301646709442, -0.2830014228820801, 0.5728076100349426, 0.4919133186340332, 0.08075904101133347, 0.21863798797130585, 0.9812970757484436, 1.2237670421600342, -1.9649516344070435, 2.179419994354248, -0.5958083868026733, 0.7622644901275635, -1.6825244426727295, -0.5499566793441772, -1.3022358417510986, 1.0799956321716309, 0.8791921734809875, -0.014258560724556446, -0.19162176549434662, 0.3743670582771301, -0.9717922210693359, -0.6341529488563538, -0.7940674424171448, 0.8150408864021301, 0.009890737943351269, -1.310197114944458, 0.33402886986732483, 0.5182516574859619, 1.3930190801620483, -0.5800811052322388, 0.3956138789653778, 2.098142623901367, -0.34563836455345154, -0.3843082785606384, 0.5018574595451355, -0.5246061086654663, -0.014510304667055607, -0.5280489921569824, -0.8127743005752563, 0.873835563659668, -0.3993484377861023, -1.0392725467681885, -0.311465322971344, -0.16472668945789337, 0.28242820501327515, 0.9269111752510071, 1.7735437154769897, -1.2672785520553589, -0.2756551504135132, -1.8735724687576294, 0.19395290315151215, 0.4982829689979553, -0.6966447830200195, -0.9639397859573364, -1.0452154874801636, -1.5603387355804443, 0.5948299169540405, 1.8879982233047485, -1.2031261920928955, 1.6032475233078003, -0.16928572952747345, -1.4659799337387085, 0.5120036602020264, 0.707841157913208, 0.33625680208206177, -0.3289906680583954, 1.6249445676803589, -0.3556702136993408, 0.5742908120155334, 0.5873076319694519, -0.979933500289917, -1.949404239654541, -1.830085277557373, -0.48046889901161194, 0.2818545699119568, -1.8850449323654175, -0.8135586977005005, 1.470281958580017, -1.9059646129608154, 0.38231244683265686, 0.2689759135246277, 0.710246741771698, -1.5820320844650269, 0.25058460235595703, -1.5030516386032104, 1.259168267250061, -0.971124529838562, 0.28988388180732727, -0.9073076844215393, 0.04323318600654602, -1.6539281606674194, 0.1895781308412552, 0.5127954483032227, -0.11008884012699127, 1.5392088890075684, 0.27489832043647766, 0.6973913311958313, 0.4140518605709076, 1.8035929203033447, -0.7651493549346924, -0.3722771108150482, -0.31940120458602905, 0.5307744741439819, 0.17046454548835754, 0.7538608908653259, 0.20826905965805054, -0.614450991153717, -0.15290267765522003, 0.3557981550693512, -2.4089739322662354, 1.3274887800216675, -0.6586859226226807, -0.4417263865470886, 0.18642425537109375, -0.33076050877571106, -1.5513845682144165, -0.3046495020389557, 1.3409161567687988, 0.9983112812042236, 1.192882776260376, -0.6232307553291321, -0.2794046998023987, 1.0149625539779663, 1.4574615955352783, -0.01902424730360508, -1.034990668296814, -2.014125347137451, 1.4032734632492065, -1.5934399366378784, 0.4007355868816376, 0.8737286925315857, 3.293656587600708, 0.165798157453537, -0.1427440643310547, 0.4477011263370514, 0.5021828413009644, 1.3561495542526245, 0.6894013285636902, -0.7972708344459534, -2.1752102375030518, 0.29329758882522583, -0.7041776776313782, 1.439232349395752, -1.3562283515930176, 0.20056940615177155, -0.7563377618789673, -0.9060184359550476, 1.7109003067016602, 0.17168456315994263, -0.18815480172634125, 0.7606294751167297, -0.8097084760665894, -0.3796728253364563, -0.6578619480133057, 2.042267322540283, 0.3399350345134735, 0.6481192708015442, 1.150268793106079, -0.19239802658557892, -0.16039803624153137, -1.690752625465393, 0.493153840303421, 0.2656588852405548, -0.4273357093334198, 1.2419788837432861, -1.8489665985107422, 0.4308423399925232, -1.4276734590530396, -1.1860284805297852, -0.22169236838817596, 1.3743988275527954, -1.2347182035446167, -1.0384308099746704, -1.25812566280365, 2.0791664123535156, 0.5366870760917664, 0.41532546281814575, 2.1341354846954346, 0.25400543212890625, -2.1830615997314453, -0.5280473828315735, -1.2447540760040283, 2.0952446460723877, -1.4729804992675781, 0.3100181221961975, -0.539620578289032, -0.46294301748275757, 0.01605016365647316, -0.1454448699951172, -0.7164522409439087, -1.2891932725906372, -0.6830203533172607, -1.0910181999206543, -0.9495899677276611, 0.11488600075244904, -0.16705387830734253, 0.4878608286380768, -0.2565340995788574, 1.98790442943573, -0.7698174118995667, -1.0494575500488281, 1.0539101362228394, -0.744595468044281, 0.11776109039783478, 1.005301594734192, -0.1142963245511055, -0.38148102164268494, 0.9936009645462036, 0.33223679661750793, -0.5661107897758484, -1.2590702772140503, -0.4079930782318115, -1.7380188703536987, -0.9224035143852234, -1.1545697450637817, 0.28946781158447266, 0.7758850455284119, -0.6944417357444763, -0.47600167989730835, -1.1635700464248657, -1.45587158203125, -0.4699931740760803, 0.5646513104438782, 0.4658673107624054, 0.5901614427566528, -0.30599918961524963, -2.8606226444244385, 0.9418735504150391, 1.3517080545425415, 0.9267686605453491, 0.918768584728241, 1.1214500665664673, 2.769683361053467, 0.32091131806373596, 1.604892373085022, 0.831809937953949, 0.6970983147621155, -0.14309222996234894, -0.3627842366695404, -1.173919677734375, 0.6045864224433899, -0.32904183864593506, -1.251124382019043, -0.7932021617889404, -1.2373532056808472, 1.657955527305603, -0.2588428556919098, 0.10751022398471832, -1.2631741762161255, -1.0777074098587036, 1.0853383541107178, -0.24821606278419495, 0.22677087783813477, 1.9037195444107056, 2.1684157848358154, 0.7132322788238525, 0.5040022134780884, 0.2436208873987198, -0.29817768931388855, 2.101930618286133, -1.2637991905212402, -0.09444450587034225, -1.9996432065963745, -1.3159613609313965, 0.41797611117362976, 0.47379565238952637, 0.34948936104774475, -0.33738651871681213, -0.7354851365089417, -0.5605399012565613, -0.3985370993614197, 1.1365535259246826, 1.515030860900879, 0.4593721032142639, -0.5600021481513977, -0.9738859534263611, 0.31604811549186707, -0.23779690265655518, -1.8960106372833252, 1.4243413209915161, -0.008583035320043564, -0.3314610719680786, -0.9046298861503601, -0.9086011052131653, 0.934103786945343, -0.9809394478797913, 0.09944872558116913, 0.6594175100326538, 1.9164307117462158, 0.6280729174613953, -0.035543978214263916, -0.4572617709636688, 0.8893720507621765, 1.0439059734344482, -0.4426550269126892, 0.6105468273162842, -0.1436976045370102, -1.686521291732788, -1.0464136600494385, 0.8411470651626587, 1.4986090660095215, 0.7053757309913635, -0.21011368930339813, -0.7444566488265991, 0.5577568411827087, 0.3439517319202423, -1.6812189817428589, 0.6770060062408447, -0.7349183559417725, 0.6983442902565002, 0.5583033561706543, 0.7546725869178772, 1.114040493965149, 0.7328285574913025, -1.0901414155960083, -1.057875394821167, 1.0810468196868896, 0.49670085310935974, -1.6323219537734985, -2.08496356010437, 0.930036723613739, 2.3133718967437744, 0.8620755076408386, -0.24423688650131226, 0.28050634264945984, -2.8282532691955566, -1.034661889076233, 1.0564669370651245, 0.8981563448905945, -2.2078051567077637, 1.232280969619751, 0.5591688752174377, 0.9769022464752197, -0.20694513618946075, 0.007602143567055464, -1.1169463396072388, 0.3367112874984741, 0.9348941445350647, -0.974236249923706, 0.3841854929924011, -0.549252986907959, 1.0872433185577393, 1.0224074125289917, -2.4502804279327393, 0.47412246465682983, 1.0586367845535278, -0.578971803188324, -0.22136518359184265, -0.15436728298664093, -1.1877418756484985, -0.11791444569826126, 1.6200151443481445, 0.49924466013908386, 1.5867520570755005, 1.035065770149231, 1.2518367767333984, 0.1309123933315277, 0.8099439740180969, 2.100221872329712, 1.5347427129745483, 0.3401186764240265, -0.7217456102371216, -0.9549883008003235, 0.25375285744667053, 0.46012696623802185, -1.9045817852020264, 0.2322242259979248, 0.3651328682899475, 0.3184094727039337, -0.5668579936027527, 2.398263454437256, 0.9107897877693176, -1.422834038734436, -0.7405896186828613, 0.8339025378227234, 0.5329341292381287, 0.1318827122449875, -0.03434879332780838, -0.22626784443855286, -0.47137749195098877, -1.0728224515914917, 0.5879554748535156, -2.8986146450042725, -0.7740153074264526, 1.2674952745437622, -0.8363261818885803, -1.2632614374160767, 1.939246416091919, 1.0507760047912598, -0.18549136817455292, -0.6394923329353333, -0.27503010630607605, -0.39684250950813293, -0.28854236006736755, -0.8049989938735962, 0.5216492414474487, -0.6191295385360718, -1.9232642650604248, -1.3860514163970947, -0.04642735421657562, -1.1060513257980347, -2.617316484451294, 1.501259684562683, 0.45867031812667847, -0.5503271222114563, -0.6016889810562134, 0.46896231174468994, 1.3587477207183838, -0.7107702493667603, 0.3593028783798218, -0.7150284647941589, 0.2684755325317383, -1.249068260192871, -1.9662644863128662, 0.01761493645608425, -0.1395605057477951, 0.6751620769500732, -0.7658265233039856, 1.3478296995162964, 0.024913568049669266, 1.5240705013275146, -0.4270039200782776, 1.488090991973877, -0.14079385995864868, 0.07266763597726822, 0.9252985119819641, -1.9058650732040405, -1.1777334213256836, 1.8262896537780762, -1.634458065032959, -0.5896575450897217, 0.49004697799682617, 1.4412212371826172, -0.5967857241630554, 0.06297177076339722, -1.2414945363998413, -0.46837958693504333, 0.9917968511581421, 0.26963186264038086, -0.10051772743463516, 0.3737700879573822, 0.07718806713819504, 0.11133663356304169, -0.42843395471572876], "dtype": "float32"}], "e": {"__nd__": [-2.429349899291992, -1.3837167024612427, 0.7435850501060486, 1.325156569480896, 0.23898783326148987, 0.958012044429779, 5.900303840637207, -0.8185284733772278, 0.5649354457855225, 3.544384479522705, 0.14535707235336304, -0.780128538608551, 1.0280704498291016, -0.9319478869438171, -2.165125846862793, 1.8018854856491089, -1.0161386728286743, 0.10074880719184875, -0.7288179397583008, 0.03995785862207413, 0.6137081384658813, -0.27994656562805176, -0.65369713306427, 0.752770185470581, -0.6493470668792725, 0.9891214370727539, 2.821488618850708, -0.5655868053436279, -0.381997674703598, -1.8334157466888428, -0.5286949276924133, -0.5572617650032043, 0.1026761531829834, -1.8823555707931519, -2.085479974746704, 0.6847056746482849, -2.8169541358947754, 0.9010722637176514, -0.9389642477035522, 2.190375328063965, 0.39857134222984314, 0.89068603515625, 0.5907553434371948, 2.357821464538574, 0.5357717275619507, 1.078123927116394, 2.1791629791259766, 0.5443918704986572, -0.9933875799179077, -2.8290421962738037, 0.6695135831832886, -1.8486324548721313, 0.21530137956142426, -1.956377625465393, 0.7809818387031555, 0.019023418426513672, -0.9216321706771851, -0.2642841339111328, -1.4031144380569458, 0.34353139996528625, -1.2387170791625977, -1.7337135076522827, 0.4987850487232208, 0.22550034523010254, 0.9524083137512207, -2.5216426849365234, 0.32695096731185913, -0.16314536333084106, -1.514503836631775, -0.581151008605957, 0.9568324685096741, 0.6482320427894592, 0.5995458364486694, 0.9158400893211365, -0.20074287056922913, -0.5910854339599609, -1.7115753889083862, 0.234821155667305, 2.3858728408813477, -0.2073422372341156, -3.9693830013275146, -0.10384970903396606, -0.9834882616996765, -0.6530823707580566, 1.0182344913482666, -0.4733424782752991, -0.6447117924690247, 0.0654217004776001, 1.093793511390686, -3.477294921875, 2.289090156555176, 0.3113817572593689, 0.0951269268989563, -0.9396501183509827, 2.1896212100982666, 3.3757758140563965, 0.5773586630821228, 1.2531239986419678, 1.0328294038772583, -0.0969012975692749, 1.6742017269134521, 0.581008791923523, -1.235843300819397, -1.52109956741333, -2.6603055000305176, 0.8343598246574402, 0.8569114804267883, 1.0635435581207275, 2.0929629802703857, -4.384026527404785, 0.8080906867980957, -0.4353238344192505, 1.1866601705551147, -0.5261086821556091, 0.806188702583313, 1.3381130695343018, 3.9227380752563477, -2.4652724266052246, 0.8938502073287964, -2.2748677730560303, -1.7112904787063599, -0.9232310056686401, -0.2241109162569046, 0.6900566816329956, -2.0446064472198486, 1.5156105756759644, 1.5858688354492188, 0.6271734833717346, 2.340599298477173, -0.46858566999435425, 0.6937010884284973, -2.454223155975342, -0.9287858605384827, 0.017307281494140625, -2.966693162918091, -0.6766283512115479, -1.337794303894043, 0.25535380840301514, -0.8830310106277466, 0.27766427397727966, -1.2533659934997559, -0.8430338501930237, -1.0945621728897095, 0.19011278450489044, 0.4695650041103363, -1.685194492340088, 0.7949070930480957, 0.9519585967063904, 1.651200294494629, -1.7249321937561035, 0.47631311416625977, 0.8676034212112427, 0.30352285504341125, 0.3978938162326813, 0.04442335665225983, 0.21710461378097534, -1.3788771629333496, 0.14523237943649292, -0.3079374432563782, -0.1759994626045227, -1.5798193216323853, 1.8557804822921753, 0.732053816318512, 0.34615859389305115, -0.43855199217796326, -3.443441152572632, -1.8817696571350098, -0.6124258637428284, 2.0802159309387207, -2.147847890853882, -1.28181791305542, 1.5050175189971924, 0.8700516223907471, 0.6209192276000977, 2.205854892730713, -0.5761252045631409, -0.3037826120853424, 0.6572239995002747, -0.02273350954055786, 1.6381163597106934, 1.1705248355865479, -0.910464882850647, -0.664989709854126, 3.025878429412842, -0.27329376339912415, 1.0412930250167847, -1.3806790113449097, -0.3621729612350464, -0.6655092835426331, -0.8493179082870483, 0.2398718297481537, 0.0940694808959961, 0.6908076405525208, 0.37405645847320557, -0.8893953561782837, 1.5036683082580566, 0.7131211757659912, 0.6117316484451294, -0.48444706201553345, -0.365003377199173, 1.2643367052078247, 0.9445410966873169, 0.0011305809020996094, -0.9149448871612549, 2.410048723220825, -0.022346198558807373, 0.7573943138122559, 0.28242844343185425, -0.2834251821041107, -1.8866136074066162, -1.0950936079025269, 0.9653013348579407, -0.340718537569046, 0.45623886585235596, -2.268847703933716, -1.5674023628234863, -1.1171091794967651, -0.30110636353492737, -3.0328209400177, 0.37667346000671387, 0.8354287147521973, -0.3793526887893677, -2.219475746154785, 4.06607723236084, 1.17082679271698, 1.2211940288543701, -0.7212358117103577, 0.0543103888630867, 1.0051188468933105, 0.614675760269165, 1.2369041442871094, -3.113903760910034, 1.998461127281189, -1.882407307624817, -0.45070287585258484, 3.3576886653900146, -0.8064736127853394, 0.022569924592971802, -1.8447000980377197, 1.1703640222549438, -1.9419965744018555, 0.8139290809631348, -0.03943866491317749, 0.588972806930542, 3.299490213394165, -1.7457298040390015, -0.7726399898529053, -1.8815479278564453, -1.991320252418518, 0.9434234499931335, -0.6664727926254272, 0.1579645723104477, -1.2473268508911133, 0.4717065095901489, -0.2105329930782318, -0.07392166554927826, -1.0175385475158691, 0.06159567832946777, 0.6574079394340515, -2.816324472427368, 0.6630992889404297, 1.2264444828033447, -1.6047656536102295, 1.8578550815582275, -1.4620635509490967, -0.8138983249664307, -1.4582382440567017, 0.14037609100341797, -0.23647183179855347, 0.6763165593147278, 0.5130277276039124, -0.8706526756286621, 0.9114302396774292, 1.3283047676086426, -0.8107377290725708, -0.2881210446357727, 0.7553657293319702, 4.15040397644043, -1.6410406827926636, -0.6733131408691406, -0.6249678134918213, -0.4914005696773529, -1.711836576461792, 1.4935258626937866, -2.130671739578247, -1.1454144716262817, 1.0662047863006592, -2.852492332458496, -1.5374377965927124, 0.7476292252540588, -3.0112807750701904, -0.46856603026390076, 1.3125615119934082, -0.9155465364456177, 1.5518321990966797, 0.26793116331100464, -0.4513639509677887, 0.33821940422058105, -0.15115468204021454, 2.2269160747528076, 1.0016324520111084, -0.36939674615859985, 0.012600407004356384, -0.3303687572479248, -1.0043035745620728, -3.431344747543335, 1.5710629224777222, -2.4224495887756348, -0.18163082003593445, 1.0349736213684082, 2.7054429054260254, -1.0704140663146973, 0.5992228388786316, 0.21199887990951538, 1.6258355379104614, 0.963499903678894, -0.13094601035118103, 0.4887055456638336, 0.6080302000045776, -0.630480170249939, 2.733088493347168, -0.42158371210098267, 0.4523792862892151, -2.165142297744751, -1.830168604850769, -0.9460330009460449, -0.2605513334274292, 0.18369868397712708, 1.7696566581726074, 1.280409812927246, -0.029457509517669678, -0.5855002999305725, -0.31062406301498413, -0.4667406976222992, -2.777486562728882, 0.7042880654335022, 0.9398247003555298, 5.634439468383789, -0.9959662556648254, -0.2471482753753662, 1.4471683502197266, -0.4534680247306824, -1.8510949611663818, -1.2750649452209473, 1.3443214893341064, 0.5079673528671265, 0.8844845294952393, -0.4578244984149933, -1.3603640794754028, 0.440940260887146, 0.36686524748802185, 0.8006850481033325, 0.6326860785484314, -0.9124964475631714, 1.6730756759643555, -1.4502991437911987, 0.17773029208183289, 1.694736361503601, -0.12515024840831757, -0.12449045479297638, -1.3355424404144287, -1.8238786458969116, -0.12314139306545258, -1.3236935138702393, -0.041207075119018555, 2.5085983276367188, 3.9918761253356934, -1.0015169382095337, -1.2342033386230469, 0.4229695796966553, -2.9106388092041016, -0.4267966151237488, 0.11399337649345398, -2.263162851333618, 0.7880771160125732, -1.9713269472122192, 0.5802685022354126, 0.8346619606018066, -0.09760693460702896, 0.17729038000106812, -0.18077918887138367, 1.2844531536102295, -0.9980311393737793, -0.30437996983528137, -1.316575288772583, -0.21074196696281433, -1.5110375881195068, -1.8050119876861572, -1.5407557487487793, 0.26896241307258606, -1.0015859603881836, -0.7789730429649353, 3.0027873516082764, 1.0017558336257935, -0.48497313261032104, -0.7216986417770386, 1.0301506519317627, -0.3536618947982788, 1.0999622344970703, 0.5400130152702332, 2.3159594535827637, -0.726987361907959, 1.3305182456970215, -1.611541748046875, -0.962772011756897, 0.9960994720458984, -0.6412962675094604, -1.374832034111023, 1.0196738243103027, -1.436350703239441, 0.22293773293495178, -0.42594385147094727, -1.703360915184021, -2.4271445274353027, 0.5087683200836182, 1.879420518875122, 4.624351501464844, 0.02432548999786377, -2.507901191711426, 0.3484662175178528, 1.1373697519302368, -1.4405280351638794, 0.6934682130813599, -1.1510785818099976, -1.0914592742919922, -0.306561678647995, 2.603666067123413, 0.6786947846412659, 0.40164971351623535, 0.4978444278240204, -1.4267867803573608, 1.8892595767974854, -0.18632864952087402, -0.3024028539657593, 1.0997575521469116, -1.0951029062271118, 1.122272253036499, -0.6587921380996704, -2.246021270751953, -0.8424428701400757, 0.4844226837158203, -0.819679856300354, 1.1679365634918213, -1.7882583141326904, 0.2589374780654907, 2.1290769577026367, -0.658453643321991, 0.031141802668571472, 0.9194759726524353, 1.4815860986709595, 2.1628401279449463, -1.1530340909957886, 1.113067626953125, -1.0075453519821167, -0.6084026098251343, -2.0683932304382324, -2.0807723999023438, 2.2459423542022705, 0.6546077728271484, -1.3189926147460938, 1.0063199996948242, -1.264752984046936, -0.707297682762146, -3.0531182289123535, 0.6087943315505981, 0.5841630697250366, -0.5596137046813965, -0.6165702939033508, 3.6749255657196045, 0.40130025148391724, -0.18485534191131592, 1.8911442756652832, 0.08191890269517899, -1.5400516986846924, -1.8813015222549438, 1.0124702453613281, -1.0009435415267944, 0.33585846424102783, 0.951836884021759, -1.0651533603668213, 0.009651124477386475, 1.1649107933044434, -0.655151903629303, 2.9508798122406006, -0.32651475071907043, 0.316240519285202, 0.7316471338272095, 0.804552435874939, 0.2549511790275574, -1.2701281309127808, 0.08851270377635956, -0.698826789855957, -0.11102133989334106, 1.3374592065811157, 1.6189475059509277, 0.5143721699714661, 2.0048160552978516, -0.5773188471794128, 1.545931339263916, 0.548533022403717, -0.00035751890391111374, 3.0984957218170166, -0.2791353166103363, 0.665895402431488, 0.8272036910057068, 2.2837414741516113, 1.0412546396255493, -1.7181843519210815, 0.17090600728988647, -0.29107484221458435, 0.8080340027809143, 0.3350660800933838, 2.1023499965667725, -0.9431940317153931, -0.6263408064842224, -0.6399394273757935, -2.4144394397735596, 1.7345423698425293, -0.6558618545532227, 0.3995867371559143, -1.6312652826309204, -0.35232657194137573, 0.39173778891563416, -0.9949973225593567, 0.03861469030380249, -0.5891146659851074, 4.689969062805176, -0.07894617319107056, -1.6583738327026367, -0.19042927026748657, 0.4238988757133484, -0.08864840865135193, -0.5254971981048584, -0.8911030292510986, -0.3657190203666687, -0.48492926359176636, -1.45901620388031, -0.2644798755645752, 1.5597949028015137, -1.4238481521606445, -0.5820013284683228, 1.244480013847351, -0.09377861022949219, 0.47961246967315674, -2.6525180339813232, 0.6078568696975708, -0.5829750895500183, -0.15307211875915527, -1.2192977666854858, -0.24817994236946106, 0.929074764251709, 0.8792880177497864, -1.1170610189437866, -1.7125741243362427, 1.8270522356033325, -0.9487614631652832, 1.141727089881897, -1.8992807865142822, 1.6061204671859741, -0.685111403465271, -1.1822195053100586, 1.6032791137695312, -1.1894705295562744, 1.502351999282837, 0.7120882272720337, -2.1452455520629883, 0.7301530838012695, 0.612682044506073, -0.8453850150108337, 1.0417171716690063, 0.8480710387229919, 0.14795401692390442, -0.24256795644760132, 1.987600564956665, -1.2133511304855347, -0.08898013830184937, 1.8314565420150757, 0.7244426012039185, 1.7046105861663818, -1.6401153802871704, 5.29335880279541, 0.03492125868797302, 0.44391196966171265, -0.5917245149612427, 0.28986331820487976, 0.1825791597366333, 1.3852338790893555, 1.1873891353607178, -0.23505854606628418, 1.1287016868591309, -0.39426547288894653, 1.4542334079742432, -2.825921058654785, -0.7102989554405212, -0.6593130230903625, 1.127421259880066, 1.122215747833252, -0.023293273523449898, -0.05554141104221344, 1.4962306022644043, -1.5784287452697754, -1.1566145420074463, -1.8383891582489014, 0.8089887499809265, 0.09049640595912933, -2.5954864025115967, 1.0213862657546997, 0.9331914782524109, 1.2300479412078857, -3.4202322959899902, 0.9821805953979492, 1.5298283100128174, -3.6532745361328125, 0.15218114852905273, 1.3153133392333984, -0.5576298832893372, 0.34806227684020996, -3.1630585193634033, -1.682807445526123, 1.5407975912094116, -0.40453943610191345, -1.1874902248382568, -0.4262818694114685, -1.0954056978225708, -0.14651688933372498, 2.0305545330047607, 1.8440667390823364, -1.2909154891967773, 0.06020444631576538, -2.304274559020996, 0.31586140394210815, 0.4062539339065552, -3.909228563308716, -1.1521754264831543, -1.0716999769210815, -1.7463581562042236, 0.3735802471637726, 0.34053540229797363, -0.7836902141571045, 3.623188018798828, -0.5584155917167664, -1.0360944271087646, -0.44772887229919434, 0.1458713412284851, 0.5722861289978027, 0.10611817240715027, 1.623157024383545, -0.6907916069030762, 0.4882844090461731, 1.0426318645477295, 1.7226495742797852, -1.6496942043304443, -3.1414079666137695, -1.539729118347168, 0.31149178743362427, -1.88357675075531, -0.9545843005180359, -0.15181922912597656, -0.0969398021697998, -0.8732813596725464, 0.28874391317367554, 2.0959036350250244, -3.8094987869262695, 0.22453252971172333, -4.92276668548584, 1.8347752094268799, -0.7007184028625488, 0.4359155297279358, -0.7004972100257874, -1.7573297023773193, -0.5303727388381958, -0.9100691676139832, 0.1941409409046173, -0.2110891491174698, 2.7053000926971436, 1.7448549270629883, 1.2346041202545166, 1.5676851272583008, 1.6458492279052734, -0.8499784469604492, 1.7823132276535034, -2.016571283340454, 0.47912126779556274, 0.23530098795890808, 2.6960952281951904, 0.23373548686504364, -2.113058090209961, -0.10136916488409042, 1.151593804359436, -1.1597504615783691, 1.1499091386795044, -0.8739104270935059, -0.5600880980491638, -0.42662888765335083, -2.3604319095611572, -2.0296077728271484, -0.24283522367477417, 1.4316635131835938, 1.2266144752502441, 3.358574867248535, -0.5965710878372192, -0.21513402462005615, 1.2158986330032349, 0.44899141788482666, 0.1951134204864502, -1.6745209693908691, -1.9637949466705322, 1.6583596467971802, -1.5904319286346436, 1.9095821380615234, 0.3782321512699127, 3.3612887859344482, 0.1518840193748474, -0.14769285917282104, -3.606269598007202, 0.9164572954177856, 0.02843773365020752, 0.7791348695755005, 1.942945957183838, -2.1663060188293457, 0.10715211927890778, 0.7374972701072693, 2.155360460281372, -2.318244695663452, -0.20358864963054657, -0.8539708852767944, -1.579939365386963, 2.6219310760498047, 2.042485237121582, 0.09119342267513275, 0.7367558479309082, -0.5890657901763916, -0.5909976363182068, -1.1585125923156738, 2.426586866378784, 0.31534722447395325, 0.49151360988616943, 1.6634113788604736, -0.11265929788351059, -0.40105968713760376, -1.6069128513336182, 0.40697944164276123, 0.7312250733375549, 0.753547191619873, 2.0600969791412354, -1.8372937440872192, -0.23995989561080933, -0.6397854089736938, -1.1366357803344727, 0.4360629916191101, 1.6268539428710938, -0.20079505443572998, -0.9184616208076477, -1.614499568939209, 2.1952648162841797, -0.0020213723182678223, 0.7640038728713989, 1.4755173921585083, 0.3062933087348938, -1.9058194160461426, -1.0272921323776245, 1.5534915924072266, 1.9991874694824219, 0.6043045520782471, 1.1239609718322754, -1.2546920776367188, -2.93107271194458, 0.013221127912402153, -0.6567053198814392, -0.8264756202697754, 0.2733958959579468, -0.5971082448959351, -4.499627113342285, -0.6940467357635498, -2.0978734493255615, -0.5738565921783447, 0.8165973424911499, 2.3258461952209473, 2.044649839401245, -1.2027976512908936, -0.8670159578323364, 0.6596248149871826, -0.47484084963798523, -0.271450936794281, -0.3350182771682739, -2.2265682220458984, 1.4737714529037476, 0.847308874130249, 0.3267463147640228, -0.6046281456947327, -1.2755929231643677, -0.6600397825241089, -1.9147330522537231, -0.7053433060646057, -1.251258373260498, -0.3626973628997803, 0.25489598512649536, -1.66896390914917, -1.512042760848999, -1.1198922395706177, -1.5867700576782227, -0.668402910232544, 1.0023152828216553, 0.5116764307022095, 1.9585585594177246, 0.4757191836833954, -2.94166898727417, 1.0849010944366455, 1.2361801862716675, -0.2019646167755127, 1.212159514427185, -0.7509076595306396, 2.841461420059204, 0.5591882467269897, -0.6207815408706665, 1.2418729066848755, 1.0376074314117432, -0.1443953812122345, -2.2939858436584473, 0.5201451778411865, 0.9119867086410522, -0.3619914650917053, -3.654245615005493, -0.9863426685333252, -1.641542673110962, 1.5828508138656616, -2.9483304023742676, -2.714193105697632, -1.2438236474990845, -1.481194019317627, 0.8101978302001953, -2.9627394676208496, 0.46426159143447876, 2.7238526344299316, 2.2862601280212402, 0.7894262671470642, 1.1738356351852417, 1.4237961769104004, -0.8964835405349731, 4.157305717468262, -1.1986908912658691, -0.15612086653709412, -3.3257884979248047, -1.7825572490692139, -0.020151495933532715, -0.7644778490066528, -1.022367238998413, -0.3769267201423645, -1.1069878339767456, -1.4853065013885498, -0.31448546051979065, 1.8798871040344238, 0.9335188865661621, 1.9395105838775635, -1.7051875591278076, -0.9568586945533752, 0.45625942945480347, 1.8652430772781372, -1.3149590492248535, 1.2198694944381714, 0.20625966787338257, -0.241010844707489, -0.6414748430252075, -0.6239999532699585, -0.6428861021995544, -0.6685149669647217, -0.6053556799888611, 1.3479602336883545, 1.7113898992538452, 2.2967095375061035, 0.05729397386312485, -0.13662466406822205, 1.194619059562683, 0.2208450436592102, -0.8912419080734253, 0.6108221411705017, -0.30115747451782227, -3.3655998706817627, -1.1397737264633179, 1.0853434801101685, 2.3530640602111816, 0.600932240486145, -0.16908372938632965, -0.5745912790298462, 1.6964302062988281, 0.32371243834495544, -2.000225305557251, -1.3462114334106445, -1.0967400074005127, -0.5292959809303284, 2.8396170139312744, 0.6769559383392334, 0.29482656717300415, -1.2138111591339111, -0.9688713550567627, -1.965055227279663, 1.7003792524337769, -0.9953333139419556, -1.5617393255233765, -3.434070110321045, 0.9418507814407349, 3.25407075881958, 0.6909891366958618, 0.385201632976532, 1.5826870203018188, 0.6262037754058838, -1.1769366264343262, 2.4583349227905273, 0.5869090557098389, -3.114093542098999, 0.7871160507202148, 0.5288248062133789, 2.296395778656006, -0.2864525318145752, -0.22417263686656952, 0.09042727947235107, 0.45790043473243713, -1.4841632843017578, -0.8475176095962524, 0.3632826805114746, -1.0217548608779907, 1.5752956867218018, 1.1277258396148682, -1.8845369815826416, 0.2541370987892151, -0.28425443172454834, 1.2045183181762695, -0.204010471701622, -0.38578566908836365, -2.8628334999084473, -1.0642526149749756, 1.246056318283081, 0.9989461302757263, 1.5471524000167847, 0.9533265829086304, 1.226452112197876, -0.07195387780666351, 0.7768833041191101, 3.439075469970703, -1.0637847185134888, -3.088043451309204, -0.8153245449066162, 4.155157089233398, -0.36216649413108826, 0.44993922114372253, -1.758792519569397, -0.26696014404296875, 1.8495337963104248, 0.42251574993133545, 0.7796208262443542, 2.347801685333252, 0.39894789457321167, -2.2108101844787598, -0.8761746883392334, 0.751923680305481, 0.5693123936653137, 0.20990073680877686, 1.5981523990631104, -2.6082921028137207, -0.7589551210403442, -0.8479837775230408, 0.2978222072124481, -2.385690212249756, -1.4979368448257446, 1.2863192558288574, -1.389108657836914, -2.458280086517334, 0.8938503265380859, 1.4618473052978516, -0.18561886250972748, -2.49230694770813, 0.5690131187438965, -0.5805128216743469, -0.6208215951919556, -0.8731406331062317, 0.9384886026382446, -0.5985006093978882, -1.9458343982696533, -1.8811781406402588, 0.0706428810954094, -0.9170703291893005, -3.1241369247436523, 1.37938392162323, -0.3800232410430908, -0.1727825403213501, -0.46509701013565063, 2.219810962677002, 2.5890421867370605, -0.9718078970909119, 0.39501580595970154, -0.6697303652763367, 0.1265295296907425, -1.1494163274765015, -0.3522423505783081, 0.96806401014328, -0.9970138669013977, 0.6750146150588989, -1.1647111177444458, 1.3774161338806152, -1.8450946807861328, 1.7334380149841309, 0.5602338314056396, 1.5089490413665771, -0.27351802587509155, -1.8982880115509033, 1.699928641319275, -1.9418151378631592, -1.306334376335144, 1.8929017782211304, -1.5985397100448608, -0.7313767075538635, 0.287712424993515, 1.3246150016784668, -0.08684837818145752, 2.627657413482666, -1.1166858673095703, -0.4960813522338867, 1.4607877731323242, -0.12564527988433838, -1.120936393737793, 1.404532790184021, -0.12785276770591736, 0.23555147647857666, -0.26715192198753357], "dtype": "float32"}}, {"a": [{"__nd__": [-1.075048565864563, 0.63822340965271, -1.2527737617492676, 0.9953588247299194, 2.1428661346435547, 1.5107449293136597, 2.3416497707366943, -1.1315574645996094, 0.19905848801136017, -0.9074109196662903, -0.8146839141845703, -1.8233400583267212, -1.3025991916656494, 0.8398433923721313, 0.6793009638786316, 1.203513264656067, 1.1259746551513672, 0.7732988595962524, -0.8645650744438171, -0.15018461644649506, 0.7290703058242798, 1.2627625465393066, -0.41189515590667725, -0.5328701138496399, -0.47255831956863403, 1.6263442039489746, 0.5661053657531738, -0.3423862159252167, -0.8065356016159058, 0.43704771995544434, -1.7451603412628174, -0.24539174139499664, -0.597165584564209, -0.611206591129303, 0.20146410167217255, 0.4217239320278168, 3.1294360160827637, 0.10353605449199677, 0.7172001004219055, -0.8291743993759155, -1.7049236297607422, 1.1798592805862427, 2.0261623859405518, -1.0221329927444458, -0.06696243584156036, 1.1544313430786133, 0.10801086574792862, -2.4552626609802246, 1.053354024887085, -0.13317951560020447, 0.7614005208015442, -0.9783194065093994, 0.9237542152404785, 1.0846787691116333, -0.727597177028656, 0.5229985117912292, -0.6384115815162659, -0.5028989911079407, -0.9179773926734924, -0.42985799908638, -0.6811418533325195, 1.2215508222579956, -0.21655601263046265, -1.1424050331115723, 1.4251651763916016, 0.5751367807388306, 1.0350061655044556, -0.04774909093976021, 0.7363600730895996, -0.08048992604017258, -0.9599102735519409, 0.13179528713226318, 0.17174561321735382, 0.16391243040561676, 0.8897827863693237, -1.7543033361434937, 0.3310903012752533, -0.7915576696395874, -0.2590629458427429, 0.488852322101593, 0.09934353083372116, -0.6553767919540405, 0.8129527568817139, -0.01988081820309162, 0.9991669058799744, 1.307692289352417, -0.32201722264289856, -1.0801235437393188, 0.13046401739120483, 0.8838015198707581, 0.37683209776878357, -1.1466997861862183, -1.6855201721191406, -0.9114753603935242, -0.9198753833770752, 0.6909968256950378, 0.11862365901470184, 0.9504358172416687, -0.03073877841234207, 1.7700449228286743, 0.07070908695459366, 0.6264184713363647, -0.3578101396560669, 0.5196537375450134, 1.1745411157608032, 0.8568023443222046, 1.3706889152526855, -0.5522089004516602, 0.5336425304412842, 1.8128997087478638, 0.3275827467441559, -0.0969712883234024, -0.1642497330904007, -0.464912086725235, 0.21931138634681702, 1.490288257598877, 0.5335245728492737, 0.04617292433977127, -0.7379665970802307, -1.0879125595092773, 2.5010814666748047, -0.4083156883716583, 0.17022646963596344, -0.42582041025161743, 0.23785382509231567, -0.3699093461036682, 0.17222860455513, 1.5378029346466064, -0.6907739043235779, 0.5964619517326355, -0.012028866447508335, 0.7870753407478333, -0.6381882429122925, 0.4642443358898163, -0.9482086896896362, -1.8067082166671753, -1.294722318649292, -0.9292933344841003, -1.1087137460708618, -0.30254927277565, 1.14255690574646, -1.1345710754394531, 0.47121214866638184, 0.16733068227767944, -0.7040384411811829, -0.9359508752822876, 0.4418719708919525, 0.4932915270328522, -0.34583714604377747, -0.14693519473075867, -0.005420089699327946, -1.3250213861465454, 0.11176911741495132, 0.31139612197875977, 1.7775471210479736, -1.2865819931030273, -0.877257764339447, 0.0858624130487442, -0.5916534066200256, -0.40923014283180237, 0.007817387580871582, -0.2966357469558716, -0.033919256180524826, 0.4440985918045044, -1.6335192918777466, 0.5664440989494324, -2.2656149864196777, 1.7177364826202393, 0.9356186985969543, -0.6099986433982849, 1.0294194221496582, -0.14371681213378906, 0.44219768047332764, 0.7676122188568115, 0.3576658368110657, 0.010484517551958561, 0.6346647143363953, -0.18099136650562286, -0.36767616868019104, -0.9409578442573547, -1.8045413494110107, 1.60584557056427, 0.7982544302940369, -1.392777919769287, 0.4129672348499298, 0.22455714643001556, -2.194443702697754, -0.8200986385345459, -0.0878778025507927, 1.7431988716125488, 1.118430495262146, 1.2829214334487915, 0.3384537994861603, 1.02362859249115, -1.156285047531128, 0.6414597630500793, 0.7556569576263428, -1.244286060333252, 0.2781482934951782, 1.5545119047164917, 1.9610120058059692, 1.4507445096969604, -0.6762915849685669, -0.05813281610608101, -0.48448294401168823, 1.2697128057479858, -0.8103867173194885, -1.0137444734573364, -1.587047815322876, 0.2358672022819519, 0.47559940814971924, -0.5610063076019287, -0.7183217406272888, -0.14615167677402496, 1.389201045036316, -1.305289626121521, 0.0013890627305954695, 0.2542071044445038, 0.42005255818367004, 1.3054454326629639, 0.8781855702400208, 0.3222116231918335, 1.5139625072479248, 1.1937681436538696, 0.34004923701286316, 0.287124902009964, -1.1359184980392456, 0.18073056638240814, -0.7350806593894958, -1.0919853448867798, -0.22658458352088928, -1.680747151374817, -0.8449296355247498, -0.47121432423591614, 0.3778185546398163, -0.2811327278614044, 0.43323978781700134, 1.6213853359222412, 0.8545868992805481, -1.1251932382583618, -1.7521800994873047, -0.8937216401100159, -0.8692778944969177, -1.968169927597046, -1.710193395614624, -1.7114152908325195, -0.6023600697517395, 1.4241218566894531, 1.7001700401306152, -1.0392520427703857, -0.4822259843349457, -1.402772307395935, -1.6681102514266968, -0.11685872822999954, -0.004366234876215458, -0.6912798881530762, 0.567509651184082, -0.4027634859085083, 1.017234206199646, -0.5783166885375977, 0.7761110663414001, -0.2839326560497284, 0.04075141251087189, 0.5383037328720093, -1.4240483045578003, -0.8297860622406006, -0.15960554778575897, -0.36899423599243164, 0.16789604723453522, -0.901286780834198, -1.4218040704727173, -0.19956038892269135, 0.1947323977947235, -0.8998470902442932, 1.122774600982666, 0.603072464466095, 0.7609068155288696, 0.6984923481941223, -0.9397217631340027, 2.049518585205078, -0.7793891429901123, 0.3184286952018738, 0.6760847568511963, 0.2927044630050659, 0.15510091185569763, -0.8365341424942017, -0.41523051261901855, 0.02215041220188141, 1.297265648841858, -0.6578723788261414, 0.696243166923523, 0.45191800594329834, 0.10751888155937195, -0.5220803618431091, -0.10516073554754257, -1.6870442628860474, 0.15506474673748016, 0.37995073199272156, -0.456035315990448, 0.5925941467285156, 0.3873142898082733, -0.6503883600234985, 0.9180743098258972, -0.44890594482421875, 2.495239734649658, -0.939577579498291, -0.7195900082588196, -0.5466392040252686, 0.3272325396537781, 1.2496540546417236, 0.18098397552967072, -0.3111860752105713, 1.9891071319580078, 1.1078020334243774, -0.19355271756649017, -0.027921944856643677, -1.4480386972427368, 0.11891230195760727, 1.798473834991455, -1.0865695476531982, -1.3405897617340088, 0.20577797293663025, -2.320896625518799, 1.4583220481872559, -1.4470795392990112, -0.6617780923843384, -0.10310116410255432, 1.2512719631195068, 0.4416014552116394, -1.8541611433029175, -1.6506880521774292, 0.2344156950712204, -2.068007230758667, -0.27538442611694336, -0.812221109867096, 0.8076767325401306, 1.2079275846481323, -2.5120229721069336, 0.42507532238960266, 0.6323766708374023, -0.7769470810890198, -0.6133223176002502, -1.2170047760009766, -0.36564919352531433, 0.3826982080936432, 1.313394546508789, 1.4115649461746216, -0.3144209682941437, -0.8625769019126892, 0.965941846370697, -2.7951712608337402, -1.4373412132263184, 0.604172945022583, 2.1898550987243652, -0.8220074772834778, 0.06294840574264526, -1.154808521270752, -0.1125173345208168, -0.11478716135025024, -0.7930143475532532, 0.3563525974750519, -0.20422978699207306, 0.4740638732910156, -0.23236599564552307, -1.234588384628296, 0.11370114982128143, 0.8318167924880981, -0.8111926317214966, 1.3190758228302002, 0.4305337965488434, -0.3789007067680359, 0.01756208948791027, -0.4366312325000763, -0.9168868064880371, 0.9046016335487366, 0.8903372287750244, -0.4643664062023163, -0.19497232139110565, -0.22038643062114716, 0.6865158081054688, 1.4569350481033325, -0.6681774258613586, 0.3016127645969391, -0.026155713945627213, 0.555271565914154, 1.4411557912826538, -0.8764351606369019, -2.061659336090088, -0.1684981733560562, -1.6012626886367798, 1.6766579151153564, -0.6937276124954224, -0.27694952487945557, 0.24981504678726196, 0.2155114710330963, -0.41337960958480835, 0.009026159532368183, 0.9583737850189209, -0.29745420813560486, 1.866323709487915, 0.37002265453338623, -2.3407201766967773, 0.9085569381713867, -1.7415587902069092, 0.35563310980796814, 2.279893636703491, 1.1001352071762085, -1.1123042106628418, -0.7926363348960876, 0.042897216975688934, 1.4778647422790527, -0.7088145613670349, 1.1337096691131592, 0.33421483635902405, -0.07489122450351715, -0.1389925181865692, 1.710214614868164, -1.4812196493148804, -1.1097111701965332, 0.9266383051872253, 1.0207372903823853, -1.2755413055419922, -0.8945516347885132, -1.1286672353744507, 0.5768077373504639, -0.6323090195655823, 1.4515880346298218, 0.7966792583465576, 0.2919238805770874, 0.2266910970211029, 0.5681440234184265, 0.8704697489738464, 0.03643528372049332, 0.2811005413532257, 0.8120283484458923, -0.04643815755844116, 1.0235629081726074, -0.1156497448682785, -0.7478850483894348, 0.20969557762145996, 1.6780147552490234, 1.5199753046035767, -1.582150936126709, 0.32524341344833374, -0.4802497327327728, -0.31170156598091125, -0.3323873281478882, -0.2193814069032669, -0.5157130360603333, -0.2698272466659546, 0.22915546596050262, -1.200433611869812, -1.2317246198654175, -1.1181411743164062, -1.132944941520691, 0.5965343117713928, -1.1064200401306152, -0.08018188923597336, 0.06325075775384903, 0.7153446674346924, 0.027215557172894478, 1.05536949634552, -0.059119898825883865, 0.24884887039661407, 0.7594911456108093, -1.4602408409118652, 0.3888775408267975, 1.5116913318634033, 0.11135940253734589, -0.2434501051902771, 1.9028993844985962, 0.10540128499269485, 1.234095573425293, -2.376509666442871, 0.10502316802740097, -0.5026662349700928, 0.38459187746047974, 1.1841169595718384, -0.14850111305713654, -0.22374920547008514, -2.820676565170288, 0.7887475490570068, 1.3310788869857788, 0.1515267789363861, -0.47520700097084045, 0.3672332465648651, 1.0364694595336914, 0.6486294865608215, 2.082751512527466, -0.5571320056915283, -1.3570626974105835, 0.49319374561309814, 1.2727442979812622, -0.4543045163154602, 0.00810531061142683, -0.640387773513794, -1.2548507452011108, -0.7337090373039246, -2.066926956176758, 0.27231284976005554, 0.46209532022476196, 0.8858374357223511, -0.8296325206756592, -0.6410772204399109, -1.6835861206054688, -1.4620620012283325, 0.8851058483123779, 2.8181865215301514, -1.0630215406417847, -0.4021267890930176, 0.9247698187828064, -0.7033829689025879, 0.5762680172920227, -0.30933821201324463, -0.3068234622478485, 0.8990276455879211, -0.26121285557746887, -1.2751420736312866, 0.8171394467353821, 0.018823761492967606, -0.5605177879333496, -0.8107805848121643, 0.5738301873207092, -0.7260957956314087, 0.11494380980730057, 0.996856153011322, -0.11418977379798889, 1.0850428342819214, -1.3847932815551758, 0.6034656167030334, 0.5276445150375366, 1.8699698448181152, 0.016824783757328987, 0.8918395042419434, -0.16264383494853973, -1.0650142431259155, -0.8826722502708435, -0.7101952433586121, 0.7751078009605408, -0.7822225689888, 0.0390169695019722, 0.8978480696678162, 1.2405638694763184, 0.9366365075111389, 1.4915721416473389, -1.1350641250610352, -0.4893532991409302, 0.3170454204082489, 1.114730954170227, 0.5029048323631287, -0.7674216032028198, -0.1902296245098114, -1.651578664779663, 0.8075640201568604, -1.786780834197998, 0.21576359868049622, 1.0769767761230469, 0.9038225412368774, -1.6122119426727295, 0.21409814059734344, -0.34767529368400574, -0.2968983054161072, 1.642608880996704, 0.28728264570236206, -1.4324098825454712, -1.079689383506775, 0.7099534273147583, 0.2247367650270462, -1.1357351541519165, -1.6788519620895386, -0.5573149919509888, -0.4441302418708801, 0.8121137022972107, -0.6177338361740112, 0.6607650518417358, 1.2025130987167358, -1.6137391328811646, -0.5980636477470398, 1.5388896465301514, -0.4610358476638794, -0.031306084245443344, 0.9480972290039062, -0.7794435024261475, -0.4357558786869049, -1.6774156093597412, 0.5296189188957214, -0.5158726572990417, -0.39977046847343445, -2.6065094470977783, 0.9347922801971436, 0.6579669117927551, -0.8335344195365906, 1.187525749206543, -2.4954049587249756, -0.11321136355400085, -1.73336923122406, 2.679318428039551, -2.668282985687256, -1.2742853164672852, -0.8150474429130554, 0.7482399344444275, 0.7725623846054077, 0.32697775959968567, -0.31227245926856995, 2.0796029567718506, -0.7877261638641357, 0.2467808574438095, 1.6861928701400757, -1.3054146766662598, -0.8808966875076294, 1.0356192588806152, 0.3859933614730835, -1.3872534036636353, -0.03912284970283508, 1.8571701049804688, -0.33031871914863586, 0.7347524762153625, 0.053481049835681915, -0.7659477591514587, 1.5432429313659668, 0.8973816633224487, 1.125185489654541, -0.22835735976696014, 0.0249031949788332, 0.1473134160041809, 0.3698478043079376, -0.4271032512187958, -0.5859496593475342, 1.234837293624878, -1.1874022483825684, -0.5280911326408386, -0.4637366831302643, -0.6006788611412048, -1.2985363006591797, -0.8573467135429382, 0.718403160572052, -0.4493582546710968, -0.8300719857215881, 1.5873746871948242, -1.4279767274856567, 0.952655017375946, 0.2660783529281616, -1.447710633277893, -0.49283432960510254, -0.4795284867286682, 1.2092522382736206, 0.8314024806022644, -0.369962215423584, -1.2715821266174316, -1.7904183864593506, -0.26150646805763245, -1.4990588426589966, -0.26957932114601135, -0.29795560240745544, 0.2862878441810608, 0.9127605557441711, 0.0176800899207592, 0.38158831000328064, -0.2642515301704407, 0.9530856013298035, 0.7274221777915955, 1.509709119796753, 1.2925992012023926, -0.8234419226646423, -1.5577775239944458, 0.5540980696678162, 1.0588551759719849, -0.7655169367790222, 1.4154642820358276, 0.006585346069186926, -0.1804799735546112, 0.26538434624671936, -1.5049426555633545, -1.747976541519165, -0.4461745619773865, -1.2806291580200195, -0.12209348380565643, -0.11171415448188782, -2.687425374984741, -0.9193198084831238, 0.4126332402229309, -2.4508540630340576, -0.45876380801200867, -1.4796396493911743, -2.0939011573791504, 0.9936670064926147, 0.9544943571090698, -0.552094578742981, -0.6313082575798035, 1.2621461153030396, 1.071995496749878, -0.14278019964694977, 0.73748379945755, -0.02943258360028267, -0.4482945501804352, 0.4842562675476074, -0.5648497939109802, 0.4548114538192749, -0.38470274209976196, -0.5965983271598816, -0.709530234336853, 0.31721562147140503, 0.7997533082962036, 0.28972935676574707, -1.0098627805709839, 0.47900187969207764, -1.0384902954101562, -1.2660861015319824, -0.33873629570007324, 0.6730307936668396, 0.5532600283622742, 0.42745715379714966, 1.2466511726379395, 0.7345409989356995, -0.4853536784648895, 0.659192681312561, -0.3920721709728241, 0.2756240963935852, 0.47298920154571533, -0.18497514724731445, -1.5960417985916138, -0.252077579498291, -0.4120783805847168, -0.2094714194536209, -0.5181533694267273, -1.2182109355926514, -1.6944867372512817, 0.5715779066085815, 0.6975346207618713, 0.28831106424331665, -0.12855735421180725, 1.8688393831253052, 0.5832528471946716, -0.05846976861357689, 0.7500482201576233, -0.14400997757911682, 0.6268437504768372, 0.20921814441680908, 0.4674629271030426, -0.003215221455320716, -0.5132883787155151, 0.44411078095436096, 1.0559078454971313, -0.9668839573860168, -0.5831406712532043, -1.1383634805679321, 0.8165098428726196, -0.27112653851509094, 0.8316695690155029, -0.8371488451957703, 0.7700929641723633, -0.6177929043769836, -0.5063008666038513, 0.48772376775741577, -0.4324682950973511, -0.6368050575256348, 0.8417524695396423, -0.10867390036582947, 1.1875962018966675, 2.4681990146636963, -0.5987967252731323, -0.37760624289512634, -1.5491162538528442, -1.212232232093811, 0.5549950003623962, 0.25763005018234253, 0.23611146211624146, 0.41737326979637146, -0.8738108277320862, -1.7078951597213745, 1.0351831912994385, 0.20803223550319672, 1.0867526531219482, -0.06653641164302826, 0.7150542736053467, -0.5564828515052795, -0.03925619274377823, -0.19601891934871674, 0.3348346948623657, 1.3506625890731812, 0.6063337922096252, -2.8945705890655518, -0.13561637699604034, -0.024130836129188538, 0.7646659016609192, 0.10819713771343231, -0.7249911427497864, -0.8079013228416443, -0.36362096667289734, -0.7309289574623108, 1.3778239488601685, 0.5670459270477295, 0.02594025433063507, 0.9307919144630432, 2.103410243988037, -0.23275578022003174, -0.05549164116382599, -0.32208940386772156, -1.2405705451965332, -0.6487307548522949, -0.9483420252799988, 0.4506577253341675, 0.42232683300971985, 0.840303361415863, -0.7878500819206238, 0.9586920142173767, -0.9866753816604614, -0.20206882059574127, 1.254223346710205, 1.2005698680877686, 1.057490587234497, 2.0672218799591064, 1.0181862115859985, -0.25948566198349, -0.22855058312416077, -0.7760991454124451, 0.2225039005279541, -0.753409206867218, -0.6217378377914429, -0.9786962866783142, -1.6783263683319092, 2.4492745399475098, 0.8737998008728027, -2.0575733184814453, 0.4345192015171051, -0.5584351420402527, -2.1215178966522217, -0.21548084914684296, 0.20568832755088806, 0.5677739381790161, 1.0889240503311157, -0.8428160548210144, -0.05281146243214607, 0.6563656330108643, 0.5351579785346985, 0.38305604457855225, 0.09684956073760986, 1.221562385559082, -1.2456209659576416, -1.4770063161849976, 1.7225122451782227, 0.4210421144962311, 1.1057555675506592, 0.2322312593460083, -0.35190078616142273, -0.4424501657485962, 0.15168914198875427, -0.45110252499580383, -0.052263882011175156, -0.06247442215681076, 0.8320020437240601, 0.8079757690429688, 1.6054329872131348, -0.673474133014679, -0.00913101527839899, 0.9834729433059692, 0.3833865225315094, -2.3714606761932373, 0.400983989238739, 0.03660105541348457, -0.4223555028438568, -0.361738383769989, -0.6592142581939697, 1.2046122550964355, 0.7354822754859924, 0.27065595984458923, 1.167049527168274, 0.9891257882118225, -0.7435221076011658, -0.9665509462356567, -0.9225010871887207, 0.817533552646637, 0.06272897124290466, 1.2783334255218506, 0.6021775603294373, -0.10893407464027405, -1.1474075317382812, -0.1649673581123352, 2.2939345836639404, -0.5807489156723022, -0.7111479640007019, -0.11607719957828522, 1.4377297163009644, 0.06134083494544029, -0.9854557514190674, -0.4985949695110321, -0.03217325732111931, 0.18070727586746216, 1.1025962829589844, 2.555060625076294, 1.4432904720306396, 0.7302070260047913, 0.08171055465936661, -0.6862578988075256, 0.671830952167511, 1.3619053363800049, 0.914198637008667, -0.022776903584599495, 0.760011613368988, -0.11526306718587875, -0.05489616096019745, -0.12569276988506317, 0.17281150817871094, 0.019008127972483635, 1.0908279418945312, 0.09707535803318024, -0.0019835426937788725, 1.114388108253479, 0.7662209272384644, 1.1958239078521729, -0.660757839679718, 1.3718549013137817, 0.07610747218132019, -0.948220431804657, -0.07613930851221085, -1.1322195529937744, -0.9030418395996094, -0.3349325954914093, 0.9994789958000183, -1.2461342811584473, 0.3410845100879669, -0.19739219546318054, 1.578460931777954, 0.24653314054012299, 0.6430248618125916, 1.4271037578582764, -1.0152946710586548, -0.2821561396121979, -1.4042589664459229, -0.7726093530654907, 0.35843759775161743, -0.1217627301812172, 0.4331483542919159, -0.3546072542667389, -0.6394955515861511, -0.10496386885643005, -0.8295284509658813, -1.0615475177764893, -1.5288280248641968, -0.11316187679767609, 0.4298005700111389, -1.192842960357666, -0.4628010392189026, -2.2494418621063232, -1.1835840940475464, 0.031936440616846085, -1.3007732629776, 0.38172438740730286, -0.436742901802063, 0.30753961205482483, -0.5876467227935791, -1.4339147806167603, -0.074213407933712, 0.6420187950134277, -0.5649855732917786, 0.3142373263835907, -0.747180700302124, -0.39599931240081787, 1.1466668844223022, -0.0828341692686081, -0.9701007604598999, 0.47163844108581543, -0.013071060180664062, -0.5749338865280151, 0.6090652346611023, -2.872256278991699, 0.49623391032218933, 1.7820613384246826, 0.2300073355436325, -1.8874317407608032, 1.2584893703460693, 2.380866527557373, 0.5041162967681885, -1.0811549425125122, -0.7012913227081299, -0.17287307977676392, 0.13843725621700287, 1.0934730768203735, 0.5030826926231384, -2.2223808765411377, -0.334605872631073, -0.18027877807617188, 1.7615790367126465, -0.5988226532936096, -1.3843231201171875, -0.3337409496307373, 0.22311042249202728, -1.817671298980713, 1.014750361442566, -0.16800162196159363, 0.7610911130905151, 0.6285060048103333, 0.4815693199634552, 1.3205077648162842, 1.5786718130111694, -0.427695095539093, -0.10159711539745331, 0.9764545559883118, 0.12282758951187134, -0.6075292229652405, 0.0664922297000885, -0.8569101691246033, -0.5155556201934814, -3.3589019775390625, 0.10519295185804367, 1.7618622779846191, 0.683835506439209, 0.6506158709526062, 0.15740159153938293, -0.7548139691352844, 0.464645117521286, 0.04016491398215294, -0.1432207077741623, 0.22390463948249817, -0.5635124444961548, 0.11680827289819717, 0.6221003532409668, 0.9591206312179565, 0.12896700203418732, 0.7837471961975098, 0.1668056845664978, -2.211940050125122, -1.890803575515747, -0.2064514458179474, -1.0948420763015747, 0.5443927645683289, -1.2894481420516968, -0.4100140631198883, -0.5429268479347229, -0.7304664254188538, -0.5769822597503662, -0.03293001651763916, 0.0814148336648941, -0.5226753950119019, 0.1590309590101242, 0.5147376656532288, -0.1971539855003357, -0.6759917736053467, -0.6646979451179504, -0.47192010283470154, -0.5246622562408447, 0.07389518618583679, 2.6164917945861816, 1.0929549932479858, 0.03495863825082779, 0.5822433233261108, 0.5778390765190125, 1.2388113737106323, 1.3967636823654175, -0.9273389577865601, 2.1884920597076416, -0.8176535964012146, 0.10884109139442444, -0.7888332009315491, -1.636675238609314, -1.7185800075531006, 1.9511656761169434, 1.4759209156036377, -0.9319403767585754, -1.0577138662338257, 1.483432412147522, -0.906586766242981, 0.4818061590194702, 2.0228891372680664, 0.7802627682685852, 1.4910304546356201, 0.2527552545070648, -1.6233625411987305, -0.7611356973648071, -1.7109591960906982, 0.3474064767360687, -1.252945899963379, 0.15064486861228943, -0.4628445506095886, -0.6567877531051636, 0.5238730311393738, -0.441705584526062, -0.7945249080657959, 0.993445873260498, -0.022145099937915802, -1.4322373867034912, 1.1764404773712158, -1.3729610443115234, 0.23678578436374664, 0.5253486037254333, -0.25039178133010864, -0.8054301142692566, -0.3138764500617981, -0.3381282091140747, -1.0793348550796509, 0.8417276740074158, -0.5895808935165405, -0.7105648517608643, 1.350166916847229, -0.7239043712615967, -1.5054947137832642, -0.06826900690793991, 0.007401301059871912, 0.028562180697917938, 2.0474374294281006, -0.5515169501304626, 1.3226284980773926, 0.8693410158157349, 1.433271884918213, -0.0894344374537468, -0.30479031801223755, -0.18791262805461884, 0.3205343186855316, 0.9628768563270569, 1.643454909324646, -1.7214065790176392, 1.0744280815124512, 0.39189451932907104, 0.07276544719934464, -0.9940438270568848, 0.3661687672138214, -1.7617310285568237, 1.2563656568527222, -0.8892465233802795, -0.08782564848661423, -0.13432082533836365, 0.598170280456543, -1.2822297811508179, 0.8506312966346741, 0.4376862645149231, 1.5599100589752197, 0.328079491853714, -0.09637028723955154, -1.835825800895691, -0.33325815200805664, 1.656310796737671, -0.420222669839859, -0.458829402923584, -0.11094678193330765, 0.870471179485321, -1.7110354900360107, 1.8128650188446045, 0.6117827892303467, 0.7397779226303101, 0.020028116181492805, 1.972849726676941, -0.40096643567085266, 0.8347805738449097, 0.1333824247121811, -1.2139606475830078, -0.10354981571435928, 0.7374048233032227, -0.2437659651041031, -0.3699452877044678, 0.013190367259085178, -0.8258195519447327, -1.0631855726242065, 0.10156859457492828, -0.8947599530220032, 0.5937013030052185, 0.7349538207054138, 1.3038464784622192, 0.9938008189201355, -0.24017870426177979, 0.4435516595840454, -1.0992984771728516, 0.4556041955947876, -0.4578123092651367, 1.3687587976455688, -0.6559396982192993, 0.20132435858249664, -0.7234850525856018, 0.6464460492134094, 0.6584309339523315, -0.9566277861595154, 1.9638391733169556, 0.5200446248054504, -1.3717724084854126, -0.8615873456001282, -1.7716999053955078, -1.9491510391235352, -0.11367933452129364, -2.0300204753875732, 0.40172699093818665, -1.5871005058288574, -0.35297051072120667, 1.7136263847351074, -0.3297751843929291, -0.012211291119456291, 1.245577096939087, -0.29026588797569275, 0.5495078563690186, -0.7806057929992676, -0.353678822517395, 0.1473129242658615, 0.0291576087474823, -0.7383667826652527, 0.23870714008808136, 0.6116370558738708, 0.6464532017707825, 0.05340602621436119, -0.9185051321983337, 0.6898705363273621, 0.855553150177002, -0.96103835105896, -2.2622036933898926, -0.1785484254360199, 1.0126886367797852, 0.8648103475570679, -1.159956932067871, 1.3127034902572632, -0.8821960091590881, 0.9127151370048523, -0.4208749830722809, 0.9544802904129028, 0.4931268095970154, -0.23305730521678925, 0.6939300894737244, -0.6612278819084167, -0.38705703616142273, -0.32571080327033997, -0.858503520488739, 0.9029361605644226, 0.9057196974754333, -0.015065072104334831, -0.9739655256271362, 0.43798327445983887, 0.25791430473327637, 0.6024418473243713, -0.03472520411014557, -0.2459702342748642, 1.3492380380630493, -0.6283196210861206, -0.2164953500032425, -1.7914868593215942, -0.40334606170654297, -0.5087606310844421, -0.3698272407054901, 0.5387773513793945, -0.6374421119689941, 0.37466102838516235, -0.4149835407733917, -0.4844694137573242, -1.8764736652374268, 0.35192158818244934, 0.5640805959701538, -1.0262235403060913, -0.6486658453941345, -1.8523021936416626, -0.4296067953109741, -1.1653554439544678, 0.29080089926719666, -0.20787115395069122, 0.5006042718887329, -0.9212510585784912, -1.7760759592056274, -0.1808185577392578, -0.6868544816970825, 1.697546124458313, 0.1246882900595665, 1.3068982362747192, 0.41240981221199036, -0.42321187257766724, -1.6098606586456299, -0.33658942580223083, 0.5056493878364563, -0.024784626439213753, 0.22168323397636414, 0.19826757907867432, -1.415190577507019, 0.19847196340560913, -0.8919408321380615, -0.1436871886253357, 1.240815281867981, -2.0490798950195312, -0.35687878727912903, 0.23976729810237885, 0.9619829654693604, -0.031222224235534668, -0.9145708680152893, 0.6736761331558228, 0.46912869811058044, -1.833521842956543, -0.02852698601782322, -1.7765955924987793, -2.685892343521118, -1.0722161531448364, -1.0104820728302002, -2.18841814994812, 1.7617148160934448, -1.3748266696929932, 0.2945359945297241, 0.6439993381500244, 2.089850902557373, -0.3929203748703003, 1.5409775972366333, -1.1714773178100586, -0.07556214928627014, -0.43305790424346924, -1.0227315425872803, 0.304866224527359, 1.863934874534607, 2.072620153427124, 0.6504458785057068, -1.1441766023635864, -0.8223114013671875, 0.5802941918373108, 0.5515611171722412, 0.5414259433746338, -0.2892455458641052, -0.9182955026626587, 0.43425172567367554, -0.03982848674058914, 1.391278862953186, -0.7055122256278992, 0.8301284909248352, 1.3365111351013184, 1.9564088582992554, 1.940241813659668, -1.0338618755340576, -1.1828582286834717, 0.13868141174316406, -0.873724639415741, -2.3764071464538574, 1.4637596607208252, 0.49218249320983887, 1.02290940284729, 0.1841057538986206, 1.1669200658798218, 0.26629653573036194, 0.14660824835300446, 1.9176180362701416, -1.1499762535095215, 0.5419232249259949, -1.5757479667663574, 1.498006820678711, -0.22378072142601013, -1.2036187648773193, -1.8132925033569336, -0.22024032473564148, 1.9678069353103638, 0.493269681930542, -1.562528371810913, -2.045823812484741, 2.8292343616485596, -0.5814045667648315, -1.4836336374282837, 0.13049769401550293, 0.5720628499984741, 1.5748786926269531, -0.2120964229106903, -0.729774534702301, 0.6671910285949707, -0.8223658204078674, 0.3475848436355591, 0.16409263014793396, 0.9718093872070312, -0.7231408953666687, -0.6982526183128357, 0.27247098088264465, 0.4514361619949341, -1.493510127067566, -0.832598865032196, 0.15209339559078217, -0.3982197046279907, 2.2722296714782715, 0.6996307969093323, 1.2368419170379639, -0.09998886287212372, 0.06290191411972046, 0.09951411187648773, 1.048712134361267, -0.3510081171989441, 1.5643380880355835, -0.7610559463500977, 0.9865989089012146, 0.24904905259609222, -0.38538381457328796, 1.5785657167434692, 0.9697596430778503, -0.2012520283460617, -0.7913442254066467, 0.8187875151634216, -0.8263174891471863, 0.5343275666236877, -0.9226882457733154, 0.09031479060649872, -0.8862617015838623, -0.21281950175762177, 1.6474394798278809, 1.7315188646316528, -0.49466416239738464, -0.28933095932006836, -1.7013806104660034, 0.28266650438308716, 1.1210873126983643, -0.8716939091682434, -0.9341865181922913, 0.3421093821525574, -0.5889579057693481, 0.17577312886714935, 1.5467561483383179, 0.4522940516471863, 0.6687882542610168, 1.8848310708999634, 0.2359023541212082, 0.6884163022041321, 0.057088453322649, -0.7665666937828064, 1.6088340282440186, -1.4045636653900146, 0.07716149836778641, -1.1711769104003906, -0.8343628644943237, -1.3014222383499146, -0.2649272382259369, 0.4622752070426941, 1.30203378200531, -1.8987412452697754, 0.22351329028606415, -0.40686625242233276, -0.6295178532600403, -0.12120742350816727, 0.19561876356601715, -0.11122468113899231, -0.0021514075342565775, 2.192739963531494, 0.10439535975456238, 0.33992916345596313, -1.1502684354782104, 1.0938206911087036, -0.5242139101028442, -0.09611140191555023, -0.7749882936477661, -0.09399706870317459, -0.9330614805221558, -1.295305609703064, -1.398228645324707, -0.38501277565956116, 0.01382573414593935, 0.7211169004440308, 1.6588133573532104, 1.8620132207870483, -0.6334415674209595, -0.7081896066665649, 0.7583281993865967, -0.8759666085243225, 1.1321731805801392, 1.0547035932540894, -0.9419142603874207, 0.7710549235343933, 1.4606415033340454, 0.4808559715747833, -0.25728338956832886, -0.13486924767494202, 0.5092883706092834, -0.7009961605072021, 0.376225084066391, -0.19421398639678955, -0.6972314715385437, -0.9319680333137512, 0.5980426073074341, -0.8595236539840698, 0.264279842376709, -1.7991377115249634, -0.1533980816602707, 0.15202021598815918, 0.39423757791519165, 0.9823877215385437, 0.5642678141593933, 0.6005803346633911, -0.22460974752902985, -0.9871042966842651, -1.11186945438385, -0.7152895927429199, 0.28806474804878235, 1.049920678138733, 0.3930620551109314, 0.249283105134964, 0.8000555634498596, -0.2788122296333313, 0.22233089804649353, -0.06124002858996391, 0.25457003712654114, -0.1700826734304428, -1.2363481521606445, -0.33278411626815796, -0.7607414722442627, 1.761845350265503, 1.4637449979782104, -0.5750356912612915, -0.8096387982368469, 0.059427082538604736, -1.4493186473846436, 1.2714520692825317, -0.01864132285118103, -0.4347485601902008, 1.069745659828186, 1.3000767230987549, 0.17678508162498474, 1.024141550064087, -1.6244977712631226, -0.7306544780731201, -0.8146520256996155, -1.5887755155563354, 0.8758096098899841, 2.3954014778137207, -1.292073369026184, -0.5451972484588623, 1.0678068399429321, -0.2810017764568329, -0.0014359918422996998, -0.7871628403663635, 0.3350633680820465, -0.9584549069404602, 0.7529599070549011, 0.5210946798324585, 0.3749486804008484, -0.7552114129066467, -0.20959509909152985, 0.46702051162719727, -0.8824822306632996, -0.9687047600746155, -2.246901750564575, -0.5314314365386963, -0.15137438476085663, 0.9375474452972412, -1.195270299911499, -1.436255693435669, 1.1880325078964233, 1.0773248672485352, -1.1318925619125366, -0.4978930950164795, -1.811165452003479, -0.07387343049049377, 1.2701352834701538, 0.7744849920272827, -1.051837682723999, 0.27004778385162354, -0.7177926301956177, 0.30751317739486694, -0.546955406665802, 0.6173134446144104, 1.2431395053863525, 0.31640195846557617, 0.3543555736541748, 0.3877732455730438, 0.8649905323982239, -0.41109782457351685, -0.16680438816547394, -0.14966659247875214, 1.305840253829956, 0.16333246231079102, 0.5672598481178284, -0.3570599853992462, -1.333630084991455, -0.29593223333358765, 0.4793241322040558, -0.29948288202285767, -0.5538923740386963, -0.35909542441368103, 0.5225579142570496, 0.4325774610042572, 0.009441223926842213, -0.9233226180076599, -0.6631020307540894, 3.570286512374878, -0.1595599353313446, 0.6182639002799988, 0.3098661005496979, -1.3553857803344727, 1.3121356964111328, 0.08807894587516785, -0.8749949336051941, 0.7565662264823914, -0.5514370799064636, 0.31906697154045105, -1.3012131452560425, -0.17265565693378448, 0.660244345664978, -0.47753599286079407, -1.3838521242141724, -0.8044763207435608, 0.6638932824134827, -1.5640664100646973, -1.3696116209030151, 0.7577922344207764, 0.1062699630856514, -0.052323486655950546, -0.4467623233795166, 0.37942981719970703, 0.8087664842605591, 0.4323456585407257, -1.732041835784912, 0.6127843856811523, -1.6365818977355957, 1.683210849761963, -0.4477090835571289, 0.5887224674224854, 0.3173922598361969, -0.637575626373291, 1.1734719276428223, 0.21714945137500763, 2.0935285091400146, 0.6572917699813843, 0.34146633744239807, 2.492621421813965, 1.3941092491149902, 0.040984924882650375, -0.7567098736763, 1.9340217113494873, 1.2497669458389282, 1.2493140697479248, -0.27568677067756653, -0.5556349158287048, -0.8355951905250549, 1.175079345703125, -0.10909204930067062, 0.697176456451416, -2.1045985221862793, 1.0617667436599731, -0.6707392930984497, -2.654589891433716, -1.237740397453308, 0.4359651207923889, 1.5310472249984741, -2.710397720336914, -1.344512939453125, 0.1391548365354538, -1.5407534837722778, -0.43253597617149353, -1.0329102277755737, 0.08004424721002579, -1.4884148836135864, -0.36509406566619873, -0.865286111831665, 0.1026621088385582, -0.09623314440250397, 0.9711006283760071, 0.20474882423877716, 1.7298612594604492, -1.6180338859558105, -0.2785531282424927, 0.7506858706474304, 0.08713827282190323, -2.6462433338165283, 0.30318209528923035, -0.5072449445724487, 2.439357042312622, -0.11781815439462662, 0.6676194071769714, -0.9634756445884705, 0.6814649105072021, -2.7957539558410645, -1.0296320915222168, -0.3909095227718353, -1.6114106178283691, 0.97292160987854, 0.006933615077286959, 1.650026798248291, -1.4236109256744385, -0.839648425579071, 0.6502833962440491, -0.512188196182251, 0.831801176071167, 0.6463663578033447, -0.7526623606681824, 0.12294923514127731, 0.5414919853210449, 0.24797643721103668, -0.20372332632541656, -0.48453640937805176, -0.22173917293548584, 0.9934295415878296, 2.080723762512207, -1.4162054061889648, -1.0058883428573608, -0.15636394917964935, 1.471055269241333, -0.8518376350402832, 2.3517117500305176, 2.3081536293029785, -2.3816113471984863, 0.702487587928772, 0.628286600112915, -0.40292420983314514, -1.844928503036499, 0.3021776080131531, -0.3693740665912628, 1.218359112739563, 0.6074314117431641, -0.9110127091407776, 2.0131146907806396, 1.892362117767334, 1.5539147853851318, 0.11474481225013733, 0.9943466186523438, 0.16560682654380798, -0.4895205497741699, -1.8240573406219482, 0.3438192903995514, -1.9332629442214966, 0.7804359197616577, -1.3083491325378418, 0.2848896086215973, -0.22053106129169464, -1.0875132083892822, -0.9770718216896057, -0.5174654126167297, 1.574944257736206, 0.44871628284454346, -0.20951911807060242, 0.8950755596160889, -1.1221411228179932, -1.4797409772872925, -0.8521938920021057, 1.094152569770813, -0.25451377034187317, 0.15299458801746368, 0.15020976960659027, -0.397347092628479, 1.2853150367736816, 0.7484840750694275, -0.8485654592514038, -1.0246504545211792, -0.030833039432764053, -0.9186582565307617, -0.00038168070022948086, 0.750859260559082, 0.06189509853720665, 0.19513818621635437, -0.5028058886528015, -0.8184067010879517, -1.0406965017318726, 1.31692636013031, -0.7820993661880493, 0.08159974962472916, 0.10883943736553192, 0.44854485988616943, -0.34249311685562134, -1.637119174003601, 2.1220216751098633, -1.0916191339492798, 0.6080378890037537, -1.1008683443069458, 0.6749172806739807, -1.0913232564926147, 1.7241355180740356, 0.46932414174079895, -1.6185442209243774, 1.716004490852356, -1.9333692789077759, -0.3497684895992279, 1.1288939714431763, -0.40299320220947266, 1.3060615062713623, 0.036716170608997345, -0.6142854690551758, 1.271960735321045, 0.036266110837459564, -0.49626216292381287, 0.9570521116256714, -0.273233026266098, -1.421105980873108, -0.7606175541877747, 2.5156099796295166, 2.3286781311035156, -1.6141629219055176, 0.8000585436820984, 1.2529182434082031, -0.3031063675880432, 0.6793004870414734, -1.0660061836242676, -0.36261045932769775, 1.0537540912628174, -0.08786819130182266, -0.18985199928283691, -0.2862972021102905, -1.4121413230895996, -1.0782663822174072, -0.5839353799819946, -0.17329417169094086, -0.9322655200958252, 1.9000128507614136, -1.8492469787597656, -1.1540828943252563, -0.708345353603363, 0.49857282638549805, -1.5290381908416748, 0.2203899472951889, -0.920688807964325, -0.859854519367218, 0.3991701900959015, 0.8085860013961792, 1.0212628841400146, -1.250342845916748, 1.023128628730774, 1.0429410934448242, 0.30733582377433777, 0.4388447105884552, 0.015101323835551739, -0.17662645876407623, 0.8669028878211975, 1.7064766883850098, -0.27521947026252747, 0.35839590430259705, 0.9557065367698669, -1.4407150745391846, 1.6849908828735352, 0.3810654580593109, 1.5009524822235107, -0.2847825288772583, -0.891306459903717, -0.13724027574062347, 0.32590821385383606, -0.942083477973938, 0.9572057723999023, 0.29764434695243835, -1.074100136756897, 0.5115475058555603, -0.13059723377227783, 0.062459830194711685, -0.3798774778842926, -0.7064502835273743, -1.2221592664718628, 0.1545306295156479, 0.0846627801656723, 0.5402213931083679, -0.6901994347572327, 0.49469950795173645, 0.9095367193222046, -1.1578755378723145, 0.2458556443452835, 2.319342851638794, -0.4839458465576172, -0.2907932996749878, -0.01719772070646286, -0.5131248235702515, 0.9138010144233704, -0.0656479224562645, -0.09169206768274307, -0.7893660664558411, 0.5728089809417725, -0.08502046018838882, -0.40456923842430115, -0.3139932155609131, 1.6392996311187744, 0.007254068274050951, 0.2850475013256073, 1.3913912773132324, -0.09627299755811691, 0.9927016496658325, -1.0264883041381836, 0.6821591854095459, -2.163881778717041, 0.15861396491527557, -0.43434834480285645, -0.7266392111778259, 1.1113063097000122, 0.760455846786499, 0.8844071626663208, 1.5432038307189941, 0.5457643270492554, 0.5949342846870422, -0.05530112236738205, 1.9313864707946777, 0.9157949686050415, -0.7917551398277283, -0.8811447024345398, -0.7560880184173584, -0.7535562515258789, 1.3189194202423096, 0.3058207333087921, 1.0828461647033691, -0.7783719897270203, -0.7866873145103455, 1.2410814762115479, -2.051985263824463, -1.2271124124526978, -2.162177085876465, -0.6988748908042908, 2.6182332038879395, -0.795486569404602, 1.0548917055130005, 1.6586053371429443, -0.6902245879173279, -0.7387391328811646, 0.32560694217681885, 0.6712690591812134, -1.6157554388046265, -0.8406364321708679, 1.6631064414978027, -0.8394145965576172, 0.36781546473503113, 1.0374282598495483, -0.3525281846523285, -0.19137704372406006, -0.5751323103904724, -0.6813009977340698, -0.6327389478683472, -1.6749026775360107, -0.5251346826553345, -0.23712575435638428, 0.713134229183197, -0.4400823712348938, -0.6851539015769958, 1.9100030660629272, -0.026851430535316467, -0.24558593332767487, -0.5448981523513794, -0.01831239089369774, -0.8920486569404602, -0.533396303653717, -0.881085991859436, -1.246883749961853, -0.46806973218917847, -1.4139965772628784, 1.73918616771698, -1.1884949207305908, 1.7720184326171875, 0.3122865557670593, 0.8840938806533813, 1.4663175344467163, -1.682665228843689, 1.6538702249526978, 0.49277955293655396, -0.49142083525657654, -0.7630104422569275, 0.3644086718559265, -1.6177973747253418, 1.8194037675857544, -0.1521380990743637, 0.8120907545089722, 0.1472570300102234, -0.39828580617904663, -0.3492928445339203, -0.8367205858230591, 1.3140381574630737, 0.03465619310736656, 0.5706873536109924, 0.5290014743804932, 0.7392350435256958, 0.5120236277580261, 1.3412905931472778, -0.5848289728164673, -0.9400897026062012, -0.05250561982393265, -0.2663779556751251, 0.1282561719417572, 1.2527656555175781, 1.0476500988006592, 0.6455318331718445, -0.7073768973350525, 0.5907521843910217, 0.2622315287590027, 0.8402878642082214, 0.07493194192647934, -0.3165616989135742, 0.21118608117103577, -0.644831657409668, -0.8053686022758484, 0.3150765895843506, 0.08684797585010529, 1.083223581314087, 0.23378674685955048, -1.0963890552520752, 1.4550319910049438, 1.7128291130065918, -1.0212355852127075, -0.5270932912826538, 1.0586167573928833, 0.2617328464984894, -1.4041814804077148, 1.2050063610076904, 0.15762192010879517, 0.3936457335948944, 1.2356406450271606, -0.6017642617225647, -0.668084442615509, 0.7158992290496826, 0.9835217595100403, 0.16535411775112152, 1.208230972290039, 0.4744436740875244, 1.1390475034713745, -0.7647711634635925, -0.16170145571231842, 1.104487419128418, -0.8053058385848999, -0.7512340545654297, 0.17443440854549408, -0.236123725771904, -0.902177095413208, -1.5975347757339478, 1.3845163583755493, 0.08590026944875717, 1.4438356161117554, -0.852173924446106, 0.09669540077447891, -0.13046741485595703, 0.11914198100566864, -1.541056513786316, 1.237330675125122, -0.2618248164653778, 0.23269023001194, -0.9678936004638672, 1.5392553806304932, -1.500089168548584, 1.8903999328613281, -0.39373689889907837, -0.34154775738716125, -0.6721630692481995, 1.3907724618911743, -0.8508502244949341, -0.8695841431617737, -0.944063127040863, 0.4801604449748993, 0.5300965309143066, -0.7220406532287598, 0.5061057209968567, 1.5214709043502808, 0.6821835041046143, 0.3744124472141266, -1.0155223608016968, 1.535480260848999, 0.6659669280052185, -1.4216700792312622, -0.7457132935523987, -2.749814748764038, 0.49121055006980896, -1.249894142150879, -0.28288814425468445, 1.3514976501464844, -1.4405348300933838, 0.8247525095939636, -0.337567538022995, -1.117944359779358, 1.2366554737091064, 0.36517366766929626, -0.7306250929832458, 0.47487035393714905, -0.6402168273925781, -0.9079326391220093, -0.4250197112560272, 0.9356526732444763, -1.6389334201812744, -1.3523606061935425, 0.8204214572906494, -1.4504297971725464, -0.2622453570365906, -0.3671126663684845, -1.346867322921753, -0.8074499368667603, -0.1908111721277237, -0.14782896637916565, 0.5629854202270508, 0.15250596404075623, 0.8446555733680725, 0.14276860654354095, -0.4922029376029968, 0.5652502775192261, 0.37791454792022705, 0.7965250015258789, -0.20952901244163513, -0.1902834177017212, -0.17570364475250244, -1.5349700450897217, 0.39327800273895264, -0.7128944396972656, -0.8087930083274841, -0.4188983738422394, 2.0145816802978516, -0.8388190865516663, 1.4401689767837524, -2.135014295578003, -0.3341568112373352, 1.8559691905975342, 0.1298631876707077, 0.3316729962825775, -0.604195237159729, -1.989652156829834, 0.47130200266838074, 0.8858662843704224, 1.0233968496322632, 1.878167748451233, -0.1048194169998169, 0.49894213676452637, -2.1443634033203125, -0.3601423501968384, 0.12572255730628967, 1.2481122016906738, 1.8234049081802368, 1.5926483869552612, 0.4448704123497009, -0.14433614909648895, -0.08417779952287674, -0.055939797312021255, 0.13451607525348663, 1.1668329238891602, -0.19751659035682678, -1.6594657897949219, -1.1954830884933472, 2.916414976119995, 1.3336786031723022, -0.49982407689094543, -0.26925310492515564, 0.7168418169021606, 0.29723161458969116, 0.3150562047958374, -1.9660652875900269, -1.378847599029541, 0.1436784267425537, 0.6942001581192017, 0.31493180990219116, 1.5697509050369263, 0.05464399233460426, 0.25792914628982544, 0.0668523907661438, 0.9762664437294006, -0.602423906326294, 2.038215398788452, -1.0597578287124634, 0.784909725189209, 0.7270870208740234, 1.5891849994659424, -1.1153937578201294, -0.8985681533813477, -0.27721095085144043, 0.7261066436767578, 0.8614540696144104, -2.569058656692505, -0.3087531328201294, 0.24839890003204346, 0.34746718406677246, 0.35757163166999817, -0.14505191147327423, 0.10848987102508545, -1.8763645887374878, 0.8646455407142639, -2.4240243434906006, -0.1935790330171585, -0.9335836172103882, 0.30713996291160583, -1.0088669061660767, 0.8451390266418457, 0.31153225898742676, 0.5844337344169617, 1.0418003797531128, 1.1036314964294434, -0.2290855497121811, 0.7738296389579773, 0.4421151578426361, 0.7228293418884277, 0.1122841089963913, -2.568896532058716, -0.2970739006996155, -0.0687665045261383, -0.5468004941940308, -0.7559257745742798, 0.094472236931324, -0.7402891516685486, 0.5936488509178162, 1.1893256902694702, -0.04538743942975998, 0.7913878560066223, -1.232759714126587, -1.0471168756484985, -0.37095344066619873, -1.0076968669891357, -0.006670055445283651, 0.024871570989489555, 1.5270124673843384, -0.5706924796104431, 1.202695369720459, -1.0510708093643188, -0.27462294697761536, 0.7770149111747742, 0.28569474816322327, -2.450955629348755, -0.0862598642706871, 0.4207497239112854, -1.529572606086731, -0.48604485392570496, -0.3419809937477112, 0.35744819045066833, 1.216838002204895, -1.3766244649887085, -0.9226536750793457, 1.5151138305664062, 1.1448619365692139, 1.662216067314148, -0.052602872252464294, 0.9870933294296265, -2.397474527359009, 0.4218154847621918, -0.9718543887138367, -1.412172794342041, -1.095406174659729, 0.9629751443862915, 1.4449180364608765, -0.5075094103813171, -0.37301570177078247, 1.1423559188842773, -0.6887819766998291, 0.23117507994174957, 0.5912147164344788, 1.1604474782943726, -0.29011282324790955, -2.7804548740386963, -0.21619412302970886, 0.6653842926025391, -0.47981855273246765, -0.5846770405769348, -0.9586490988731384, 0.4075322151184082, 0.14442561566829681, 0.2538650929927826, -0.015118052251636982, 1.2416189908981323, 0.31048789620399475, -0.08177822828292847, -1.1172020435333252, -0.41754692792892456, -0.017902741208672523, 1.4427471160888672, 0.4600135087966919, -0.3376632332801819, 0.7311762571334839, 0.6120651364326477, -0.45325252413749695, 0.4491333067417145, -1.0806190967559814, 0.09492747485637665, 1.9822105169296265, 0.9592180252075195, 0.6159011125564575, 0.8608480095863342, 0.695300281047821, -1.1480028629302979, -0.4475531578063965, -0.6745718121528625, 0.46219778060913086, 0.9061958193778992, 2.0098371505737305, -0.13756045699119568, -1.104506015777588, 0.29439517855644226, -0.025278137996792793, 0.2680017948150635, 0.030295953154563904, 0.5059695839881897, -0.5544782876968384, 0.30892840027809143, 0.9802942276000977, 0.19188739359378815, 0.5556138157844543, -0.6099476218223572, 0.8657662868499756, -0.08300809562206268, -0.631192147731781, -0.19826218485832214, -0.3597106337547302, -0.1471385508775711, -0.523465096950531, 1.6850649118423462, 0.9002399444580078, 0.47505947947502136, 0.4662785232067108, -0.07770387828350067, 0.8114545345306396, -0.21667027473449707, -0.06306801736354828, -0.7473912239074707, 1.1903975009918213, 1.1598162651062012, 1.0741612911224365, 1.095296025276184, -2.312026023864746, 0.09705091267824173, 0.9984279274940491, 0.48992064595222473, -1.5396058559417725, 0.2887379229068756, 0.6610885858535767, -0.5385056734085083, -1.1382567882537842, -0.6047691702842712, -2.92169189453125, 0.2239842265844345, -0.8611741065979004, -0.34814217686653137, -0.20817647874355316, -0.13226987421512604, 1.4966621398925781, -0.8784251809120178, -0.28378865122795105, -2.1414244174957275, -0.7248339653015137, -1.9569417238235474, 0.6731711626052856, -0.0986836701631546, -0.6767418384552002, 0.7787857055664062, -1.0858794450759888, -0.2546267509460449, -0.8847774863243103, -0.9944509863853455, -0.30859577655792236, -0.6748923063278198, -0.21795545518398285, -1.1592957973480225, -1.6072081327438354, 1.2586127519607544, 0.2981046140193939, -0.4301948547363281, 0.1420215219259262, -1.0471539497375488, 0.3074738085269928, -1.4907586574554443, 0.3969714045524597, 0.3044697046279907, -0.11661414057016373, 0.6518802046775818, -0.12338438630104065, 1.6021801233291626, 0.5760362148284912, -1.5402849912643433, -1.0214461088180542, 1.579156517982483, -0.5751088261604309, 0.3096616864204407, -0.5687583088874817, 1.4532824754714966, 1.0483020544052124, 0.7671726942062378, -0.42278388142585754, -0.31329452991485596, 1.3447492122650146, -0.5519164800643921, -2.112281560897827, 1.5256413221359253, 0.9573380351066589, -1.2847871780395508, 0.2754720151424408, 0.5975174903869629, 0.21808400750160217, 1.6373897790908813, 1.2376309633255005, -0.7564480900764465, -0.12708312273025513, 0.9282832741737366, 0.9475933909416199, -0.3148077726364136, 1.1349914073944092, -0.7951467037200928, 0.8369782567024231, -1.7157403230667114, 0.4909124970436096, 2.0398099422454834, 1.5122525691986084, -0.7863904237747192, -2.8961586952209473, 0.2923137843608856, 1.0733124017715454, 0.12330663204193115, 0.25093814730644226, -1.350144863128662, -0.5247480273246765, -1.2595685720443726, -1.2799580097198486, -0.5543888211250305, 1.3848638534545898, 0.34593823552131653, -0.7895117402076721, -0.18683169782161713, -0.9644506573677063, -0.6024739742279053, 0.1475801318883896, -1.5267271995544434, 0.2409612536430359, -0.6705007553100586, 1.0232514142990112, -0.9748265743255615, 0.1566420942544937, -0.8996762037277222, 0.45644187927246094, -1.7826719284057617, -0.5809178352355957, -0.9138688445091248, -1.6285051107406616, -0.42413583397865295, -0.24876977503299713, -0.6491518616676331, -1.0675519704818726, -0.7434971928596497, -0.7764566540718079, 0.9737842082977295, 0.46493270993232727, 1.2509759664535522, 0.8840399980545044, 1.7512153387069702, 0.030972784385085106, 0.2767595946788788, 0.10047470033168793, -0.362280935049057, 0.6764311194419861, 0.41441091895103455, 0.5218102931976318, 0.30401942133903503, 1.0747910737991333, -1.3216980695724487, 0.6231931447982788, 0.022708851844072342, -0.5525521039962769, -0.2589882016181946, 0.5546296238899231, 0.872077465057373, -0.1502871960401535, 0.7703831195831299, 0.4366331100463867, 1.0833320617675781, 1.0756345987319946, 0.8331667184829712, -0.2641705274581909, -1.0115488767623901, 1.4344143867492676, -0.7175403833389282, 0.8899715542793274, 0.15810811519622803, 1.691528081893921, -0.022295404225587845, -0.21225623786449432, -1.9235199689865112, 0.6894814968109131, 0.07428566366434097, -0.5020357370376587, -0.296444296836853, 1.33477783203125, 0.4783438742160797, 0.9188759922981262, -0.13806043565273285, 0.40359607338905334, 1.0220285654067993, -0.16230559349060059, 1.3670581579208374, -0.12186826765537262, 1.9932668209075928, -0.41792261600494385, -0.31184545159339905, 0.25360214710235596, -0.198933944106102, 0.03983632102608681, 0.11492440104484558, 1.3325234651565552, -0.3309478759765625, 1.2841169834136963, 0.994513988494873, 0.2856611907482147, -0.04757365584373474, 1.9934837818145752, -2.320129156112671, -0.3068768084049225, 0.023454684764146805, 0.6652982234954834, 0.18769381940364838, 1.0439180135726929, 0.6865600943565369, 1.208250641822815, -0.4371156394481659, 0.6102017760276794, 1.7705351114273071, -0.47922253608703613, 0.34457340836524963, -1.2218971252441406, 0.5415335893630981, -0.613460898399353, 0.7513121366500854, -0.9836022853851318, 0.17430612444877625, -0.8716813921928406, 0.09554453939199448, 0.5316299796104431, 2.3642501831054688, 0.5340889096260071, 0.5881783366203308, 1.922155737876892, 1.1691478490829468, -0.9129571318626404, 0.6355729103088379, 0.22813309729099274, 0.9140201807022095, 0.6203571557998657, -1.632766842842102, -0.4589295983314514, 1.856222152709961, 0.9202741384506226, -0.7761565446853638, -0.414250910282135, -0.21564960479736328, -1.2829378843307495, -3.2390079498291016, 0.5209661722183228, -1.0445300340652466, -0.7054965496063232, -0.7828716039657593, -0.4287485182285309, -0.4834049940109253, 1.4897578954696655, -0.359178364276886, 0.5434268116950989, -0.07421767711639404, 1.232263445854187, 0.06839657574892044, -2.0478355884552, 1.7602680921554565, 1.6195762157440186, 0.6290784478187561, -1.632546305656433, 1.271926760673523, 0.3000664710998535, -0.8731777667999268, -0.6124951839447021, 1.1117973327636719, -0.6928720474243164, -0.22290585935115814, -0.8152444362640381, 0.4665774703025818, -0.9470061659812927, 0.5202494263648987, -0.36400914192199707, 1.9993058443069458, -0.5102139115333557, 0.04350727051496506, 0.9254182577133179, 0.8289088606834412, -1.5688652992248535, 0.14713901281356812, 0.3732875883579254, 0.6585323810577393, -0.18969376385211945, 1.4805667400360107, 1.3742753267288208, -0.3767971992492676, -1.0863932371139526, -0.07022836059331894, 0.5857216119766235, 1.1146835088729858, -1.130732774734497, -0.6186981797218323, -1.916623592376709, -0.9250350594520569, 1.1739128828048706, 2.1816914081573486, 0.8316411972045898, -0.03929794207215309, -0.1627182811498642, 2.0721590518951416, 1.3377366065979004, 0.342122346162796, -0.4014434814453125, -0.6343620419502258, 0.13158579170703888, 1.9483529329299927, -0.09304112941026688, -0.21014994382858276, 0.4085165560245514, -0.1314278095960617, 1.0766867399215698, -0.8976821899414062, 0.20812371373176575, -0.5863267183303833, 0.9476468563079834, 0.24950715899467468, -0.6668586730957031, -0.46295225620269775, 0.01278245635330677, 1.5112813711166382, 0.5898044109344482, 0.32922154664993286, -0.9738402366638184, -1.5076431035995483, -1.5534634590148926, -0.6211667656898499, 0.38403433561325073, 0.027503805235028267, 1.114670753479004, -1.1197870969772339, -0.23577281832695007, -0.2863295078277588, -1.0220237970352173, -2.209699869155884, -0.8497530221939087, 0.8886198997497559, -1.7853374481201172, 0.8176527619361877, 0.8446223139762878, -0.1924487203359604, 0.24313686788082123, -0.16929325461387634, -0.02441699616611004, 1.4093947410583496, 1.1155835390090942, 0.9670115113258362, 0.2318372130393982, -0.1476326584815979, 0.2328444868326187, -1.3103801012039185, -0.9038159847259521, 0.047381505370140076, -1.8194992542266846, -0.03149985149502754, -0.172993466258049, -0.22212882339954376, -0.17823055386543274, -0.7141103148460388, 2.0320358276367188, -1.9871256351470947, -0.797328770160675, -1.8512874841690063, 0.25703707337379456, 0.8264469504356384, 1.7175540924072266, -0.8143184781074524, 0.5786975622177124, -0.9879243969917297, -0.16460460424423218, 0.5630753636360168, -1.5796629190444946, -0.27067258954048157, -0.7335817813873291, 0.2735593020915985, 0.7141014933586121, -1.5735341310501099, -0.21802979707717896, -1.4893815517425537, -2.4122202396392822, -1.2119766473770142, -0.7661239504814148, -1.322095274925232, -1.224990963935852, 1.1877943277359009, 0.007600078824907541, 0.9865913987159729, 1.3404377698898315, 0.08216717094182968, 2.506687879562378, 0.38258421421051025, 0.08033277839422226, -0.8700048327445984, -0.3970571458339691, 0.3176994323730469, 0.042528219521045685, 0.19579698145389557, 0.30367806553840637, -1.412689208984375, -0.8442835211753845, -0.4524292051792145, -0.5966315865516663, -1.4032821655273438, -0.1515142172574997, -0.9476966261863708, 0.9292663335800171, -0.8807024955749512, 0.1605234593153, -2.0733981132507324, -0.24395613372325897, 0.9139878153800964, -0.03728187456727028, 0.5674391388893127, 0.12295264005661011, -0.39133909344673157, 0.4119398891925812, 0.3267303705215454, 0.3048764169216156, -0.675815761089325, -0.5780598521232605, -2.8459317684173584, 0.00031582728843204677, -1.1566011905670166, -0.855841875076294, 0.719936728477478, 0.08551790565252304, 0.2880842983722687, -0.8276312351226807, 0.37575197219848633, -0.07876527309417725, 0.09461095184087753, -0.21335454285144806, 1.54560124874115, -1.2197989225387573, 0.1454555094242096, 0.19011050462722778, 0.8498849868774414, 0.4317305386066437, -0.5563278794288635, 0.08224255591630936, 1.450370192527771, 1.0075387954711914, -0.4568757116794586, 0.13074713945388794, -0.18158596754074097, 1.273815393447876, -0.4710753262042999, 1.2688868045806885, 1.6044450998306274, -0.16545109450817108, 2.21448016166687, -0.2583432197570801, -0.21993988752365112, 0.7218661308288574, -0.5329793691635132, 0.08123308420181274, -1.1918468475341797, -0.42383745312690735, 0.06737471371889114, -1.364235758781433, 0.6924426555633545, 0.7283409833908081, -0.12546902894973755, 0.6393386721611023, 0.6432555913925171, -0.812708854675293, 0.5251984000205994, 0.19700033962726593, -0.6730917096138, 0.35934585332870483, -1.0825728178024292, -2.8366503715515137, 0.42184141278266907, 0.22632746398448944, 1.7928082942962646, -0.32221174240112305, 0.8788105845451355, -0.8740854263305664, -1.1256844997406006, -0.3049001395702362, 0.23474064469337463, -0.010306081734597683, 1.1369733810424805, 1.2154638767242432, -1.2155307531356812, -0.45480698347091675, 1.8726719617843628, 1.4341121912002563, -1.0681358575820923, -0.6473856568336487, 2.095716953277588, -0.8317267894744873, 1.3729077577590942, -1.8565852642059326, -0.06273168325424194, -0.8769047260284424, 0.4913386404514313, 1.4308311939239502, 0.627220630645752, 1.179795265197754, 1.3416520357131958, 1.8303284645080566, 0.49361690878868103, -1.7817912101745605, -1.4173986911773682, 1.0528526306152344, 0.7731364965438843, 1.2343432903289795, -0.19346782565116882, -0.3615552484989166, -1.7643001079559326, -1.0488007068634033, -0.08563003689050674, -0.6375126838684082, -1.6148853302001953, 1.4765770435333252, 0.029858361929655075, 0.2665244936943054, 0.11086724698543549, -0.028920425102114677, -1.070191740989685, 0.8543024063110352, -0.5800677537918091, -1.0328586101531982, 1.2379571199417114, 0.08964353054761887, 0.5981334447860718, -1.9157811403274536, 1.744152545928955, -0.8516539335250854, -1.1595442295074463, -0.42190128564834595, -0.9862055778503418, -0.9045524597167969, 0.4766611158847809, 1.4492028951644897, -0.2510985732078552, 0.005345165263861418, 0.06724829226732254, 0.19015845656394958, -1.2640454769134521, 1.6914395093917847, -1.4360071420669556, -0.6553221940994263, 2.0652248859405518, -0.11648282408714294, -1.0083272457122803, 0.12194812297821045, 2.360511302947998, 0.08412276208400726, -1.1081973314285278, 0.2513388395309448, 0.3310564160346985, -0.819326639175415, 0.2671341001987457, -2.2360525131225586, 0.48878705501556396, 0.2515065371990204, 0.1310998648405075, -0.6087847948074341, 1.4142669439315796, 0.7252517938613892, 1.412636637687683, -0.5597255825996399, 0.8192731738090515, 1.8046321868896484, 1.4892109632492065, -0.8845078349113464, 0.014962228015065193, -0.6082407832145691, -0.24800287187099457, 0.036632053554058075, 1.815423846244812, 1.5750423669815063, -0.10105457901954651, 0.9355946779251099, -0.564344048500061, -1.228369951248169, -1.8748444318771362, -0.8342218995094299, 2.047058582305908, 0.22917519509792328, 0.5761849880218506, 1.095882534980774, -1.9561913013458252, -0.2158418893814087, 0.050515249371528625, 1.3077490329742432, 0.43718472123146057, -1.681326150894165, -1.7240054607391357, 0.32490643858909607, 0.2207748293876648, -1.4149986505508423, -1.4006260633468628, 0.09053339809179306, -1.3257967233657837, 0.36523598432540894, -0.24323518574237823, -0.27990350127220154, 1.9807639122009277, -0.8382891416549683, -1.1439954042434692, -0.5341749787330627, -2.4348392486572266, 0.7746961116790771, 0.6457025408744812, 0.07481909543275833, -0.046212416142225266, -0.09973155707120895, -1.5914552211761475, -0.19205255806446075, -0.07239701598882675, 0.7995855808258057, 0.2576087713241577, 0.9932488203048706, 0.6517404913902283, -2.0609066486358643, -0.8009156584739685, 0.6050236821174622, -0.4604676067829132, -0.04647153988480568, 0.8254296779632568, -1.0351957082748413, 0.15554329752922058, 1.6202391386032104, 0.022849924862384796, -0.7388949394226074, -0.007565599400550127, -0.39536458253860474, -1.276892900466919, 0.6304140686988831, 0.004235670901834965, -1.9015190601348877, 1.1986329555511475, -0.963470995426178, 0.05679740756750107, 0.44539302587509155, -0.1426936239004135, -1.2888950109481812, -2.379431962966919, -0.5794515013694763, 0.2137448489665985, -1.245520830154419, -0.426196426153183, 1.276164174079895, -1.0234445333480835, 4.036861896514893, 0.36645764112472534, -1.2229506969451904, 0.414054274559021, -0.24901263415813446, 1.324722409248352, -1.285361409187317, 0.6888108253479004, -1.124700903892517, 0.3363484740257263, 1.5278546810150146, 0.14091891050338745, 1.3659188747406006, -0.3092799186706543, 0.9436260461807251, -0.6644560098648071, -0.2614412009716034, 0.015903571620583534, -1.509752631187439, -0.946412980556488, -0.6136409044265747, 1.7274078130722046, 0.14391441643238068, -1.1831753253936768, -1.2470006942749023, -0.11820031702518463, -1.1768467426300049, 0.17385804653167725, -1.1402838230133057, -1.093242883682251, 0.8201685547828674, -1.9197845458984375, 2.3038978576660156, -0.7901695966720581, -1.5226995944976807, 0.5894994139671326, 1.0887129306793213, 1.3148207664489746, 0.5304538607597351, -0.718711256980896, -0.17637406289577484, 0.13519801199436188, 0.8804975152015686, 0.9818411469459534, 0.7133980989456177, 0.8757429718971252, -2.277080774307251, -0.5607978105545044, 1.1874595880508423, -0.47382596135139465, -0.07698626816272736, -0.733051598072052, -0.133950874209404, -0.6794551014900208, -2.0792694091796875, 0.3853945732116699, 0.43081772327423096, -0.9136263728141785, -2.110457181930542, -0.3631592094898224, -1.3108571767807007, 0.9884555339813232, -0.3056862950325012, -1.6905783414840698, -0.7789376974105835, 1.8001834154129028, -0.2737944722175598, -0.9268800616264343, -0.37000006437301636, -1.2932252883911133, 0.7845473885536194, -1.537399172782898, 0.5140547752380371, 0.9770203828811646, 0.8702061176300049, -0.19542378187179565, -0.4933713376522064, -0.021411368623375893, 0.20489871501922607, 0.5278820991516113, -0.05066649615764618, 0.2149815857410431, -0.5082175731658936, -0.8633909225463867, 1.1045982837677002, 0.40414854884147644, 0.7050737142562866, 1.6793781518936157, -0.7543193101882935, -0.6481146216392517, -0.41274493932724, 0.8714406490325928, -1.4786388874053955, -0.7113156914710999, 0.7960575222969055, -1.6210150718688965, 0.6995994448661804, 1.614615559577942, 0.41320276260375977, -0.8708067536354065, -0.46646422147750854, -1.0207054615020752], "dtype": "float32"}, {"__nd__": [-0.08438274264335632, 0.959157407283783, 1.0297125577926636, -2.2978312969207764, -0.7942619919776917, 0.33191195130348206, -0.3615522086620331, -2.505873441696167, 0.21162240207195282, 0.5489720702171326, -0.006984233390539885, -0.0752776637673378, 0.9512256979942322, -1.4551748037338257, -0.30825385451316833, 0.40839579701423645, 3.484982967376709, -0.7484485507011414, 0.7993671298027039, 1.7676334381103516, 0.7851652503013611, 0.888940691947937, 1.2867451906204224, -1.8161876201629639, 0.5249518752098083, -0.4530232548713684, 1.5785975456237793, 0.44907161593437195, 0.2757357358932495, 0.2008703500032425, -1.2742719650268555, 0.956822395324707, -1.6443063020706177, -0.19053404033184052, -0.3506781756877899, 1.0255675315856934, 0.023885836824774742, -0.8568097949028015, 0.7107119560241699, 1.7924931049346924, -1.0334968566894531, 1.4402668476104736, -0.6653204560279846, -0.21275721490383148, -0.18522782623767853, -1.0497725009918213, 1.6853686571121216, 0.8281928300857544, 0.6482651829719543, -1.375177025794983, 0.7059811949729919, -0.8937869668006897, 1.4381047487258911, -1.7980921268463135, 0.7693324685096741, -1.5853897333145142, -1.4961650371551514, 0.2483585625886917, 0.904735803604126, 1.1727416515350342, -0.3899655044078827, -0.5443766117095947, 0.9735826849937439, -1.1724793910980225, -1.1373041868209839, 0.5658564567565918, 0.4937354624271393, 0.11106667667627335, 1.3079437017440796, -1.6698001623153687, 0.5107475519180298, 2.2923803329467773, -0.3990460932254791, -1.2782952785491943, 1.314501404762268, 0.6080613136291504, 1.291145920753479, 0.7674800753593445, -0.42366504669189453, -0.17702126502990723, -1.1738595962524414, 2.7459514141082764, -0.9123489856719971, -0.3111708462238312, -0.6545354127883911, 0.5069372057914734, 1.5194206237792969, -0.473714679479599, 0.5140010714530945, 0.811817467212677, 2.3173434734344482, -0.17857663333415985, 0.3304615914821625, 1.7544214725494385, -1.4312540292739868, 0.041342224925756454, 0.14261819422245026, -1.0999292135238647, -0.15937365591526031, 0.27342361211776733, 1.2083377838134766, 0.9987149238586426, -0.5109549760818481, -0.9129632115364075, 1.0336456298828125, -1.5120999813079834, -0.5107467174530029, -1.1190903186798096, -1.5848567485809326, 0.3069054186344147, -1.4240621328353882, -0.6109195947647095, 1.3113574981689453, 1.6284644603729248, -1.3577311038970947, -0.9250946044921875, -0.8695799708366394, -1.3493984937667847, 1.3723739385604858, 0.9272035360336304, 0.5213491916656494, -0.9856082797050476, 0.7895569205284119, 0.21093609929084778, -0.015414925292134285, 0.6058946251869202, 1.4050202369689941, -0.052581723779439926, -0.4056719243526459, -1.035693883895874, 1.4494526386260986, 0.5110726356506348, -2.4512593746185303, 0.15277695655822754, 0.8364701271057129, -1.6097352504730225, -1.5070089101791382, 1.0727275609970093, -0.28616511821746826, -1.4939179420471191, 0.03732287511229515, 0.36660081148147583, -0.9711063504219055, -0.8147364854812622, 0.035007212311029434, -0.06772412359714508, 1.8021186590194702, 1.3489638566970825, 0.5208491683006287, -0.6823938488960266, -0.6271253824234009, -2.281367063522339, -0.08094191551208496, -0.41998371481895447, -0.6837112903594971, 0.7327607870101929, 1.270538568496704, -1.8895344734191895, 0.4057096540927887, 0.44624581933021545, -0.6414676904678345, -0.7393257021903992, 1.7395951747894287, 0.7403997778892517, 2.1564114093780518, -0.7910337448120117, -1.085361361503601, -0.28107118606567383, 0.8740528225898743, 1.3317290544509888, 1.98874032497406, -0.6612593531608582, -1.4727414846420288, 1.005771517753601, 0.9144768714904785, 0.5816971659660339, 1.4728405475616455, -0.8237137198448181, -1.655031442642212, 2.093451499938965, 0.31126147508621216, -0.3052235245704651, 0.8207834959030151, 2.2696313858032227, 0.1928243339061737, 1.7787336111068726, 0.10148503631353378, 1.968629002571106, 0.20561060309410095, 1.2392641305923462, -0.005673936568200588, 0.6271430850028992, 0.24211660027503967, -0.4159497916698456, 0.12067936360836029, 0.7307035326957703, -0.10527107119560242, -0.5388365983963013, 0.4382268190383911, 1.4897342920303345, -0.07796918600797653, -1.0161038637161255, -0.8898144960403442, 0.0325234979391098, 1.6848777532577515, -1.0555013418197632, 1.1689560413360596, -0.1913403868675232, -0.6864349246025085, -0.5414464473724365, 1.1800533533096313, -0.47580984234809875, 1.3906669616699219, 2.076165199279785, -0.8839995265007019, -1.3202062845230103, -1.795446753501892, 0.5422350168228149, -0.9140052199363708, -0.6517276167869568, -0.8150538206100464, 1.3709487915039062, 0.6678017973899841, 0.6048237681388855, -1.7871655225753784, -0.8825697302818298, -0.07611773163080215, -0.14268988370895386, 0.2945314347743988, 1.616516351699829, 0.6723906993865967, 0.7925763726234436, -0.15966854989528656, -0.4460895359516144, -0.5132283568382263, -0.13712790608406067, -1.7090779542922974, -0.62137371301651, -0.009337487630546093, -0.5262873768806458, -0.08273644745349884, -1.9148746728897095, -0.6019083857536316, -0.027512559667229652, 1.1423003673553467, 0.0812908187508583, -0.2684426009654999, -0.6280617713928223, -0.0009032441885210574, -0.5644707679748535, 0.6985507011413574, -0.5965278148651123, 0.6282672882080078, -0.7070732116699219, -1.5611156225204468, -1.5259274244308472, 0.8745753765106201, -0.6417968273162842, -0.5534705519676208, 2.5317113399505615, -1.2485449314117432, -0.09974800795316696, -1.4484683275222778, -0.6997363567352295, 0.4681895077228546, -0.37385329604148865, 1.86037278175354, -1.0216000080108643, -0.7720773220062256, -1.4923840761184692, -0.04905305802822113, -0.8543766736984253, -0.21356256306171417, -0.8947174549102783, 0.3927055597305298, -0.7271826863288879, 0.024538643658161163, -0.18843436241149902, -0.8201452493667603, -0.05400001257658005, -0.06360217928886414, 0.47011956572532654, -2.0591416358947754, 1.4529294967651367, 0.0776621401309967, 0.3861325979232788, 0.4918534755706787, -1.430822491645813, 0.7024717926979065, 0.257769912481308, -0.535028338432312, 0.7913720607757568, -0.29269543290138245, -1.052369475364685, 0.8573254346847534, 0.7611470818519592, 0.30841386318206787, -0.15712833404541016, 1.5901455879211426, -0.6004446148872375, 0.2072206288576126, -0.19361773133277893, -0.20576132833957672, 0.47145751118659973, -0.9937095046043396, 0.5011529922485352, 0.2716117203235626, -0.34256982803344727, -0.7300257086753845, -1.3856085538864136, 0.08400580286979675, 1.7011717557907104, -0.6354141235351562, -0.9344778656959534, -0.6780100464820862, 0.2671155333518982, 0.5004255175590515, 0.3539360463619232, -0.0516141876578331, -1.2299989461898804, 1.0293141603469849, -0.3343892991542816, 0.3955498933792114, 0.09100126475095749, 0.008612123318016529, 0.8169299364089966, 0.15213264524936676, 2.099600076675415, 1.4850690364837646, 0.9261180758476257, -0.24994516372680664, 0.8367198705673218, 0.3751840889453888, 0.7503980398178101, 0.11267036199569702, -0.986906886100769, -0.928646981716156, -0.0345439612865448, 0.897329568862915, -0.03561825305223465, -0.5636928081512451, -0.1442623734474182, 0.7014939785003662, -0.44616034626960754, -0.4867613911628723, 0.14104576408863068, 0.6908890604972839, -1.1773767471313477, -0.3603038787841797, 0.01724523864686489, -0.40638357400894165, 0.5402539372444153, 0.8768092393875122, -0.7163369655609131, 2.4293460845947266, 0.4194319546222687, 1.6129344701766968, 0.7852514386177063, -1.7921645641326904, -1.2335612773895264, 0.2935985028743744, 0.5514148473739624, 0.6386204361915588, -1.9395798444747925, -0.6098182201385498, 0.5035402774810791, 1.2945208549499512, 0.5498773455619812, -1.3581515550613403, -0.5292479395866394, -0.3099837303161621, -0.40480515360832214, 1.0550148487091064, 1.1893855333328247, 1.3035436868667603, -1.0808250904083252, -1.243548035621643, -1.4963301420211792, -0.24152463674545288, 0.10842785984277725, -1.3638567924499512, 0.27388855814933777, 0.25395455956459045, -0.19615726172924042, 0.46118855476379395, -0.983917236328125, -1.0650261640548706, 1.4651631116867065, -0.3668307662010193, 0.9832844734191895, -0.6488187909126282, -2.2623093128204346, 0.5431658029556274, -1.2313342094421387, -0.8678023815155029, 0.8795678615570068, -1.5248461961746216, -0.2655368149280548, 1.1448193788528442, 2.0264527797698975, -0.9775325059890747, 0.16719138622283936, -0.6573379039764404, 1.2321842908859253, -0.4846906363964081, 1.3390021324157715, -1.2423696517944336, 0.7883921265602112, 1.492455244064331, 0.6716196537017822, 0.022903714329004288, -0.4902096688747406, -1.318139672279358, 0.9580745697021484, 0.9423396587371826, 0.3498082458972931, -1.2637698650360107, -0.9038633704185486, -0.14465540647506714, 1.7052226066589355, 1.8681730031967163, 0.966923713684082, -1.4307969808578491, -2.078376293182373, 0.004857118707150221, 0.42842021584510803, 1.6243282556533813, -0.9289014339447021, 1.0762677192687988, -0.3847845494747162, 0.5200170874595642, -0.6264866590499878, -0.5661135315895081, -0.6753919720649719, -0.2821667790412903, -1.060333251953125, -0.9239559173583984, 0.18793578445911407, -1.3412362337112427, -0.4589356780052185, -0.6813256144523621, -0.3407086730003357, 1.1262061595916748, -0.8115136623382568, 1.5833477973937988, -1.9342855215072632, -0.9796671271324158, 0.6542446613311768, -2.5051684379577637, -0.2970242202281952, 0.3932947814464569, -1.4854943752288818, 1.4498211145401, 1.8380764722824097, 0.04116823151707649, 0.001527488580904901, 0.8024422526359558, 0.9817739129066467, -0.4581528604030609, -0.4150106906890869, 2.062080144882202, 0.24526450037956238, -0.16342918574810028, -0.8727446794509888, -2.970841884613037, -0.15362326800823212, 0.0581609345972538, -0.07512512058019638, -0.19801518321037292, 0.819071888923645, 0.2874387800693512, 0.19404400885105133, -0.19190378487110138, -0.3508937954902649, 0.11012996733188629, -0.7064120769500732, 1.1520624160766602, 0.2762795686721802, 0.004392136353999376, 2.054560899734497, 0.8199785351753235, -1.1596810817718506, -0.8625797033309937, -0.08207641541957855, -0.496858149766922, 1.3287544250488281, 0.7739410400390625, 0.9091895222663879, 1.0929731130599976, 0.9307626485824585, -1.8086295127868652, -1.3874152898788452, -1.013373613357544, 0.5302985906600952, 0.23235274851322174, 1.8649542331695557, 1.3090331554412842, 1.3746647834777832, -0.6593653559684753, -0.060819126665592194, 1.0122674703598022, -1.099778175354004, -0.6723588705062866, 1.8388270139694214, 2.161330223083496, -1.7733820676803589, -0.058083318173885345, -1.2222166061401367, -2.025723934173584, -0.8623833060264587, 0.5357939600944519, 0.634160041809082, 0.2488320767879486, 1.2963510751724243, -0.871598482131958, -0.29229363799095154, 1.4939208030700684, 0.7359438538551331, -0.8364891409873962, -1.281491756439209, 0.007843316532671452, 0.1593749076128006, 0.2853654623031616, -0.203835129737854, 0.0721101313829422, -0.6778606176376343, -0.10417953133583069, -0.7051849365234375, -1.3279759883880615, 0.07245223224163055, -0.8629637360572815, -0.8585455417633057, 0.5264712572097778, 0.30012479424476624, 1.863537073135376, 1.687742829322815, -0.41283276677131653, -0.32375770807266235, -0.7124590873718262, -1.434324860572815, 1.5358283519744873, 0.6421278715133667, -0.6146174669265747, 1.7755722999572754, 0.6587690711021423, 0.7480593323707581, -0.48836180567741394, -2.456220865249634, 0.6314692497253418, -0.7956108450889587, -1.5424354076385498, 0.5299584865570068, 0.5627626776695251, -0.994897186756134, -0.07554437965154648, 0.46044498682022095, -0.003632782958447933, -0.6194184422492981, 1.4885627031326294, -0.32573986053466797, 1.1530436277389526, -0.10686280578374863, -0.3367473781108856, 1.268477439880371, -0.6757029891014099, 0.34115439653396606, 0.5111801028251648, 1.1353431940078735, -0.07921165227890015, -1.6005401611328125, -1.7652040719985962, -1.8395315408706665, -1.203870177268982, -0.1703934371471405, -0.36493247747421265, -0.5494616031646729, -0.15997883677482605, -0.8053856492042542, -0.1064523383975029, 0.0476749949157238, 2.1362619400024414, -0.31386375427246094, -0.29333534836769104, -0.35465484857559204, -0.8874759078025818, 0.7780729532241821, -1.3689734935760498, 2.021597385406494, -1.6687272787094116, -0.1122661754488945, 0.9805395603179932, 0.5063618421554565, 0.7279291152954102, -1.5801314115524292, 0.870521068572998, 0.5589762330055237, 1.57964026927948, -0.036256641149520874, 0.07964684069156647, -1.6556074619293213, 0.5477957129478455, -1.0990852117538452, 0.3790096938610077, 1.8164076805114746, -0.9205844402313232, 0.24249279499053955, -0.5448453426361084, -0.6105738878250122, -1.956322193145752, -0.43978506326675415, 0.3271610140800476, 1.2972785234451294, 1.5898505449295044, -0.157427579164505, 0.9445458054542542, -0.10521458089351654, -0.9305113554000854, -0.48745545744895935, 0.8745076656341553, 0.6967790126800537, -1.7624396085739136, 0.5308611989021301, -0.34231770038604736, 0.624088704586029, 0.1588325798511505, -1.3003919124603271, -1.4020651578903198, -0.0969766229391098, -0.0043665640987455845, 0.1124027669429779, -1.2104681730270386, 1.4089258909225464, -2.4366631507873535, 1.3403542041778564, -0.20815172791481018, -0.5583915114402771, 1.7114160060882568, -0.138078972697258, 0.03952696919441223, -0.5565257668495178, 0.5313630700111389, 0.5481865406036377, 1.509291648864746, 1.1863081455230713, 0.005335490684956312, -0.6454775333404541, -0.4307372570037842, -0.7756108045578003, -0.4545840322971344, -0.5955211520195007, 0.3209695518016815, 0.12351097911596298, -0.7531969547271729, 0.26172536611557007, -2.2747037410736084, -0.12018220126628876, -2.385711193084717, -0.42523518204689026, 0.6081069111824036, 0.15935029089450836, -1.175630807876587, -0.2523946464061737, -0.20421984791755676, -0.6550514101982117, 0.07541905343532562, -0.7404826283454895, -0.9301268458366394, -1.0746978521347046, 0.38072875142097473, -1.024503469467163, -1.606255054473877, 0.7332476377487183, -0.8491495251655579, -1.5625104904174805, 0.5404251217842102, -0.7306989431381226, -0.6906322836875916, -0.9618486762046814, 0.5483133792877197, 0.5818698406219482, 0.9001405835151672, -0.8049007654190063, 2.2998623847961426, -0.7311121821403503, -1.2603882551193237, -0.4465976059436798, -1.9267491102218628, -0.907494306564331, -0.5362973213195801, 0.0725669115781784, 1.263525128364563, 0.38056012988090515, -0.3389480412006378, 0.0005729454569518566, -1.2482820749282837, 1.4355229139328003, 0.17702455818653107, 0.43181076645851135, -0.5804747939109802, -0.8798193335533142, -1.04619300365448, -0.9165594577789307, -0.6289050579071045, 0.7991191148757935, -0.7612695097923279, 0.552646279335022, 0.8052698373794556, 0.33990100026130676, 0.8291682004928589, -2.3447911739349365, 0.24081948399543762, -0.48186415433883667, 1.3376587629318237, 0.3419002294540405, -0.34540095925331116, 0.03017556667327881, 0.20906151831150055, 1.7649122476577759, -0.40067344903945923, 0.2894350588321686, -0.5523137450218201, 0.27944448590278625, 0.374349445104599, 0.2029723972082138, -0.1427917182445526, 1.556554913520813, 1.111990213394165, 0.5221898555755615, -0.5551561713218689, 0.4853259325027466, 0.18093492090702057, 0.9797019362449646, 0.6132094860076904, -0.6021135449409485, 0.6444925665855408, 1.9227378368377686, 0.6200080513954163, -1.0319639444351196, -0.36755526065826416, 1.046463966369629, 1.3343009948730469, -1.4333224296569824, 1.1671968698501587, 1.796816110610962, 1.5901994705200195, -1.1967910528182983, 0.1444716453552246, 1.4824817180633545, 0.9070338606834412, -0.23950013518333435, -0.8296528458595276, 0.07779628038406372, 0.6743477582931519, -0.8726616501808167, -0.034600064158439636, 1.4062508344650269, 1.4596526622772217, 0.32087773084640503, 1.5056960582733154, -0.011965411715209484, -0.5122495889663696, -0.5484695434570312, -0.9611060619354248, -0.3605343699455261, -0.24126167595386505, 2.239704132080078, 1.1068835258483887, 2.3788564205169678, -0.9086253046989441, 0.8155571818351746, -0.3627539873123169, -1.3489985466003418, 0.6888210773468018, -0.5805227756500244, 0.8552298545837402, 2.7027385234832764, 0.18823453783988953, -0.02364247664809227, -0.612963080406189, -1.4444798231124878, -0.16859735548496246, -0.7645784020423889, 0.23855897784233093, -0.2785816192626953, 0.3859015703201294, 0.5652822852134705, 1.2659257650375366, -1.0718467235565186, -0.7410696148872375, 1.5061473846435547, 0.9699507355690002, -0.6547232866287231, 2.6063551902770996, -2.783040761947632, -0.3566131591796875, -0.28216007351875305, 0.5264424681663513, -1.3176950216293335, 0.033745937049388885, 0.24637563526630402, 1.2035579681396484, -1.8967968225479126, -0.6682316660881042, -0.6169450879096985, 1.0043237209320068, 1.9569562673568726, 1.723488211631775, -0.7153761386871338, 0.21994532644748688, -0.1809999644756317, -0.04221063107252121, -0.7256357073783875, 0.17679466307163239, -0.10367380082607269, 0.5659136772155762, -0.17512647807598114, 2.5598249435424805, -1.6761751174926758, -0.60069340467453, -0.5788553357124329, 0.1753847450017929, 0.9307531118392944, -0.05086464434862137, 3.0514626502990723, 0.386836975812912, 1.479782223701477, -0.04443634673953056, -0.7152699828147888, 2.5251317024230957, -0.8221867680549622, -1.494250774383545, -0.7003505229949951, 0.22068218886852264, 1.4411213397979736, -0.24732591211795807, -0.23239867389202118, 0.3019033968448639, 0.8261533975601196, -0.8007031679153442, 0.3311728537082672, -0.5354435443878174, 0.4687831699848175, 0.5422608852386475, 1.0670572519302368, 1.8672266006469727, -0.4736842215061188, -0.8562108874320984, 0.2054133117198944, 0.1193690150976181, 0.11804255843162537, 0.9109196662902832, 0.24267807602882385, 0.9694296717643738, 0.15575185418128967, -0.007920640520751476, -0.7401852011680603, 0.7849828004837036, 0.14992330968379974, 1.3251848220825195, 0.7372468709945679, -0.4273032248020172, 0.32448825240135193, -0.006565895397216082, -0.05090158060193062, -1.2859046459197998, 0.2897597849369049, 0.15993936359882355, -1.1311924457550049, 0.36434102058410645, 0.4347505271434784, -1.0589011907577515, -1.0534732341766357, 0.31622231006622314, 0.8094483017921448, 0.3483237624168396, -1.7141166925430298, -1.0368491411209106, 0.57792067527771, -0.9642291069030762, 1.0855422019958496, 1.1430068016052246, -0.7843714952468872, 0.27800440788269043, 0.7343724966049194, 0.504582405090332, -0.41711536049842834, -0.16924192011356354, -0.053139299154281616, 1.6075047254562378, 0.6784626245498657, 1.1805531978607178, 0.27834102511405945, 0.5731985569000244, 0.8924188017845154, -0.13760881125926971, -0.6938839554786682, 0.4223777949810028, -0.3704933226108551, -0.5144904255867004, -0.7294053435325623, -1.3442238569259644, 0.15042799711227417, -0.9051602482795715, -0.7513537406921387, -0.2879590690135956, -1.1843923330307007, 0.5357567667961121, 0.8793273568153381, -1.0881133079528809, -0.3431694805622101, -0.2099655270576477, -0.22903338074684143, 0.0907745510339737, 0.3542237877845764, -0.616798996925354, 0.7311879396438599, -1.8786473274230957, -1.0036325454711914, 0.923576831817627, -0.40360236167907715, 0.3532513976097107, 1.1688780784606934, 1.695472002029419, -1.0681794881820679, -0.7261345386505127, -0.71238112449646, -0.6879856586456299, 0.28544750809669495, -1.8882838487625122, 0.42292073369026184, -0.03800452873110771, 1.451102375984192, -1.5014880895614624, 0.6297032237052917, -0.38264673948287964, -0.6905807852745056, 0.846336841583252, 0.8568008542060852, -1.3381500244140625, 0.8143829703330994, -0.26906266808509827, -0.38254353404045105, -0.47693657875061035, -0.08145956695079803, -0.5915555357933044, 0.09256411343812943, -2.393141746520996, 0.895389974117279, 0.27658611536026, -0.5598190426826477, 0.36511287093162537, -0.2173769474029541, 0.5730311870574951, 1.024169683456421, 0.44652968645095825, -2.9936306476593018, 1.2776663303375244, 0.05469633266329765, 0.6509475111961365, -0.7911726236343384, -0.17500662803649902, -0.24041436612606049, 0.5309014320373535, 0.6903164386749268, 0.3312128484249115, 0.23278848826885223, 0.837731122970581, 1.071086049079895, 0.6402269005775452, -0.3575899302959442, 0.7913153767585754, -0.6153507828712463, 0.21778684854507446, 1.4048219919204712, -1.460753083229065, -1.1530992984771729, -2.2297072410583496, 1.797162413597107, 0.09333734959363937, 0.3884681463241577, 0.18319128453731537, -0.20741598308086395, -0.9344338774681091, 0.1363980770111084, -1.0165081024169922, 0.528692364692688, 1.4516034126281738, 2.1341211795806885, -0.972778856754303, -0.4837501347064972, -0.9734019637107849, -1.7613117694854736, -1.2656809091567993, -0.25929611921310425, 0.8466737866401672, 1.0796568393707275, -0.9903574585914612, 0.5308247208595276, -0.3139764368534088, 0.12163402885198593, 0.9010128378868103, 0.8625656366348267, 0.8286707997322083, 0.2179015725851059, 0.19095823168754578, 0.18192094564437866, 0.5139487385749817, -0.15091075003147125, 0.37772026658058167, 0.07799471169710159, 1.2764687538146973, -0.996125340461731, -0.4394712448120117, 0.029825305566191673, -0.13735581934452057, -2.231111526489258, 0.9947652220726013, 1.3877649307250977, -1.6870874166488647, -1.3796932697296143, 1.2231448888778687, -1.5228241682052612, 0.050099436193704605, -0.15340307354927063, -0.46309730410575867, 0.28908389806747437, -1.1158970594406128, -1.6366666555404663, -0.6461630463600159, -0.2500836253166199, 0.13131417334079742, 0.7220139503479004, 1.3789159059524536, 0.4393039345741272, -0.130629301071167, 0.39082831144332886, -1.245161771774292, -0.7769599556922913, -0.8237189650535583, -1.5997017621994019, 0.6764191389083862, -1.1050488948822021, -1.4713339805603027, -1.0606129169464111, 1.3554744720458984, 1.136262059211731, 0.16314665973186493, 0.5505777597427368, -1.8864744901657104, -0.9827779531478882, 0.1874512881040573, -1.125654697418213, -1.4943907260894775, -0.833364725112915, 0.6408116817474365, 1.3223586082458496, -1.1194660663604736, 1.211863398551941, -1.3444457054138184, 0.001998670632019639, -0.1603420227766037, -0.4746394157409668, 1.5477951765060425, 0.6671985387802124, -0.05667434632778168, 0.926959216594696, -0.7595123648643494, 0.3129051625728607, 1.3105800151824951, 0.3165878653526306, 1.8120276927947998, -1.134909749031067, -1.2792260646820068, -0.35198500752449036, -0.28561335802078247, -0.5818275809288025, 1.3178093433380127, 0.27901211380958557, -1.0331438779830933, -0.7341096997261047, 0.3557940125465393, -1.2739505767822266, 0.2947116196155548, -1.074306845664978, -0.1258293241262436, 0.16599637269973755, -1.5090265274047852, -0.5271863341331482, -0.46874502301216125, 0.03341799974441528, 0.6419749855995178, -0.1618071049451828, -1.422439694404602, -0.4933808445930481, -0.9687903523445129, -0.293128103017807, 0.860010027885437, 0.7915260791778564, 1.0400466918945312, -0.8873574137687683, -1.7351921796798706, -1.0134779214859009, -0.1056780219078064, -0.6658706068992615, -1.2508617639541626, -0.15099401772022247, 1.1157608032226562, -0.3824310898780823, 0.31377530097961426, 1.944779634475708, -0.618161141872406, 0.7211335301399231, -1.4932713508605957, -0.8855515122413635, -0.22533096373081207, 1.0109710693359375, 0.08940444886684418, -0.5234969854354858, 0.585303008556366, 1.4424819946289062, 1.4031636714935303, -0.8727417588233948, 0.5864878296852112, -0.5784208178520203, 0.6661890745162964, 1.0468974113464355, -1.121594786643982, 2.0807158946990967, -1.892486333847046, 0.13638898730278015, -1.7481904029846191, -0.5296163558959961, 1.5014469623565674, 1.263790488243103, 1.8532295227050781, -0.3368939757347107, -0.977336049079895, -0.9764273166656494, 2.8578908443450928, 0.9787566661834717, -0.9597652554512024, -0.002445325953885913, -0.10014744848012924, 0.9254592657089233, -0.608773410320282, -1.7538660764694214, -0.7795118093490601, 0.5242006778717041, 0.0556543804705143, -0.5817098617553711, -1.2829058170318604, 1.1758133172988892, 0.5782124996185303, -0.8773086071014404, -0.57065749168396, 1.7912510633468628, -0.3072395920753479, 1.3055815696716309, -0.16871339082717896, -0.1687186360359192, -0.47579291462898254, 0.579369843006134, 0.4698966443538666, -0.8763327598571777, 1.7015738487243652, -0.4070344567298889, -0.5473876595497131, 1.1575478315353394, -0.7492503523826599, -0.1965184360742569, 0.2582390308380127, 0.9046216607093811, -1.5987184047698975, -0.9472683668136597, -2.388170003890991, 0.5245774388313293, 0.2520569860935211, 0.5487064123153687, 0.27119746804237366, -0.6634455323219299, 0.551437258720398, -1.0850911140441895, -0.9641745686531067, -0.37154075503349304, 0.953434407711029, -1.3323073387145996, 2.9071974754333496, -0.7738221287727356, -1.5998307466506958, 0.09712269902229309, -0.9105037450790405, 0.6006659269332886, 0.7966787815093994, -1.0628905296325684, 0.4351896047592163, -0.6157110929489136, 0.1966170370578766, -0.5573701858520508, -0.2005770206451416, 0.257394939661026, 1.3534663915634155, -0.4807671010494232, 0.6927824020385742, 1.4696139097213745, -0.8647260069847107, 1.015681266784668, -0.3728271722793579, 0.27738022804260254, 0.398347944021225, -0.321277916431427, 1.4082576036453247, 1.193102478981018, 0.08081332594156265, -0.09383457154035568, 0.009509816765785217, 0.4990459382534027, 0.487729012966156, -1.4177701473236084, 0.6194242835044861, 0.43601903319358826, -0.40247154235839844, -0.8120660185813904, -1.7247025966644287, 0.263720840215683, -0.7193929553031921, -0.48239657282829285, 2.485227584838867, -1.5059465169906616, -1.993618130683899, 0.6613315343856812, -1.8435086011886597, -0.6372010111808777, -0.07644107937812805, -1.0435491800308228, 1.0508556365966797, -0.08618751168251038, 0.033768948167562485, -0.8145765066146851, -1.1498594284057617, -0.3591119647026062, 1.7379331588745117, -0.20468363165855408, 0.42058342695236206, 0.33417245745658875, -1.0194553136825562, 0.6219314932823181, -0.6501817107200623, -1.860425591468811, -1.9641778469085693, 0.5097112059593201, 1.7626289129257202, 0.5251356363296509, 0.9716977477073669, -1.256974458694458, -3.0134518146514893, 1.6378430128097534, 0.846518337726593, -1.7222651243209839, 0.12597869336605072, -0.1429564505815506, -0.24359117448329926, -1.1675422191619873, 0.2899375557899475, 0.5900917053222656, -1.456485629081726, 0.26476117968559265, -1.575052261352539, 0.5665566921234131, -0.3204837441444397, 0.8017621040344238, 0.3618961274623871, -1.7177035808563232, -1.1553285121917725, -0.024943700060248375, 0.478960245847702, 1.1065313816070557, -1.274240255355835, -0.23532648384571075, 2.286496877670288, 0.01263608317822218, -1.0130029916763306, -1.8532190322875977, 0.3480275869369507, 0.8260510563850403, -0.8480691909790039, 0.76231449842453, 1.0353409051895142, 1.3546031713485718, 0.6166759133338928, -0.32162824273109436, -0.7858481407165527, 0.8569878935813904, -1.4115411043167114, 1.4692384004592896, 0.5786890983581543, -0.3329566717147827, -1.8767188787460327, -0.9139052629470825, -1.56350839138031, -1.1333575248718262, 0.4065071642398834, 2.167449951171875, -1.1179108619689941, -0.895626962184906, 2.00111722946167, 0.016373705118894577, 0.27626147866249084, -0.7713962197303772, 1.5044795274734497, 0.018451938405632973, 0.6934525966644287, 0.7637917995452881, 1.4862737655639648, 0.34687289595603943, -0.6495770215988159, 0.4899497628211975, -0.13491934537887573, -1.8117743730545044, -0.671086311340332, -1.5895981788635254, -0.23199324309825897, -0.5478343963623047, -0.4306943714618683, -1.7244911193847656, 0.46312281489372253, 0.09958241879940033, 0.9824413657188416, -1.2784708738327026, -0.5837920904159546, -0.7525588274002075, 0.36435645818710327, -0.7865164875984192, 0.4508539140224457, -1.1941001415252686, 0.3486131429672241, 0.055900901556015015, -0.14091756939888, -0.27329984307289124, -0.1939658522605896, 0.8564445376396179, 0.9265218377113342, 0.8083868622779846, 0.04907398298382759, -0.603178083896637, -0.695859432220459, -0.3434448540210724, -0.1647671014070511, -0.6696131825447083, -1.2888689041137695, -1.1374318599700928, 0.6525192856788635, -0.7853220105171204, -0.1470777541399002, -0.531914472579956, 0.17941264808177948, -0.3647334575653076, 1.0077340602874756, 0.08253588527441025, -1.2336097955703735, -0.10815674811601639, -1.2079309225082397, -0.49813637137413025, 0.7663899064064026, -1.4956152439117432, 0.42025691270828247, -0.19599807262420654, 0.8943592309951782, -0.11099473387002945, 0.8049574494361877, -0.012487414292991161, 1.0451594591140747, 0.1163821667432785, -0.8547670841217041, 0.05695726349949837, -0.08352714776992798, 1.6156673431396484, -0.8951653838157654, -1.632948875427246, 1.1244776248931885, -0.7414587140083313, -1.7852996587753296, -1.0354658365249634, -0.8701393604278564, -0.5306159257888794, 2.0462472438812256, -0.3605806529521942, 0.4910145699977875, 0.3499988615512848, -0.6991572380065918, -1.8533275127410889, 1.7127516269683838, -0.6128252148628235, -0.9560985565185547, 0.4216102361679077, -0.9842842221260071, -0.6726956963539124, 0.8197473287582397, -0.604439377784729, 0.06600470840930939, -0.5004515051841736, 1.3474863767623901, -1.7117820978164673, -2.8801686763763428, -0.034656353294849396, -0.4565531313419342, 2.668346643447876, -0.03908208757638931, -0.022007513791322708, 0.6454780697822571, 0.16915041208267212, 0.4890744388103485, -1.01749849319458, -0.4294147491455078, 1.3238276243209839, -1.2961876392364502, -0.493846595287323, 0.2551254332065582, 0.6259955167770386, -0.0618908554315567, 0.5305620431900024, 0.7036232948303223, 0.010001414455473423, -0.8231812715530396, 0.3706135153770447, -0.6590616703033447, -0.6379029750823975, 1.146597146987915, 0.3806455731391907, -1.1855577230453491, -1.827617883682251, -0.8923346996307373, 0.15526293218135834, 0.497585266828537, -0.12240032106637955, 1.8967208862304688, -0.20936331152915955, -1.081843614578247, -0.1967717707157135, 0.824948787689209, 0.34316277503967285, 0.24716463685035706, 1.208105444908142, 0.9531487822532654, 1.461211919784546, -0.4901021420955658, 0.3357972502708435, 0.20189616084098816, 0.10811498016119003, 1.8285613059997559, 0.4842786192893982, 0.48804420232772827, -1.1399955749511719, -0.267088383436203, 0.5450031161308289, 0.1720922291278839, -0.9244106411933899, 0.6542150974273682, 0.004272859543561935, 1.0874037742614746, 0.7080245018005371, 1.0472064018249512, -0.789326548576355, -0.5659956932067871, 0.9636215567588806, 2.9888863563537598, 0.818343460559845, 0.719423770904541, -0.21605156362056732, 2.197730541229248, 0.4869183599948883, 0.16939251124858856, 0.5893929600715637, -0.2858429551124573, 0.8573213815689087, 0.34623444080352783, 0.033826254308223724, -1.205207347869873, -0.9197461605072021, -0.1883896440267563, -2.0437490940093994, -0.7241643071174622, 0.1034209355711937, 0.07579824328422546, 2.230302095413208, -1.894201397895813, -0.1264057755470276, -0.3251960575580597, 0.35198840498924255, -0.21489833295345306, -0.7115517854690552, 0.23306481540203094, -0.23182746767997742, 2.1520140171051025, -0.29979419708251953, 0.5032884478569031, 0.9575659036636353, 0.4722955524921417, -0.12298090010881424, -0.02412586845457554, -0.14743486046791077, -0.8024295568466187, 0.12421351671218872, 1.6595667600631714, 1.4928590059280396, -0.7984800338745117, 0.8912374973297119, 0.04383694380521774, -0.08573231846094131, -0.20368345081806183, 0.9018455743789673, -0.83552485704422, -0.8132481575012207, 1.2641416788101196, -1.6776349544525146, -0.45806002616882324, 0.48246878385543823, -0.011263099499046803, -0.23545220494270325, 1.8936632871627808, 1.365177035331726, 0.4123299717903137, 1.069063663482666, 0.020316822454333305, -0.2456233650445938, 1.274470329284668, 1.4366943836212158, -1.072190523147583, 1.603885293006897, 2.1403324604034424, 1.0511467456817627, -1.0100356340408325, 0.8446277976036072, -0.5013301372528076, 1.3299152851104736, 0.10630775988101959, 0.030717087909579277, -0.7454341053962708, 0.042867813259363174, 0.27471283078193665, -1.2494934797286987, -0.48938626050949097, -0.8807559609413147, 1.123692512512207, 0.2566780745983124, -0.39888688921928406, 1.1264088153839111, 0.19734084606170654, -0.14085671305656433, -0.3168952465057373, 1.232397198677063, 0.19003704190254211, -1.6373344659805298, -0.542034924030304, -0.20272256433963776, -2.2339274883270264, 1.0014278888702393, 1.3300750255584717, -0.8331894874572754, 1.1078757047653198, 0.5655213594436646, 0.29767850041389465, -0.795070230960846, 1.8893755674362183, 0.41634389758110046, -0.6495128273963928, 0.8613080978393555, 0.18685001134872437, -0.788052499294281, 0.783926784992218, 0.43908876180648804, -1.8530898094177246, 0.02702290564775467, 0.4691193699836731, 0.9492044448852539, 1.1903976202011108, 0.20981109142303467, 0.3499864935874939, -1.0445362329483032, -0.059370256960392, -0.6388620734214783, 0.8875414729118347, 1.2528129816055298, -0.5005710124969482, -2.104534387588501, 0.37312859296798706, 1.070427417755127, -0.19651274383068085, -1.35186767578125, 0.7117499113082886, -0.7700091600418091, 0.12448181211948395, -0.1370229721069336, 0.7707749009132385, -1.297911286354065, 1.2619658708572388, 0.3634452819824219, 2.1817891597747803, -0.9703449010848999, -2.1004245281219482, -0.07635363936424255, -1.3810765743255615, 0.05343792587518692, -1.9365161657333374, 0.8332490921020508, 0.10805122554302216, 0.7154265642166138, -0.08888257294893265, -1.323721170425415, 1.0867055654525757, 1.2784245014190674, 1.5493563413619995, -0.46915605664253235, 0.18947383761405945, 0.6076681613922119, -0.5045809149742126, 0.07174279540777206, 0.16100773215293884, 0.9479650259017944, -1.7915608882904053, 1.5388338565826416, 0.11617796868085861, 0.15888290107250214, -0.4711078405380249, 1.1245050430297852, 0.7542927861213684, 0.17949381470680237, 1.0654432773590088, -0.4785926043987274, 0.9013564586639404, -0.7541329264640808, -0.25993266701698303, 1.8968720436096191, -0.5276482701301575, -0.6510735750198364, -0.09663031250238419, 0.304864764213562, 0.643919825553894, 0.22110812366008759, 0.13279855251312256, 0.17734551429748535, 2.6952855587005615, -1.4516453742980957, 0.45589786767959595, 1.0056383609771729, -0.7798035740852356, 0.7619462013244629, 0.7441129088401794, -2.49627947807312, 2.116532564163208, 1.209014654159546, 0.6752270460128784, -0.6275039315223694, 0.009860610589385033, -0.5059984922409058, -0.5378624796867371, 1.2323687076568604, -1.972277283668518, -2.0832488536834717, -0.44921356439590454, 0.9052203297615051, -0.8623208403587341, 0.8986598253250122, -0.515605628490448, 0.04343581199645996, 0.08971168100833893, 0.4967164695262909, 0.7232087254524231, -0.03518795967102051, -0.8157806992530823, -1.2772353887557983, 1.3762617111206055, 0.5273616909980774, 1.2169959545135498, -0.5243619680404663, 0.7723337411880493, 1.0831482410430908, 1.4309183359146118, 0.12340940535068512, 1.4871481657028198, -1.1292555332183838, -0.06398084759712219, -0.8256797790527344, -1.1391786336898804, 0.26604241132736206, 1.1163603067398071, -0.5852137804031372, -0.34536731243133545, -0.4235819876194, -0.3969068229198456, 0.9794490337371826, 1.178123116493225, -1.3403587341308594, 0.6338253617286682, 0.5769339203834534, -0.3426724970340729, -1.5812665224075317, -0.07099278271198273, -1.1167491674423218, 0.5380673408508301, 0.5124229788780212, 1.5495591163635254, -0.4168234169483185, 0.3237552046775818, -0.20110933482646942, 0.5753999948501587, -0.47843167185783386, -0.3676352798938751, -0.9708173274993896, 0.1654462218284607, 0.23545633256435394, 1.8489266633987427, 0.514757513999939, -0.11227350682020187, 1.1593581438064575, 1.0983684062957764, -0.3066542446613312, 0.3081248998641968, -0.19796904921531677, -0.12407417595386505, 0.1844007670879364, -0.10535397380590439, -0.5482580065727234, 0.4842533767223358, -0.43507587909698486, -1.7824065685272217, -0.5566074848175049, 1.1351494789123535, 0.38950616121292114, 0.6453471779823303, 0.2672523558139801, -0.759345293045044, -1.4587411880493164, -1.3216253519058228, 0.026701342314481735, -0.2612505853176117, 0.16981637477874756, -0.5680928826332092, 2.275559186935425, -0.7809404134750366, 1.4128285646438599, -0.33230262994766235, 1.368714690208435, 1.9734169244766235, 1.8542194366455078, 1.5581591129302979, 0.6240490674972534, -1.44435715675354, -1.6084665060043335, 0.311471164226532, 0.9954422116279602, -1.2996279001235962, 1.2532001733779907, 0.11254657059907913, -1.0242154598236084, 0.29750338196754456, 2.3958969116210938, -0.532027006149292, 0.385820209980011, 0.4376291334629059, -0.31634506583213806, 1.2258275747299194, 1.5633655786514282, 0.727514922618866, 1.6325454711914062, 0.24961361289024353, 0.8760640025138855, -0.9889234900474548, 1.232572317123413, -0.07126645743846893, -1.7165855169296265, -0.023539261892437935, -1.448440670967102, -1.0981407165527344, 0.7907008528709412, 0.755804181098938, -1.076499104499817, -2.3691580295562744, 0.5509849190711975, -1.312650442123413, 0.6718279123306274, 0.6804459095001221, 0.3558708131313324, 2.0354437828063965, 0.6928777694702148, -0.2046106606721878, -0.547593891620636, -0.08770567178726196, 0.2626189589500427, -1.425778865814209, -0.39719682931900024, -0.06923463940620422, -1.6688536405563354, -1.4614193439483643, 0.203951895236969, -1.579924464225769, 1.1455109119415283, 0.4338573217391968, 0.20565669238567352, -0.7766925692558289, -1.1614645719528198, 0.11115686595439911, 0.28888702392578125, 0.9417546391487122, -2.8431854248046875, 0.4121614098548889, 0.7099647521972656, -0.7370193600654602, 0.2953961491584778, -0.6857980489730835, -0.5967335104942322, -1.5586283206939697, 0.6605083346366882, 0.014492183923721313, -0.4513337016105652, -0.45136499404907227, 0.2533683478832245, 1.4060606956481934, 1.4134942293167114, 0.10088140517473221, -0.9079704284667969, 0.543297529220581, -1.781048059463501, 1.5162098407745361, 0.7398636937141418, -0.5872739553451538, -0.25624075531959534, -0.7941716909408569, 0.33975306153297424, -0.990093469619751, 0.0042442213743925095, -0.7251123189926147, -1.900951862335205, -0.08640560507774353, -0.4937237799167633, -0.9898462891578674, -1.383514165878296, -0.6169363260269165, -0.5696225166320801, -0.9122312664985657, -0.7499688863754272, -0.3812601566314697, -1.028260588645935, 0.747689962387085, 0.2297155261039734, 0.5489402413368225, -0.9885371923446655, -0.5459762215614319, 0.0749150738120079, -0.12720999121665955, -0.7415739297866821, -0.6906618475914001, 0.5168371200561523, 0.8563808798789978, 1.5316452980041504, -0.33282729983329773, 0.6848935484886169, 0.7763163447380066, 0.2391977608203888, 0.1119682714343071, -0.018552973866462708, 0.838995635509491, 0.84661865234375, -0.8489099740982056, -0.33484768867492676, -0.864144504070282, 0.1645277887582779, -0.5136919617652893, 0.33084914088249207, 1.37104070186615, -0.5106077790260315, 1.0619045495986938, 0.21827638149261475, -0.16839981079101562, -0.15022802352905273, -0.1192423403263092, 0.6888893842697144, 1.8788459300994873, 0.1094270646572113, -0.3037714660167694, 1.1992064714431763, 0.6966105103492737, -0.729374885559082, 0.8606025576591492, -0.29687172174453735, 0.26444125175476074, 0.7383976578712463, -0.5295526385307312, -0.16492488980293274, 0.47479549050331116, -1.445091724395752, -0.3749501705169678, -1.814292073249817, 0.20337872207164764, -0.19183498620986938, -1.0569311380386353, -0.7521114945411682, -1.7632743120193481, -0.16254669427871704, -0.9335660338401794, -0.005041591357439756, 0.82036292552948, -0.5381419658660889, -1.1296502351760864, 1.2007660865783691, -0.6725906133651733, 1.0600894689559937, 0.399906188249588, 1.1169791221618652, -0.5033147931098938, -1.7148722410202026, -0.5634086728096008, 0.08310487121343613, 1.7325700521469116, -0.8841493129730225, 0.4452737271785736, -1.3569018840789795, -0.3912760317325592, 1.04180908203125, 0.9348971247673035, 0.4098690152168274, 0.05691298469901085, 0.4461587965488434, 0.9168674349784851, -0.166049525141716, -0.16438938677310944, 0.13268038630485535, 0.29127809405326843, -1.7646023035049438, -0.28687289357185364, -0.3795807957649231, -0.11015457659959793, -0.6964725852012634, 0.979057788848877, 0.7114121913909912, 0.04804787412285805, 2.95741605758667, 0.7799949049949646, 1.265699028968811, 0.3307632505893707, 0.5960196256637573, 1.0406795740127563, -0.6925880312919617, -1.4632467031478882, 2.012899398803711, -1.6523816585540771, -1.3419588804244995, -0.25534874200820923, -0.6288350224494934, 0.1088632270693779, 1.5640757083892822, 0.8046324253082275, -2.0810930728912354, -0.6055541634559631, 0.2036105841398239, -0.1690049171447754, -2.0052683353424072, -0.6418989300727844, 1.0629721879959106, -0.03021341934800148, -0.5749692916870117, -1.35654878616333, 0.9667123556137085, 2.441476583480835, -0.8007373213768005, 0.9815280437469482, 0.059402987360954285, -1.0871845483779907, -1.4566655158996582, 0.3417322635650635, -0.1174745038151741, -0.7203549146652222, 0.24668695032596588, 0.4527475833892822, 2.5529747009277344, -0.11294999718666077, -0.7321699261665344, -1.0113532543182373, 0.5204601883888245, 0.8246545791625977, -0.15854474902153015, 0.9949341416358948, 0.14067603647708893, -0.2035696804523468, -0.4583733081817627, 0.07764671742916107, 0.2761838138103485, -0.3142560124397278, -1.0097591876983643, -1.0368677377700806, 1.1874758005142212, -0.5345369577407837, 1.8546991348266602, -1.340250015258789, -0.04945939779281616, -0.28245609998703003, -1.2920069694519043, -1.7588521242141724, -0.8605932593345642, 1.1884735822677612, -0.9020102620124817, 0.3018525540828705, 0.37482133507728577, 0.5773208141326904, -2.360050678253174, -0.7009514570236206, -2.1005260944366455, 0.3891691267490387, 0.661761999130249, 1.0360552072525024, 0.6896098852157593, 1.174634575843811, 0.9921529293060303, 0.3795561194419861, 0.8721656203269958, -0.8940142393112183, 0.18695662915706635, -1.046280860900879, -0.6527648568153381, 0.12821511924266815, 1.3738882541656494, -0.2148071974515915, 0.63434237241745, 0.049480222165584564, 1.3298906087875366, 0.6997799873352051, -0.5140925645828247, 0.9759872555732727, 0.3168642222881317, -0.11729485541582108, 1.7073822021484375, 0.011633767746388912, -0.6350380182266235, -2.1679790019989014, -1.5238959789276123, 0.8872843980789185, -3.318207025527954, 0.9785914421081543, -1.195913314819336, -1.2089858055114746, 0.9347200989723206, -1.7994825839996338, -0.6514396667480469, -0.9606796503067017, -0.2923455238342285, 1.1065324544906616, 1.822791337966919, -1.7144705057144165, -1.0787526369094849, 2.5342705249786377, 0.7566525936126709, 2.03903865814209, 1.9315170049667358, 0.10748536139726639, -0.7671712636947632, 0.9419021010398865, 0.465327650308609, 1.6018707752227783, 0.5057203769683838, 2.515563726425171, -1.596454381942749, -1.6713742017745972, -0.47255590558052063, -1.1325061321258545, 0.13260510563850403, -0.6940637826919556, 1.3426803350448608, 0.17726820707321167, 0.46578723192214966, -0.5269616842269897, 0.009393708780407906, -0.36429035663604736, -0.9257249236106873, 0.292041540145874, -1.9813727140426636, -0.47209078073501587, -0.6459798812866211, 1.2038891315460205, -0.32037296891212463, -1.7900313138961792, -0.1644279807806015, 0.34769004583358765, 0.4562933146953583, 0.8684993386268616, 0.7828713655471802, 0.500903308391571, 1.3952851295471191, -0.34207436442375183, 0.5578741431236267, 0.20667842030525208, 0.4176352620124817, 0.5167784094810486, -0.2935008108615875, -0.23415564000606537, 0.05351148918271065, -2.2525107860565186, 0.36851024627685547, 0.9700836539268494, -0.19662129878997803, -0.3926700949668884, -0.22716741263866425, 1.1194394826889038, -0.5218908786773682, 1.025266408920288, -1.4576786756515503, -0.6311224102973938, 1.5303572416305542, -0.563112199306488, -1.4628405570983887, 0.2690751850605011, -0.48826566338539124, -0.08660133928060532, 0.5360593795776367, -0.49560126662254333, 0.4366922080516815, -0.2786898612976074, 0.027160149067640305, -0.2824920415878296, -1.38395357131958, -0.8901569843292236, -0.684282660484314, 0.7503078579902649, -0.856674075126648, -1.0401198863983154, 0.14473873376846313, 0.7038787603378296, 0.08703187108039856, -0.3441610038280487, -1.5623241662979126, 0.41530686616897583, 1.084520697593689, -0.28104883432388306, 0.20837999880313873, -0.858920156955719, -0.685768187046051, 0.12657158076763153, -0.161007821559906, 0.49448350071907043, 0.18308772146701813, -0.4820522665977478, -0.6312243342399597, 0.5201894640922546, -1.207971215248108, 0.4873019754886627, -0.37749508023262024, -1.4157623052597046, -0.9672249555587769, -0.40862056612968445, 2.4389820098876953, -0.17639559507369995, -1.2984966039657593, 0.9155725836753845, -1.0904974937438965, -1.4000341892242432, 0.3212434947490692, 0.9652705788612366, 1.9689420461654663, 0.3174821734428406, -0.8159971237182617, 0.02459694817662239, 1.3279169797897339, 1.603293776512146, -0.8524019718170166, 1.6706289052963257, -0.2314881682395935, -1.1944411993026733, 0.6653188467025757, 0.13061486184597015, -1.486008644104004, -0.9862831234931946, 0.017934363335371017, 0.4401722848415375, -1.1427700519561768, 1.0992437601089478, -0.8849238157272339, -0.977422297000885, 0.6899287700653076, 1.213960886001587, -0.33730924129486084, -0.2322370409965515, 0.3348209261894226, 1.1075410842895508, -0.4435546100139618, -0.20692668855190277, -0.6350822448730469, -1.6971114873886108, 0.6484492421150208, -0.4355412423610687, 0.9786455035209656, -0.5152884125709534, 1.3162997961044312, 0.20981739461421967, 1.000900149345398, -1.3026347160339355, -0.699661135673523, -0.5888822674751282, 1.8573037385940552, 0.4189995527267456, -0.5224697589874268, -2.257063388824463, -0.9998037815093994, 1.3192514181137085, -0.5174021124839783, 0.5796229839324951, -1.6926093101501465, 0.37162235379219055, 1.0309486389160156, 0.411387175321579, -1.0179460048675537, -1.1097710132598877, 0.5603137016296387, -1.0967857837677002, -1.4675540924072266, 0.11683053523302078, -1.1527843475341797, -1.207453727722168, -0.24649806320667267, -0.16238678991794586, -1.4249083995819092, -0.03483868017792702, 1.1879286766052246, 0.6601441502571106, 1.0070898532867432, 0.9389218688011169, 0.02158287912607193, 0.20455066859722137, -1.0270625352859497, -0.9577014446258545, -0.8138734698295593, -1.2097572088241577, 1.9267843961715698, 1.288076639175415, -1.6067025661468506, 0.5958104133605957, 0.0485258549451828, 0.3873905837535858, 0.37004318833351135, 0.9126068353652954, -1.2261561155319214, -1.0041929483413696, -0.4739840030670166, 0.8909783363342285, -0.0040915049612522125, -0.09522517770528793, -0.26173269748687744, 0.7525641918182373, 0.3517623841762543, 1.397526741027832, 1.9028090238571167, 0.6414739489555359, -0.8676348328590393, -1.229722499847412, -0.12288501113653183, 0.22080574929714203, 0.046938806772232056, 0.31497687101364136, 0.0467599593102932, 1.3551899194717407, -0.02771497704088688, 1.2516273260116577, 0.22487565875053406, -0.3376784324645996, 0.11301762610673904, -0.5219265818595886, 0.12705697119235992, -2.173874616622925, 0.07154014706611633, 0.9641303420066833, 1.1165989637374878, -0.7131199836730957, -1.0691282749176025, 1.2146930694580078, -1.3858007192611694, -0.09858199208974838, -0.32641884684562683, 0.26649922132492065, 0.8899468183517456, 1.3365172147750854, -1.964390754699707, -0.3831944763660431, -2.0792295932769775, -0.7440024614334106, -1.5841493606567383, -0.3483312726020813, -0.6066698431968689, 0.2043686807155609, 0.4897875189781189, 0.030248722061514854, 0.6247978210449219, 1.020386815071106, -0.2216227501630783, -0.12095677852630615, -0.8577427864074707, -1.0087170600891113, 0.6084603667259216, 1.461830973625183, 0.06525696069002151, -1.582666277885437, -0.8418800234794617, 0.1439545899629593, 0.011040208861231804, -1.0005865097045898, 0.06763122975826263, 0.018681855872273445, -1.5087103843688965, -1.0774493217468262, 0.5108246803283691, -0.9952746629714966, -0.3000502288341522, 1.1313873529434204, 0.7274208664894104, 0.596291184425354, -1.3109022378921509, -0.8217450976371765, -0.11586257070302963, -0.13312695920467377, 0.6800580024719238, 0.6435085535049438, 0.9830313324928284, 1.3931546211242676, -0.16015344858169556, -1.0427374839782715, 0.8471361398696899, 0.12141110002994537, 0.14870479702949524, -0.5077730417251587, -0.1819935441017151, 2.309354305267334, -1.8656221628189087, 0.6879533529281616, 0.2887348234653473, 0.652887225151062, -0.5002766251564026, 0.004345486871898174, -0.46375134587287903, -0.27500835061073303, -0.44483277201652527, 1.6371688842773438, 1.0553972721099854, 0.5773827433586121, 0.42880013585090637, -0.5583151578903198, -1.481461763381958, 0.33831077814102173, -0.9340448975563049, -0.5810448527336121, -0.9510481953620911, -0.49103814363479614, -0.5529094338417053, 1.264194369316101, 1.4465031623840332, -0.02124761790037155, 0.541317880153656, -0.06131754070520401, 0.26640504598617554, -0.45824408531188965, 0.36099186539649963, -1.5934765338897705, -2.510453701019287, 1.8234201669692993, 0.13090737164020538, -0.005060211755335331, 0.5215069055557251, 0.13616330921649933, 0.7791450619697571, -1.1288950443267822, -0.67979896068573, 1.2564243078231812, 0.24343685805797577, -0.7543305158615112, -1.3893976211547852, 1.7684932947158813, 1.0134586095809937, 0.5476924777030945, -1.4833333492279053, -0.6638368964195251, -1.1973196268081665, -1.9561352729797363, -0.6186386346817017, -0.4509485065937042, 2.0621652603149414, -1.3919777870178223, 0.9517075419425964, 0.2964707911014557, 0.6648410558700562, 0.09623889625072479, 0.943835973739624, -0.13190694153308868, 1.136327862739563, 0.7013256549835205, -0.9884242415428162, -0.2347487062215805, 1.8009089231491089, 1.7164148092269897, -0.8626373410224915, 0.3358798027038574, 1.6762572526931763, 0.5070697665214539, -0.63663649559021, 0.6607042551040649, 0.17615056037902832, -1.6644023656845093, 0.5014516115188599, 0.1956189125776291, 0.6653293371200562, 1.8295105695724487, 1.8376010656356812, 1.607598900794983, 1.686248779296875, -0.9423644542694092, -1.036793828010559, 0.38926613330841064, 0.009735135361552238, 0.46871763467788696, -0.44741663336753845, 0.6148169636726379, 1.632997989654541, -0.0985686182975769, -0.07991823554039001, -0.5002676844596863, -2.0685150623321533, -0.20776642858982086, -0.20925085246562958, 0.26831915974617004, -0.6929259300231934, 1.4130582809448242, 1.0703016519546509, 1.9330663681030273, -2.6911892890930176, -0.5521084666252136, 0.7103496789932251, 0.49236875772476196, 0.4483449161052704, 0.032628875225782394, 1.1220213174819946, -0.2730626165866852, -0.11452760547399521, 2.589585065841675, -0.2044467329978943, 1.0634642839431763, 0.24870160222053528, 0.8891845941543579, 0.13373003900051117, 1.065890908241272, 1.290838599205017, -0.030330486595630646, -0.40796899795532227, -0.5143083333969116, -0.7964395880699158, 1.611721396446228, 1.9388848543167114, -1.7393443584442139, -0.7072798013687134, 0.23553074896335602, 0.9998807907104492, -1.1055439710617065, -0.4619038701057434, 0.2251373827457428, 0.9623318314552307, -0.8107923865318298, 1.1095657348632812, -0.7945343852043152, 1.0130478143692017, -0.5576779246330261, -0.2742217183113098, 0.8606982231140137, 0.488957017660141, 0.8228431344032288, -0.08047891408205032, -0.1931498497724533, 1.3112602233886719, 1.9687261581420898, 0.5054379105567932, 0.5023000240325928, -0.7023511528968811, 1.9364233016967773, -0.30222031474113464, 0.47881880402565, 1.4453110694885254, -2.7567243576049805, -0.27825024724006653, -1.31974196434021, -0.40723612904548645, 0.043483980000019073, -1.4395112991333008, 1.1671744585037231, 2.4111170768737793, -0.2733227014541626, 0.021679608151316643, 1.758480429649353, 0.4295736253261566, -1.0879656076431274, 0.6125932931900024, -0.5986605286598206, 1.3499176502227783, -0.6350637078285217, 1.4948408603668213, -1.2446765899658203, -0.1724323332309723, -2.096357583999634, 2.621976852416992, 0.07641811668872833, -0.27198949456214905, 0.4625967741012573, 0.1928776651620865, 0.5325186848640442, 2.3613147735595703, 1.6997705698013306, -0.4014967978000641, 0.3815673887729645, -0.033014461398124695, 0.5603039264678955, -0.9990184903144836, -0.32818126678466797, -1.5081464052200317, 0.6034889221191406, -0.6312136650085449, 1.2480789422988892, 0.21740387380123138, -1.400570273399353, 1.8029427528381348, -2.024548053741455, 2.9839236736297607, 1.8156884908676147, 1.7646232843399048, 1.8839285373687744, 1.3172839879989624, -0.013717986643314362, -0.6472011804580688, 0.2075345665216446, -2.451782703399658, 0.4809681475162506, -0.019390780478715897, 0.27464771270751953, -0.43593138456344604, -1.4770374298095703, 0.11259832978248596, -0.4155661463737488, -0.4049959182739258, 0.06912162899971008, 0.8862210512161255, 1.3444191217422485, -1.2153971195220947, 0.19670109450817108, -2.0028605461120605, -1.1944774389266968, -1.1534245014190674, -0.28362563252449036, -0.08792088180780411, 0.2973158061504364, 0.015298034064471722, -0.13488881289958954, -1.429579257965088, 0.45734918117523193, -1.0425994396209717, 0.6147720217704773, -0.3725418448448181, 2.0121264457702637, -1.060279130935669, 0.4756602346897125, 1.0848058462142944, -0.49623605608940125, -0.34185463190078735, 1.7097560167312622, 0.983443558216095, 0.3454320728778839, -0.5505116581916809, -0.5310323238372803, -1.6211519241333008, 1.718400478363037, 0.08746013045310974, 0.6862483024597168, 1.4492263793945312, 0.865128755569458, -0.0832860916852951, 1.4887546300888062, -1.8013560771942139, 1.3847578763961792, 0.5873798131942749, 0.011972538195550442, 0.7790274620056152, 1.2487391233444214, -0.5858578085899353, 0.41025853157043457, -0.26500335335731506, 0.0009971198160201311, 0.9690607786178589, -0.30675649642944336, -0.9217151999473572, -0.16509845852851868, 1.730567216873169, 1.4903066158294678, 0.9495775699615479, 1.7467710971832275, -0.031258609145879745, -0.5018876194953918, -0.6876056790351868, 0.5458019375801086, 0.6132803559303284, -1.0979254245758057, 2.6111979484558105, 0.961665689945221, -1.4689913988113403, -1.75819730758667, 0.2760692238807678, 0.21743394434452057, -1.1640727519989014, -0.018569961190223694, -0.07169289886951447, -0.19609400629997253, 1.2190656661987305, -1.3789595365524292, -0.18526101112365723, -1.3724366426467896, 0.009922435507178307, 0.13068629801273346, 2.171049118041992, -1.0387661457061768, 0.13689593970775604, -0.23585209250450134, -0.0004091660666745156, -0.7986379265785217, -1.3062998056411743, -0.7685028910636902, -0.8960750102996826, 0.8996874094009399, -0.27451518177986145, 1.3953421115875244, -0.9873231053352356, -1.913283109664917, -0.6734384298324585, -0.4551069736480713, 2.0527145862579346, -1.2158671617507935, 0.7838758826255798, -0.4044475853443146, -0.7057141065597534, 0.9159167408943176, -0.6659210920333862, 1.886397361755371, 0.6517815589904785, 0.8118091821670532, -0.21460866928100586, 0.4296376705169678, 1.0245953798294067, -0.7569696307182312, -0.06119510531425476, -0.8984799981117249, -0.3213730454444885, 0.17869827151298523, -1.9188278913497925, 0.01305533666163683, -0.6574828624725342, 0.8039892315864563, -1.2342537641525269, -0.41760924458503723, -0.30346670746803284, 1.0987108945846558, -0.047502290457487106, 1.5451606512069702, 0.8268771767616272, -0.9928683638572693, -1.4015086889266968, -0.2934398055076599, 0.5345712304115295, -0.26106536388397217, -1.1552506685256958, 0.3802092969417572, -0.4147556722164154, -0.9794937968254089, 0.12420783936977386, -0.3276590406894684, -1.7691361904144287, -0.27421483397483826, 0.29067957401275635, -0.957682192325592, -1.6462244987487793, 0.5102518200874329, 0.9099146127700806, -0.3700123131275177, 0.03470037877559662, -0.9439483284950256, -0.11613169312477112, -1.5975968837738037, -0.5305702686309814, 0.029791317880153656, -0.8327968120574951, -0.32907387614250183, -0.11399123817682266, 1.1146669387817383, -0.7159637212753296, -1.5991712808609009, -0.9493930339813232, -0.9363129734992981, 0.2689962089061737, 1.085500717163086, -1.4473892450332642, -0.08706393837928772, 0.28589242696762085, -0.5059764385223389, -0.38087767362594604, -1.056312918663025, 0.9849520921707153, 0.4286854565143585, -1.2945125102996826, 0.010541040450334549, -1.5170398950576782, 0.19278940558433533, 0.6971812844276428, -2.1530778408050537, -0.5779186487197876, -0.4445229768753052, 1.9542529582977295, 0.39522916078567505, 0.5080580711364746, -0.29508984088897705, 0.001000995864160359, 0.5233620405197144, 0.027851320803165436, 0.4692448079586029, -0.25449037551879883, -1.3646854162216187, 0.15975530445575714, -0.033886589109897614, -1.5859111547470093, -1.1383464336395264, -0.6909943222999573, 1.3107168674468994, 0.47095200419425964, 1.4923968315124512, -2.359754800796509, 0.24665425717830658, 1.4105793237686157, -0.5737679600715637, 0.33174756169319153, -1.027807593345642, 0.2620120942592621, -0.15724734961986542, -1.5682963132858276, 1.0209134817123413, -0.03489190712571144, 0.35307008028030396, -1.7690443992614746, 0.765885591506958, -0.6512317657470703, 0.6859904527664185, -1.1671257019042969, 1.148340106010437, -0.37358078360557556, 0.0398082509636879, -0.542968213558197, -1.7020930051803589, -0.22679145634174347, 1.166976809501648, 1.2212055921554565, -0.37223750352859497, 1.272823452949524, -0.1398024559020996, -1.1771469116210938, -0.09480076283216476, 0.04231884330511093, 1.028875708580017, -2.169290542602539, 1.203652262687683, 0.8576732873916626, 0.6141934394836426, -1.3708382844924927, -1.5402003526687622, 1.0172713994979858, -0.2900179624557495, 0.996673583984375, -1.1762717962265015, 0.6530174612998962, 0.5260802507400513, -1.3334910869598389, -0.30101290345191956, 1.7765775918960571, -0.771280825138092, 1.1031838655471802, -0.009343351237475872, -0.9562903642654419, -0.563671350479126, -1.0532844066619873, 1.7368940114974976, -1.5748997926712036, -1.2248843908309937, -0.008893350139260292, 0.958797812461853, -1.3419239521026611, -0.9059275388717651, -0.266335666179657, -0.38361304998397827, 0.013671759516000748, -0.6851789951324463, -0.06832952797412872, 0.6846504807472229, -1.1941864490509033, -1.6545677185058594, -0.7934851050376892, 0.07878555357456207, 2.5848820209503174, 0.8030512928962708, 0.4699844419956207, -2.9683918952941895, -1.9291274547576904, 0.5984709858894348, -0.6315966844558716, -0.7686052322387695, 0.7451121807098389, 1.1476413011550903, 0.5188243985176086, 0.1336398869752884, -1.8686552047729492, -0.2683287560939789, 0.8022715449333191, -0.2539771795272827, -1.5430728197097778, -2.4806768894195557, -0.4104160964488983, 0.5079609155654907, -0.7989580035209656, -0.26173049211502075, 0.3666597008705139, -0.5630687475204468, -0.17526043951511383, -0.6815651059150696, -0.7875736355781555, 0.4361855685710907, 0.5218626260757446, -0.2642683684825897, -1.4791181087493896, 0.3758366107940674, 0.7722018361091614, 0.6928211450576782, 0.6756012439727783, -0.024023108184337616, -0.26115936040878296, 0.5191236734390259, 0.5968677401542664, 0.5899055600166321, -0.873755156993866, -1.834809422492981, 0.3539878726005554, 0.3213264048099518, 0.25850000977516174, 0.5130177736282349, -1.545249104499817, -0.6434493064880371, 0.625659704208374, -0.4889014959335327, -0.18505659699440002, 1.4145488739013672, 0.737933874130249, -0.5062530636787415, 0.28059783577919006, -0.7274827361106873, 0.42446067929267883, -1.336965799331665, -1.6473642587661743, 1.5080560445785522, 0.998719334602356, -0.8954718112945557, 0.47810253500938416, -2.393439531326294, 0.05299091339111328, 1.1232452392578125, 0.7584363222122192, 0.17059296369552612, -0.04808465391397476, 0.2397274225950241, 2.068917989730835, -0.4332948625087738, 0.24840931594371796, 0.7830125093460083, -0.8386057019233704, 1.5290390253067017, -1.8767285346984863, -0.20121344923973083, -0.4181830585002899, -0.05255497246980667, -1.2578223943710327, -0.36031702160835266, 0.3055993914604187, -0.7970383763313293, 0.8911043405532837, -1.9436858892440796, -0.14532341063022614, 1.6022260189056396, -0.9211487770080566, 0.08654060959815979, -0.3517085909843445, -0.5520389080047607, 0.6562939286231995, -1.2986652851104736, 0.522024393081665, -1.3743579387664795, -0.2568081021308899, 0.7702167630195618, -0.5766310095787048, 0.03511640056967735, 2.21675968170166, 0.15450114011764526, 0.2733666002750397, 0.6449499726295471, 0.5715423226356506, -0.001013062079437077, -0.2657351493835449, 0.22902081906795502, 0.34397074580192566, 1.3181095123291016, 0.07623742520809174, 0.7872352004051208, 1.9347259998321533, -1.630581021308899, -0.2226509302854538, -0.2612513601779938, -0.16851627826690674, 1.3848055601119995, -0.28453269600868225, 0.17696955800056458, 1.6363568305969238, -2.115297555923462, -1.2642204761505127, -1.1577556133270264, 0.0112222321331501, 0.5108578205108643, -0.8758467435836792, 1.3603806495666504, 1.594428300857544, -0.7878562808036804, -0.5531755685806274, -1.0200974941253662, 0.6221686601638794, -0.4489533603191376, 0.9251927137374878, 1.094044804573059, 0.011566142551600933, -0.8388446569442749, -2.5072414875030518, 0.3161245286464691, 0.49922454357147217, 0.048331841826438904, -0.8405028581619263, 1.0554578304290771, -0.06476625055074692, 0.6571632623672485, 0.7843882441520691, 0.9735424518585205, 0.6596660614013672, -1.5768284797668457, 0.3574007451534271, -0.09761934727430344, -0.5344736576080322, -0.8937587738037109, -0.9605467915534973, -0.4825194776058197, -0.30066484212875366, 0.04643252119421959, -1.8899720907211304, -0.9040243625640869, 0.6517100930213928, 1.0687012672424316, -0.2940533459186554, -0.49651435017585754, 1.8432635068893433, -1.0362122058868408, 1.0136630535125732, 0.8229461908340454, 0.35546815395355225, 1.047365665435791, 0.6581311821937561, -0.15559585392475128, 0.7194499969482422, 2.492011547088623, 0.18992504477500916, 1.0135672092437744, 1.9028509855270386, -0.89543217420578, -0.009730338118970394, -0.9435257911682129, 0.13680818676948547, -0.4958488345146179, 1.1798063516616821, -0.2554091215133667, 0.7040203213691711, 0.21009676158428192, 0.46009770035743713, -0.8798585534095764, 0.9744028449058533, -0.07933405786752701, -0.18753351271152496, -0.7630581259727478, 0.47712063789367676, -0.4740854799747467, -0.7128042578697205, -0.02701110579073429, -2.22501540184021, 1.1788321733474731, -0.4501776695251465, 0.9694119095802307, -1.2082538604736328], "dtype": "float32"}, {"__nd__": [-2.0163350105285645, -0.9255067706108093, 0.38360825181007385, 1.2636175155639648, 1.3740959167480469, 1.1140544414520264, -0.7519196271896362, -0.5291501879692078, 0.37532296776771545, 0.7309287786483765, -0.9443048238754272, -0.9530680179595947, -1.3230432271957397, 1.1864603757858276, -0.9254799485206604, 0.2899118661880493, 0.07196220755577087, 0.4772261679172516, -1.525555968284607, 0.05075256898999214, 0.8878450989723206, 0.6089035868644714, -0.3634834289550781, 0.7572380900382996, 0.2526753544807434, -0.31332916021347046, 0.08753372728824615, 2.27520751953125, -0.46087726950645447, -1.4980342388153076, 0.7024935483932495, 0.8037551045417786, 0.535528302192688, -0.0928567498922348, 1.0214471817016602, 1.983779788017273, 1.484076976776123, -2.036027193069458, -0.81255042552948, 0.2745230197906494, -2.125659704208374, 1.799070119857788, 0.82892906665802, -0.7470197677612305, -0.9733206629753113, 1.1900867223739624, -0.28954556584358215, 0.34309542179107666, -1.0202194452285767, -1.0697063207626343, -0.4216064214706421, 0.07177141308784485, 0.8490346670150757, -0.48254096508026123, 0.809961199760437, -0.4288567900657654, -0.8414871692657471, -0.39709821343421936, -0.44674891233444214, -1.2165982723236084, -1.8341470956802368, 0.5311822891235352, 0.23078875243663788, 0.7491198778152466, -1.5551023483276367, -0.33416950702667236, -1.3346447944641113, 1.323528528213501, 1.6895853281021118, 0.62850022315979, 1.4196950197219849, -1.4341109991073608, -0.12736019492149353, 0.23806753754615784, 0.5747923851013184, 0.11989305168390274, 0.7405656576156616, 0.32571977376937866, -0.2594550549983978, 1.2769094705581665, -0.3746349513530731, 0.7734950184822083, 0.6396242380142212, -1.0351791381835938, -0.3653961718082428, 2.14564847946167, 0.11328727751970291, 0.7695674896240234, -0.34169280529022217, 1.193582534790039, 0.17962823808193207, 1.0963046550750732, 1.012523889541626, -0.7127367258071899, 0.1357974261045456, 0.7785962224006653, -0.8609858155250549, 0.19649116694927216, -0.08978724479675293, 0.7811301350593567, -0.9017329812049866, 0.56773442029953, 1.0248819589614868, -0.36805739998817444, -0.27396878600120544, 0.8566177487373352, -0.37141311168670654, 0.6328177452087402, 1.3248978853225708, -1.1808547973632812, 0.5893692970275879, 1.3422967195510864, -1.2961814403533936, 0.33067587018013, 0.5967631936073303, -0.2614883482456207, 0.3748628795146942, -0.9127757549285889, 0.1516885608434677, -0.622958242893219, 0.8088592886924744, 0.12791456282138824, 0.16486133635044098, 1.053086280822754, 0.49333351850509644, -0.5030860304832458, -0.13058894872665405, -0.5847992300987244, 0.6429122686386108, 1.3863160610198975, 2.0209808349609375, 0.9259190559387207, -0.618033766746521, 0.8591115474700928, -0.9111837148666382, -1.2478030920028687, -0.9006377458572388, 0.47496098279953003, -1.5056792497634888, 0.634263813495636, -0.013837608508765697, 1.7764109373092651, 0.09751413762569427, -0.2857936918735504, -0.9881327748298645, -0.719590961933136, 0.3226592540740967, -1.2821581363677979, -2.5190320014953613, -0.4705454409122467, -0.09168461710214615, -0.17305399477481842, -0.30059537291526794, -0.8019385933876038, -1.4530211687088013, 1.483402132987976, 0.7742848992347717, 0.40878963470458984, -0.4222627580165863, -0.723238468170166, -0.18999935686588287, 0.668760359287262, 0.139714315533638, 0.03098159097135067, 0.6264334321022034, -0.6061606407165527, -0.8082452416419983, -0.3295974135398865, -0.14803488552570343, 1.7806276082992554, 2.278878688812256, -1.0242700576782227, -0.7542235851287842, 0.028527405112981796, -1.8943989276885986, -0.5827973484992981, 1.3332651853561401, -0.7301682233810425, -0.457870215177536, 1.0689845085144043, 1.766133427619934, -0.20969733595848083, -0.0071473801508545876, 0.30371755361557007, 0.9110375046730042, -1.711135745048523, 0.31814098358154297, -0.4007216989994049, 0.2990729808807373, 0.6389296054840088, -2.3141350746154785, 1.017167091369629, 0.10049862414598465, -0.31702226400375366, 0.5351295471191406, 0.535685122013092, 2.0775949954986572, 0.7908558249473572, 0.875169575214386, 0.30275532603263855, 0.7424070835113525, -0.8176949620246887, 0.8128767609596252, 0.2511305809020996, 0.8681150674819946, 0.9213606119155884, -1.3527017831802368, 0.4838619530200958, 0.7507984042167664, 1.0595717430114746, -1.8941679000854492, 0.4269596040248871, -1.3025435209274292, -0.6847702860832214, 0.35628923773765564, 0.5777961015701294, 0.1192890852689743, -0.3090815842151642, -0.3921477198600769, 0.7716019749641418, 0.8379608392715454, 0.14864026010036469, -1.7088377475738525, -2.221191167831421, -1.5511735677719116, -0.4694649577140808, 0.7527967095375061, -0.38479524850845337, -0.5387306809425354, -0.16804999113082886, -0.28126123547554016, -0.4739505648612976, -0.8127437233924866, -0.7317058444023132, -0.2302631139755249, -0.02386036328971386, -0.5976517796516418, 0.8103948831558228, -1.1052262783050537, 0.43995988368988037, -0.7270091772079468, 0.5776807069778442, -1.0319244861602783, -0.039792001247406006, 0.9158586859703064, 0.7681058645248413, -0.8581251502037048, -0.6286773085594177, -1.1886467933654785, -0.38422709703445435, -1.5789512395858765, -0.4395332634449005, -0.8751083612442017, -0.15675358474254608, -0.2987304627895355, -0.13184893131256104, -0.041748445481061935, 0.8800116777420044, -0.3035595715045929, -0.5773972272872925, 1.0130571126937866, -3.0971109867095947, -1.019729495048523, -1.0572736263275146, 1.4847759008407593, 0.5819591283798218, 0.5084769129753113, -0.01966206356883049, 0.336016446352005, 1.3687275648117065, -1.6939301490783691, -0.906567394733429, 0.4282998740673065, -0.8430169224739075, 0.7515931725502014, 0.7778748869895935, 1.1019514799118042, -1.632545828819275, 0.16527636349201202, 1.048998475074768, -0.41798093914985657, 1.0076035261154175, -0.2671416103839874, -1.1490713357925415, 1.6766353845596313, -1.4191858768463135, 0.11228486150503159, -0.12970218062400818, -0.5926908850669861, -0.5835670232772827, -1.7263965606689453, 0.2556520700454712, -1.0831040143966675, 1.840080738067627, -0.41977426409721375, -1.638999581336975, 0.34753647446632385, 0.8471285700798035, 1.1387896537780762, -0.35189640522003174, -0.06916315108537674, -0.15696501731872559, 0.5754462480545044, 1.3689063787460327, 0.9642802476882935, -0.38158077001571655, -0.17306064069271088, -0.09135939925909042, -0.5886223316192627, -1.061176061630249, -0.4469180405139923, 0.4961455166339874, 0.491525262594223, -1.0655330419540405, 0.2914392352104187, -0.24413561820983887, 0.2832781672477722, 0.6673806309700012, 0.4403979480266571, 0.8925960659980774, 0.6661436557769775, 0.7923353910446167, -0.15531106293201447, -1.7028580904006958, -2.2327675819396973, -0.21667206287384033, -0.29054492712020874, -1.6113299131393433, -1.4713199138641357, -2.279813528060913, 1.109352707862854, -2.393707036972046, -2.201585292816162, 0.47504401206970215, 0.7667658925056458, -0.6595613360404968, 0.40642428398132324, 0.4855404496192932, -0.6273621320724487, 2.556462287902832, 0.5122396349906921, 0.4091632664203644, 0.02029004693031311, 0.4552302062511444, -0.9181354641914368, -0.06615927815437317, 0.16647689044475555, 0.33903518319129944, -0.3498587906360626, 0.24571675062179565, -0.48562997579574585, 1.3965805768966675, -1.1302235126495361, -1.4082945585250854, -0.033376410603523254, 0.6972333192825317, -0.5449796915054321, 0.057453688234090805, -0.002383718965575099, 0.3030148446559906, 1.4646986722946167, -1.0689305067062378, -0.7277539372444153, -0.021302977576851845, -0.32653284072875977, 0.3123958110809326, 1.6418626308441162, 0.6397521495819092, -0.4425518214702606, 0.7762947082519531, 0.6204574108123779, 2.0173356533050537, -0.9313530325889587, 0.35496747493743896, 1.057826042175293, 0.8622568845748901, 0.2554866373538971, -0.6251978278160095, -0.8561926484107971, -0.9530501961708069, 0.23290792107582092, 0.5000895261764526, -0.26311859488487244, -0.6120239496231079, 0.18965527415275574, -1.501322865486145, 2.425767421722412, -0.717445433139801, 0.29125019907951355, 0.03459315001964569, 0.03395271301269531, 0.36576855182647705, -1.0095770359039307, 1.9647681713104248, 0.3223738372325897, -1.4534173011779785, 1.5097935199737549, 0.7723209261894226, 0.013544132001698017, 0.3050192594528198, -0.7297810316085815, 2.175368547439575, 0.2674781382083893, -0.41943588852882385, -1.4646966457366943, -0.8922489881515503, -1.7795804738998413, -0.11409483850002289, -0.46279609203338623, -0.8534331321716309, -0.24168464541435242, 0.32696712017059326, -0.01265833992511034, -0.5646469593048096, 0.6135839223861694, -0.18182338774204254, 0.22043044865131378, -0.17548872530460358, -0.4313609004020691, 1.8690146207809448, -0.8516082167625427, -1.8459765911102295, -1.5370651483535767, -1.114999532699585, 1.8916741609573364, 2.9882140159606934, -1.9543182849884033, 0.9558712840080261, -0.03397338464856148, 1.9980748891830444, 0.46614140272140503, -0.5081937909126282, 1.00330650806427, 0.23991839587688446, 0.2005196064710617, -1.7144794464111328, 0.37340396642684937, 0.27805519104003906, 0.14899663627147675, -0.17256613075733185, 0.5070222020149231, -0.09324616193771362, -1.2329517602920532, 0.40586793422698975, -1.386931300163269, 0.3102174401283264, 0.22359879314899445, 1.8553746938705444, 0.995359480381012, 1.2268757820129395, 1.0719975233078003, -0.3248131573200226, 0.8674632906913757, 0.9680169224739075, -1.0820595026016235, -0.4298873245716095, 1.5905954837799072, 1.7430678606033325, 1.930702805519104, 1.3842220306396484, -1.9675213098526, -0.7491607666015625, -1.6806347370147705, -1.442848563194275, 0.6762962341308594, -0.4931815266609192, -0.7472770810127258, -0.8275762796401978, -0.7203295230865479, -1.1252087354660034, -1.2876790761947632, 0.9612085819244385, -0.0555557943880558, -1.7300422191619873, -0.5790344476699829, -0.03262937068939209, -0.8385641574859619, 0.49544021487236023, -0.5639927983283997, 0.3287719786167145, 0.5996101498603821, 0.10115988552570343, -0.34357333183288574, 0.5827063322067261, 0.9541518688201904, 1.4253884553909302, 0.4618019461631775, -1.4597084522247314, 1.4503017663955688, -1.5191454887390137, -1.0523018836975098, 2.2104945182800293, 0.3605567514896393, 2.130758285522461, 0.09129612892866135, -0.061534322798252106, -2.2275328636169434, -0.02872343361377716, 0.6772514581680298, 0.3700277805328369, 2.5382254123687744, 0.9162164330482483, 0.6070560216903687, 1.1540567874908447, 0.571623682975769, 0.6848692297935486, -0.6428627371788025, -1.783159852027893, -0.1738455444574356, -0.3439140319824219, 0.29602551460266113, 0.8280916810035706, 0.01394633762538433, -1.2732388973236084, 0.2603145241737366, 0.9248871207237244, -1.0545185804367065, 0.9055227041244507, -0.658136248588562, -0.48140838742256165, 1.4489654302597046, -0.08055926859378815, 0.9471999406814575, 0.8172805309295654, 0.3478477895259857, 1.9477280378341675, 0.770301342010498, 1.1680691242218018, 0.5553953051567078, -1.6523301601409912, -1.8539999723434448, -0.26231929659843445, 0.25813838839530945, 1.81028151512146, 1.1078909635543823, -0.3214717209339142, 0.14279890060424805, 0.3252955377101898, 0.5582308173179626, 0.4928613305091858, -0.690991222858429, 0.7089021801948547, 0.5907830595970154, -0.21083630621433258, 1.7743695974349976, 1.0026956796646118, 1.4298827648162842, 0.02870769239962101, -2.145831346511841, 0.8566006422042847, -0.9256680607795715, -1.110885500907898, -1.210333228111267, -1.4868301153182983, -0.1298336386680603, -1.2591043710708618, -1.7631205320358276, -1.500781536102295, -0.7586806416511536, -1.626328468322754, -0.9956400394439697, 0.40872976183891296, -0.8984672427177429, -0.4855295717716217, 0.001703457091934979, -2.499281167984009, 0.8875236511230469, 1.2757543325424194, -2.159212112426758, -0.37901055812835693, 0.3237491846084595, -0.38668787479400635, 0.4167930483818054, -0.38058018684387207, -0.3344692587852478, 0.5438068509101868, -0.029589124023914337, 0.5155104994773865, -0.7269397377967834, -0.4900762736797333, -0.4974652826786041, 0.08921527862548828, 0.7960538268089294, -0.5178375244140625, -0.6481157541275024, -1.0594258308410645, 0.6961745619773865, -0.37752068042755127, -0.9530309438705444, 0.118794746696949, -0.18815116584300995, 0.7051593661308289, 0.08931359648704529, 0.23699991405010223, -0.718363881111145, -0.8636738061904907, 1.362691879272461, -0.2115524411201477, -1.2597527503967285, 0.7619953155517578, 0.4889882504940033, 0.24664857983589172, -0.47680312395095825, 0.2793143689632416, 0.36643508076667786, -1.7215020656585693, 0.4035674035549164, -1.183614730834961, 1.9788821935653687, 0.9978394508361816, -0.6381261944770813, -0.37241214513778687, -0.1617724597454071, -0.5866013765335083, -0.3950662612915039, -1.1256643533706665, 1.6926770210266113, 0.4455350935459137, -0.1525246948003769, 1.1984343528747559, -0.3917912244796753, 3.196833610534668, -0.789657473564148, 1.7896150350570679, -1.1609171628952026, -0.674431324005127, -0.36239469051361084, 1.5381053686141968, -0.41322848200798035, 0.5208322405815125, -2.419229745864868, -1.6712666749954224, 0.06527438014745712, -0.5147956013679504, 1.0607490539550781, -0.04059777036309242, 0.8945168852806091, -0.10625836253166199, -1.152929425239563, 1.286584496498108, -1.388570785522461, -1.630589485168457, 0.29200488328933716, -0.720959484577179, -0.6203187108039856, -0.7243924140930176, 0.8659632205963135, -1.4433492422103882, -1.7074881792068481, -0.42951688170433044, -1.682005524635315, -0.046196404844522476, 0.40244191884994507, 1.3770711421966553, -0.6346554756164551, -1.3980556726455688, -0.6960002779960632, 1.2463996410369873, 0.35632652044296265, 2.10442852973938, -0.43571075797080994, 0.047186657786369324, 1.7884931564331055, -0.5177680850028992, -0.5380910634994507, 0.309291809797287, 0.4577150344848633, 0.48327428102493286, -0.03072563000023365, 0.43798309564590454, 0.2490352839231491, -0.6633133292198181, 0.7658238410949707, 1.1536881923675537, 1.4750205278396606, -0.7083858251571655, -0.8801367282867432, -1.076808214187622, -0.7264198064804077, -1.650653600692749, -0.17533789575099945, 2.232602596282959, -0.22066760063171387, 0.18364134430885315, -0.7645716667175293, -0.04396240785717964, -0.1990624964237213, 0.37930575013160706, 0.20340387523174286, -1.0824476480484009, 1.1207131147384644, 1.6594185829162598, -0.49815019965171814, -0.16111348569393158, -0.2517407536506653, 0.5525582432746887, 1.6375305652618408, -0.47344934940338135, -0.7383447289466858, 2.7828590869903564, 0.9607905745506287, 1.149875521659851, -0.6676532030105591, 0.9076538681983948, 1.2531640529632568, -0.04918168857693672, -1.1009987592697144, 2.6995456218719482, 0.5388391613960266, -0.7911364436149597, -0.425385057926178, 0.30193933844566345, -0.8039065599441528, -0.793915867805481, -0.563205361366272, 0.5980572700500488, 0.807390034198761, 0.3779589831829071, -0.4307738244533539, 0.9451679587364197, 0.37569665908813477, -1.1806252002716064, 0.3912506401538849, 0.18600326776504517, -0.7733114361763, -0.3619808256626129, -1.375003695487976, -0.738122820854187, -0.442141592502594, 0.06249396875500679, 0.5616331100463867, 0.5397085547447205, -0.03550932928919792, -0.49140220880508423, -0.9512001872062683, 0.04038062319159508, 0.9332349896430969, 0.7226928472518921, -1.8539106845855713, 0.30420342087745667, -0.23530147969722748, 0.4921668469905853, 0.3852613568305969, 1.408639907836914, 1.4956207275390625, 0.8304891586303711, 0.3037680983543396, -1.127558946609497, 1.4079653024673462, 0.18372821807861328, 0.9929913878440857, -0.31192439794540405, 0.7969760894775391, 1.6134625673294067, 0.5727787017822266, -0.230446919798851, 0.9267260432243347, 0.06225699931383133, -1.3927639722824097, 0.8984447717666626, -1.0428756475448608, -0.46532297134399414, 1.5505410432815552, 1.3808609247207642, -0.13525177538394928, -0.6446143984794617, -0.16094407439231873, 0.7565294504165649, -0.8172081112861633, 0.3946399390697479, -2.112943410873413, 0.7031930088996887, -1.4791827201843262, -1.2249219417572021, 0.450893372297287, 0.36718589067459106, -0.4708763659000397, -0.9141148328781128, 1.376530647277832, -1.8798034191131592, -0.15514293313026428, -0.3788355588912964, 0.0068940962664783, -0.7458012104034424, -0.23351198434829712, -0.3055916130542755, -1.0026413202285767, -0.21047015488147736, -0.03187389671802521, -0.06284041702747345, 0.7857437133789062, -0.7823168039321899, 0.6532485485076904, -0.08287299424409866, -0.3578195571899414, 1.5877374410629272, 0.7077108025550842, -1.1715797185897827, 0.2623783349990845, 0.532753050327301, 1.6886768341064453, -0.16532227396965027, -0.2846795916557312, -0.15350040793418884, 1.065163016319275, -1.1178566217422485, -1.159295678138733, -1.3137352466583252, -0.4327959418296814, 0.41833847761154175, -0.7150732278823853, 1.6961193084716797, 0.6369820833206177, -0.6197000741958618, -0.7239825129508972, -0.6306047439575195, -0.03602662682533264, -2.0486481189727783, 0.6144443154335022, 0.9322928190231323, -1.8329616785049438, -1.8874629735946655, -0.703802764415741, -0.4989798367023468, -0.6294662952423096, -1.659883737564087, -0.660914421081543, -1.415658712387085, -1.1472655534744263, 0.13700374960899353, -1.8952137231826782, 0.17358523607254028, -1.5839996337890625, -1.612566351890564, -1.3802435398101807, -0.6234017014503479, 0.7783971428871155, 0.03914880007505417, 0.9184171557426453, 2.109015464782715, -0.8942842483520508, -0.743682861328125, -0.12170983850955963, -1.9875394105911255, 0.9202451705932617, -0.43346261978149414, 0.2417117804288864, -0.480343759059906, 0.15603385865688324, -1.9312970638275146, 1.4814404249191284, -0.6682287454605103, -0.7200725078582764, 1.4024535417556763, 0.15114682912826538, 0.6754053831100464, -0.1453486829996109, 2.0540452003479004, -0.1579592525959015, -1.081985592842102, 0.11767079681158066, -0.9456418752670288, 0.11237119138240814, -0.051984094083309174, 0.6876486539840698, 0.5596652030944824, 0.48449328541755676, 1.1903351545333862, 0.6623539924621582, -0.9671978950500488, -0.17227418720722198, -0.4095202684402466, 2.900468349456787, 1.2859814167022705, 0.5896905660629272, -0.9023555517196655, 0.3317165970802307, 2.0063722133636475, 0.23996785283088684, -0.5283768177032471, 0.9840270280838013, -0.5596197843551636, 0.45515915751457214, 0.07852290570735931, -0.43938711285591125, 1.8203116655349731, -1.7563434839248657, 0.9882360696792603, 0.3382788598537445, 0.3779890835285187, -0.11652319878339767, 1.572782039642334, 0.3221452236175537, -3.6518449783325195, 0.34220850467681885, -0.3205110430717468, 0.4253382384777069, 0.4884290397167206, -0.8480269908905029, -0.5249989032745361, -0.4594961106777191, -0.05514857545495033, 0.47191664576530457, 1.2586685419082642, -0.09246214479207993, -0.6438656449317932, -0.8502578139305115, -0.2442200779914856, 0.827149510383606, -0.5952632427215576, 0.35796111822128296, -0.5555786490440369, -0.5877032279968262, 0.0826316848397255, 0.27099931240081787, 0.6350783109664917, 0.20248065888881683, -0.848400354385376, -0.7909390330314636, -2.0960264205932617, -1.3601946830749512, -0.8982388377189636, 1.0615938901901245, -0.5535134673118591, 1.0439419746398926, -0.7724981307983398, 0.4671417772769928, 0.14837969839572906, 0.04313927888870239, -0.7833214402198792, -0.0008121214923448861, 1.1993907690048218, 0.011997269466519356, -1.0199110507965088, -1.102041244506836, -0.10429972410202026, -0.6220577955245972, -0.44768160581588745, -0.43095213174819946, 1.207769751548767, 0.25999394059181213, -0.5996502637863159, -0.8143736124038696, -0.2980901002883911, 2.0554215908050537, -0.27597174048423767, -2.4413485527038574, -0.3552999794483185, 0.408149778842926, -0.7504522800445557, 0.12496326863765717, 2.022061824798584, 1.0981602668762207, 0.956015408039093, -1.269910454750061, -0.32118988037109375, -0.13009683787822723, -0.03470582515001297, 1.8733106851577759, 1.4918582439422607, -0.21308307349681854, 0.5366378426551819, -0.9550389051437378, 0.8958097100257874, -1.812831997871399, 1.0055385828018188, -1.0847126245498657, 1.7460757493972778, 1.1803934574127197, -0.2961002290248871, -0.31774240732192993, -1.13429856300354, -0.9780536890029907, -0.9967585802078247, -0.8516044616699219, 0.5573511719703674, -0.5448479056358337, 0.7501227259635925, -0.4038865864276886, 0.11421485245227814, 0.3205254375934601, 0.04881307855248451, -1.1357712745666504, 0.9193231463432312, -1.2090661525726318, 0.4503325819969177, 0.3417980968952179, -0.3393259048461914, -0.0505293607711792, 0.26700860261917114, 1.9762036800384521, 0.20757746696472168, -0.6735451817512512, 1.0493533611297607, 1.2651076316833496, 0.12217390537261963, 1.0423299074172974, 1.4743919372558594, -2.0151941776275635, 0.7997177839279175, -0.44855910539627075, -0.7305944561958313, -0.43407657742500305, -0.07905884832143784, -0.9706024527549744, -0.46559572219848633, -1.237406611442566, 0.18187308311462402, 0.2780511975288391, -0.545799732208252, -0.16135630011558533, 1.4877182245254517, -1.0386621952056885, 1.207231879234314, 1.0891324281692505, 0.482093870639801, 1.0558785200119019, -1.6286120414733887, 1.6355286836624146, -0.8697181940078735, 1.402134656906128, 1.4082553386688232, -0.383298397064209, 0.8633766770362854, 0.7577259540557861, 0.4382680654525757, 0.2228212058544159, 0.6038089394569397, -1.2693463563919067, -1.5435386896133423, 1.3522049188613892, -1.145909070968628, -1.0147552490234375, 0.5640782713890076, -0.33426907658576965, -0.2722738981246948, -0.023453380912542343, 1.5745583772659302, 0.6191698312759399, 0.13374054431915283, 2.193121910095215, 0.503546953201294, 0.19601553678512573, 0.9844714999198914, -0.25492867827415466, -1.1505602598190308, -0.7416772246360779, -0.3199114203453064, 1.662881851196289, 1.2476266622543335, -2.084928512573242, -1.3502174615859985, -0.04720265045762062, 0.13837110996246338, 0.0385623425245285, -0.44120147824287415, 1.0903213024139404, 0.27141207456588745, -0.9918729066848755, 0.010471954010426998, 0.7539780735969543, 0.8967874050140381, -0.2162122130393982, -0.8974509835243225, 1.996423363685608, -0.27065080404281616, 0.2998392581939697, -0.5647873878479004, 2.4529759883880615, 1.6182366609573364, -0.4810754656791687, 0.7805948257446289, -0.24690085649490356, 0.7191961407661438, 0.42747581005096436, 0.5032474994659424, 1.7382017374038696, -1.0437572002410889, 0.32860231399536133, -1.026257872581482, 0.6831524968147278, -1.0161707401275635, 1.8251745700836182, 0.5079380869865417, -1.9877538681030273, 0.0430205836892128, -1.6269631385803223, 0.10704457014799118, -0.24975180625915527, -1.657169222831726, -0.35501208901405334, 0.042504843324422836, 0.8279479146003723, -1.150490164756775, 1.0486565828323364, 0.08318613469600677, -0.8146984577178955, 0.22892332077026367, -0.6558173894882202, -0.18017953634262085, 0.883869469165802, 1.0568124055862427, 0.7480083107948303, -1.271851897239685, -0.5186411738395691, 0.2154070883989334, -0.06068268418312073, -0.04068725183606148, -0.6633334755897522, -1.5299115180969238, -1.0049104690551758, -2.6512067317962646, -0.21543218195438385, 0.6075392961502075, -1.085250735282898, 0.9211417436599731, 0.1284361630678177, -1.046064019203186, 0.10449135303497314, 0.35724616050720215, 0.062388814985752106, -0.5160682201385498, 0.06266902387142181, -0.35467302799224854, 0.5631104707717896, -0.8254377841949463, -0.12769097089767456, -0.6297628283500671, 0.4868045747280121, 1.2699214220046997, -1.1309584379196167, -0.5763018131256104, -1.0601236820220947, -0.12117084115743637, 0.65446937084198, 1.516654133796692, -0.4673607051372528, 0.626556396484375, 0.05291379615664482, 0.1857663094997406, -0.8654783964157104, -1.1425426006317139, 0.11046401411294937, 0.48932379484176636, -0.5228866338729858, 0.6234016418457031, -0.6976694464683533, 0.7562820911407471, 0.5093309283256531, -0.11909569799900055, -0.24698692560195923, 0.7334330081939697, 0.7610599398612976, 0.14368963241577148, -0.6451826691627502, 1.3288768529891968, 0.913184404373169, -1.4519920349121094, -0.533193051815033, 0.4970240890979767, 0.515684187412262, -1.257516622543335, -0.24846893548965454, -0.20661896467208862, 2.0315747261047363, -0.35268673300743103, 0.11103183776140213, -0.609226644039154, 0.735634446144104, -0.3038085997104645, 0.3586483299732208, -2.0245866775512695, 0.43863677978515625, 0.6177243590354919, 0.719229519367218, -1.6965911388397217, -0.26910749077796936, 0.9172617197036743, 0.08127179741859436, -0.1101205125451088, -0.2314131259918213, -0.28813138604164124, -0.12911337614059448, 0.030654899775981903, 0.3656562566757202, -0.692528486251831, -0.7158613801002502, 0.1565278023481369, 1.0608363151550293, 0.3448162376880646, 1.7046140432357788, -2.0851712226867676, -0.1293591856956482, 0.06883327662944794, -0.9759795069694519, -0.34775489568710327, 0.48653385043144226, 1.3665553331375122, -0.8970914483070374, 1.4367468357086182, 0.6250324249267578, 1.2360918521881104, 0.6250888705253601, -0.14373821020126343, 2.4405019283294678, -0.7277624011039734, 0.6092057228088379, -0.7299343943595886, 0.580409049987793, -0.3878139555454254, 0.9444729089736938, 2.331791639328003, 0.8797932863235474, 0.9685366153717041, -0.8828604221343994, -0.9636096358299255, -1.586044192314148, -0.7299932837486267, -0.8683289289474487, 0.7024204730987549, 1.2056431770324707, -0.5711462497711182, -0.16021323204040527, -1.2405173778533936, -0.2715819180011749, 0.6284934282302856, 0.8539855480194092, -0.6983948945999146, 1.08051598072052, 0.5495074987411499, 0.3316258490085602, 0.9359042644500732, -0.5126401782035828, -0.9382205009460449, 0.6523670554161072, 1.5140085220336914, 1.568246841430664, -3.5367507934570312, 0.24416594207286835, -0.4837157726287842, -1.5654648542404175, -0.02848339080810547, -0.03712569549679756, 0.7847025394439697, -2.0545802116394043, -1.2414089441299438, -0.4545206129550934, 1.6480270624160767, 1.8038997650146484, 1.0207276344299316, 1.1811596155166626, -0.32965782284736633, 0.15773308277130127, 0.3950284719467163, -0.03153672441840172, -0.7727863788604736, 0.9771885275840759, 1.388202428817749, -1.6108980178833008, -0.04683134704828262, 2.18829083442688, -0.7274758815765381, -0.6362422704696655, -0.7161933183670044, -1.6599136590957642, 1.679973840713501, 1.2376726865768433, 0.17251063883304596, 2.523258924484253, -1.5469850301742554, 0.053069107234478, 0.35357511043548584, -0.5078761577606201, 0.19655953347682953, 0.277332067489624, -0.46500203013420105, 0.9913117289543152, 1.6076527833938599, -0.6997571587562561, -0.30754563212394714, -0.8549047708511353, -2.3130438327789307, -0.7369852662086487, -0.24625074863433838, 1.3574062585830688, -0.7771835923194885, -0.7420952916145325, 1.2371704578399658, -0.5897462368011475, 0.5209718942642212, -0.6899796724319458, -2.8880465030670166, -1.0891340970993042, -1.2456117868423462, 0.2712049186229706, -0.0989936962723732, 2.9955103397369385, -1.1479955911636353, -0.485886812210083, -0.0015163037460297346, 2.724930763244629, -1.1507282257080078, 2.068493366241455, -1.1238257884979248, -0.5592308044433594, 1.6693599224090576, -0.9875428080558777, -1.1114659309387207, -0.159878671169281, 0.8160409927368164, -1.1256487369537354, 0.3799724876880646, -0.8940539956092834, -1.5892455577850342, 0.09375568479299545, 1.8592662811279297, 0.9873727560043335, -0.9801833033561707, -1.1685371398925781, 0.4110542833805084, 0.6455370187759399, -1.1601203680038452, 1.3684614896774292, -0.18424955010414124, 0.6006526947021484, 0.9944568276405334, 1.2334568500518799, 0.4122390151023865, -0.15334369242191315, -0.8496698141098022, 0.06048837676644325, 1.6718542575836182, -0.8979787230491638, -1.1242480278015137, -1.9462991952896118, 0.40292903780937195, -0.36220550537109375, -1.2633895874023438, 0.6312986016273499, 0.23361831903457642, -0.05856223404407501, -1.1102941036224365, 0.21029172837734222, -1.6030195951461792, -0.5154950022697449, 0.7124845385551453, 0.29616692662239075, -0.8198035955429077, -0.6430083513259888, -0.5863555073738098, -1.0039139986038208, -0.5156927704811096, -0.7482802271842957, -0.972662627696991, 0.11798905581235886, -1.7128119468688965, -1.2812477350234985, -1.0525466203689575, -0.7743269801139832, -1.9323127269744873, 0.9042506814002991, 1.748435139656067, 2.5594494342803955, -1.4481040239334106, -0.23128943145275116, -0.6691489815711975, -0.2723831832408905, 0.8506926894187927, -0.20209020376205444, -1.502925992012024, 0.025562429800629616, 2.1202359199523926, 0.27947965264320374, 1.8463574647903442, 0.16644014418125153, 0.6654872894287109, 2.043811321258545, -1.083339810371399, 1.8540432453155518, 1.6921427249908447, -0.20414642989635468, 1.1167733669281006, -0.08072419464588165, 0.679093599319458, 0.2677091360092163, -0.7640908360481262, 1.932965636253357, 1.839335322380066, 0.272763192653656, 1.139988899230957, 0.990871250629425, -0.07871965318918228, 1.918291687965393, 0.9047101736068726, -1.3266386985778809, -2.3816945552825928, -0.5092493295669556, 0.364678293466568, -0.009063957259058952, -0.23486317694187164, -0.843963086605072, -0.42161184549331665, 1.083521842956543, 0.8452929854393005, -0.37633177638053894, -0.6266149282455444, 0.843645453453064, -0.7994807362556458, 0.41184961795806885, 0.3371798098087311, -0.8921062350273132, -0.22528980672359467, 0.08259847015142441, -0.826298713684082, -1.5141369104385376, -0.20883603394031525, -0.33841434121131897, 0.6795595288276672, 0.0037877624854445457, 0.38132578134536743, -2.4307968616485596, -1.2987980842590332, 0.2850798964500427, 0.595710039138794, -1.5692483186721802, -0.3094683587551117, 1.285721778869629, -1.2845544815063477, -1.744677186012268, -0.24760466814041138, 1.5193989276885986, 0.23413367569446564, 0.5356814861297607, 1.0455214977264404, -0.45646727085113525, 0.014435102231800556, -0.9242308139801025, 0.22042958438396454, 0.09860453754663467, 0.23472337424755096, -0.37570086121559143, 0.6852111220359802, -0.6941906213760376, 0.5294783711433411, 1.4448920488357544, -0.450479120016098, 0.3381670415401459, -1.0489054918289185, -0.4346904456615448, 0.7238126397132874, -1.0004405975341797, -0.7010570168495178, -1.8781481981277466, -0.9248014092445374, 0.8410851955413818, 0.9691550135612488, 0.6565114259719849, -1.140475869178772, -0.2417908012866974, 0.822114109992981, 0.6172361373901367, 0.25157374143600464, 1.0114808082580566, -0.9975172281265259, 0.3730984926223755, 0.5038080215454102, -0.9893071055412292, -0.557686984539032, 1.3804560899734497, 0.5162340402603149, -1.1930652856826782, -0.13508909940719604, -0.1441875696182251, -0.20621025562286377, 0.9270473718643188, 0.9005844593048096, 1.4376156330108643, -1.064685583114624, 0.9274474382400513, -0.92522132396698, 0.9891881942749023, 0.7656145691871643, 1.67154061794281, 0.9044482111930847, -0.049040473997592926, 0.46779847145080566, 0.9272975921630859, -1.634045958518982, -0.9927528500556946, 0.24633339047431946, 0.6353446245193481, 0.7122240662574768, -1.0575774908065796, 1.2811839580535889, 0.017295939847826958, 0.5810567140579224, 0.5013132095336914, -0.5642091035842896, -0.9966331124305725, 1.9445866346359253, 0.12946176528930664, 1.7665932178497314, -1.088820457458496, -0.6470896601676941, -0.7983394265174866, 1.035443663597107, -0.07133256644010544, -0.8810502290725708, 0.8482173681259155, -0.9921136498451233, -1.6806840896606445, -0.8541999459266663, -1.0061836242675781, -1.0097520351409912, -0.6416950225830078, -1.5603713989257812, -0.1026802659034729, -0.8858567476272583, 0.2752700746059418, 0.2723906338214874, -0.00897715799510479, -0.6608016490936279, -0.3242669105529785, -0.7144652605056763, -0.2645014524459839, 0.8080707788467407, 1.0229519605636597, 1.2955725193023682, 0.10758401453495026, 1.0257281064987183, -1.663912296295166, 1.463429570198059, 0.9402675628662109, -0.8900086283683777, 0.08762884885072708, 0.2756994664669037, -0.44703003764152527, -0.6139171719551086, -0.6308621764183044, -0.09570587426424026, -0.4565677046775818, -1.0560033321380615, 0.373444139957428, 0.5299738049507141, 0.22345107793807983, 0.48372992873191833, 1.8942159414291382, 0.6306276917457581, -0.14359912276268005, -0.6924879550933838, 0.2744394540786743, 1.0096462965011597, -0.7956940531730652, 0.5086081027984619, -0.7870272994041443, -1.5979290008544922, -0.5602918863296509, 1.4551082849502563, 0.028768768534064293, -1.2396363019943237, 0.3257319927215576, 0.5362436175346375, 0.40495437383651733, -1.2182800769805908, 0.8642805218696594, -0.7343359589576721, 1.8940309286117554, -1.7056102752685547, -0.9273290634155273, -0.37300434708595276, -1.2879096269607544, -0.19287987053394318, -1.151254415512085, -1.1622287034988403, 0.4017118811607361, -1.8244798183441162, 0.7201316952705383, -0.13801945745944977, -1.7765123844146729, 0.7162402868270874, 1.3340246677398682, -0.4111005365848541, 1.559584617614746, 0.28265121579170227, 0.19673599302768707, 0.9639763832092285, -0.9870074987411499, 0.5659297704696655, -0.5811846256256104, 0.08680196106433868, 0.12255576252937317, -1.310813546180725, -0.040930815041065216, -0.7899596691131592, 0.07021845132112503, 0.010498023591935635, -1.2428511381149292, -0.9621251225471497, 1.5675143003463745, 1.6675599813461304, -0.17285796999931335, 0.33654171228408813, 0.4455892741680145, -1.1542682647705078, 0.9958256483078003, 0.0840073823928833, -0.273642361164093, 1.2068499326705933, 0.2481287270784378, -1.4824968576431274, -0.5468732714653015, -0.647239089012146, -0.9537843465805054, -0.7646051049232483, -0.7406564950942993, -0.6886059641838074, 1.6450108289718628, 0.14332732558250427, 0.8161700367927551, 0.973572850227356, 1.5778306722640991, 0.1856623888015747, 0.05604221671819687, 0.3583318591117859, -1.1993800401687622, -0.6137336492538452, 1.2466051578521729, 0.711138904094696, -0.17295685410499573, -0.5730998516082764, 0.4185279309749603, -1.7071373462677002, -1.0057158470153809, 1.7251685857772827, 0.41414618492126465, 0.18110167980194092, -0.664694607257843, -0.15798602998256683, -0.877715528011322, 0.41480305790901184, 0.34093549847602844, -0.05974028259515762, 1.560912847518921, 0.6748831868171692, -1.426287293434143, 0.7383451461791992, 1.932879090309143, 0.04575417563319206, 1.2606852054595947, 0.4199831783771515, -0.7103785276412964, -0.5512009859085083, -0.40501388907432556, -1.3553355932235718, -0.04277867078781128, -0.7823934555053711, 1.6640000343322754, 0.0865367129445076, -1.1315784454345703, -1.289035439491272, 1.2562612295150757, -0.062014833092689514, -1.2441189289093018, -0.5893151760101318, 0.21708691120147705, 0.06375880539417267, 2.0592877864837646, 0.4270263612270355, -0.4951971471309662, -1.9862221479415894, 0.5777851939201355, 0.4634423553943634, -0.0933958888053894, 0.9514537453651428, 0.09017129242420197, 0.2945552468299866, 2.2129669189453125, -1.1156723499298096, 0.9783809185028076, 1.0404696464538574, -0.5839176177978516, 2.796064615249634, -1.5474294424057007, 0.5200605988502502, -1.0185657739639282, -1.1204761266708374, 0.09555662423372269, -0.297442227602005, 0.5305417776107788, 0.01569552533328533, -0.4715377390384674, -0.3673073947429657, -0.8340559005737305, 0.2711638808250427, 1.1173981428146362, -0.16207903623580933, -0.19838778674602509, 0.15836060047149658, -1.7386316061019897, -1.8596081733703613, -0.3464709222316742, 0.9855260848999023, 1.383841872215271, 1.1108745336532593, 0.11863992363214493, -0.3956076204776764, 0.06843239814043045, -0.5414628386497498, 0.01043765526264906, 0.44356364011764526, -0.24307742714881897, -1.6646125316619873, 1.5030040740966797, 2.409996747970581, -0.8034069538116455, -1.0092679262161255, 1.2081841230392456, -1.4518648386001587, -0.957606315612793, 0.5569695234298706, 2.4612607955932617, 1.118066430091858, -1.2511417865753174, 2.1264801025390625, -0.8117713928222656, 0.6410897374153137, -1.4855866432189941, 0.5672357082366943, 0.46535035967826843, -1.3010666370391846, -1.2827726602554321, 0.9608470797538757, 0.727810263633728, -0.8432033658027649, -0.03747784346342087, 0.47655653953552246, -0.9785563945770264, -1.7011003494262695, 0.03222768381237984, 0.005584741476923227, 0.5058760643005371, -0.7881384491920471, -0.8150101900100708, -0.7450195550918579, -0.041864342987537384, 1.3089879751205444, 0.13479657471179962, -0.6599065065383911, 1.397247314453125, -0.22584134340286255, 0.9033990502357483, 1.6558339595794678, 1.8537657260894775, -0.634947657585144, -0.9179152250289917, -0.9122977256774902, 0.19870953261852264, 1.0855051279067993, -1.2179539203643799, -0.44805479049682617, -0.30520695447921753, -0.6011412739753723, 0.5135771632194519, 1.7749568223953247, -0.4277186691761017, -0.5702415704727173, -0.7720230221748352, -0.2657378613948822, 0.7246763110160828, -0.5392472147941589, 1.2492703199386597, -0.11655460298061371, -1.1024675369262695, -0.29187774658203125, 0.5129658579826355, -0.15804795920848846, -1.136824131011963, 0.3802189826965332, -0.9092440009117126, -0.49360498785972595, 0.7059529423713684, 0.7252012491226196, 0.3949006199836731, -0.7573437690734863, 1.4785457849502563, -0.39301547408103943, 2.6834254264831543, -0.7501972913742065, -0.943978488445282, -0.24130521714687347, 0.8937933444976807, -1.174442172050476, 0.258134126663208, -1.0611923933029175, -0.10890838503837585, 0.9408141374588013, 0.9190828204154968, -0.8101726770401001, 0.17539933323860168, 0.4402290880680084, -0.6108365654945374, -0.5101075768470764, -0.0609450601041317, 0.2824288308620453, 0.11721636354923248, 2.936758279800415, -0.5473152995109558, -0.23214703798294067, 1.1085457801818848, 0.4756435453891754, -2.3422250747680664, -0.18748758733272552, 1.0018876791000366, 0.016927605494856834, -0.6539515256881714, 0.1420321762561798, 1.3336753845214844, 0.39581969380378723, -0.7403610944747925, -0.9588177800178528, -1.3304970264434814, 0.35107868909835815, 1.1820299625396729, 0.5119249820709229, 1.4410769939422607, -0.35725656151771545, 1.825779914855957, -1.2949784994125366, 1.113531231880188, -0.12345916032791138, -0.20230913162231445, -0.8926509618759155, -0.4220227003097534, 0.7704566717147827, -0.7228363752365112, 0.3273560404777527, -0.6824595332145691, -1.7751786708831787, 0.6581061482429504, -0.6124827265739441, -0.05038636177778244, 1.5400443077087402, 1.5540965795516968, -0.827839195728302, 0.030187806114554405, 1.0559406280517578, -0.2453324943780899, -1.3967450857162476, 1.195875883102417, 2.1131677627563477, 0.6795874834060669, -0.43242546916007996, -1.044381856918335, -0.6882398128509521, -0.41303443908691406, -0.37347355484962463, -0.9489995837211609, -0.7065227627754211, 0.7541643381118774, -0.24744948744773865, 0.04590725526213646, 0.0836116150021553, 1.2639269828796387, 1.2896610498428345, 0.11290343850851059, -1.5026392936706543, 1.3154526948928833, 0.6696085929870605, -0.2584955096244812, 1.2120836973190308, 0.27187204360961914, -0.34383806586265564, 2.0809099674224854, 1.8732844591140747, -2.7562994956970215, 1.4087406396865845, -0.12491171807050705, 1.8642230033874512, -1.1960930824279785, 0.5483894944190979, -0.5223768949508667, -0.036508675664663315, -0.005775514990091324, 0.5632385015487671, -1.017295479774475, -0.5695107579231262, 0.23893438279628754, -0.9105603694915771, 0.13907265663146973, -1.1755530834197998, 0.036659255623817444, -1.3291689157485962, -1.7245993614196777, -0.320789635181427, 1.3862725496292114, 0.7937653660774231, 1.2911163568496704, 0.45351049304008484, -0.039580874145030975, 0.6247383952140808, -1.1190754175186157, -0.08561733365058899, -0.8479124307632446, 1.2600891590118408, 0.3777611255645752, -0.5516225695610046, 0.0006956930155865848, 0.9029250144958496, -0.08403051644563675, 0.16402693092823029, 0.5257033109664917, -0.058178503066301346, 2.353140354156494, 0.40430837869644165, -0.6508431434631348, -0.5761693716049194, 2.4679534435272217, 0.21923433244228363, -1.0489031076431274, -1.751774549484253, 0.6681013703346252, 2.620075225830078, 0.6766191720962524, -0.21402117609977722, 1.2501310110092163, -0.2653684914112091, -0.27702364325523376, -0.20224174857139587, -0.5196956396102905, -1.3691210746765137, 0.23029781877994537, 2.192519187927246, -1.4159573316574097, -1.076983094215393, 0.5480534434318542, -0.2649409770965576, -0.0751633420586586, 0.3155289888381958, 0.6811484694480896, 0.2145485281944275, 0.6929923892021179, 0.803051233291626, -0.5196067094802856, 0.939633846282959, 0.11645569652318954, -1.0605665445327759, 1.6205720901489258, -0.5971072316169739, 0.7212260365486145, -0.5321394205093384, 1.1349875926971436, -1.472944974899292, -0.3448421061038971, -0.3623587489128113, -0.6011108756065369, 2.2101221084594727, -1.517700433731079, -0.751102089881897, -0.37072688341140747, 1.3837536573410034, 0.0659257173538208, -0.8823648691177368, 0.23675474524497986, -0.6420255303382874, -0.45657527446746826, 2.105245351791382, -0.704488217830658, -0.08116213232278824, 0.21421130001544952, -1.9144103527069092, -0.06896188855171204, -1.4260770082473755, -2.3698878288269043, 0.38611459732055664, -0.1317918300628662, 0.5490804314613342, -1.9877885580062866, 1.7500159740447998, 1.3817569017410278, -0.999082088470459, 0.36750754714012146, -0.3982219398021698, 0.6931172609329224, 0.260563462972641, 0.12983135879039764, 0.18368308246135712, -0.7153245210647583, -0.9403003454208374, -0.4300597310066223, 0.6187267899513245, -0.29886865615844727, 0.11865159869194031, -1.014106035232544, 0.09670655429363251, 0.6557224988937378, 0.03943448141217232, -0.9475647211074829, 1.6035356521606445, 0.6411387324333191, -0.82625412940979, 0.9816039800643921, -0.22776472568511963, 0.6090492010116577, -0.1569603830575943, -1.5935333967208862, -0.3690710961818695, 2.325486898422241, 1.3659584522247314, 0.24371017515659332, -0.4772581160068512, -1.1016814708709717, -0.4345988929271698, 0.7941039800643921, 1.2432888746261597, 1.883293867111206, -1.3906919956207275, -0.7081348896026611, 0.5212379693984985, -0.678414523601532, 0.8168045878410339, -0.6868788599967957, 0.7033562660217285, -0.571514904499054, 1.538068175315857, -0.09274876117706299, -0.5594701170921326, 0.6836684346199036, -0.4839741885662079, 3.506115198135376, 0.5331635475158691, 0.003804673906415701, -1.7211037874221802, 1.0444358587265015, 1.1213208436965942, -0.9318666458129883, -0.739163339138031, 0.867568850517273, 1.5371311902999878, -1.4869543313980103, -1.0934195518493652, 0.6092524528503418, 1.7772270441055298, -1.268657922744751, 0.16481398046016693, 1.8029900789260864, -0.1634330004453659, 1.1119146347045898, 0.9736723303794861, 0.2681760787963867, -1.1420068740844727, 0.6932086944580078, 0.3924880921840668, 1.1720914840698242, -1.1775422096252441, 0.8293356895446777, 2.503267288208008, -0.014920014888048172, -1.414707064628601, 0.3266642987728119, -1.5407003164291382, 0.07727031409740448, 0.652540922164917, 0.47946977615356445, 1.4129515886306763, -0.40761077404022217, 0.37516871094703674, 0.72648024559021, -0.158040851354599, -0.30445149540901184, 0.5266232490539551, 0.22633883357048035, -0.1319170445203781, -0.2589973509311676, 0.037454746663570404, 0.20358014106750488, 0.6056633591651917, -1.8014932870864868, -0.6097589731216431, -0.4229277968406677, 1.1128885746002197, 0.9256511330604553, -0.5935396552085876, -0.7311745882034302, -0.2962592542171478, 0.20730911195278168, -0.9386100172996521, -1.0321192741394043, -0.3838776648044586, 0.44164496660232544, 1.2641745805740356, 0.9089990258216858, -0.9965055584907532, -1.4432796239852905, 1.372743844985962, -1.6007814407348633, -0.36880210041999817, -0.6795968413352966, 0.8551036715507507, 0.20729795098304749, 0.13611796498298645, 0.33128267526626587, -1.1011420488357544, -0.029130086302757263, -0.31350475549697876, 1.1936637163162231, -0.8699985146522522, -0.6352912783622742, 3.2895023822784424, 0.09617523103952408, 1.0097161531448364, 0.00614504050463438, -0.20481917262077332, -0.21272413432598114, -1.4987398386001587, -1.4787555932998657, -0.9117355942726135, -0.36706802248954773, 0.098939910531044, -0.8615943193435669, -0.35254067182540894, -0.8244408965110779, 0.40809983015060425, -0.8056463599205017, -0.046492788940668106, -0.024081170558929443, -1.466741919517517, -0.12957961857318878, -1.4720591306686401, 1.9908676147460938, -0.6262732148170471, -0.10863503813743591, -1.5806852579116821, 0.13596515357494354, 1.201399326324463, 0.07981152832508087, -0.3421527147293091, 0.508978009223938, 0.38370105624198914, 1.1272873878479004, -0.975843071937561, 1.3029588460922241, 0.18328848481178284, 1.3256253004074097, 0.41444510221481323, -0.04546251520514488, 1.4431017637252808, -0.7003583312034607, -0.1388346254825592, -0.9141151905059814, 1.0861778259277344, 1.0430015325546265, 1.7342805862426758, -1.0904853343963623, 0.30336281657218933, -0.8889000415802002, 0.7469557523727417, -0.44346949458122253, -1.4929687976837158, 2.134972095489502, 1.0956220626831055, 1.867072343826294, 1.5939654111862183, 0.85914146900177, 0.22635981440544128, 0.7865124344825745, 0.17356941103935242, 0.0692000538110733, -2.061284065246582, 0.6166687607765198, 0.5059787631034851, -0.655001163482666, 0.5826405882835388, 0.176731139421463, -1.119398593902588, 0.8488093018531799, -0.820732057094574, 1.379767656326294, 1.1129403114318848, 0.4506869912147522, -0.7006354331970215, 0.20384642481803894, 0.43939507007598877, -0.833432674407959, -0.10160297900438309, -0.5118871927261353, -0.828948438167572, 1.5543098449707031, 0.12234214693307877, -0.19267314672470093, -0.6823992729187012, 1.4146100282669067, -1.017913818359375, 1.7760906219482422, 1.4039157629013062, -1.9848958253860474, -0.547568678855896, 1.880827784538269, 0.18423044681549072, -0.6133361458778381, 0.7618906497955322, -0.2716769278049469, 0.46013614535331726, -0.6086834073066711, 0.9442059993743896, 1.0154448747634888, 0.7895936965942383, -0.8518662452697754, -0.5524824261665344, -0.6402040719985962, 2.7099878787994385, -1.0815831422805786, 0.05545007064938545, -1.5673655271530151, 1.3328911066055298, -1.36276376247406, -0.36236417293548584, 0.24830302596092224, -0.5447466373443604, 1.8159211874008179, 0.22508420050144196, 1.122940182685852, 0.16504159569740295, 0.2709173262119293, 1.1698530912399292, 0.3955039083957672, -0.5762081146240234, -0.9458226561546326, 1.7086100578308105, 0.33973273634910583, 0.3781549036502838, 0.6381163001060486, 1.5183593034744263, 0.3309737741947174, -1.0158990621566772, -1.2875256538391113, 0.013618066906929016, -1.5432740449905396, 1.0273045301437378, 1.4675438404083252, -2.0890259742736816, -0.48912331461906433, -1.0056629180908203, 2.5277411937713623, -0.713499128818512, 1.41917884349823, 1.9542230367660522, -0.012767712585628033, 0.26624196767807007, 1.3117445707321167, -2.542872190475464, -0.49348312616348267, -0.3590090572834015, -1.8466581106185913, -1.3710774183273315, 0.050396762788295746, -0.1961432546377182, -2.429131031036377, 0.36137229204177856, 0.7007247805595398, 0.525566041469574, -0.5462015271186829, -1.705047369003296, -0.6798537969589233, -0.42307165265083313, -0.391167551279068, -1.0625618696212769, -0.3485499918460846, -1.1506344079971313, -1.438439130783081, 0.8360992074012756, -0.5881630182266235, -1.2413355112075806, 0.4466787874698639, -0.08441387116909027, -0.913279116153717, -0.3194009065628052, 0.9229016900062561, -1.116990327835083, 0.2926354706287384, 1.257426381111145, 0.44441983103752136, 1.0583302974700928, -1.4639798402786255, -0.22680556774139404, 2.057893991470337, 0.3699129521846771, 1.435631275177002, -0.1785230189561844, 0.4689123034477234, 0.7938748598098755, 1.3722283840179443, -0.01062766369432211, 1.2100805044174194, -0.8585001230239868, -1.3248189687728882, -1.0725483894348145, 0.07843223214149475, -0.2934703826904297, -1.1070724725723267, -0.20904168486595154, 0.18972207605838776, 0.9239144325256348, 0.34253451228141785, -1.3786563873291016, -0.4479069709777832, 1.1133464574813843, 1.2321217060089111, -0.9592640399932861, 0.01801794394850731, -0.6901292204856873, -0.6774841547012329, -0.42447060346603394, 0.2910897135734558, 0.08754652738571167, -0.5489351749420166, -1.0997307300567627, -0.7742263674736023, 2.5595510005950928, 0.34159356355667114, 2.2094576358795166, 1.5934165716171265, 0.864742636680603, 0.19887107610702515, -1.483976125717163, 0.2887224853038788, 1.9366835355758667, -1.2259210348129272, -0.4304855167865753, 1.3072830438613892, 0.01539615262299776, 1.0884929895401, -0.05543563514947891, 0.35971689224243164, -0.23990188539028168, -0.3466477394104004, -0.13473708927631378, 0.5629335641860962, -0.08107862621545792, -2.0858547687530518, 0.049939680844545364, 1.4901032447814941, -0.26185914874076843, -1.391861081123352, -0.29700034856796265, -1.4736754894256592, 1.1813043355941772, 0.1907128244638443, 0.2574215531349182, -0.33540958166122437, 0.5428804755210876, -0.04704377055168152, 1.2886909246444702, 0.6749991178512573, -1.4545007944107056, -1.0920079946517944, -0.232520192861557, 0.5086432099342346, -0.3160690665245056, -1.156107783317566, -0.8982300758361816, -0.30236515402793884, 1.398597002029419, -1.0044711828231812, -0.6742178201675415, 0.3051773011684418, 0.10867467522621155, 0.35020628571510315, 0.38794514536857605, 0.5961377620697021, 1.074716329574585, -0.32618945837020874, 1.458117961883545, -0.04166603088378906, -0.41022032499313354, -0.4560752809047699, 0.7828580141067505, 1.1069937944412231, -0.7540169954299927, 0.7795153260231018, 0.45502519607543945, -0.795555591583252, 0.040308594703674316, 0.7580757141113281, 1.7182834148406982, 0.343365877866745, -0.3120296895503998, -0.44405779242515564, -0.554999053478241, 1.0694465637207031, -0.7027478814125061, -0.19690026342868805, -0.6291764974594116, 0.2881583273410797, 0.7006592154502869, 1.2202669382095337, 1.5566136837005615, -0.1793001890182495, -0.702651858329773, 1.0918296575546265, 0.512694239616394, 0.19607298076152802, -0.06985591351985931, 0.45170238614082336, -0.8868038058280945, -0.4790067970752716, -0.04045544192194939, 0.4294811189174652, 0.581886887550354, -0.34865036606788635, 0.15116235613822937, 0.251848965883255, 0.9104933142662048, 0.6495924592018127, -0.6939979791641235, 0.7463437914848328, 2.214308500289917, -1.450210690498352, 0.6734189987182617, -0.3320496678352356, -1.3621366024017334, 0.3973310887813568, 1.1463677883148193, -1.1517666578292847, 0.1414720118045807, 1.1846977472305298, -0.4591203033924103, 0.4339296221733093, 0.16235068440437317, -1.4433118104934692, 0.9192861914634705, -0.4252298176288605, 0.9335656762123108, -1.3096686601638794, -1.3720089197158813, -0.6082119345664978, 1.875728726387024, 0.41937434673309326, 0.5659946799278259, -0.5476245284080505, -1.5373789072036743, -1.8738003969192505, -1.0028116703033447, 0.2647077739238739, 1.7946735620498657, -1.6198756694793701, -2.0702688694000244, 0.8496018648147583, 1.1468586921691895, 0.20313987135887146, -1.075600266456604, 1.1634544134140015, -0.9396464824676514, 2.5249953269958496, -0.2310248613357544, -0.7809872627258301, -0.6712377071380615, 2.5521743297576904, 0.640145480632782, 0.4615764617919922, -1.1167634725570679, 0.29266685247421265, -0.1393326073884964, 1.4176980257034302, -0.8030950427055359, 1.489815354347229, -0.9481914043426514, -0.8970127701759338, 0.18089377880096436, 0.15504395961761475, -1.702558994293213, -1.4731968641281128, -0.9732030034065247, 0.6805655360221863, 1.071445345878601, 0.8194394111633301, 0.39394018054008484, 0.637917697429657, -0.6139837503433228, 0.1904846876859665, -0.3025640547275543, -0.4864412844181061, 1.9227827787399292, 0.016698040068149567, 1.1466126441955566, -1.0500547885894775, -0.3056796193122864, -0.6404657959938049, -0.425436794757843, -0.18996457755565643, 0.34238773584365845, 0.08924593776464462, -0.5352374911308289, 1.610087513923645, -0.7339664697647095, 1.0734467506408691, -0.5387275815010071, -1.0232229232788086, 1.301312804222107, -1.3671008348464966, -0.9443212747573853, 0.562061071395874, -0.3198715150356293, -1.25115966796875, 0.10474646836519241, 1.3995327949523926, 0.588312566280365, 0.24584200978279114, 0.5633948445320129, -0.5222573280334473, 0.9357953667640686, -0.03383545204997063, -1.4725615978240967, -0.8300255537033081, 0.04346414655447006, 0.09605979174375534, -0.10831271857023239, -1.0272785425186157, 0.2693692147731781, -0.9330117702484131, -1.8520616292953491, -0.5845918655395508, 1.321129560470581, 1.0728341341018677, -0.04261666163802147, 0.9979974627494812, 0.574230432510376, -0.6686820387840271, 0.8008933663368225, -0.1386842131614685, -0.22431284189224243, 0.6672856211662292, -0.8392317295074463, 0.4162485599517822, 1.3891047239303589, 1.2415136098861694, -0.020227884873747826, 1.1674162149429321, -1.3102375268936157, 0.657924234867096, -0.5315548777580261, 1.2836781740188599, 0.04890110343694687, 0.2846449613571167, 0.3957344591617584, -0.9817530512809753, -2.888334035873413, -0.755208432674408, -0.0025603624526411295, 1.5394593477249146, 0.013722959905862808, 0.7135823965072632, -0.014722361229360104, 0.15147653222084045, 0.7671987414360046, -0.17785872519016266, 1.0586323738098145, 2.009891986846924, 1.3938673734664917, -0.7727487683296204, 0.9979193806648254, 0.8960654735565186, 1.206149697303772, -0.07555881887674332, 1.350879430770874, 0.4550231695175171, -0.2610028386116028, 1.128585696220398, -0.0702224150300026, -1.3606723546981812, 0.1698692888021469, -1.2287483215332031, -0.5888563394546509, 0.22142496705055237, -2.0267627239227295, -0.6015469431877136, -0.355290949344635, -0.9277233481407166, -0.9321287274360657, -1.6036407947540283, -0.7964826822280884, -0.8381372690200806, 0.5013593435287476, -0.0889144316315651, 1.1643424034118652, 0.7921421527862549, 0.4361116290092468, 0.7434359788894653, -0.5069146752357483, 1.8041900396347046, 1.047780156135559, -0.08036424219608307, -0.06044945865869522, -0.059291549026966095, -0.9677571654319763, 0.9436582326889038, -0.9471096396446228, 0.5461894869804382, 0.3600673973560333, -0.14991843700408936, 0.5783041715621948, 0.7821505665779114, -1.207737922668457, -1.9751346111297607, -0.44330623745918274, 0.7064408659934998, 0.3689936101436615, 0.4519987106323242, 0.3983112871646881, 1.5206221342086792, -0.9167868494987488, -0.11246553808450699, 2.4987967014312744, -1.419619083404541, -1.439673662185669, -0.6336644291877747, 0.0280892476439476, 0.3578948974609375, -0.12253470718860626, 0.7881909608840942, 1.5781539678573608, 0.07409827411174774, -1.1692293882369995, -0.32504889369010925, -0.13391119241714478, 0.4733155369758606, -0.9472783803939819, 1.329697608947754, 0.2583334147930145, 1.417024850845337, -0.5288952589035034, -0.5169742703437805, 2.7700355052948, -2.033677101135254, -0.330036461353302, -0.21914835274219513, -0.6251352429389954, -1.0752325057983398, 1.0014139413833618, -0.42603906989097595, -0.6401760578155518, 0.2556679844856262, -0.23905429244041443, -1.1728013753890991, -1.4299665689468384, 0.06950365751981735, 0.8120620846748352, 0.038787033408880234, 0.6280543208122253, 0.6359273791313171, -0.9289060235023499, 0.8548980951309204, 0.22516992688179016, 2.226271152496338, 0.6387186050415039, -0.6932483911514282, 0.0626356303691864, -2.1060667037963867, -0.5855450630187988, 0.37642529606819153, 0.4549987316131592, 0.8220441341400146, -0.4176500737667084, -0.894442617893219, -0.6390635967254639, -1.1127086877822876, 1.559025764465332, -1.0624313354492188, -1.2696025371551514, 1.886849284172058, -0.41671401262283325, 0.6062943935394287, 0.3378629684448242, -0.6005632877349854, 0.051506832242012024, 1.2843437194824219, -0.4414013624191284, -1.9701560735702515, -2.348335027694702, -1.2497445344924927, -0.30102190375328064, 0.7527587413787842, 1.0004489421844482, 1.2233021259307861, 1.1226329803466797, 1.749029278755188, -0.1559094488620758, 0.2001800686120987, -2.3928310871124268, -0.038285400718450546, 1.5720880031585693, -0.25077396631240845, 0.7174420952796936, -0.11560932546854019, 0.30828866362571716, -0.6549186110496521, 0.36342334747314453, -0.9245873093605042, 0.28034141659736633, -0.8446357846260071, -0.6065889000892639, 0.9843772053718567, -0.9218365550041199, 0.5391356945037842, 0.8198424577713013, -0.9163621664047241, -1.051850438117981, -0.9259433150291443, 0.7457025051116943, -0.18449178338050842, 0.261726051568985, 1.673288345336914, -0.45387452840805054, 1.0863796472549438, -0.07134819775819778, -0.2681407928466797, -1.3152114152908325, -0.8504343032836914, 2.3882009983062744, 0.23946821689605713, -0.8283657431602478, 1.3063011169433594, 0.6679354310035706, 0.8107677698135376, 0.8587501645088196, -1.8561965227127075, -1.7199857234954834, -0.29996657371520996, 0.2524610459804535, -0.019781136885285378, 0.8203749656677246, -1.2517753839492798, 1.0478432178497314, -2.185358762741089, 0.09760542958974838, 0.533918559551239, 0.567483127117157, 0.6458538174629211, 0.10296420753002167, 0.024368418380618095, -1.3444478511810303, 0.05655045434832573, -0.2615676820278168, 0.2725834548473358, -0.6978421807289124, -1.443569540977478, 1.2898499965667725, -1.9091005325317383, -0.3699147403240204, 1.3758126497268677, 1.4322824478149414, -1.807528018951416, 0.750200629234314, -0.6365975141525269, 0.3699677288532257, 0.6453078389167786, 0.873731791973114, -1.949281096458435, -0.9166214466094971, 0.6258437037467957, 0.6033297777175903, 1.6677427291870117, 0.8078658580780029, 1.2237193584442139, 1.128374695777893, 0.7716090083122253, -1.413130521774292, 1.9174449443817139, 0.3751521408557892, -0.4880635738372803, -0.26756736636161804, -0.6770778298377991, -0.03939466550946236, -1.7488455772399902, 0.9101247191429138, -1.0750771760940552, 1.0190784931182861, -0.5961913466453552, 0.6326571702957153, 1.3447954654693604, 0.4431767761707306, -0.004686305765062571, 0.1603747308254242, -0.25913333892822266, -0.2500024437904358, 2.0348641872406006, -1.1824864149093628, -0.8133600950241089, -0.4963669180870056, -0.06426604092121124, 0.6218498945236206, -0.13867473602294922, 1.0276765823364258, -1.0874618291854858, 0.3590082824230194, 0.19314375519752502, -1.0609867572784424, -0.13595902919769287, -0.38342148065567017, -0.7087506651878357, -0.8616306185722351, 0.8797003626823425, -0.8314269185066223, 0.9069012403488159, 1.0239306688308716, 0.5824205875396729, 0.6068452000617981, -1.5714576244354248, 0.13480238616466522, 0.7072891592979431, 0.8230388760566711, 1.2952220439910889, 0.7208809852600098, -0.09737975895404816, -0.5446614027023315, -0.7292628884315491, 0.048702824860811234, -0.2382012903690338, 1.3802729845046997, -0.5265732407569885, -0.9647971391677856, -0.6687635779380798, 2.3563385009765625, 0.11777795106172562, -1.2619754076004028, 1.4961624145507812, -0.26560547947883606, 0.20721587538719177, -0.6919222474098206, 0.652760922908783, 0.7630888223648071, -0.7913463115692139, 0.06050426512956619, -1.39030921459198, 1.4037553071975708, -0.9820578098297119, -0.8865704536437988, -0.9962871670722961, -1.0784451961517334, 0.1327226608991623, -0.6368729472160339, 0.6350025534629822, 0.4973960220813751, 1.1565189361572266, 1.3919037580490112, -0.7032210826873779, 0.06383143365383148, -0.931801974773407, -0.22960636019706726, 0.6441109776496887, 0.3948991000652313, -0.6743289828300476, 0.47970035672187805, 0.0526345819234848, 0.5680440664291382, -2.421506881713867, -0.9434317350387573, 1.005179524421692, -1.6100029945373535, -0.452265202999115, -0.2842598259449005, -0.7604522705078125, -0.38941147923469543, 0.14827999472618103, -0.18513359129428864, -0.5806896686553955, 0.556186318397522, 0.8508696556091309, 0.8295640349388123, -1.9394482374191284, 1.3632736206054688, 0.41503292322158813, -0.19868065416812897, 1.020389437675476, 0.8304176330566406, 0.8951219916343689, -0.777593731880188, -1.7143545150756836, 1.5267441272735596, 0.5768613219261169, -0.4503306746482849, -0.4258701205253601, -0.1974683552980423, 0.8079025745391846, -1.22199285030365, -0.8563238382339478, 2.4190754890441895, -1.3039499521255493, 0.5009877681732178, 0.23017145693302155, 0.15076526999473572, 1.296686053276062, -0.38385140895843506, 1.2127106189727783, 0.044750768691301346, -0.6527836918830872, -0.7355043292045593, -0.5309150218963623, 0.9441918730735779, 1.0740773677825928, 0.09287507832050323, -0.7829422354698181, -0.9059673547744751, -0.16792307794094086, 1.3812329769134521, 0.7675371170043945, -0.3057364225387573, 1.5012964010238647, -1.613687515258789, -2.0499746799468994, -1.5443974733352661, 0.9040413498878479, -1.1538856029510498, -0.5085256695747375, 0.0026104908902198076, -0.5612697005271912, 0.8516573309898376, 0.37691396474838257, -0.25648966431617737, 0.7769456505775452, -2.0145034790039062, 1.1824088096618652, -0.5216083526611328, -0.9361113905906677, -0.010008575394749641, 0.5792025327682495, 0.020904913544654846, 0.3922869563102722, -0.20806817710399628, -0.6321131587028503, 1.5720707178115845, 1.0710529088974, -1.4577512741088867, 0.5546501874923706, -0.8498597741127014, -0.8075747489929199, 0.40600308775901794, 0.9808505773544312, 1.5923848152160645, 1.0851777791976929, -0.5353401899337769, -0.5761337876319885, -0.7427382469177246, -0.31643879413604736, 1.6235020160675049, 0.3715224266052246, 0.246229887008667, -3.174307107925415, 0.29268962144851685, -0.6954219937324524, -0.23719163239002228, 0.36714160442352295, 0.7084499597549438, -0.6782594919204712, -0.793034553527832, 0.09421011060476303, -0.6777404546737671, 1.5014313459396362, -0.44366753101348877, -1.7598484754562378, -0.028692232444882393, -0.6250943541526794, -0.1600687950849533, 0.020316462963819504, 1.1008994579315186, 1.2239471673965454, -0.414341002702713, -1.8282039165496826, -2.1543970108032227, 1.785525918006897, -0.14187443256378174, 0.5879095792770386, -0.46032291650772095, 0.17608729004859924, 0.7237237095832825, -0.48287639021873474, -0.1339513659477234, 0.4740351140499115, -1.2251001596450806, -0.9602874517440796, 0.5196647047996521, -0.4706956148147583, -0.2582699954509735, -1.5966445207595825, 0.3959295451641083, 0.3661576509475708, 1.9547468423843384, -0.05534158647060394, -0.24083152413368225, 0.8280977606773376, 0.047208599746227264, 1.5650429725646973, 0.032938312739133835, -0.09795787185430527, 0.826858401298523, 1.255361557006836, 0.5483384132385254, 1.3822224140167236, -1.2251803874969482, 0.07856377959251404, 0.5065398812294006], "dtype": "float32"}], "e": {"__nd__": [-1.9256194829940796, -0.3133500814437866, -0.9063886404037476, -1.0235490798950195, -0.3279012441635132, 1.6154887676239014, -1.5985482931137085, 2.306389570236206, 0.4174481928348541, 0.23278552293777466, -0.9386149048805237, -0.8158112168312073, -2.5621089935302734, -0.03565859794616699, -1.13487708568573, 0.7814216613769531, 3.995964527130127, -0.1015482246875763, -2.216660976409912, -0.2147187739610672, 1.4602857828140259, 1.7314245700836182, -0.8934875130653381, 1.7250301837921143, 0.0046049803495407104, -1.0501009225845337, 0.9811862707138062, 2.1214516162872314, -0.6832679510116577, -1.410244345664978, 2.926302433013916, 0.5689587593078613, 1.517451524734497, 0.023598909378051758, 0.9507980942726135, 2.4162862300872803, 1.558826208114624, -2.1247379779815674, -0.30282771587371826, -1.2117663621902466, -0.36362648010253906, 3.498382329940796, -0.5191181898117065, -0.5295535922050476, -0.9609173536300659, -0.02180349826812744, -0.10750743746757507, -1.6903353929519653, -0.3373667001724243, -0.8865609169006348, 0.115928053855896, 0.9461805820465088, 2.177489995956421, -2.4328932762145996, 0.25019705295562744, -1.2580132484436035, 0.11368191242218018, -0.5219974517822266, -1.2772759199142456, -1.7207106351852417, -1.5685253143310547, -0.1338014006614685, 0.019953563809394836, 2.088566303253174, -3.1759486198425293, -0.00872465968132019, -0.8236255645751953, 1.3182251453399658, 2.652702808380127, 0.7629023194313049, 0.9294232130050659, -1.131986141204834, -0.19589461386203766, 0.02853904664516449, 1.744413137435913, -0.9468308687210083, 1.1680514812469482, -0.28178495168685913, -0.1496991366147995, 1.1903722286224365, -0.49125030636787415, -1.0261378288269043, -0.10207235813140869, -1.028992772102356, -1.0193862915039062, 2.8085663318634033, -0.3759923279285431, 1.2812378406524658, -0.27463415265083313, 1.9110679626464844, 1.05287766456604, 1.30107843875885, 0.455524206161499, -2.3118486404418945, 1.4523727893829346, 0.8071635961532593, -0.8440679311752319, -0.8489208817481995, -0.08488829433917999, 1.2651021480560303, -0.8162925243377686, 1.1933479309082031, 1.2077068090438843, -0.8424821496009827, 0.9400904178619385, -0.43895310163497925, -1.0714879035949707, 1.2507894039154053, 0.4791508913040161, -0.624466061592102, 0.12287110090255737, 1.401538372039795, -1.5115715265274048, -0.42641696333885193, 0.29899731278419495, -1.6401458978652954, -0.08907940983772278, -0.9750814437866211, -0.8610775470733643, -1.6316745281219482, 2.1127960681915283, 0.5303539037704468, 0.2992648184299469, 0.9632653594017029, 0.48966702818870544, -0.727212131023407, 0.11139573156833649, -0.6656595468521118, 0.9231398105621338, 0.7685640454292297, 2.0035455226898193, 1.328171730041504, 0.9463311433792114, 0.9300373792648315, -1.704331874847412, 1.660518765449524, 1.0505203008651733, -0.5219175815582275, -1.1884040832519531, 1.0862475633621216, 0.028805900365114212, 1.3604762554168701, -0.3600829839706421, -0.42212408781051636, -1.0127792358398438, -0.6562045216560364, 1.1189649105072021, -0.6167256832122803, -2.6991610527038574, -0.3702777624130249, -0.08828554302453995, 2.849806070327759, -0.30964216589927673, -0.932719886302948, -2.6683502197265625, 0.5406453013420105, -0.3403049111366272, 0.24654965102672577, -0.6623022556304932, -0.9058557152748108, -0.19501395523548126, 0.888070821762085, 0.08070854097604752, 0.3597921133041382, -2.896106243133545, -1.0542370080947876, 1.6507658958435059, -0.812403678894043, 0.6697452664375305, 0.968274712562561, 4.326126575469971, -0.9292359948158264, -1.4054664373397827, 0.8005698919296265, -1.56732177734375, -0.5766985416412354, 2.2680251598358154, -0.5810831785202026, 0.1506454050540924, -0.9008650779724121, 1.20444917678833, -0.699839174747467, 0.6480466723442078, -2.857374906539917, 0.9906676411628723, -1.3117084503173828, 0.09543778002262115, -2.0151917934417725, 0.2810043692588806, 2.799213409423828, -2.320481061935425, 1.8217424154281616, 0.18244391679763794, -0.7428003549575806, 0.39558979868888855, 1.0044020414352417, 1.9980461597442627, 1.4613226652145386, 0.9970616102218628, 2.618565082550049, 0.589508593082428, -2.291802167892456, 1.414650797843933, 0.24923989176750183, 0.05182051658630371, -0.4188230037689209, -2.3000082969665527, 0.6778322458267212, 1.8402035236358643, 0.9318622946739197, -1.3329352140426636, 0.6938918828964233, -2.30148983001709, -0.9882053136825562, -0.8717638254165649, 2.3010478019714355, 0.11679510027170181, -0.17124159634113312, -0.7760779857635498, -0.0791928768157959, 0.12219232320785522, 0.5903759002685547, -0.6978108882904053, -1.4991717338562012, -2.158897876739502, -0.7228727340698242, 0.8392602205276489, -0.4105836749076843, -0.7552350759506226, -1.9332621097564697, -0.4336146116256714, -1.8060710430145264, -0.6778350472450256, -0.5215020775794983, -0.4241703152656555, 0.014690777286887169, -1.33809232711792, -0.19709134101867676, -1.1132060289382935, 1.0321348905563354, -0.582040011882782, 2.289045810699463, -0.5086988210678101, 0.014357391744852066, -1.0376958847045898, 0.6289834976196289, -0.6964260339736938, -1.52311372756958, -1.1901824474334717, 0.2024003267288208, -1.9158105850219727, 0.3972594439983368, -1.9231274127960205, -0.07412590831518173, -0.2919142544269562, 0.9229940176010132, 0.45458152890205383, 1.1385040283203125, -0.8665688037872314, -2.0415282249450684, 0.04404759407043457, -3.06878924369812, -1.07875657081604, -1.4339443445205688, 0.8180513978004456, 0.8921773433685303, 0.21155110001564026, 0.3573024570941925, 0.2063877135515213, 2.7137937545776367, -1.6241862773895264, -0.7360676527023315, 0.38671231269836426, -0.03790801763534546, 1.1925129890441895, 0.3393310308456421, 1.1206231117248535, -1.7641657590866089, 0.9359846711158752, 0.9383244514465332, -0.3684100806713104, 1.1573030948638916, -1.6592957973480225, -0.7237923741340637, 1.68868088722229, -1.7421989440917969, -0.09194771200418472, -0.16139549016952515, 0.31860166788101196, -0.7531467080116272, -2.0989065170288086, 0.6132873296737671, -1.1145743131637573, 2.3895020484924316, -0.509931206703186, -2.923088550567627, 0.3953605890274048, 0.787427544593811, 0.41362708806991577, -0.7077163457870483, 0.011096358299255371, -0.031038299202919006, 0.38654205203056335, 1.1572662591934204, -1.5152631998062134, -0.8524528741836548, -0.3685097098350525, 0.0959027037024498, -0.8275104761123657, -2.7927074432373047, -0.43171432614326477, -0.03323546051979065, -0.7723814249038696, -2.1007494926452637, 0.422669917345047, -0.25159400701522827, -0.4413573741912842, 0.7094680070877075, 0.3475711941719055, 2.2290754318237305, -0.7137444019317627, 0.7235254645347595, -1.0733414888381958, -1.5701489448547363, -2.245229959487915, -0.7572984099388123, -0.30622997879981995, 1.0158408880233765, -0.8155112862586975, -3.996985673904419, 1.5219342708587646, -2.1975667476654053, -2.977468729019165, 0.2683960795402527, 0.6752526760101318, -1.4566631317138672, -0.7153140306472778, 0.5723156929016113, -0.24592947959899902, 2.53393816947937, 0.9501991271972656, 0.4976426064968109, -0.8334314823150635, 0.61836838722229, -1.104418158531189, 0.11908945441246033, 1.141711711883545, 0.7092270851135254, -0.03906899690628052, 0.2623746395111084, 0.6502817273139954, 0.6200513243675232, -0.6004790663719177, -2.976968765258789, -2.0303170680999756, 0.7236359119415283, -2.4076101779937744, -0.030900713056325912, 0.2033337652683258, 1.2812466621398926, 1.5693233013153076, -1.1815458536148071, -0.42500704526901245, 0.42938944697380066, 0.4263416528701782, 0.36964893341064453, 2.7186667919158936, 0.19369569420814514, -2.2340567111968994, 0.5484355688095093, 0.7379104495048523, 2.0102264881134033, -1.392005443572998, -0.7355644702911377, 2.237013816833496, -0.10004192590713501, 0.8329485654830933, -0.3334548771381378, -0.80296391248703, -0.878612756729126, -1.7541428804397583, 0.31708335876464844, -0.18652266263961792, -0.6068933010101318, 0.4457401633262634, -2.9193010330200195, 3.359193801879883, -3.738112688064575, 0.3530605137348175, -1.5399036407470703, -1.0538944005966187, 1.935194969177246, -1.1600065231323242, 1.6571624279022217, 0.1353524625301361, -1.8170127868652344, 1.4960299730300903, 0.517837405204773, -0.32698720693588257, 4.0870361328125, -1.0914902687072754, 1.7840203046798706, -0.32975074648857117, -2.5653574466705322, -1.637068748474121, 2.1605334281921387, -3.146355152130127, -0.991026759147644, -1.6457703113555908, -0.8246225118637085, -0.2078360617160797, 0.6744349002838135, -1.507046103477478, -0.24444422125816345, 0.543010950088501, -0.23044411838054657, -1.9408870935440063, 1.1633315086364746, -0.27083516120910645, 3.44913911819458, 1.0553057193756104, -3.0793275833129883, -0.25714337825775146, 1.2307956218719482, 1.8944758176803589, 2.717319965362549, 0.4035372734069824, 0.21583479642868042, 0.2802148461341858, 1.9108476638793945, 0.7615860104560852, -1.0535314083099365, 0.9826800227165222, 0.05006535351276398, -0.028607815504074097, -1.6652395725250244, -0.5723230242729187, 0.2563204765319824, 1.1520870923995972, -0.2688029110431671, -0.636252224445343, -0.6111149191856384, -3.014780044555664, 0.14192846417427063, -2.1473336219787598, 0.9131372570991516, 0.5492277145385742, 1.7118456363677979, 2.2873075008392334, 1.3070210218429565, 1.162123203277588, 1.4584242105484009, -0.9183170199394226, -1.087212085723877, -1.128700852394104, -0.4289761185646057, 0.7027572989463806, 1.6643474102020264, 1.9017243385314941, 1.0873463153839111, -1.9114006757736206, -0.49031609296798706, -1.6709728240966797, -1.6600301265716553, -1.5800318717956543, -0.268854558467865, -0.7246596217155457, -0.9411422610282898, -0.7423803806304932, -1.3246119022369385, -0.7407119870185852, 0.9816610813140869, -0.2923834025859833, -0.8961397409439087, -0.5674682259559631, 0.3224601149559021, -0.3954903185367584, 0.8225875496864319, -0.5646450519561768, -0.13093438744544983, -1.7132842540740967, -0.813535749912262, -1.4917349815368652, 0.5702695846557617, 1.1902623176574707, 1.91335129737854, 1.2639682292938232, -0.8699813485145569, 3.7266931533813477, -2.037703037261963, 1.4021217823028564, 1.526229977607727, -0.9292087554931641, 1.8898411989212036, 0.09317941963672638, -1.2558281421661377, -3.870173931121826, -1.0373274087905884, 2.040111541748047, 0.3534659445285797, 3.0059895515441895, -0.05800825357437134, 1.1648668050765991, -0.024773359298706055, -3.067162036895752, 3.2776637077331543, -0.6942726373672485, -5.227594375610352, 1.9795427322387695, 0.0028733909130096436, 0.791511595249176, 0.3820343017578125, 0.15734030306339264, -1.6742498874664307, 0.5277414321899414, 0.6621070504188538, -1.4447498321533203, -0.032910287380218506, -1.3416645526885986, -0.5055308938026428, 1.4445691108703613, -0.2097773551940918, 1.11095130443573, 0.9652843475341797, 0.35613641142845154, 1.2719985246658325, 0.7821975946426392, 0.40291327238082886, 2.3943674564361572, -1.6086077690124512, -2.309338092803955, -1.8677736520767212, 0.2669961452484131, 2.077944755554199, 0.8047981262207031, -2.1189417839050293, 0.5071949362754822, 0.555226743221283, 0.005998194217681885, 1.6148226261138916, -0.6310678720474243, 1.285435438156128, -0.17168915271759033, 1.4522294998168945, 2.7569711208343506, 0.15360039472579956, 1.6688642501831055, -0.7500258684158325, -1.4419130086898804, 0.4564841091632843, 0.2580302357673645, -1.2116992473602295, -2.139780044555664, -2.2902731895446777, 0.005147606134414673, -1.159757137298584, -1.7670329809188843, -2.0606260299682617, -3.1585590839385986, -1.6960687637329102, -1.3965247869491577, 0.44045713543891907, -1.4516115188598633, -0.12111800909042358, 0.9695870876312256, -2.867621898651123, 1.2504377365112305, 1.5309076309204102, -2.069248676300049, 2.3080592155456543, 1.3075239658355713, 0.430303692817688, -0.5608864426612854, -0.2753223776817322, -0.5756039023399353, -0.11692792177200317, 0.2285749912261963, 0.9971823692321777, -0.8907581567764282, -0.5120561718940735, -0.564343273639679, -0.20835807919502258, 1.024692177772522, -0.3632946014404297, 0.8405501842498779, -0.647343635559082, 1.402390480041504, -1.1856956481933594, 3.3965225219726562, 0.01384919136762619, 0.4570114016532898, 0.2830893397331238, 0.9537481069564819, 4.180067539215088, -0.8169167637825012, -1.8325860500335693, 5.595051288604736, -0.11480946093797684, -1.3612455129623413, 2.111393928527832, 0.8988708853721619, -0.6024633646011353, -0.35287538170814514, -0.2878997027873993, -1.5480151176452637, -1.912519931793213, 0.2691100239753723, -2.2131600379943848, 4.532693862915039, 1.385244607925415, -0.29931193590164185, 0.12832874059677124, -2.367298126220703, -0.5804423689842224, 1.3591159582138062, -1.0909099578857422, 1.008981466293335, 0.41946545243263245, -0.82235187292099, 2.273733615875244, -1.973372220993042, 3.7941508293151855, -0.7114866971969604, 1.8051568269729614, -1.1375190019607544, -1.1553784608840942, 0.23643189668655396, 1.5949287414550781, -0.4186204671859741, 0.38736492395401, -1.7799922227859497, -2.3246374130249023, 1.5289263725280762, -2.25529408454895, 1.2392072677612305, -0.4417479932308197, 0.12547796964645386, 0.008357122540473938, -1.090185284614563, 2.0812902450561523, -0.8823651075363159, -1.4847289323806763, -1.8930127620697021, -1.3056128025054932, -0.6228772401809692, -1.5049376487731934, 0.5078471899032593, -1.156402587890625, -1.1294472217559814, 0.6367150545120239, -1.7659411430358887, -0.23134662210941315, 0.6054882407188416, 1.2990885972976685, -1.2858755588531494, -1.5077532529830933, -0.7381798624992371, 1.084134817123413, 0.19563333690166473, 2.256303071975708, -1.2908906936645508, -0.33385586738586426, 1.5245187282562256, 0.021628737449645996, -0.6555771827697754, -0.10100817680358887, -0.5271545648574829, 1.305973768234253, 0.5081822872161865, 0.4312363862991333, 0.5389321446418762, -0.4687208831310272, 2.0437450408935547, 3.8849198818206787, 1.2338966131210327, 0.2273685336112976, -0.7958150506019592, -0.969356119632721, -2.1999711990356445, -2.1855781078338623, 0.19609002768993378, 4.205296993255615, -1.2757612466812134, 1.2654238939285278, 1.8745567798614502, -0.4877317249774933, -2.0381336212158203, 0.8803284168243408, 0.541972815990448, -0.9908576011657715, 2.4752063751220703, 1.6050821542739868, -0.7481188774108887, -0.16113035380840302, 0.3078572750091553, 1.2477192878723145, 1.5375382900238037, -0.27705687284469604, -0.5150344967842102, 3.30775785446167, 1.7030961513519287, 0.8591285347938538, -1.1706221103668213, 1.1391820907592773, 2.021941661834717, 0.2155369073152542, -1.9372637271881104, 2.2692017555236816, 0.25796979665756226, -2.369253158569336, -0.29214924573898315, 0.09596306085586548, 0.86368727684021, -0.5427761077880859, -0.3955637216567993, 0.6179487705230713, 0.7254228591918945, 0.8644113540649414, -0.6202880144119263, 0.891629695892334, 1.2572124004364014, -1.2510669231414795, 0.2369893342256546, 0.14348635077476501, -0.6993234157562256, -2.258193016052246, -3.259256362915039, -0.43965062499046326, -0.8293822407722473, 0.20241880416870117, 0.5383725762367249, 2.3706140518188477, 0.32214686274528503, -0.45619678497314453, -0.46779969334602356, -0.23651280999183655, 1.321883201599121, 0.5067873001098633, -2.025729179382324, 0.30083879828453064, -0.9201827049255371, -0.14438709616661072, 1.6177136898040771, -0.32867276668548584, 0.5683107376098633, 2.1928725242614746, 0.4217306077480316, -1.5294990539550781, 2.1623177528381348, 0.3842254877090454, 0.35408157110214233, -0.35998639464378357, 0.45555323362350464, 1.1878447532653809, 0.5877421498298645, -1.125954508781433, 2.1553921699523926, 0.02738596498966217, 0.3953949213027954, 0.8689117431640625, -0.7361422777175903, -0.258217453956604, 3.0394060611724854, 1.8179123401641846, -0.2691507935523987, -0.06759929656982422, 0.1004038155078888, 1.7494006156921387, -0.02324146032333374, -0.9982461929321289, -2.4884603023529053, 0.4225578308105469, -0.7306045889854431, -1.186295986175537, 1.0624291896820068, -1.1368417739868164, -0.4782657325267792, -0.9094804525375366, 1.171289324760437, -3.830808162689209, -0.2573692202568054, 1.834290623664856, -0.025458406656980515, -0.7390788197517395, 0.06157377362251282, -0.24442969262599945, -1.920426368713379, 0.6554762721061707, 0.23759455978870392, -1.1637271642684937, 2.1221652030944824, -1.1535749435424805, 0.720858097076416, -2.673305034637451, -1.1079232692718506, 1.653411865234375, 0.6784976720809937, -0.7471641302108765, 0.22051411867141724, 0.37292158603668213, 0.5472922325134277, -1.0201283693313599, -0.5668917894363403, -0.6719214916229248, 0.2739064693450928, 0.7582616806030273, -2.859818935394287, -1.1691800355911255, -0.15693536400794983, 0.20103538036346436, -0.7597105503082275, 0.19606924057006836, 0.8169919848442078, -0.5927982330322266, -0.8533223867416382, -0.4946892261505127, 0.5335444211959839, -0.7858023643493652, 0.9879181385040283, 1.4988163709640503, -2.127314567565918, 0.3922070264816284, -0.748248279094696, -6.777588367462158, -0.46137821674346924, -2.486246109008789, -0.5666419267654419, -1.2615317106246948, -0.6278754472732544, -0.3298124670982361, -3.522339344024658, 0.7638518810272217, -1.5956541299819946, -0.6666638255119324, -1.5126019716262817, -0.7124234437942505, 0.8076363801956177, 1.048346757888794, 1.9157898426055908, 1.6198711395263672, -1.8165923357009888, -0.5463054180145264, 0.47789818048477173, -1.7397353649139404, 0.2631666660308838, -0.22388096153736115, 0.11183388531208038, -0.5730062127113342, 0.14979517459869385, -1.9386717081069946, 2.2393274307250977, -0.4721507430076599, 0.8362818956375122, 1.2975586652755737, 0.15121915936470032, -0.05254673957824707, 0.15560315549373627, 1.6985080242156982, 0.3734186589717865, -1.0550016164779663, 0.298144668340683, -1.0630217790603638, 0.11669952422380447, -0.11330076307058334, -0.2581114172935486, 0.6380904316902161, 0.6711504459381104, 0.07144355773925781, 0.3914583921432495, -1.3874064683914185, 0.8045633435249329, -1.2707700729370117, 2.920304536819458, 2.3207263946533203, 0.7994433045387268, -0.7156298160552979, 1.5214052200317383, 1.911034107208252, -1.9719107151031494, -1.1588042974472046, 0.17118006944656372, -0.46857213973999023, 0.8548543453216553, 0.12356992810964584, -0.9366307258605957, 2.0282833576202393, -1.7508983612060547, 0.9786334037780762, 2.1107075214385986, 2.111502170562744, 1.587357997894287, 1.7760286331176758, 0.36898159980773926, -4.264274597167969, 0.24975864589214325, -1.2655153274536133, 0.8114754557609558, 0.49686771631240845, -1.2390456199645996, -0.4409254193305969, -0.3857033848762512, -0.07405628263950348, 0.3154945373535156, 1.2443866729736328, -0.4065759479999542, -0.7588409781455994, -0.8513205051422119, 0.7356918454170227, -0.006585657596588135, -1.0056334733963013, 0.49669748544692993, -0.8697792291641235, -0.580794632434845, -0.2532505691051483, 0.3179619610309601, -0.19278699159622192, 1.8989777565002441, -0.5122511386871338, 0.1321566104888916, -1.5930836200714111, -1.2397061586380005, -1.128966212272644, 3.73783016204834, -0.8168550729751587, 0.577019453048706, -1.7891398668289185, 1.1656498908996582, 0.06783892959356308, 2.6947786808013916, -1.1100739240646362, -0.014434373937547207, 1.022700548171997, -0.6383697986602783, -1.243208408355713, -0.8573403358459473, -0.03181369602680206, -1.3241182565689087, -1.3572163581848145, 1.6148490905761719, 1.1156126260757446, 0.14435064792633057, -0.14333590865135193, -0.5936468839645386, -0.11485153436660767, 2.755577325820923, -0.273015558719635, 0.671586275100708, -0.01350778341293335, 0.2873527407646179, -0.9226188063621521, -0.08959411084651947, 2.3337619304656982, 1.0556336641311646, 1.6135516166687012, -1.5221933126449585, -1.2619004249572754, -1.0847444534301758, -0.05636553466320038, 2.6197307109832764, 1.5573943853378296, -0.04330900311470032, 0.42324918508529663, -0.9619783759117126, 0.4989233911037445, -1.611101746559143, 0.3369103670120239, -0.6690020561218262, 3.6548166275024414, 1.3276503086090088, 0.3788263499736786, 0.6781195998191833, -2.5993666648864746, -0.8682637810707092, -2.5155887603759766, 0.17280900478363037, 0.7566909790039062, -0.8535224199295044, 2.715271472930908, -0.3569301962852478, -0.7491093277931213, 0.25922855734825134, 0.08620578050613403, -2.7818503379821777, 0.8376448750495911, 0.19810950756072998, 0.2738862931728363, 0.6656659841537476, -4.218456745147705, -1.0376570224761963, 0.3482794165611267, 1.235356092453003, -0.899417519569397, -1.2830582857131958, 0.7069507837295532, 2.6017277240753174, -0.33959004282951355, 1.1429473161697388, 1.99271821975708, -2.0537590980529785, 0.7258215546607971, -0.38864874839782715, -1.4697357416152954, -0.8613024950027466, -0.8109688758850098, -0.9505149722099304, -0.14507606625556946, -0.8859502077102661, 0.0836881548166275, 0.3375049829483032, -0.6046712398529053, 0.4317486584186554, 1.4477089643478394, -0.9757208228111267, 1.2139098644256592, 1.166534185409546, 0.221481591463089, 1.6747223138809204, -0.2975780963897705, 1.4179500341415405, -1.9510489702224731, 1.606162190437317, 4.776651382446289, -0.4780265986919403, 0.8950469493865967, 1.2647442817687988, 0.5956432819366455, 1.661712646484375, 1.2748652696609497, -0.9185271263122559, -1.36086106300354, 1.2764389514923096, -1.1696850061416626, -0.90249103307724, 0.3344649076461792, -0.3550431728363037, -0.07109984755516052, 0.222035214304924, 2.0997769832611084, 1.166694164276123, 0.8886719942092896, 1.8382303714752197, 0.42188915610313416, -3.6537177562713623, -0.17473071813583374, -0.20754313468933105, -0.4889792799949646, -0.6474047303199768, 0.36215054988861084, -0.9720771312713623, 2.1589949131011963, -1.6746928691864014, -0.42982184886932373, -0.20985376834869385, 0.7957568764686584, -1.0102382898330688, -2.713780403137207, -1.093942403793335, 2.0600266456604004, 0.2610703706741333, 0.008357932791113853, 0.516121506690979, 1.3270891904830933, 0.5295250415802002, 0.45221763849258423, 1.9522024393081665, 1.111473560333252, 0.10786852240562439, -1.0727459192276, 1.455446720123291, 1.0765677690505981, 0.14843469858169556, 2.202575206756592, -0.43960970640182495, 0.8821104764938354, 0.6150631904602051, 0.1984437108039856, 1.1561179161071777, -1.2654392719268799, -0.6977702379226685, -1.0100009441375732, 0.17357099056243896, -2.5148978233337402, 1.4205470085144043, 0.2535575032234192, -2.0538580417633057, 0.0014564543962478638, -0.41154778003692627, 0.2725159525871277, -0.09125588834285736, -1.693238377571106, 0.1853560507297516, 0.13790321350097656, 1.8386836051940918, -1.8166366815567017, 1.74996817111969, 0.5244889259338379, -0.8734104633331299, 0.2347816377878189, -0.6261113882064819, -1.996988296508789, 1.8408573865890503, -0.28364241123199463, 0.6561380624771118, -2.2262253761291504, -0.40677106380462646, 0.2614285945892334, -0.2703482508659363, -0.1632695347070694, -0.3612065017223358, 1.6662461757659912, 0.05919623374938965, -1.8764005899429321, -0.8006370067596436, 0.5431017279624939, -0.8612618446350098, 1.291327714920044, -0.029070422053337097, -1.7037676572799683, -0.4159873127937317, 0.23055924475193024, -0.1260852813720703, -1.038116455078125, -0.6893430948257446, -0.8466958999633789, 0.854692280292511, 0.8076280355453491, -0.49566322565078735, -0.8302820324897766, 3.9610798358917236, 1.22446870803833, -4.026504993438721, -0.3537450134754181, -1.7490317821502686, -0.26138433814048767, 2.2676522731781006, 2.0930917263031006, -2.2391390800476074, 0.029194951057434082, 2.167118549346924, 0.20536896586418152, -2.758951187133789, -1.1415621042251587, 0.026862867176532745, 0.6127637624740601, 0.21614032983779907, 0.8050141334533691, -1.2724852561950684, 0.6284998059272766, 0.48874184489250183, -0.12676866352558136, 0.812461793422699, -0.5166747570037842, 0.8197881579399109, 0.9286702275276184, -0.9839828014373779, 2.6453638076782227, 0.5125911235809326, -0.15450406074523926, -0.4926716983318329, 0.4221886396408081, 1.0387226343154907, -0.9935532808303833, -0.46359342336654663, -1.4061071872711182, 0.9154448509216309, -0.43463268876075745, 0.5070586204528809, 0.13906556367874146, 0.24230483174324036, -0.11581359803676605, 0.8657882213592529, -1.5541430711746216, 2.6317145824432373, 1.433878779411316, 4.950350284576416, -2.719071865081787, -0.2977611720561981, -0.1966235637664795, 0.19021913409233093, 0.9428342580795288, -0.4260542094707489, -2.1475722789764404, 0.18884748220443726, 0.035191893577575684, 1.5532323122024536, -0.30580511689186096, 0.8816664814949036, 0.7605778574943542, 1.6266624927520752, 0.35912367701530457, 1.6780658960342407, -2.5286829471588135, 0.060813724994659424, -0.5812699198722839, -0.6946498155593872, -0.38063758611679077, 0.30594009160995483, 0.9820420742034912, -1.0686957836151123, 1.189380407333374, -2.436784267425537, 1.321932077407837, 1.3266618251800537, 1.1271991729736328, 3.443546772003174, 0.6055259108543396, 0.9381123781204224, -0.4767652750015259, 0.4127543568611145, -0.6944674253463745, 1.6389224529266357, 2.0537304878234863, 0.9358720779418945, 1.0305826663970947, -0.8865412473678589, -1.1261543035507202, -2.004761219024658, -2.0101492404937744, -0.30730414390563965, 0.6958518028259277, 1.5976365804672241, -0.9268175959587097, -0.6050387024879456, -1.0816409587860107, -0.24660085141658783, 0.7471486330032349, 4.207149028778076, 0.24782085418701172, 1.512125015258789, -0.6352592706680298, 1.0751978158950806, 1.260087013244629, -0.4843701720237732, -1.5004611015319824, -0.017492592334747314, 1.4817174673080444, 1.5542333126068115, -3.142113447189331, 2.4018468856811523, -0.6100950241088867, -0.5851304531097412, 0.18156777322292328, -0.309943825006485, 0.16571414470672607, -1.6166152954101562, -1.9661802053451538, -0.6435940265655518, 2.0347559452056885, 0.8206239342689514, 0.551155686378479, -1.949403166770935, -0.4246121048927307, -0.5096818804740906, -1.7387436628341675, -0.4072788953781128, 1.3677077293395996, 1.3263009786605835, 2.1170854568481445, -1.8137061595916748, 0.0012862831354141235, 2.0651190280914307, -0.6985387802124023, -0.5719679594039917, -0.599197268486023, 0.40129101276397705, 1.7325215339660645, 2.642526149749756, 0.0911037027835846, 2.1255977153778076, -3.189859628677368, -0.0760839506983757, -0.058274030685424805, -1.6192824840545654, 0.19733832776546478, -0.16071102023124695, 0.2804417312145233, 0.3935290575027466, 2.0391290187835693, -0.7649840116500854, -0.3299948573112488, 1.8659123182296753, -0.3259924650192261, -1.0886609554290771, -2.0539958477020264, -0.13664984703063965, -1.8252339363098145, -0.43715012073516846, 2.1095340251922607, 0.69901442527771, 0.6473461985588074, -1.900954008102417, -3.8919882774353027, -0.9824750423431396, -1.881877064704895, -0.3206386864185333, -0.20050093531608582, -0.5025713443756104, -3.0421741008758545, -1.5028643608093262, 1.295244812965393, 2.390655279159546, 0.10703039169311523, 1.451897144317627, -1.6087415218353271, -1.138045072555542, 1.6543240547180176, -0.8675757646560669, -1.080742359161377, 1.933271884918213, 0.8030229210853577, -0.5499939918518066, 1.40078866481781, 2.013705253601074, -0.9162282347679138, 0.765328586101532, 1.279725193977356, 0.9686619639396667, 0.6028086543083191, 0.42623722553253174, -1.9157353639602661, 0.5313540101051331, -1.7205052375793457, 1.289168119430542, -2.1965930461883545, 0.7239806652069092, 1.0090564489364624, 3.117403984069824, 1.8824501037597656, -0.4697141647338867, 0.3361731767654419, 0.6062968373298645, 1.8478615283966064, -1.4406349658966064, 1.0410048961639404, -2.023077964782715, 0.512931227684021, -0.43171587586402893, -0.8363507986068726, 1.02811861038208, 2.656700611114502, -0.5972462296485901, -2.3096442222595215, 0.21669577062129974, -1.948075294494629, -1.61138916015625, 0.7853279709815979, 0.41640976071357727, -1.2665635347366333, 0.41691339015960693, -0.9817095994949341, -0.8968403935432434, -1.2788760662078857, -0.6419222950935364, -0.6012519598007202, 0.16687379777431488, -1.877465844154358, -2.786308765411377, -1.1212658882141113, -0.9619508981704712, -1.8892425298690796, -1.8404457569122314, 1.39992356300354, 3.507352590560913, -1.2985591888427734, -0.20485447347164154, -0.6886535286903381, 0.6655422449111938, 0.8896527290344238, 1.0571353435516357, -1.4934223890304565, 1.0567156076431274, 2.1492207050323486, 0.6088930368423462, 1.9362682104110718, 0.08543888479471207, 0.34033095836639404, 2.752195358276367, -2.4203779697418213, 0.9248676896095276, 1.2959609031677246, 1.4431285858154297, 1.023255467414856, 0.6904470324516296, 0.7920190095901489, 3.638777732849121, -1.3884429931640625, 1.6900782585144043, 1.738069772720337, 1.4622957706451416, 0.6161152720451355, 2.911015272140503, 0.4554763734340668, 2.8114662170410156, 1.0489469766616821, -0.7469367384910583, -2.49993634223938, 0.758699893951416, 0.09129396080970764, 0.035079218447208405, -1.17812979221344, -0.5260878801345825, -1.6000306606292725, 0.9190974831581116, 0.8718593716621399, -1.1108499765396118, -4.374477863311768, 0.8406298160552979, -0.7737060189247131, -0.1267133355140686, 0.11704370379447937, -1.0216753482818604, -0.6956541538238525, -0.4765140116214752, -3.33990478515625, -1.8038520812988281, -0.007906526327133179, -0.4990203380584717, 0.6036842465400696, -0.0083192503079772, 0.32231420278549194, -2.4323105812072754, -1.2768676280975342, 0.19914358854293823, 0.7216923832893372, -0.8111504912376404, -1.0072197914123535, 0.6846596002578735, -1.321138858795166, -0.8258838057518005, -0.07581394910812378, 2.3520021438598633, 0.033020734786987305, -0.160056471824646, 1.0926471948623657, -0.43024370074272156, -0.13654032349586487, -2.7188074588775635, -0.14596204459667206, -0.4239523112773895, -0.008300930261611938, -0.18826894462108612, -0.37304896116256714, 0.3849388360977173, 2.0706238746643066, 1.9065262079238892, -0.19156098365783691, 0.6330649852752686, -0.9969177842140198, -0.9051488637924194, 0.6584983468055725, -0.7518853545188904, 0.0980755090713501, -1.9786335229873657, -1.0306485891342163, 0.7210970520973206, 1.8306761980056763, 1.0477598905563354, -1.1441484689712524, 0.04558810591697693, -0.4517195224761963, 0.4565966725349426, 0.1315801441669464, 0.7883440256118774, -0.05086725950241089, 2.059630870819092, 0.9952889680862427, -1.150896668434143, -0.3444215655326843, -1.0631333589553833, 0.16794639825820923, -1.1442692279815674, 0.48372673988342285, -0.25654157996177673, 0.007505476474761963, 1.2040541172027588, 0.8911532759666443, 1.1696608066558838, -1.0083602666854858, 0.8794890642166138, -0.5776150226593018, 1.884507417678833, 0.731197714805603, 1.613877773284912, 4.833895683288574, -2.8216683864593506, 0.5404863357543945, 1.1905889511108398, -1.6131283044815063, -0.681296706199646, -0.6583706140518188, 0.6309999823570251, 0.8130106925964355, 1.2445300817489624, 0.8914284706115723, 0.10626982897520065, 1.5617396831512451, -0.2659298777580261, -0.47435256838798523, -0.9769788980484009, 2.1788275241851807, -0.5733137726783752, 2.064134359359741, -3.233102560043335, -1.4609923362731934, -1.6509618759155273, 0.7850043773651123, -0.07139551639556885, -0.8135649561882019, 0.7799705266952515, -1.856492042541504, -2.309800863265991, -1.2779792547225952, -0.5321953296661377, 0.25721704959869385, -0.5456879138946533, -1.3350485563278198, -0.09274078160524368, -0.6577730774879456, -3.979605197906494, -0.45310738682746887, -0.07139335572719574, 0.34149622917175293, -0.3485510051250458, -0.3616873025894165, 1.2496107816696167, 2.3558573722839355, 2.2365565299987793, 0.4970090985298157, -3.7689123153686523, 0.9480763077735901, -2.946794033050537, 2.1175811290740967, 1.467585563659668, -0.5308679342269897, 0.01132192462682724, 0.2851453721523285, -0.03931081295013428, -0.5874543190002441, -0.2893558144569397, -0.49104803800582886, -0.629984438419342, -1.3975369930267334, 1.3454275131225586, 0.424454003572464, 0.28998714685440063, 0.3151441812515259, 2.151911497116089, 0.6076211929321289, -0.3233610689640045, -1.1325277090072632, 0.021000325679779053, 1.4941864013671875, -1.0555044412612915, 0.569320023059845, 0.4503280520439148, -1.9575371742248535, 0.13474935293197632, 1.0946892499923706, 0.039228469133377075, -1.7617950439453125, 0.12834078073501587, -2.302384853363037, 0.10348573327064514, -0.9608696699142456, 0.663018524646759, -1.9017407894134521, 2.1392035484313965, -1.7750210762023926, -1.6132609844207764, -0.04080462455749512, -0.26604723930358887, -0.18425776064395905, -1.761678695678711, -1.3261141777038574, 1.1876652240753174, -1.9246721267700195, 0.23580214381217957, 0.7022852301597595, -1.8159278631210327, 1.7154629230499268, 0.11843752861022949, 0.538271427154541, 1.5063889026641846, 0.3927677869796753, 0.03003619611263275, 1.3701283931732178, -1.1459404230117798, -0.018544375896453857, -1.8139652013778687, -0.3850476145744324, -0.08116891980171204, -1.5414520502090454, -0.3860137462615967, -1.5540692806243896, 0.4707566499710083, -0.22122584283351898, 1.3174172639846802, -1.1728349924087524, -2.829784393310547, 1.6173733472824097, -0.6444491147994995, 0.4697422385215759, -2.2541258335113525, -1.1201176643371582, 0.9140622019767761, 1.4676579236984253, -0.38472485542297363, -0.446893572807312, -0.05146162211894989, -2.1928341388702393, -1.841507911682129, -1.1985347270965576, -0.9744544625282288, -0.34095317125320435, 0.3212836980819702, -0.6124318242073059, 1.5370166301727295, -2.373131036758423, 3.0336573123931885, 1.6444506645202637, 1.755704641342163, -0.24497345089912415, 0.6894527673721313, 0.514812171459198, -2.3615593910217285, -0.6913712024688721, 0.14609789848327637, 0.67283034324646, -1.5145491361618042, -0.2977703809738159, 0.6434440612792969, -1.512400507926941, -0.9549385905265808, 1.092910647392273, 0.3943612575531006, 0.7084754109382629, -1.7065787315368652, -0.2195763885974884, -0.7780255079269409, 0.4302566349506378, -6.791445732116699, -0.4998531639575958, 1.3296608924865723, 3.1279940605163574, -1.3344123363494873, 1.247035264968872, 1.2159444093704224, -1.6553726196289062, -4.656619071960449, -0.8248571157455444, -0.9743312001228333, 0.45996546745300293, -0.39542028307914734, -1.3588440418243408, -0.9302662014961243, -2.536807060241699, 3.3200197219848633, -1.2681653499603271, -0.9014965295791626, -0.5360720753669739, 0.6988860368728638, -0.738402247428894, -1.3075122833251953, -0.5657950043678284, 0.23933330178260803, -0.0374339297413826, 1.708866834640503, 0.43482890725135803, -1.3056178092956543, -4.643795967102051, -1.371284008026123, -0.06702461838722229, -0.2836901843547821, 0.18008828163146973, -0.5677316784858704, 2.8418076038360596, 5.515746116638184, -1.4095855951309204, 2.0230841636657715, 0.3309735059738159, -0.5581381916999817, 4.319384574890137, -1.8916637897491455, 0.4217914342880249, 0.34156203269958496, -1.4759533405303955, 0.4101906418800354, -1.1501613855361938, -0.22054964303970337, 1.5376758575439453, -0.33635422587394714, -1.700088620185852, -0.7290900945663452, -0.011257141828536987, 1.7424523830413818, -0.705748975276947, -0.06114007532596588, -0.7131905555725098, -2.4426114559173584, -1.7136242389678955, -0.6881968379020691, 1.4388270378112793, 1.0675097703933716, 1.2149416208267212, 1.0248628854751587, -0.6102876663208008, 0.14545902609825134, -1.4104176759719849, -0.17521636188030243, 0.09514924883842468, -1.8187214136123657, -1.1013892889022827, 1.5315792560577393, 2.587372303009033, -0.6384212970733643, -0.8874197602272034, 1.6042217016220093, -1.6000415086746216, -0.8523212671279907, 0.36802318692207336, 2.4645092487335205, 1.6217281818389893, -1.251326560974121, 1.7997993230819702, -0.9220936298370361, 0.5324743390083313, -2.0563464164733887, 0.24846124649047852, -0.20626017451286316, -0.949114978313446, -0.68888920545578, 0.8418141603469849, 0.5839653015136719, -0.8312265872955322, 0.05199868232011795, 0.1985468864440918, -2.1840617656707764, -4.185144424438477, -0.4426136612892151, -1.5497535467147827, 0.2815992832183838, -2.281848669052124, 2.587428092956543, 0.12521040439605713, -2.5638139247894287, 2.37985897064209, 2.927272319793701, -0.09731560945510864, 1.7488652467727661, -0.6269978284835815, -0.7939949631690979, 1.701846718788147, 1.7846300601959229, -1.9377094507217407, -0.9071259498596191, -2.101290702819824, -0.31046801805496216, 0.9800863265991211, -1.8398712873458862, -0.2074371874332428, 2.7784969806671143, 3.0394339561462402, -0.6607504487037659, 3.0810887813568115, -0.11497321724891663, -0.8357821702957153, -1.4437992572784424, -1.579667568206787, 0.750518262386322, -2.3481061458587646, 1.2513387203216553, 0.15843476355075836, -0.788072943687439, -1.4084590673446655, -0.30199235677719116, 0.47055792808532715, -0.7262628674507141, -0.13344526290893555, -3.403296709060669, -1.735980749130249, -0.07933801412582397, 0.4731218218803406, 1.4097175598144531, -1.8167803287506104, 1.4334516525268555, 0.11114808917045593, 2.7588396072387695, -0.6453676223754883, -2.0968432426452637, -0.6469476222991943, 0.9803603887557983, -2.881894111633301, -1.2660402059555054, -0.9985106587409973, -0.8022499084472656, 0.958112895488739, 0.8424521684646606, -0.6318882703781128, -1.1500084400177002, 0.7598867416381836, -0.5709984302520752, -0.23401635885238647, -1.4177451133728027, -4.508312702178955, 0.27427685260772705, 4.002381801605225, -0.3374250531196594, -0.49543553590774536, 1.202664852142334, 0.28116318583488464, -0.8738670349121094, 0.44475477933883667, 1.006201148033142, 0.5017052292823792, -0.8848461508750916, 0.10894297063350677, 1.4214977025985718, -0.14113494753837585, -0.8116288185119629, 0.15086668729782104, -1.246540904045105, 0.20029020309448242, 2.0011188983917236, 0.0012714862823486328, 1.1505528688430786, -0.5903169512748718, 2.7453317642211914, -1.211448311805725, -1.1828349828720093, -0.12551313638687134, 0.00854867696762085, -0.8599589467048645, -0.3776858448982239, 0.3192913830280304, -0.6578550338745117, 0.45421332120895386, -0.195470929145813, -2.101463556289673, 0.7356644868850708, -0.30906838178634644, 0.0693267434835434, -0.145582914352417, 1.5595203638076782, -0.7623593807220459, 0.7939784526824951, 1.151110053062439, -0.7873239517211914, -1.473644495010376, 1.1090984344482422, 3.717845916748047, 0.5700388550758362, -0.6569128036499023, -1.6666617393493652, 1.0138872861862183, -0.6661348938941956, 0.23225119709968567, 0.24901479482650757, -0.5759771466255188, 0.8207781314849854, -0.24642348289489746, 1.6663321256637573, 0.8589407205581665, 1.9360558986663818, 1.5847103595733643, 0.7662727236747742, -1.6266202926635742, 0.6379343867301941, 0.7707891464233398, 1.226130723953247, 1.609526515007019, -0.563514769077301, -0.07293930649757385, 2.4264638423919678, 2.057631015777588, -2.498476505279541, 0.9272931814193726, 4.794344902038574, 1.7771751880645752, -1.5165390968322754, 2.5373997688293457, -1.0031945705413818, 0.5023090839385986, 0.2744426429271698, 0.3639577031135559, -1.4445679187774658, -1.190234661102295, -0.6417679786682129, -0.7721199989318848, 0.3137097954750061, -2.674732208251953, 0.16883975267410278, -0.9819550514221191, -1.8415690660476685, -0.1900922656059265, 2.0550341606140137, 2.053478956222534, 2.2170729637145996, 0.4920544922351837, -0.7053387761116028, 0.6269571185112, -1.681150197982788, -1.1134700775146484, -0.8175796866416931, 0.9651979207992554, 0.744254469871521, -0.5710353255271912, -0.3560400605201721, 0.3071324825286865, 0.359433114528656, 2.3022732734680176, 0.7894178628921509, -0.1756885051727295, 5.3664021492004395, 1.4551153182983398, 0.13819009065628052, -0.9999115467071533, 2.1220288276672363, 1.7468572854995728, -2.6220221519470215, -1.0739043951034546, 0.6961469054222107, 2.4008235931396484, -0.022960245609283447, -0.2745310664176941, 1.5160796642303467, -0.023969292640686035, -0.32133814692497253, -1.6352590322494507, -0.5619397163391113, -1.2179393768310547, 0.26877403259277344, 2.7752721309661865, -0.12943804264068604, -1.0523282289505005, 0.5754737854003906, 1.2995364665985107, 0.501436173915863, 0.9635968208312988, 1.124798059463501, -0.13402101397514343, -0.2853397727012634, 0.8394160270690918, -0.1298300325870514, 1.197800636291504, -1.953591227531433, -2.4664697647094727, 1.4557363986968994, -0.15228387713432312, 0.7855372428894043, -0.12198945879936218, 1.811110496520996, -1.628885269165039, -0.15314684808254242, -0.31935903429985046, -0.49213114380836487, 3.8251023292541504, -1.7199478149414062, -0.6587851047515869, -0.4034547805786133, 1.2493335008621216, 1.5532310009002686, 0.5242325067520142, 4.4185872077941895, 0.17571592330932617, -0.973932147026062, 2.168130397796631, -0.9890401363372803, 1.9642606973648071, 0.6260008215904236, -1.932926893234253, -0.3525265157222748, -1.121260643005371, -2.6423351764678955, -1.3194880485534668, -0.21265265345573425, -0.1710246205329895, -2.155019998550415, 2.378852128982544, 1.7730090618133545, -1.179672122001648, -0.39338937401771545, -0.4209694564342499, 0.46827709674835205, 0.6296941637992859, 0.0715004950761795, 0.23185904324054718, -0.6411212086677551, -0.02931874990463257, 1.226372480392456, 2.2628064155578613, -0.34478551149368286, 2.796532392501831, 0.1280200481414795, 0.09192405641078949, 0.6925737857818604, -0.11449779570102692, 1.7629257440567017, 0.5386972427368164, 0.32996684312820435, -1.0361430644989014, 0.6894428133964539, 0.34918105602264404, -0.25698351860046387, -4.6184000968933105, -1.3175429105758667, 0.3483588993549347, 2.063901901245117, 2.286318778991699, -0.6378176212310791, -1.0769319534301758, -2.2106106281280518, 0.04179370403289795, 0.9953053593635559, 0.6135498285293579, 1.4308280944824219, -1.1062428951263428, -1.4218904972076416, 0.2768346667289734, -0.8086198568344116, 2.9263830184936523, -0.8299333453178406, -0.19846928119659424, -0.6084129810333252, -2.118884563446045, 0.25099053978919983, 0.083091139793396, 0.40757322311401367, -0.055732935667037964, 3.6750824451446533, 1.9413312673568726, -0.00012250849977135658, -1.0111665725708008, -1.63660728931427, 0.5648341774940491, -1.580138921737671, -2.3148815631866455, 0.24105817079544067, 2.6229400634765625, -0.9731115102767944, -0.21884620189666748, 3.5584845542907715, 2.6582083702087402, -2.0568201541900635, 0.588840663433075, 1.5128071308135986, -0.8326027989387512, 3.42107892036438, 1.8447110652923584, -0.21539103984832764, -1.2538620233535767, 1.8411576747894287, 0.6870559453964233, 1.2628796100616455, -1.2870701551437378, 0.3657287061214447, 2.766294002532959, 0.5904502868652344, -1.0118881464004517, -0.20041927695274353, -1.2369215488433838, 0.37093687057495117, 1.3779001235961914, 0.03408002853393555, 1.3184181451797485, 0.15374314785003662, -0.1872778832912445, 1.083601474761963, -0.5487520694732666, -1.0633653402328491, 0.5065675377845764, 0.3480689525604248, -1.8500338792800903, -0.22107189893722534, -0.6197131276130676, 0.4888151288032532, 1.8909385204315186, -1.234097957611084, -0.8935666084289551, -2.254840135574341, 0.8040652275085449, 0.8892064690589905, -0.3658756911754608, -2.593552827835083, -0.5782043933868408, 0.270283967256546, 0.8028623461723328, -1.655859351158142, 0.5046197175979614, 0.5335900783538818, 1.2038947343826294, 0.8654977679252625, -0.980087161064148, -1.474777340888977, 1.435182809829712, -1.15587317943573, -0.9803322553634644, -1.839315414428711, 0.2816743850708008, -0.3163977563381195, 0.24966171383857727, 0.02987012267112732, -1.4752552509307861, 0.2756115198135376, -0.77275550365448, 2.4344916343688965, -2.9801278114318848, -0.7161983251571655, 2.273998260498047, 0.18091556429862976, 0.2432606816291809, 0.0014127977192401886, -0.06655383110046387, -0.24585625529289246, -1.0724118947982788, -1.310866117477417, -0.856377363204956, -0.06769487261772156, -0.987338662147522, -1.508815884590149, -1.4399924278259277, -1.6613296270370483, 1.177879810333252, -0.5173137187957764, 0.05860297009348869, 0.5822780728340149, -1.6903319358825684, -0.023318827152252197, -1.8601387739181519, 2.1351730823516846, -0.2384793758392334, -0.06786836683750153, -1.5580781698226929, 1.747612476348877, 0.6084529161453247, -0.22700105607509613, -0.3109849691390991, 0.047336310148239136, 0.4399346113204956, 1.6136139631271362, -1.5093153715133667, 1.4650146961212158, -0.5226906538009644, 1.8332966566085815, -0.002170354127883911, 0.278868168592453, 0.6946344375610352, -0.8810156583786011, 1.6241331100463867, -0.9339216351509094, 4.421881198883057, 0.7710088491439819, 1.8092702627182007, -0.32494592666625977, 0.06052657961845398, -0.7977087497711182, -0.7106306552886963, -0.25499656796455383, -2.4634552001953125, 2.1338558197021484, 2.146519422531128, -0.10940361022949219, 2.48652982711792, 0.2394159436225891, 0.45962971448898315, 0.7944794297218323, 0.19011694192886353, 0.2686505913734436, -1.2132301330566406, -0.5695293545722961, 0.48712846636772156, -0.7758826017379761, -0.3053087592124939, 0.490779310464859, 1.0495104789733887, 0.9331216216087341, -0.5304447412490845, -0.47707366943359375, 1.2768877744674683, 0.5301076173782349, -0.5809543132781982, 1.5515445470809937, 1.050003170967102, -0.6425110101699829, -1.0638248920440674, -2.454845428466797, 0.24891430139541626, 1.5772205591201782, 1.088356614112854, 1.042717695236206, -0.12716364860534668, 1.210698127746582, -2.4313578605651855, 3.203004837036133, 0.7301594614982605, -2.835782527923584, -1.4901678562164307, 1.7245343923568726, -0.41261595487594604, 0.941288411617279, 0.5307609438896179, 0.5082839727401733, -0.1402818262577057, -0.7768394947052002, 5.650429725646973, 0.9351022839546204, 1.4755706787109375, -1.0492573976516724, 0.04268723726272583, 0.4236769676208496, 2.938333749771118, -1.2399871349334717, -0.31711068749427795, -1.5691317319869995, -0.09842777252197266, -1.7376635074615479, -0.3422060012817383, 0.4297218918800354, 0.05021947622299194, 1.8165448904037476, 1.93896484375, 1.4266154766082764, -0.1750156283378601, 0.9574346542358398, 1.183063268661499, 0.3027907907962799, -1.0374960899353027, 0.08908778429031372, 1.631351113319397, -2.058260917663574, 2.2263612747192383, 1.4314441680908203, 0.1352325677871704, 0.7452409267425537, -1.0716068744659424, -1.460903525352478, -0.23600263893604279, -1.121469259262085, -0.08383297920227051, -0.5507204532623291, -2.023824453353882, -1.4732142686843872, -1.0068674087524414, 2.5301482677459717, -0.7836439609527588, 1.4419784545898438, 2.132204055786133, -0.7876659631729126, 0.854073703289032, 1.940577745437622, -2.709360361099243, -1.1767339706420898, -0.2840556502342224, -1.6554919481277466, -1.3749737739562988, -0.14841416478157043, -0.20541398227214813, -2.91660737991333, 0.3654502332210541, 0.045541584491729736, 0.9044961333274841, -0.8501931428909302, -1.6513572931289673, -0.9232169389724731, -0.43294447660446167, -2.155168056488037, -1.0780625343322754, -0.4093557894229889, -1.985170602798462, -2.2873353958129883, -0.4038931727409363, 0.7166132926940918, -2.759197473526001, 0.6746029257774353, -0.11609311401844025, -0.6471988558769226, 0.11660242080688477, -1.13480806350708, -1.6841844320297241, 0.03930997848510742, 2.377103328704834, 1.2912856340408325, 2.0163750648498535, -0.4462631940841675, -0.3626900315284729, 1.88189697265625, 0.1993972659111023, 1.4293341636657715, -0.26116496324539185, 1.996086597442627, 0.9885538816452026, 1.4065545797348022, 1.8261637687683105, 1.9412329196929932, -2.0492215156555176, -0.34075653553009033, -1.0789881944656372, 1.1494886875152588, -0.9491145014762878, -1.2633898258209229, -0.2118528187274933, 1.0750185251235962, 0.8566584587097168, 0.33676937222480774, -0.3604393005371094, -0.21307101845741272, 0.5211495757102966, 2.831735134124756, -1.3369110822677612, 0.3552897274494171, -1.0030618906021118, -0.5927979946136475, 0.9482458233833313, 0.03842461109161377, 0.2602696418762207, -0.6017827987670898, -0.892673671245575, -0.8492685556411743, 3.2003695964813232, 0.1697000414133072, 1.9528629779815674, 0.99276202917099, -0.4400883913040161, 0.07485617697238922, -1.2491480112075806, 0.5807472467422485, 1.8803271055221558, -2.5393853187561035, -3.141761541366577, 2.028465986251831, 0.23690561950206757, 0.8124628067016602, 0.10129829496145248, 0.36556047201156616, 0.01605011522769928, 0.2342473268508911, -0.8133923411369324, 2.1302576065063477, -1.4370394945144653, -1.9268019199371338, 0.306155264377594, 1.3683435916900635, -2.687589645385742, -0.9731571674346924, 0.40955615043640137, -1.3998345136642456, 0.2984622120857239, -0.27459168434143066, 0.4314817190170288, 1.099440097808838, -0.6073017716407776, -0.064827561378479, 0.35993003845214844, 0.6448975801467896, -0.9110851287841797, -1.7849887609481812, -0.5164006948471069, 5.1236042976379395, -1.0499093532562256, 0.8009916543960571, -0.882088303565979, -0.3036349415779114, 0.6944871544837952, -1.0759226083755493, -1.6556044816970825, 1.7501156330108643, 0.48554763197898865, 2.0901830196380615, 0.4721592664718628, 1.1916905641555786, 1.3342998027801514, -2.031813859939575, 0.8475355505943298, 0.039162494242191315, 1.8544249534606934, -0.6160342693328857, 1.5856616497039795, -0.8946243524551392, -0.15095162391662598, 0.7088778018951416, -1.400255799293518, -1.4309124946594238, -1.6562737226486206, 0.5858505368232727, 1.110705852508545, 0.18664033710956573, -0.7123443484306335, -0.4112433195114136, -1.2926483154296875, 0.32074499130249023, 0.0321427583694458, -0.01462806761264801, 1.124520182609558, 1.0861756801605225, -0.3784794211387634, 1.517198085784912, 4.492101192474365, -0.1635948270559311, -0.8788471221923828, 1.15821373462677, 0.4488782584667206, -0.9297805428504944, 0.1379511058330536, 0.553778350353241, -0.6845307350158691, 1.4873348474502563, -2.4692091941833496, 1.4313257932662964, 0.6201796531677246, 0.17205509543418884, 0.4196797311306, 0.4677475094795227, 0.9189831018447876, 0.5791501998901367, -1.0386801958084106, 1.0147932767868042, 3.9833874702453613, -1.5562344789505005, 0.606833815574646, -0.19989368319511414, 0.7302675247192383, 0.09930792450904846, 1.2965136766433716, -0.912970244884491, 0.03191480040550232, 3.574925422668457, -0.4829831123352051, 0.0236242413520813, 5.338906764984131, -1.8239803314208984, 0.9720549583435059, -0.6724165081977844, 0.8006563782691956, -1.2661163806915283, -0.8352968692779541, -0.859122633934021, 1.8915404081344604, 1.4645206928253174, 0.3570442795753479, -0.7202306985855103, -1.1973893642425537, -1.9821637868881226, -0.7362520098686218, -0.18075212836265564, 1.3921314477920532, -1.6275675296783447, -1.989109992980957, 0.8291137218475342, 1.055328369140625, 2.3507964611053467, -1.7172701358795166, -1.070067286491394, -1.6430461406707764, 2.5922772884368896, -0.2785928547382355, -2.9848711490631104, 0.40043890476226807, 2.4830849170684814, 0.6627166867256165, -0.07784229516983032, -0.908504843711853, -0.5367618799209595, 0.5561856031417847, 0.7438833117485046, -0.6832284331321716, 2.015014886856079, -0.08247584104537964, -1.2913377285003662, 0.153162881731987, 0.3910531997680664, -0.9924675226211548, -2.6809334754943848, -0.5934613943099976, 0.1865020990371704, 0.9490212202072144, -0.8685047626495361, 0.36506468057632446, 0.8924721479415894, 2.803093433380127, -1.2818512916564941, -0.4662248194217682, -3.023190975189209, 1.446663498878479, -0.023000970482826233, 0.23169827461242676, -0.7837836742401123, 1.898130178451538, -0.8100234866142273, -0.4608345329761505, -0.9969832897186279, 1.139771819114685, -0.9119807481765747, -1.010705828666687, 1.8580832481384277, -1.0250756740570068, 1.8881940841674805, -5.380529403686523, -1.6716573238372803, 1.4814236164093018, 0.11187219619750977, -2.996992588043213, 0.5292969346046448, -0.18839043378829956, -0.5620024800300598, 0.03546898066997528, 1.6889177560806274, 0.413061261177063, 2.340407133102417, 0.5359338521957397, -1.3036446571350098, 0.8776810765266418, 0.8736194372177124, -2.101022720336914, -0.2942544221878052, -1.8747875690460205, 0.277146577835083, 0.4428490400314331, -1.7917208671569824, 0.5110782384872437, 0.03740423917770386, -2.253948211669922, 1.0659096240997314, 2.7133612632751465, -0.6466341018676758, 0.8754276037216187, 0.3122302293777466, 3.2078840732574463, -0.6616829037666321, 0.7727354168891907, 0.05337207019329071, -2.256617307662964, -0.08728861808776855, -0.8420848846435547, 0.5187711715698242, 1.102029800415039, 1.5216983556747437, 0.1464814692735672, 0.5963138937950134, -1.1576361656188965, 0.5828309655189514, -0.593792736530304, 2.071133613586426, -1.3058820962905884, 0.06222859025001526, 1.6349005699157715, 1.3076105117797852, -1.8213759660720825, -1.0881602764129639, -0.19437658786773682, 1.7867194414138794, 0.013121779076755047, 0.7355312705039978, -2.9770379066467285, 0.7632893323898315, 0.410502165555954, -0.4246549606323242, 1.2949588298797607, 2.2746591567993164, -0.6719306707382202, -0.8170047402381897, 0.7699475288391113, 0.6933448314666748, 1.2510788440704346, 1.7653127908706665, 0.4680596590042114, 0.5269157886505127, 0.06177684664726257, 0.6253545880317688, -0.47471141815185547, -2.5066027641296387, 0.12937942147254944, -1.2199764251708984, 1.6013325452804565, 0.7316817045211792, -2.054182291030884, -2.051356077194214, 2.360511064529419, -3.0788941383361816, -1.2969895601272583, -1.5990428924560547, -0.7750564813613892, 0.5537956953048706, 1.157395362854004, -0.18564224243164062, 1.2402206659317017, 0.79112309217453, -1.7052218914031982, 1.004103183746338, -1.3259692192077637, 2.0989465713500977, 2.4627833366394043, 1.1783820390701294, -0.24319446086883545, 0.3654129207134247, -0.9624652862548828, 0.9559128284454346, -1.9162174463272095, 1.1550772190093994, 0.9531165361404419, -0.4044584035873413, 0.19280606508255005, 1.0060691833496094, 0.7171992063522339, -0.38604772090911865, -0.430225670337677, 0.31081995368003845, 0.4056617319583893, 0.45521119236946106, 0.4142363369464874, 1.5555720329284668, -1.7873342037200928, -2.9145607948303223, 2.866933584213257, -0.32533586025238037, -1.4580429792404175, -0.6000732183456421, 1.8223462104797363, -1.4262421131134033, -0.23401159048080444, 0.651703953742981, 1.5785582065582275, 0.20555774867534637, -1.9047746658325195, 0.88892662525177, 0.10863174498081207, -0.18667876720428467, -1.0223745107650757, 2.32611346244812, 1.8119200468063354, 1.8341776132583618, 0.47411155700683594, 0.580843985080719, 0.2821934223175049, -1.1021721363067627, -1.3663949966430664, 0.2762962579727173, -1.4633784294128418, -1.068271517753601, 0.344421923160553, 2.1025593280792236, -0.5866209864616394, 2.2906200885772705, -0.3211601972579956, -1.1382874250411987, -2.3213696479797363, 0.3700638711452484, 0.7926204204559326, 0.0005762800574302673, 0.5651304721832275, 0.6901941299438477, 1.781801462173462, 0.8438757061958313, 0.5226343870162964, 1.7465858459472656, 2.37072491645813, -0.6299746632575989, 0.35023000836372375, -1.0850716829299927, -0.5437096953392029, 0.6244598627090454, -1.2594468593597412, 1.064260482788086, -1.6986119747161865, -0.8835026025772095, -0.33572694659233093, -1.1448073387145996, 2.011120557785034, -0.9058079719543457, -1.4051158428192139, 1.5882247686386108, -0.5006556510925293, 0.7957009077072144, 5.372704029083252, -0.600649893283844, -0.28469347953796387, 2.1039681434631348, -1.6265788078308105, -1.9265203475952148, -2.08620285987854, -0.9435107707977295, -0.2879831790924072, 0.8271090984344482, 0.9894616007804871, 1.5641566514968872, 0.30258291959762573, 1.7126898765563965, -0.2770443260669708, 0.13761967420578003, -2.489710569381714, 0.44295036792755127, 1.9703985452651978, -0.3822938799858093, -0.6595292687416077, -1.058980941772461, 0.18539083003997803, -0.5129925012588501, 0.6262489557266235, -1.0354907512664795, 0.1456645429134369, -1.4866626262664795, -1.2176861763000488, 1.1591453552246094, 1.2593202590942383, 0.42838770151138306, 1.1045573949813843, -0.9087529182434082, -0.2432994842529297, -0.9102824330329895, -0.08523082733154297, 0.7280632257461548, 0.2227889448404312, 2.2797224521636963, 0.8993335366249084, 1.3742412328720093, -0.13509374856948853, -0.4568031430244446, -1.3145675659179688, -1.2757751941680908, 2.4028284549713135, 0.3319095969200134, -0.6570703983306885, 0.8159070611000061, 0.49498867988586426, 0.9068921804428101, 0.189747154712677, -2.113835573196411, -2.958806037902832, -0.7222949266433716, 0.6663386821746826, -1.3242634534835815, 3.4767143726348877, -1.3269803524017334, 1.3789634704589844, -2.179445505142212, 0.47479358315467834, -0.7153444886207581, 0.24899935722351074, 0.7173709869384766, -2.833940267562866, 1.488472819328308, -1.3071784973144531, -0.1720220446586609, -3.9689838886260986, -0.36442407965660095, -1.5919232368469238, -2.7171692848205566, 1.3630657196044922, -2.9160852432250977, -0.5534694194793701, 1.4327715635299683, 1.091721534729004, -3.8156492710113525, 0.44592541456222534, 1.499353289604187, 0.9727754592895508, 1.3085572719573975, -0.9303664565086365, -2.0964725017547607, -1.826716661453247, 0.5088270306587219, 0.5951424241065979, 1.2957472801208496, 4.63514518737793, -0.03867197036743164, 1.0549321174621582, 0.3800528943538666, 0.8006160259246826, -0.3567795753479004, 0.40552619099617004, -0.5653604865074158, -0.1570689082145691, -0.6430595517158508, -0.7382485866546631, -1.2994139194488525, 1.6836398839950562, -0.7641733884811401, 3.2184054851531982, -0.6653316617012024, 1.2925083637237549, 1.362695336341858, -1.2247395515441895, 0.47536662220954895, 1.381704568862915, -0.991931140422821, 1.3031725883483887, 3.142836332321167, -1.1867254972457886, 0.5761324167251587, -0.1594117283821106, -0.06910837441682816, 0.6039392948150635, -0.21162199974060059, 1.010394811630249, -2.2464005947113037, 0.45712995529174805, -0.2555229067802429, -3.527250289916992, 0.056769683957099915, 0.41667115688323975, -0.6991429328918457, 5.2400126457214355, 0.9472552537918091, -1.3522624969482422, 0.1608290672302246, 0.38528066873550415, 0.09207737445831299, 0.4381241798400879, 0.14718401432037354, 0.4990035891532898, 0.9959284067153931, 0.8910566568374634, 1.2138640880584717, -1.92189621925354, -0.2919856905937195, 0.5886567831039429, -0.587105393409729, -1.2154953479766846, -4.714910507202148, 0.7690768241882324, -0.9758686423301697, -0.9767513275146484, -0.509568452835083, 2.2654058933258057, 0.09715158492326736, -1.580147385597229, 0.42266845703125, -0.18601755797863007, 0.6153087615966797, -0.9864323139190674, 0.9773802757263184, 3.536205291748047, -1.1048774719238281, 1.6412466764450073, -1.2315318584442139, 1.7930265665054321, -1.008384346961975, -0.37569278478622437, -1.1083358526229858, -1.0482943058013916, 0.9041711091995239, -1.0188653469085693, 3.7199156284332275, -0.11288097500801086, 1.2609199285507202, 1.4489740133285522, -1.429140567779541, 2.228147506713867, -0.9900556206703186, -1.0591039657592773, 0.46554654836654663, 0.4399113655090332, -1.0702661275863647, 1.9413731098175049, 0.4770210385322571, 0.24704143404960632, -2.0329039096832275, -1.976925253868103, -0.030562639236450195, -2.673710346221924, -0.33943381905555725, -0.3304130434989929, -0.6711454391479492, -1.1502902507781982, 0.6079461574554443, -0.18896996974945068, 0.31744104623794556, 0.7515661716461182, 1.020310878753662, 0.7982253432273865, -2.433504104614258, -0.2937551736831665, 0.1528792679309845, -0.3130651116371155, 0.984001636505127, 0.13820761442184448, -0.6877326369285583, -1.069506287574768, -2.0403683185577393, 1.5171886682510376, 0.6156939268112183, -0.4408144950866699, -0.28341352939605713, -0.5876860618591309, 0.3054383397102356, -1.218218445777893, 2.839632034301758, 2.2448861598968506, -2.8476481437683105, 0.448668897151947, 0.268716037273407, 0.20095184445381165, 2.008206367492676, -1.945458173751831, 1.9652241468429565, 0.15633079409599304, 1.0590076446533203, -0.6260536313056946, 0.4520080089569092, 1.534341812133789, 1.2158374786376953, 0.9052236080169678, -0.9718894958496094, -0.792778730392456, -0.32852375507354736, 2.1383678913116455, 0.7688392400741577, -0.48877766728401184, 1.2437164783477783, -1.4979934692382812, -0.03609490394592285, -1.5336542129516602, 1.9793407917022705, -1.7522575855255127, -2.047184467315674, 0.15055224299430847, -0.4929678440093994, 0.8489773273468018, -1.7137997150421143, 0.012795776128768921, 0.6683498620986938, 0.8121521472930908, 0.8779870271682739, 0.9741860628128052, 0.5076106190681458, -0.011335046961903572, -0.021998822689056396, -0.13136808574199677, -1.158933162689209, -1.9511655569076538, -1.2782881259918213, 2.6340484619140625, -1.2791475057601929, -1.94937002658844, 1.2382712364196777, -0.30445921421051025, 0.3835259675979614, 0.42121049761772156, 0.5358822345733643, 3.3943674564361572, 1.0294215679168701, -0.4678460359573364, -0.5335777401924133, -1.5679786205291748, 0.43652284145355225, 1.5667834281921387, -1.1248914003372192, -0.19365331530570984, -2.0182647705078125, -0.01987728476524353, -0.5740278363227844, -0.49918484687805176, 0.3802177906036377, 1.0716007947921753, 1.1801058053970337, -1.1632241010665894, -0.11366783827543259, -0.403045117855072, 1.4034374952316284, 0.2426932454109192, -0.5748016834259033, 0.6154941916465759, -0.951781690120697, 0.3370514512062073, 0.40707018971443176, 4.419112205505371, 1.5076563358306885, -1.3538850545883179, -2.1326940059661865, -2.6140973567962646, 2.607234001159668, -1.1536847352981567, 0.5079247951507568, 0.2425944209098816, 2.344650983810425, 0.686607837677002, -0.9829413890838623, -0.1746940016746521, 0.2905622124671936, -1.2302366495132446, -0.9124823212623596, 0.5490759611129761, -0.21869653463363647, -1.2769041061401367, -1.878769040107727, 0.6804583072662354, 0.5142913460731506, 2.7274248600006104, 0.6083527207374573, -0.8723562955856323, 0.8608424663543701, -0.11621572822332382, 2.6933302879333496, -0.30644509196281433, -0.47535720467567444, 1.9823248386383057, 1.2364646196365356, -3.044206142425537, 1.8693190813064575, -0.8331626653671265, -0.37363219261169434, 1.7398111820220947], "dtype": "float32"}}, {"a": [{"__nd__": [[[1.6911777257919312, -1.2196186780929565, 1.6678640842437744, -0.0003382449212949723, 0.9093248844146729], [0.4181351065635681, -1.9970226287841797, -0.5498265624046326, 0.21705204248428345, -0.581890881061554], [-0.5293996930122375, 0.21176007390022278, -0.18828608095645905, -0.1980505883693695, 0.6134865880012512], [-0.4632808268070221, -1.3659629821777344, -0.9836968183517456, 0.2632957398891449, -1.3274171352386475], [-0.5249884724617004, 0.5244264006614685, -0.689629316329956, -0.5756297707557678, -1.3888087272644043], [0.844525933265686, 0.7245283722877502, -0.2703724503517151, 0.6887357831001282, -0.6421647071838379], [-1.186708927154541, -1.4518399238586426, 0.5924875140190125, -0.8201508522033691, 0.4004804193973541], [-0.2549649775028229, 1.0084348917007446, 0.22316507995128632, -1.0548417568206787, -1.456915259361267], [-0.1702953428030014, 0.9577697515487671, 0.011441228911280632, 1.3621207475662231, 1.1220569610595703], [-0.34257808327674866, -0.5652394890785217, -0.42009857296943665, -0.12935949862003326, 0.3690343201160431], [0.2537211775779724, -1.0616375207901, 0.10684766620397568, -0.17195037007331848, 0.7673531174659729], [-1.1560642719268799, 1.4907505512237549, 0.11641968041658401, -0.3658483028411865, -0.23098917305469513], [-0.6894827485084534, 0.3873290717601776, 1.0478399991989136, 0.3358295261859894, 0.2674199640750885]], [[0.9429970383644104, 1.6675899028778076, 1.1559191942214966, -1.9966527223587036, 0.25967469811439514], [0.43644487857818604, 0.18731659650802612, 0.4014277160167694, 1.7736929655075073, 0.45130258798599243], [0.14854292571544647, 0.020402444526553154, -1.4072933197021484, -1.1799602508544922, -1.5458072423934937], [1.698012113571167, 0.19940191507339478, -1.0274252891540527, -0.22506116330623627, 0.12213967740535736], [-0.4461352825164795, 1.2698646783828735, 0.4469345808029175, 1.0222736597061157, -0.6153554320335388], [-1.1455284357070923, -0.18760980665683746, -0.0657796636223793, 0.80008864402771, -1.8588272333145142], [0.9788088202476501, 0.02467905916273594, 0.29458633065223694, 0.974183201789856, -0.5745551586151123], [0.011948115192353725, 0.5335407257080078, -0.2176821231842041, -1.1705853939056396, 0.609496533870697], [-0.1701427698135376, -0.28549185395240784, 0.5286744236946106, -0.5606153011322021, 1.4638910293579102], [0.9722857475280762, 1.508067011833191, -0.5400266647338867, 0.09651423245668411, -0.20238818228244781], [-0.47549116611480713, -0.8481131196022034, -0.1857476383447647, -0.0402691513299942, -0.8091081380844116], [-2.2228453159332275, -0.891063392162323, 1.7245615720748901, -0.3866286277770996, 1.0234789848327637], [-0.8149884343147278, -1.812621831893921, 0.32193106412887573, 0.7003971934318542, 0.4310087561607361]], [[0.007563864346593618, -0.00783684104681015, 0.9472079277038574, -0.4826298952102661, 0.1983519196510315], [1.9046251773834229, -1.8918079137802124, 0.38919326663017273, 2.099806070327759, -1.6537306308746338], [-1.6778984069824219, -0.6481953859329224, -0.20383338630199432, 1.5960966348648071, -0.6872761249542236], [-1.6532806158065796, 1.5817108154296875, -0.1429557502269745, 0.18662016093730927, 0.03669142723083496], [-1.1301591396331787, -0.6461141109466553, 0.6306129097938538, 0.819435715675354, -1.506881594657898], [0.1884486973285675, -0.9672539234161377, 0.8599482178688049, -0.13739600777626038, -0.2903268039226532], [0.3699176013469696, 1.8037314414978027, 0.2735344171524048, 2.415268659591675, 0.6653304100036621], [0.372285395860672, 0.36848098039627075, 0.38733339309692383, 0.22090871632099152, 2.5045225620269775], [-0.997568666934967, 0.25185519456863403, 0.1315745711326599, 0.3846410810947418, 0.09879091382026672], [1.0921865701675415, -1.050122618675232, 1.3304955959320068, 1.4455502033233643, -0.32593491673469543], [0.6332345604896545, -1.514504313468933, 1.2691106796264648, 1.234407663345337, 1.8036022186279297], [0.16141892969608307, 2.3420796394348145, 1.0902388095855713, 1.380439043045044, 1.0684943199157715], [0.24721890687942505, -0.6029185652732849, 0.5244542956352234, -0.24119094014167786, 1.3726534843444824]], [[-0.7798293828964233, -0.8610816597938538, 0.6569234132766724, -0.5155271887779236, 0.8908116817474365], [0.5935653448104858, 1.115140676498413, 0.8013765215873718, 0.057941146194934845, -0.24916112422943115], [-0.0990372747182846, 0.8486425280570984, 0.004171975888311863, 1.79574453830719, 2.022317886352539], [-0.6197938323020935, 0.9167895913124084, -1.2366828918457031, -0.3754108250141144, 1.4752908945083618], [-1.1912459135055542, -0.32617971301078796, -0.14927871525287628, -1.54719877243042, -1.0150724649429321], [0.6906181573867798, 0.19164973497390747, 1.1453312635421753, -0.2984282970428467, -2.311521530151367], [-0.039928898215293884, -1.2179712057113647, 0.7360718846321106, -0.906365156173706, 2.0085766315460205], [-1.7341434955596924, -0.4382059574127197, 2.3357224464416504, 0.8186226487159729, 0.34400951862335205], [0.9242737293243408, 1.4903833866119385, -0.824518620967865, 0.2792227864265442, 1.5693399906158447], [1.0914430618286133, -0.521561324596405, 0.46899446845054626, 0.865727424621582, -1.1557679176330566], [1.2995928525924683, 1.3881504535675049, -1.4173293113708496, -0.36480724811553955, -0.7342181205749512], [-1.3655438423156738, -0.6649008393287659, -1.6504648923873901, 0.8633127212524414, 0.4664930999279022], [-1.5655295848846436, -1.4273449182510376, 0.8286135196685791, -0.45190516114234924, 1.416000247001648]], [[-0.5016205906867981, -0.31103605031967163, 0.15022234618663788, -0.9969701766967773, -0.3676964044570923], [0.40239304304122925, 2.560121774673462, -0.2873799502849579, 0.26928892731666565, -1.0292538404464722], [-0.20960187911987305, -0.9483274221420288, -0.919377326965332, -0.844023585319519, -1.1211966276168823], [0.9026616811752319, 0.6253151297569275, -1.5830163955688477, 1.2692381143569946, 0.4386492073535919], [-0.3475271761417389, -1.2053484916687012, 0.09038082510232925, 0.3770468235015869, -1.4485679864883423], [-1.6671500205993652, 1.2366365194320679, -0.9386356472969055, 0.3097990155220032, -0.6785504817962646], [-0.6599479913711548, 1.0851304531097412, -0.8094278573989868, 1.6435123682022095, 0.6912254095077515], [2.1363441944122314, -0.003006810788065195, -0.09107450395822525, -0.06309468299150467, 0.3809841275215149], [1.6560624837875366, -1.1591871976852417, 2.0282557010650635, 0.32699671387672424, -0.3731776773929596], [-0.4217158854007721, 1.0223500728607178, 0.702307939529419, 0.7548294067382812, 0.3036291301250458], [1.3827804327011108, 0.46156805753707886, -0.41391870379447937, 0.48253554105758667, 0.26181256771087646], [0.23804382979869843, 1.2655866146087646, 0.25805261731147766, -1.177024006843567, -1.4321883916854858], [-0.901594877243042, -1.2071504592895508, -0.5539895296096802, -0.43282249569892883, 1.7323517799377441]], [[0.6978372931480408, 1.3214071989059448, 0.07362422347068787, -0.2390313446521759, -1.743773341178894], [-0.44657450914382935, 0.7409195899963379, -0.9767245054244995, 1.1277836561203003, 0.8197848200798035], [0.006039902102202177, -0.046014126390218735, 1.4025917053222656, -0.37469619512557983, -0.38562098145484924], [-0.35002344846725464, -1.4227564334869385, -0.44405627250671387, -1.1489900350570679, 1.1141490936279297], [-0.027296703308820724, -1.6411621570587158, 1.23264741897583, -0.49864518642425537, -0.5042688250541687], [-0.4725611209869385, -1.2501375675201416, -0.37135180830955505, 0.5463628768920898, -1.3787562847137451], [-0.25642165541648865, -0.7192246317863464, 0.49852874875068665, -0.05182979255914688, -2.4473648071289062], [0.39149922132492065, 0.6253200173377991, -1.1263761520385742, -0.5875007510185242, -0.3013913333415985], [-0.4139549732208252, -0.7925102710723877, 1.7007321119308472, -0.5413146615028381, -0.28155088424682617], [-0.08327765017747879, -0.055116936564445496, 1.8041938543319702, 0.7934857606887817, 1.8644465208053589], [-0.6172692179679871, 1.0739035606384277, 0.2691221535205841, 0.5315267443656921, -1.6609437465667725], [-0.5169835686683655, -0.11706855148077011, -0.4684400260448456, 0.3242899477481842, 0.48931363224983215], [0.7011098265647888, 0.3204008936882019, -1.1626924276351929, 0.5541406273841858, -1.0098140239715576]], [[0.9887136816978455, -1.0713852643966675, 0.8147621154785156, -2.5474188327789307, 1.361038088798523], [0.05198041349649429, 0.41318732500076294, -0.44082486629486084, -0.3912615478038788, -0.6988546252250671], [0.8313417434692383, -0.276138573884964, -0.35255464911460876, -0.6558283567428589, -1.540932536125183], [-0.9906859993934631, -0.3207358717918396, -1.3069041967391968, 1.3587138652801514, -0.7574526071548462], [-0.09430418163537979, 0.24115177989006042, -1.2634072303771973, 0.41039395332336426, -0.4350056052207947], [0.05003426969051361, -0.29467687010765076, -0.023918945342302322, 0.767245352268219, 0.13925312459468842], [-0.2102949321269989, 2.1005759239196777, 0.03450121730566025, 0.22016838192939758, -0.11063922941684723], [0.328766405582428, -1.286694049835205, 1.1543582677841187, 1.7156461477279663, -0.9442220330238342], [1.122574806213379, 0.6372643709182739, -0.10865114629268646, -1.23435640335083, -1.3668147325515747], [0.10037699341773987, 1.7375712394714355, 1.16423499584198, 1.6213535070419312, -0.9021843075752258], [-0.6753729581832886, -0.4312799870967865, 1.5239267349243164, -0.8847332000732422, -0.834437906742096], [0.6156591773033142, -0.07394442707300186, -1.2363593578338623, 1.0011963844299316, -2.082218885421753], [-0.3820728063583374, -0.11905062198638916, -0.6439498066902161, -0.1346030980348587, 0.37967172265052795]]], "dtype": "float32"}, {"__nd__": [[[-1.1222540140151978, 0.3676680326461792, -0.5415120124816895, -0.3572787344455719, -0.8168017268180847], [-0.15530231595039368, -0.29958629608154297, -0.11256483942270279, -1.0367586612701416, -0.17093005776405334], [-0.059269774705171585, 1.5506352186203003, 0.4375338852405548, -0.37696248292922974, 0.7131194472312927], [0.6831359267234802, -0.25972622632980347, -0.4356919229030609, 1.9108861684799194, 0.043740447610616684], [0.4232657253742218, -0.9885511994361877, -0.11194286495447159, -1.6248286962509155, 0.04431184381246567], [0.5924383997917175, -1.3552559614181519, -0.9989222288131714, -0.093989297747612, 0.5909005403518677], [0.03550980985164642, -1.1149230003356934, -0.8112338185310364, -0.22376210987567902, 0.06906501948833466], [-0.6951424479484558, -0.8341448903083801, 0.13831180334091187, 0.4224035143852234, -0.06577226519584656], [-0.5969136953353882, -2.0549705028533936, 0.07145509868860245, 0.006800655275583267, -0.539033055305481], [-1.252885341644287, -0.3148682415485382, 1.9824575185775757, -1.7266002893447876, -1.0478858947753906], [0.157459557056427, -1.393567681312561, -0.7200751304626465, 0.05378430709242821, -0.5051009654998779], [-0.4798695147037506, -0.20258618891239166, -0.9695115685462952, 0.7789375185966492, -0.951137900352478], [-0.9285954236984253, -0.4082508981227875, -0.3607700765132904, 0.8224391341209412, 0.272234171628952]], [[-0.32901236414909363, -0.5319584608078003, -1.0698868036270142, 0.8358714580535889, -0.8403887152671814], [0.8829288482666016, -1.4808751344680786, 0.43965306878089905, -0.508335292339325, -1.3408783674240112], [-2.509638786315918, 0.12500685453414917, 0.7186859250068665, -0.00905691273510456, -1.7037409543991089], [1.7827681303024292, -0.46175065636634827, -0.9043912887573242, 0.44215449690818787, -0.8486358523368835], [1.344854712486267, -0.018126197159290314, -0.3256239891052246, -0.0060583315789699554, -0.526534378528595], [0.07123719900846481, 1.6341781616210938, 1.1445006132125854, 1.087816596031189, 0.755795419216156], [-0.037990640848875046, 1.503899097442627, 0.21201331913471222, 1.0913376808166504, 0.12168990075588226], [-0.5788062214851379, -1.249621033668518, 0.5808078646659851, -1.1473486423492432, 0.007867047563195229], [1.0299136638641357, 0.4169272780418396, -0.5892786383628845, 0.040352243930101395, 1.7931205034255981], [-0.24756093323230743, 1.057126522064209, 0.5829051733016968, 1.5201308727264404, -0.13864605128765106], [1.1773301362991333, 0.24348515272140503, 0.6766063570976257, -0.9610138535499573, -0.5664607286453247], [-0.355289101600647, 0.02736815996468067, 0.0373401939868927, -0.3693905174732208, 0.6289555430412292], [-0.4656226634979248, 0.618336021900177, -0.9558016657829285, -0.025451060384511948, -0.19680018723011017]], [[0.5684874057769775, -1.4221659898757935, 0.16428032517433167, -1.2930686473846436, 0.6699332594871521], [-0.7601678967475891, 1.6111068725585938, 0.3239963948726654, 1.2642252445220947, 0.5073423981666565], [0.8026838302612305, -0.9044198393821716, -0.2202538251876831, -1.2854949235916138, 0.38874128460884094], [-0.47790956497192383, 0.8010618090629578, 0.7316423654556274, 0.38682541251182556, -0.2655317783355713], [0.444835901260376, -0.021985230967402458, -0.21606886386871338, 0.0766945630311966, 0.8348917365074158], [0.2222265601158142, 0.06731710582971573, -0.3279549181461334, -1.0258405208587646, -0.07046562433242798], [0.09225071966648102, 0.5638959407806396, -0.7065956592559814, -0.7710735201835632, -1.3671029806137085], [0.3842991590499878, -1.0516481399536133, 1.1567214727401733, -0.7719455361366272, -0.5328934192657471], [-0.3269735872745514, 0.28735342621803284, 1.4981951713562012, -0.7210862040519714, -0.6856248378753662], [1.7892476320266724, 1.8209950923919678, 1.7599209547042847, -0.46207883954048157, 2.107201099395752], [-0.02993505261838436, 1.6144676208496094, -0.8420647978782654, -0.04883426055312157, -1.265917420387268], [0.18104583024978638, 1.41506826877594, 0.7026144862174988, 0.8696553111076355, 1.1584910154342651], [0.9890459179878235, -0.3880910277366638, -0.2989305555820465, -0.03526288643479347, -0.2515586316585541]], [[1.0185987949371338, 0.6275083422660828, 0.390814870595932, 0.7465614080429077, 0.3761078417301178], [-0.2737594246864319, 1.0665907859802246, 0.6112574934959412, 1.160912036895752, -0.1630202829837799], [2.4740231037139893, 0.9906359910964966, 0.5877669453620911, 1.3195730447769165, -1.0081812143325806], [-1.8416550159454346, -0.610526442527771, 1.2558363676071167, -0.13602855801582336, -1.8709094524383545], [-0.6271805763244629, -0.44897252321243286, 1.5484998226165771, 0.06058374047279358, 1.5960859060287476], [-0.3344630300998688, -0.07134811580181122, 0.07877229899168015, -0.3502759039402008, -2.8800129890441895], [-1.36708402633667, 1.5114535093307495, -0.41531410813331604, 0.8984308242797852, -1.7077487707138062], [0.2813897728919983, -0.5874650478363037, -2.414994239807129, 0.36507856845855713, -0.5385885238647461], [-0.5253926515579224, 1.5539710521697998, 0.8943729996681213, -0.5372474193572998, 1.5607483386993408], [-1.4437862634658813, -0.4209657609462738, -1.098444938659668, -0.6970869302749634, 0.8578500151634216], [0.6059119701385498, 1.3254773616790771, 1.0876893997192383, 0.3527935743331909, -0.6450039148330688], [0.06373175978660583, -0.42626768350601196, -0.20916692912578583, -0.2402706742286682, 0.7909700274467468], [-0.5784673094749451, 1.1032829284667969, 0.7562487125396729, 0.1995832771062851, 0.6970626711845398]], [[-0.8355973958969116, 1.4081648588180542, 0.4708620309829712, 0.5236989855766296, 0.2946663796901703], [0.21605059504508972, -0.8662965297698975, -1.35696280002594, -0.4180373549461365, 1.2519489526748657], [-0.6086483597755432, -0.5956190228462219, 0.19327139854431152, -1.6619000434875488, 0.18575425446033478], [0.5290964841842651, 1.31858491897583, 0.2181076854467392, 0.6201506853103638, -0.49741309881210327], [-0.6855758428573608, -1.948272466659546, 1.703507661819458, -0.5047262907028198, -0.7478125095367432], [-0.2721562087535858, 0.3787209689617157, 0.23800460994243622, 0.5600739121437073, 0.8714345693588257], [2.4667532444000244, -0.990811824798584, 1.081518530845642, -1.7189483642578125, 0.8321517705917358], [0.682258665561676, 0.9565033912658691, -0.753421425819397, -0.2923193573951721, 0.8761852383613586], [-2.6145706176757812, -0.4665283262729645, -0.2019680291414261, -1.3528004884719849, 0.6187406778335571], [-0.41120705008506775, 1.1827011108398438, 0.4387317895889282, 0.1892811506986618, 0.6244680285453796], [0.15857389569282532, -0.8292313814163208, -2.0339789390563965, 0.6534440517425537, 0.23743890225887299], [-0.776679515838623, -0.1708085983991623, -0.9077247977256775, -0.1503024399280548, -0.14233613014221191], [-0.40465018153190613, -1.1208980083465576, -0.029389187693595886, -1.4458184242248535, -0.9983586668968201]], [[0.5426714420318604, 0.058246906846761703, 0.5954392552375793, 1.3080118894577026, 0.626380443572998], [1.0517451763153076, 0.5999457836151123, 0.23551645874977112, -0.054075099527835846, 0.8959712386131287], [0.9540044069290161, -1.0437533855438232, 0.3940906822681427, 0.8623314499855042, 0.030649244785308838], [-1.800113320350647, -0.019689731299877167, -1.8904482126235962, 0.3556062579154968, -0.910856306552887], [0.7444168925285339, -0.9329853653907776, 1.1476422548294067, 1.3134841918945312, -0.973352313041687], [0.46975597739219666, 1.1902800798416138, -0.93993079662323, -1.9746352434158325, 0.2541891932487488], [-0.4268108606338501, -0.09487096965312958, 0.5238611698150635, -1.8217573165893555, 1.1330331563949585], [-0.5452636480331421, -0.09618942439556122, 1.6793655157089233, 0.644263744354248, 0.08621610701084137], [-1.7177164554595947, 1.46415114402771, -0.7489407062530518, 0.6601549983024597, -0.1871398389339447], [1.1306288242340088, -1.2267426252365112, -0.27425116300582886, -0.011093898676335812, -0.5838776230812073], [-2.242546796798706, 0.9813042879104614, 0.7076417803764343, -0.8067735433578491, 0.09018845856189728], [0.7189261317253113, 0.6733496189117432, 1.4565967321395874, 0.20384232699871063, -0.23226670920848846], [-1.0563331842422485, 0.06297251582145691, -0.21429936587810516, 1.2292691469192505, -1.222732424736023]], [[-0.22375182807445526, 1.0564066171646118, 1.1717381477355957, -0.3614741861820221, 0.8568149209022522], [-1.1567846536636353, 1.7272536754608154, -0.832133412361145, 0.6708118319511414, -0.27058738470077515], [-1.965981125831604, -0.6902666091918945, 1.990035057067871, 0.22228814661502838, -1.3534481525421143], [-1.2626479864120483, 0.35444727540016174, 0.04317198693752289, 0.15633390843868256, -0.18960799276828766], [0.3343106806278229, 0.2551833689212799, -0.17198483645915985, -2.024754762649536, -0.32506123185157776], [0.18654850125312805, 0.9394649863243103, 1.5700557231903076, 0.19759736955165863, 1.636437177658081], [0.14885789155960083, 1.3744792938232422, 0.3383893370628357, 0.6799372434616089, -1.2165435552597046], [-0.45582881569862366, 1.0569032430648804, 0.9980818629264832, 0.7561705112457275, -0.34243735671043396], [-0.7603070139884949, -0.5652067065238953, -0.2589614689350128, 0.2941918671131134, -0.860085666179657], [-0.1453855186700821, -0.5789157152175903, -2.7525811195373535, -1.3570764064788818, -0.00672368798404932], [-0.40309014916419983, 0.4369201958179474, -2.727189779281616, 0.3740667998790741, -0.7710449695587158], [-0.3389413356781006, -1.428425669670105, 0.5602115392684937, -0.34594666957855225, 0.940296471118927], [-1.2723881006240845, 0.21390798687934875, 0.710085391998291, 1.2612143754959106, 1.0836840867996216]]], "dtype": "float32"}, {"__nd__": [[[0.47760218381881714, 0.35039740800857544, -1.0212595462799072, -1.3641576766967773, 1.823736310005188], [1.4342418909072876, -0.4057863652706146, -1.6083935499191284, 1.1950972080230713, 0.8667015433311462], [-0.37736397981643677, 0.11475156992673874, -0.06345001608133316, 0.9459690451622009, -0.10307972133159637], [1.2630467414855957, 0.5612764954566956, 1.0289561748504639, -0.6195361018180847, 0.4862310290336609], [-2.115266799926758, 0.9611249566078186, -1.787000298500061, 0.2160053700208664, 0.023175135254859924], [1.077210783958435, -0.2681546211242676, 1.3934485912322998, 1.0568805932998657, -0.4655199646949768], [0.6092107892036438, 0.5929430723190308, 0.42757052183151245, -1.6805317401885986, 0.1295839250087738], [-0.06795621663331985, -0.518487811088562, -1.4578666687011719, 0.3853212893009186, 0.421421617269516], [-1.3042430877685547, -1.2343292236328125, -0.04095456376671791, 0.38780662417411804, -0.13918140530586243], [-1.054925560951233, -0.31633374094963074, 0.06919194757938385, -0.7546601891517639, -0.8212921619415283], [0.9159188866615295, 1.2507919073104858, 1.1917498111724854, 0.8176285028457642, 0.4121674597263336], [0.7935105562210083, 0.4599859416484833, -0.2383018285036087, 1.3075770139694214, 0.7517992854118347], [-0.6932148933410645, -0.4829784333705902, -0.009923332370817661, 0.47518211603164673, -1.404350996017456]], [[-0.4871804714202881, 0.7175902128219604, 0.09214618802070618, 1.5408071279525757, 0.523646891117096], [-2.1049184799194336, -2.576308012008667, 2.022498369216919, -0.5105258822441101, -1.4779280424118042], [0.15364181995391846, -0.45050349831581116, 0.3696869909763336, 0.6744785904884338, -0.34696105122566223], [2.070316791534424, -0.3218803405761719, -0.387705534696579, -1.4168089628219604, 0.3578742742538452], [-0.13568203151226044, 0.3521594703197479, 0.5555621981620789, 0.24538123607635498, 1.0800503492355347], [-1.033263087272644, -1.651768445968628, -1.281827449798584, -0.08689271658658981, 2.4777984619140625], [0.07064755260944366, 1.063728928565979, 0.7579878568649292, -0.3933418095111847, 0.2786644399166107], [1.0668169260025024, -1.1397411823272705, -0.9833121299743652, 1.2975642681121826, -0.0830039456486702], [0.05269862338900566, -0.23597824573516846, -1.4686042070388794, 0.5467206239700317, -0.6217241883277893], [-0.6552548408508301, 1.5971184968948364, 1.6328201293945312, -0.5654687285423279, -1.2171276807785034], [0.29569128155708313, -1.1136281490325928, 0.28639519214630127, -1.211938738822937, 0.4455603063106537], [-0.24673449993133545, 1.287824273109436, 1.270784616470337, 2.051332950592041, 0.0030253229197114706], [-0.8918061256408691, -0.4780891537666321, -0.4727473258972168, -0.8041452765464783, 0.3153788149356842]], [[0.37462297081947327, -0.44898247718811035, -0.7989804148674011, 0.4571182131767273, -1.413705825805664], [-0.12794239819049835, -1.2271662950515747, -0.343730092048645, 1.216438889503479, -0.384317547082901], [0.19895197451114655, 0.4021473824977875, 1.050512433052063, -0.6913776397705078, 0.023058343678712845], [-1.4245624542236328, 0.5345568060874939, 0.16986119747161865, 0.02822261117398739, -0.22242297232151031], [-1.1689070463180542, 1.855183720588684, -1.2053842544555664, -0.472872793674469, -0.5198063850402832], [0.25402766466140747, -2.0936267375946045, 1.9524061679840088, -0.7320082783699036, -0.2668583393096924], [-0.2887183129787445, -0.4372043311595917, 1.5890883207321167, 2.220784902572632, 1.1869007349014282], [-2.4512546062469482, 0.35057124495506287, 0.8835334181785583, -0.9201022386550903, 1.50522780418396], [0.7411214113235474, -1.194444179534912, -0.5908377766609192, -0.44462689757347107, -1.4892629384994507], [-1.3100521564483643, 0.9438408613204956, 1.9815142154693604, 1.2740904092788696, 0.8541412949562073], [2.220139741897583, 1.8462467193603516, 0.4624856114387512, -0.1118537038564682, 0.9901623129844666], [-1.8174562454223633, -1.0167235136032104, -0.5830399394035339, -0.28079450130462646, 2.302478790283203], [0.04318754002451897, -1.1624770164489746, 0.10402167588472366, -1.480309247970581, 0.7639991044998169]], [[2.693903684616089, 1.4954625368118286, 1.5613573789596558, -0.26821407675743103, -0.3801072835922241], [0.7315121293067932, -0.6302905678749084, -1.779502034187317, 0.8020333051681519, -0.4793926477432251], [-1.333048939704895, -0.372022807598114, 1.020154595375061, -0.21892094612121582, -0.36823344230651855], [-0.8620932698249817, -1.218673586845398, 0.8310229182243347, -0.018096690997481346, 0.2226988524198532], [-0.00477781193330884, -0.18724501132965088, -1.073284387588501, -0.5465608239173889, 0.6029288172721863], [0.3396937847137451, 0.3257264494895935, 0.7271485924720764, 2.53166127204895, -0.5573675632476807], [1.0002394914627075, -1.6997641324996948, 0.46798211336135864, -0.2688300609588623, -0.8287737369537354], [-0.02278251200914383, -0.3300248980522156, -1.884631633758545, 0.0035265511833131313, -0.1962667554616928], [0.7816384434700012, 0.4745239019393921, 1.2047702074050903, -0.5597480535507202, 2.0078234672546387], [0.5680550932884216, 0.3988134264945984, 0.05848699435591698, -0.3948267698287964, 0.6679845452308655], [0.9586800932884216, 1.8242568969726562, 0.916202962398529, 0.10877516120672226, 0.2736433446407318], [-0.10860224068164825, -0.27585381269454956, 0.22065739333629608, 2.1781647205352783, -1.0585625171661377], [-0.2690194547176361, -0.16998402774333954, -1.2443941831588745, 1.5510470867156982, -0.8572679162025452]], [[1.0416626930236816, 2.0975263118743896, -0.41813403367996216, 0.2900371551513672, -1.3700752258300781], [0.1094023585319519, -1.0475764274597168, 0.4837299585342407, -1.510905385017395, -0.517853856086731], [-0.7357834577560425, -0.07110010832548141, 0.8777740597724915, 0.8222469687461853, -0.5567687749862671], [1.208673119544983, 0.3648480772972107, 0.6874605417251587, 0.7582134008407593, 1.7406998872756958], [-0.2939882278442383, -1.7595053911209106, 0.3472899794578552, -0.7934375405311584, -1.1579842567443848], [-1.140863060951233, 1.2201224565505981, 0.46245455741882324, -1.680749535560608, -1.2900996208190918], [-1.3079564571380615, -0.2621401846408844, 1.1091039180755615, -0.5221646428108215, 0.8621181845664978], [0.41109609603881836, 0.19483508169651031, 0.9099339246749878, -0.4291428327560425, -0.9306094646453857], [-0.8684312701225281, 0.06719990819692612, 0.34622007608413696, -1.3212004899978638, -0.89677494764328], [-1.3052364587783813, 0.24809396266937256, 0.7787829637527466, -1.5037074089050293, 1.3394241333007812], [-0.5509675145149231, -0.10382094234228134, -0.40589800477027893, 0.42930808663368225, -0.2606644928455353], [-0.16791477799415588, 0.3920767605304718, 0.2644345462322235, -0.15506380796432495, -0.7211809158325195], [-0.28405654430389404, 0.7662083506584167, -0.4183485209941864, -1.2245256900787354, -0.06711976230144501]], [[1.159727931022644, 0.9096888303756714, -1.2778412103652954, 0.41495174169540405, 1.2542577981948853], [0.7364850640296936, -1.9150012731552124, -0.8867743611335754, -1.1114228963851929, 0.9850388765335083], [-0.08427946269512177, -0.5053601264953613, 1.230561375617981, 0.18020334839820862, 0.23050802946090698], [1.2805403470993042, 1.5956531763076782, -0.4963149428367615, -1.5608916282653809, -1.3228555917739868], [0.024520747363567352, -0.02710559032857418, -0.13544416427612305, 1.0696322917938232, -0.16451187431812286], [-1.066399097442627, 1.1896857023239136, -1.6162711381912231, -1.8886029720306396, 1.0309001207351685], [1.2780641317367554, 1.2798490524291992, -0.8458110094070435, 0.6142866015434265, 0.22843849658966064], [1.321405291557312, 1.1745069026947021, -0.2440539449453354, -0.47263145446777344, -0.951258659362793], [-0.1948472261428833, -0.7563669681549072, 0.6626554727554321, 0.9987863302230835, 2.44490385055542], [0.7855221033096313, 0.08784865587949753, 0.004775389097630978, 0.9251866936683655, -1.3001359701156616], [-0.5526026487350464, 0.25649574398994446, -0.041854552924633026, -0.9296025037765503, 1.5429104566574097], [0.46239715814590454, 0.4881797730922699, 0.8410766124725342, 0.46167421340942383, 0.7075210213661194], [1.7687232494354248, -0.4180344045162201, 0.4986254870891571, -0.5099034309387207, 0.9814503788948059]], [[1.135603666305542, -1.7802670001983643, 0.220381960272789, 0.19154596328735352, 0.9611940979957581], [-0.27680903673171997, 0.7621917724609375, -0.4985666275024414, -0.3891329765319824, -1.5241773128509521], [0.0813496857881546, 0.13944777846336365, -0.8205835223197937, 0.9938976168632507, 1.4237538576126099], [0.8454287052154541, -0.19171839952468872, 0.40751346945762634, 1.0761101245880127, 0.09693575650453568], [0.36404523253440857, -0.3473477065563202, 0.29150575399398804, 1.1990923881530762, -0.37575608491897583], [0.2759759724140167, -0.6829253435134888, -0.9856480360031128, -0.025850562378764153, -0.3903341293334961], [0.262037992477417, 0.5845404863357544, -1.097748041152954, -0.013701211661100388, 0.38588857650756836], [0.053788501769304276, 1.3876054286956787, -2.1830313205718994, 1.26213538646698, 1.2824617624282837], [0.35242247581481934, -2.256290912628174, 1.1297662258148193, -0.44298869371414185, 1.6087839603424072], [0.7795220017433167, -0.8673478364944458, -0.8662974834442139, -0.06076144799590111, 1.2624276876449585], [0.41956183314323425, -0.508150041103363, -1.5609689950942993, -0.8119393587112427, 0.4341125190258026], [-0.07141515612602234, -1.5489360094070435, 0.18803875148296356, 0.184182271361351, -0.07012064754962921], [-0.0055509163066744804, -0.09705378115177155, 0.6291203498840332, 0.3204348683357239, -1.6174359321594238]]], "dtype": "float32"}], "e": {"__nd__": [[[-1.4203288555145264, -0.09801739454269409, -1.9244279861450195, -1.3640367984771729, 1.080998182296753], [1.3693045377731323, 0.1924942433834076, -1.5465023517608643, 0.9700666069984436, 0.9661641716957092], [-0.34598657488822937, 0.44311419129371643, -0.14583155512809753, 1.0206266641616821, 0.3344094753265381], [0.9465630054473877, 0.9160529375076294, 1.4575449228286743, -0.11640793085098267, 0.42816922068595886], [-2.3374764919281006, 0.44270259141921997, -1.709801197052002, 1.151305079460144, -0.03836553916335106], [1.577540397644043, -1.2500760555267334, 1.663529634475708, 0.9921467900276184, -0.844975471496582], [0.5670709609985352, 2.21163272857666, -0.05307537317276001, -1.4970130920410156, 0.15724311769008636], [0.10928075760602951, -1.3596686124801636, -1.4270002841949463, -0.060247570276260376, 0.5172462463378906], [-1.2025914192199707, -3.2025179862976074, -0.04013703018426895, 0.39706993103027344, -0.7440072298049927], [-0.6257145404815674, -0.13835777342319489, -0.7636356353759766, -0.5313080549240112, -1.2079980373382568], [0.955869734287262, 2.730255603790283, 1.1148114204406738, 0.8083802461624146, 0.024576663970947266], [1.3482705354690552, 0.15798047184944153, -0.3511720597743988, 1.022603988647461, 0.9715018272399902], [-0.05296438932418823, -0.6411058902740479, -0.38795265555381775, 0.7513814568519592, -1.331550121307373]], [[-0.7974381446838379, -0.16949832439422607, -1.1445565223693848, -0.12813794612884521, 0.3054192066192627], [-1.7195687294006348, -2.8537003993988037, 2.1989872455596924, -1.412156581878662, -2.0830698013305664], [-0.21914726495742798, -0.4479530453681946, -0.6417149305343628, 0.6851654052734375, 2.286694049835205], [5.097478866577148, -0.41395431756973267, 0.5414890050888062, -1.5163207054138184, 0.25422215461730957], [-0.7356691956520081, 0.32914164662361145, 0.4100295901298523, 0.23918795585632324, 1.4040560722351074], [-1.1148673295974731, -1.958356261253357, -1.3571122884750366, 0.7834569811820984, 1.0729053020477295], [0.033461976796388626, 1.1008437871932983, 0.8204441070556641, 0.66982102394104, 0.20874688029289246], [1.059901237487793, -1.8064649105072021, -1.109743595123291, 2.6406338214874268, -0.07820900529623032], [-0.12253373861312866, -0.3550075888633728, -1.78014075756073, 0.5240985155105591, 2.003208875656128], [-0.8959547877311707, 3.191336154937744, 1.3180358409881592, -0.4187544584274292, -1.1890673637390137], [-0.26411882042884827, -1.3201310634613037, 0.1607171595096588, -1.1732394695281982, 0.9038882851600647], [0.5430182218551636, 1.2634375095367432, 1.3351800441741943, 2.194149971008301, 0.6467480659484863], [-0.5123290419578552, -1.5988984107971191, -0.7804495692253113, -0.8219711184501648, 0.23055621981620789]], [[0.3789229393005371, -0.4378371834754944, -0.6433727741241455, 1.0811917781829834, -1.2808232307434082], [-1.5757774114608765, -4.275071144104004, -0.21763287484645844, 3.8710665702819824, -1.223325252532959], [-1.1478699445724487, 0.988388180732727, 1.095407485961914, -2.7431516647338867, -0.24411426484584808], [-0.6344438195228577, 1.801604986190796, 0.06526871770620346, 0.1004120334982872, -0.23216570913791656], [-1.6716424226760864, 1.8693886995315552, -1.3416401147842407, -0.41002652049064636, -1.777889370918274], [0.2959059774875641, -2.1587395668029785, 1.6703819036483765, -0.5910618901252747, -0.24640028178691864], [-0.2545931339263916, 0.5799124240875244, 1.3958101272583008, 0.35843515396118164, 0.2773255705833435], [-2.308185577392578, -0.036941081285476685, 1.3315702676773071, -1.0906317234039307, 0.17058420181274414], [1.0673000812530518, -1.122072696685791, -0.3937133848667145, -0.7219862937927246, -1.556996464729309], [0.6441401243209839, -0.968427300453186, 4.323081016540527, 0.6061322689056396, 0.16733086109161377], [2.201183795928955, -0.5988714694976807, -0.6061877608299255, -0.17213508486747742, -1.2930490970611572], [-1.7882319688796997, 2.2974791526794434, 0.1829776167869568, 0.9197115898132324, 3.5403199195861816], [0.2876983880996704, -0.9284897446632385, -0.052753739058971405, -1.471804141998291, 0.4186962842941284]], [[1.8995704650878906, 0.9551265835762024, 1.8180928230285645, -0.65308678150177, -0.04506602883338928], [0.5690180063247681, 0.5591081976890564, -1.2896546125411987, 0.8692978620529175, -0.43877434730529785], [-1.5780694484710693, 0.46867305040359497, 1.0226067304611206, 2.1506950855255127, -2.4070963859558105], [0.27935320138931274, -1.778397798538208, -0.7220484614372253, 0.03296989947557449, -2.5374367237091064], [0.7423484921455383, -0.04079928994178772, -1.3044424057006836, -0.6402959227561951, -1.017214059829712], [0.10870754718780518, 0.3120526075363159, 0.8173689842224121, 2.6361935138702393, 6.099844932556152], [1.0548256635665894, -3.540670871734619, 0.1622810661792755, -1.0831364393234253, -4.258917808532715], [-0.5107527375221252, -0.07259422540664673, -7.525387763977051, 0.30238813161849976, -0.3815463185310364], [0.2960318326950073, 2.790536403656006, 0.4673430323600769, -0.7097597718238831, 4.4571685791015625], [-1.0077555179595947, 0.618372917175293, -0.4566776156425476, -0.9983140230178833, -0.3234909772872925], [1.7461190223693848, 3.6642189025878906, -0.6254110932350159, -0.019926495850086212, 0.7472169399261475], [-0.1956307590007782, 0.0075719356536865234, 0.5658800601959229, 1.970736026763916, -0.6895804405212402], [0.6365882158279419, -1.7447493076324463, -0.6177562475204468, 1.460854411125183, 0.12977302074432373]], [[1.4608155488967896, 1.6595362424850464, -0.347400039434433, -0.23207509517669678, -1.478422999382019], [0.19633960723876953, -3.2654011249542236, 0.8736938238143921, -1.6234781742095947, -1.8064271211624146], [-0.6082096099853516, 0.4937417507171631, 0.7000846862792969, 2.2249298095703125, -0.7650358080863953], [1.6862682104110718, 1.1893792152404785, 0.34219250082969666, 1.5453323125839233, 1.522510051727295], [-0.05573199689388275, 0.5888417959213257, 0.5012544393539429, -0.9837429523468018, -0.07472693920135498], [-0.6871378421783447, 1.6884626150131226, 0.23905494809150696, -1.5072392225265503, -1.8814120292663574], [-2.935885429382324, -1.337300181388855, 0.2336927056312561, -3.3472774028778076, 1.4373226165771484], [1.8686354160308838, 0.19195905327796936, 0.9785513877868652, -0.4106990396976471, -0.5967968106269836], [-5.1983232498168945, 0.6079935431480408, -0.06342273950576782, -1.7635618448257446, -1.127675175666809], [-1.1318238973617554, 1.4572285413742065, 1.0869077444076538, -1.3608324527740479, 1.5290307998657227], [-0.331694632768631, -0.4865676760673523, 0.4360038936138153, 0.7446180582046509, -0.19850000739097595], [-0.35279855132102966, 0.17590367794036865, 0.030193790793418884, 0.021845772862434387, -0.5173287391662598], [0.0807739794254303, 2.1193008422851562, -0.4020672142505646, -0.5987429618835449, -1.7966281175613403]], [[1.5384242534637451, 0.9866567254066467, -1.2340024709701538, 0.1022959053516388, 0.16199231147766113], [0.26680248975753784, -1.470489740371704, -1.1168090105056763, -1.172407865524292, 1.7195425033569336], [-0.0785173699259758, -0.4573327302932739, 1.7833096981048584, -0.1429089605808258, 0.21868903934955597], [1.910622239112854, 1.6236668825149536, 0.34315043687820435, -1.9694796800613403, -2.3376853466033936], [0.004200620576739311, 1.5040746927261353, 1.2791941165924072, 0.4146696925163269, 0.3263193368911743], [-1.2883875370025635, -0.298328161239624, -1.2672260999679565, -2.967470407485962, 0.6804351806640625], [1.387507677078247, 1.3480825424194336, -0.5846511125564575, 0.7087079286575317, -2.5445070266723633], [1.1079349517822266, 1.1143577098846436, -2.135651111602783, -0.8511369228363037, -0.9772434234619141], [0.5162100195884705, -1.9167218208312988, -0.6110920906066895, 0.6414347887039185, 2.4975931644439697], [0.6913660168647766, 0.15546295046806335, -0.49002689123153687, 0.9163838624954224, -2.388744592666626], [0.8316524028778076, 1.3103219270706177, 0.14858752489089966, -1.358424186706543, 1.3931125402450562], [0.0907241702079773, 0.4093517065048218, 0.15874838829040527, 0.5277782082557678, 0.5938697457313538], [1.0281176567077637, -0.3978579640388489, 0.747789740562439, 0.17128455638885498, 2.2161827087402344]], [[0.9143771529197693, -2.91208553314209, 1.175069808959961, 1.1123721599578857, 2.127351760864258], [-0.3369391858577728, 1.4758710861206055, -0.13174152374267578, -0.6515958309173584, -1.335076093673706], [-1.5530524253845215, 0.3300570249557495, -1.5221796035766602, 0.8481147289276123, 3.5093259811401367], [2.096316337585449, -0.30540233850479126, 0.35109180212020874, 1.2885231971740723, 0.2405548393726349], [0.3325183391571045, -0.28580978512763977, 0.5087926387786865, 0.36814528703689575, -0.23435263335704803], [0.2853097915649414, -0.9597639441490173, -1.0232020616531372, 0.12575510144233704, -0.16245514154434204], [0.2307339310646057, 3.471738815307617, -1.0860731601715088, 0.13599947094917297, 0.5204859972000122], [-0.09607270359992981, 0.027694344520568848, -1.030887246131897, 2.5594563484191895, 1.6057987213134766], [-0.5010790228843689, -2.6164770126342773, 1.157902717590332, -0.8061262965202332, 2.7843618392944336], [0.7649286389350891, -1.8732551336288452, -4.070948600769043, -2.2610621452331543, 1.26849365234375], [0.6917980313301086, -0.6965849995613098, -5.717006206512451, -1.1428886651992798, 1.077501654624939], [-0.28008750081062317, -1.4433119297027588, -0.504584014415741, -0.16217829287052155, -2.0280237197875977], [0.4805939793586731, -0.12251965701580048, 0.17186099290847778, 0.15067151188850403, -1.2059917449951172]]], "dtype": "float32"}}, {"a": [{"__nd__": [-81.08938598632812, -76.07254791259766, 110.61808013916016, -132.0750274658203, 120.73637390136719, -206.57949829101562, 116.94261169433594, 46.30970001220703, -121.24415588378906, 107.08273315429688, -88.12992095947266, -33.4168701171875, -120.87408447265625, 25.02926254272461, 57.87395095825195, 102.26753997802734, -18.699254989624023, -83.68363189697266, -6.97778844833374, -42.35639953613281, 123.43115997314453, 42.67376708984375, -168.921630859375, 132.90792846679688, -51.08287811279297, -118.34219360351562, -50.22642135620117, -158.72117614746094, 37.22941589355469, -194.10020446777344, -95.56647491455078, -120.38326263427734, 121.82318115234375, 42.81501388549805, -80.64897155761719, 82.0942611694336, 151.9713897705078, 98.04230499267578, 77.63731384277344, 19.155038833618164, -74.52376556396484, -41.85972595214844, -29.247909545898438, 77.22937774658203, 14.180599212646484, 74.01007080078125, 145.82533264160156, -126.95803833007812, 9.320198059082031, -102.78915405273438, -230.6041259765625, 68.80241394042969, 14.759434700012207, -84.33519744873047, 42.462196350097656, -203.45716857910156, 261.3716735839844, 74.72876739501953, -61.66438293457031, 11.140280723571777, -52.204437255859375, -205.769775390625, -32.886390686035156, -31.119834899902344, -139.66827392578125, 79.29442596435547, 117.1885757446289, 38.78455352783203, 181.9114227294922, 45.42392349243164, -78.40531158447266, 194.10536193847656, 39.254432678222656, 46.30974197387695, -71.31613159179688, -138.80372619628906, -1.0372352600097656, -33.52787399291992, 45.51702117919922, -91.59334564208984, -132.11676025390625, -106.57505798339844, -48.52863311767578, -109.78305053710938, 81.92060089111328, 14.073049545288086, -115.56980895996094, 10.311728477478027, -77.6794204711914, 92.67057037353516, 22.696197509765625, -96.40512084960938, -49.437461853027344, 35.07069396972656, -117.98722839355469, -19.03114128112793, 137.12193298339844, 2.0350539684295654, -11.108266830444336, 12.33866024017334, 0.43203920125961304, 10.4353666305542, -248.61500549316406, -121.70272064208984, -133.284912109375, -123.41785430908203, -117.26512908935547, 189.0908966064453, 16.07786750793457, 11.404318809509277, -55.75514602661133, -87.22644805908203, -12.487760543823242, 45.57849884033203, -20.445711135864258, 137.14410400390625, 62.6136360168457, -247.7266082763672, 249.18963623046875, -101.76090240478516, 16.04143714904785, 77.69011688232422, -99.93836212158203, -115.8137435913086, 149.50765991210938, -110.43798065185547, 231.4469757080078, 122.60234069824219, -5.800921440124512, 122.98020935058594, 142.87904357910156, -13.10739803314209, -175.74256896972656, 107.21106719970703, 75.93484497070312, -187.1097412109375, 138.98318481445312, -61.93680191040039, -127.66676330566406, -90.10697937011719, 111.5914306640625, 170.39401245117188, 138.0290069580078, 22.598365783691406, -28.392780303955078, 61.29039001464844, -17.726482391357422, -2.9804444313049316, -16.19080924987793, 3.032404661178589, -136.2450714111328, -102.9192123413086, -14.23539924621582, 75.09931182861328, -64.38126373291016, 157.00953674316406, 160.95277404785156, -132.65203857421875, 2.6312780380249023, 43.30961227416992, -69.41830444335938, 233.1388702392578, -238.30715942382812, -6.933366298675537, -1.483771800994873, -63.125091552734375, -56.59352493286133, -134.4640655517578, 3.4495818614959717, 132.7674102783203, -130.44578552246094, 6.881765365600586, 97.38607788085938, 141.0583038330078, -45.10757064819336, 96.99870300292969, -186.12136840820312, 105.12284088134766, -135.9103240966797, 90.53811645507812, 110.50688171386719, -3.653874635696411, -46.98450469970703, 15.226024627685547, 6.984302520751953, -29.067588806152344, -27.94375991821289, -53.258365631103516, 123.97376251220703, 24.072200775146484, -40.33577346801758, 107.42948913574219, -33.16303634643555, -18.304332733154297, 13.852713584899902, -29.652023315429688, 82.86035919189453, 60.52596664428711, 214.3280487060547, 3.819575786590576, 223.7882843017578, -106.11137390136719, 7.116659164428711, 49.502098083496094, -217.53900146484375, 117.90718078613281, 2.256598949432373, -138.7140655517578, 128.25567626953125, 100.02696228027344, -83.1213607788086, 26.769203186035156, 13.734867095947266, -69.93769836425781, 95.68416595458984, -145.9114990234375, 83.4370346069336, -65.09542846679688, 109.00423431396484, 240.38462829589844, 127.81275177001953, -108.31697845458984, 70.33192443847656, -30.835586547851562, 89.2441177368164, 163.5940704345703, -129.55101013183594, -206.8031768798828, -36.24748992919922, -32.646202087402344, -109.10216522216797, 13.111238479614258, -10.381834030151367, 16.363845825195312, 1.9258919954299927, -117.98480987548828, 155.55792236328125, 49.42527389526367, -65.59781646728516, 123.27653503417969, -86.31582641601562, -192.058837890625, 18.464868545532227, -85.06085968017578, -118.27973175048828, 101.1450424194336, -25.378660202026367, 26.30365562438965, 123.74574279785156, -113.91938018798828, 70.75323486328125, 12.49770450592041, 119.01521301269531, 47.70676040649414, 26.705795288085938, 64.64900207519531, -14.873705863952637, -122.57901763916016, -74.86613464355469, -66.32795715332031, -28.823036193847656, -12.078944206237793, -5.270649433135986, 92.53477478027344, -50.2591552734375, -50.438297271728516, -0.9616522789001465, -22.578907012939453, -60.62947463989258, 121.05973815917969, 18.7310848236084, 150.23077392578125, 54.37075424194336, -23.381710052490234, 138.95440673828125, -53.824222564697266, 46.95292663574219, 144.1782684326172, 102.06831359863281, -74.9982681274414, 21.366331100463867, 26.737377166748047, -20.908222198486328, -40.54225540161133, 9.106273651123047, -200.1902313232422, 147.14144897460938, 123.87566375732422, 101.59979248046875, 88.57267761230469, -107.76950073242188, 17.99998664855957, 325.81988525390625, -84.55799102783203, 150.2008514404297, -46.549015045166016, -168.1520538330078, -27.930883407592773, 103.57672882080078, -67.29547119140625, 59.7647590637207, 44.913352966308594, 101.70536041259766, -33.24584197998047, 49.84211730957031, 24.20924949645996, 37.337120056152344, 49.16741180419922, -121.22395324707031, -61.579925537109375, 92.57929229736328, 125.84974670410156, 85.97933959960938, 96.19475555419922, 35.081871032714844, -36.769954681396484, 127.36932373046875, 64.74224090576172, 28.1488094329834, -13.38095474243164, 29.88055419921875, -133.43807983398438, -105.45929718017578, -9.290221214294434, -164.19155883789062, 29.382488250732422, -77.82006072998047, -30.599546432495117, 61.694580078125, -103.51573181152344, 49.5137825012207, -32.73887252807617, 77.89789581298828, 104.6629409790039, 199.57577514648438, 159.57371520996094, 17.09355354309082, -146.8334503173828, 69.85061645507812, 174.0719451904297, -19.093236923217773, -40.86423110961914, -76.44119262695312, 47.67707061767578, 24.461088180541992, -52.386653900146484, -179.26065063476562, -52.63918685913086, -29.67738151550293, 53.317283630371094, -21.545642852783203, 6.743775844573975, -43.52754592895508, -42.357704162597656, 18.424413681030273, 57.968963623046875, -101.8519515991211, -171.607177734375, 49.94496536254883, -5.981855392456055, 132.1875762939453, 83.36689758300781, 58.311729431152344, -12.636451721191406, 100.11431884765625, 40.52619171142578, -46.11903381347656, -200.81944274902344, -144.85926818847656, 41.4263801574707, 243.89572143554688, 106.98739624023438, -16.300521850585938, 12.230113983154297, 34.90938186645508, -68.44222259521484, 41.211639404296875, -71.63320922851562, -37.83247375488281, 106.26338958740234, 164.1607208251953, 49.32911682128906, -98.77881622314453, 23.063127517700195, -113.57120513916016, 163.12841796875, -119.56660461425781, -44.44466018676758, -165.5232696533203, 52.96417999267578, -50.152896881103516, -121.99468994140625, -115.04951477050781, -129.5459747314453, -113.73551940917969, -26.51885223388672, -152.22525024414062, -116.34040069580078, -13.43692684173584, 81.1991958618164, 1.751125454902649, 210.4575653076172, 102.8471450805664, 76.0870361328125, 241.73443603515625, -48.732757568359375, 155.42877197265625, -81.42198181152344, -98.7167739868164, 70.39739227294922, -75.27297973632812, -106.3215103149414, 21.353656768798828, -84.49244689941406, 121.29222106933594, 239.20721435546875, 161.666015625, 24.25376319885254, -55.272762298583984, -49.40568161010742, 47.77804946899414, -201.57957458496094, 108.071533203125, -194.43438720703125, -71.25114440917969, -45.52661895751953, 31.655899047851562, 35.735626220703125, 96.73890686035156, -45.70808410644531, 139.12454223632812, -117.17792510986328, 57.38308334350586, -113.79472351074219, 129.7201385498047, -66.7710189819336, -15.397040367126465, 113.28718566894531, 2.8609981536865234, 4.7794413566589355, 53.92442321777344, -175.1727294921875, -3.4140310287475586, -58.717132568359375, -72.30045318603516, -12.390758514404297, -111.51988220214844, -22.19293785095215, 105.09705352783203, 23.184688568115234, 119.51664733886719, 34.134464263916016, -59.79450225830078, -63.09810256958008, -115.34243774414062, -70.70854949951172, 99.59699249267578, -15.482561111450195, 56.99829864501953, 134.96096801757812, -134.34213256835938, -50.69981384277344, -140.04933166503906, 87.10807800292969, 52.037261962890625, 61.50221252441406, -18.055076599121094, -1.573682188987732, -2.4099574089050293, 132.5821075439453, -226.90773010253906, 30.26603126525879, 117.60963439941406, -89.29075622558594, 2.5202198028564453, 43.07917022705078, 43.742183685302734, -174.62184143066406, 1.2222304344177246, 46.64261245727539, -178.3135223388672, -9.774680137634277, -71.5462417602539, -0.47029346227645874, 32.23923873901367, -145.1321563720703, 98.49595642089844, -59.19093322753906, 15.73443603515625, -47.415748596191406, 107.23466491699219, 162.3248291015625, -17.60211181640625, -62.06580352783203, 103.5630111694336, 120.36253356933594, 56.917388916015625, -70.60440063476562, -1.3375426530838013, -43.57609176635742, 127.60395050048828, 42.42036437988281, 159.6876983642578, 48.013607025146484, 122.6644515991211, 185.9315643310547, 40.02859878540039, 131.91952514648438, -143.07421875, -1.0006661415100098, -57.611572265625, -113.8636474609375, -68.68547821044922, 40.625675201416016, 3.9319489002227783, 82.00727844238281, -128.95797729492188, -9.72697925567627, -18.795740127563477, 98.26971435546875, -45.40098190307617, 7.358049392700195, 38.93702697753906, -121.15111541748047, 21.417179107666016, -69.63133239746094, 0.20726384222507477, 105.0906753540039, 99.45870971679688, -5.507408142089844, 14.738628387451172, -9.205195426940918, -74.44243621826172, 145.06080627441406, -7.950112819671631, -39.55217742919922, 68.3034439086914, -55.16263961791992, 119.95455169677734, -169.90304565429688, -106.45365905761719, -78.13536071777344, -10.405716896057129, -45.08250427246094, -23.876155853271484, 79.11956787109375, -38.62077713012695, 120.30452728271484, 41.45818328857422, 15.507203102111816, -76.20317077636719, -3.385261058807373, 10.697380065917969, -37.49237060546875, -25.819751739501953, 52.409385681152344, 154.18130493164062, -32.74869918823242, -150.16273498535156, 112.75118255615234, -117.71170043945312, 81.161376953125, 49.188716888427734, 2.64614200592041, -11.890118598937988, -151.40505981445312, 140.1417694091797, -45.97740936279297, 106.38957214355469, -42.313541412353516, -6.766296863555908, -148.15213012695312, 5.494205474853516, -202.46536254882812, -62.79411315917969, 190.4724884033203, 6.376922130584717, -95.90420532226562, 47.06503677368164, -40.62492752075195, 0.2446550875902176, -96.19095611572266, 29.846054077148438, -61.02864074707031, 57.95058059692383, 9.856281280517578, -78.54385375976562, -70.96659851074219, 6.665185928344727, -121.04984283447266, 12.923847198486328, 93.19957733154297, 126.82501983642578, -159.1058349609375, 24.283084869384766, 118.7803726196289, 184.423583984375, -112.13082122802734, -71.6465072631836, 25.698888778686523, 178.00181579589844, 6.607577323913574, 7.061422824859619, 27.903541564941406, 198.88177490234375, -252.4751434326172, 52.58387756347656, -105.94541931152344, -5.272496700286865, 79.07987976074219, 113.02818298339844, -49.767024993896484, -94.631591796875, 51.08994674682617, -34.96342849731445, -36.60511779785156, 63.67428970336914, 58.98836135864258, 197.47186279296875, -2.5392673015594482, -61.31267547607422, -38.962425231933594, 64.9571533203125, 173.63209533691406, -41.10274887084961, -130.87716674804688, 51.64145278930664, -11.27315902709961, 4.535286903381348, -29.92064094543457, 122.05918884277344, 66.4744873046875, 94.32904052734375, -128.0109100341797, 87.97447204589844, 217.54586791992188, 29.16562271118164, -2.0871081352233887, -22.411909103393555, -2.641416549682617, -36.74677658081055, -0.9496009945869446, 43.43230056762695, -133.03350830078125, 16.03729820251465, 6.331874370574951, 39.634056091308594, 0.696341872215271, -52.80579376220703, 88.51611328125, -27.48383903503418, -133.43124389648438, -88.20648193359375, 23.474056243896484, -118.15755462646484, 136.02105712890625, 92.56075286865234, -78.5918960571289, -88.23068237304688, 13.661652565002441, 31.307050704956055, 102.3182144165039, 122.14887237548828, 136.671630859375, -70.912841796875, -58.023677825927734, -26.935821533203125, -11.615201950073242, -153.01060485839844, -119.509521484375, 18.444847106933594, 18.319889068603516, 35.48931884765625, 162.34609985351562, 97.63999938964844, -49.1818962097168, -10.607796669006348, -21.283695220947266, 51.056861877441406, -75.5149154663086, 9.961181640625, -202.47361755371094, 59.36308670043945, 103.42500305175781, -51.0240478515625, 98.92559051513672, -12.958793640136719, -88.51216125488281, -74.15283966064453, -61.41131591796875, 27.00408172607422, -48.49555206298828, 67.3771743774414, 50.47505569458008, -53.017967224121094, -20.230037689208984, -96.6680679321289, -31.566253662109375, -16.602624893188477, 106.36526489257812, -139.1956024169922, 11.637545585632324, 69.58685302734375, 48.42924499511719, 14.926746368408203, 25.21721076965332, -135.27700805664062, 2.5607752799987793, 90.73164367675781, 54.295616149902344, 126.18595123291016, 129.4379425048828, -79.59781646728516, 13.415318489074707, -99.62069702148438, 55.38560104370117, -117.11237335205078, 2.1239724159240723, -4.288614749908447, 78.3499526977539, -93.51822662353516, -63.11842727661133, -60.661991119384766, -100.01155090332031, -208.42013549804688, -167.16177368164062, 58.25593948364258, -40.38225555419922, 23.061279296875, -82.9549331665039, -69.53318786621094, -52.035301208496094, 87.2146987915039, 120.17815399169922, -96.8079605102539, 149.8350830078125, 19.572175979614258, 115.22351837158203, -11.692072868347168, -98.69187927246094, -106.28962707519531, -119.33646392822266, 160.28089904785156, -163.2159423828125, 49.23957443237305, -69.92794799804688, 105.54415130615234, 72.12269592285156, 71.43738555908203, 6.8602800369262695, 41.08760452270508, 107.85240173339844, -76.7381362915039, -53.407562255859375, -74.24639892578125, -253.43881225585938, 5.991458892822266, -28.53030776977539, 203.6112823486328, -31.7209415435791, -24.415803909301758, -165.79566955566406, 46.77764892578125, 23.539457321166992, 72.1077880859375, -75.49852752685547, -80.1498031616211, 12.948862075805664, 1.8544535636901855, -148.4665069580078, -73.03915405273438, 129.56573486328125, -0.35196059942245483, -71.98040008544922, -15.80669116973877, 85.31372833251953, -133.09121704101562, -45.278255462646484, 85.63433837890625, -127.08229064941406, 102.87716674804688, -84.34132385253906, 13.297118186950684, -74.81761932373047, 123.05841064453125, -140.16476440429688, -97.43757629394531, -130.35586547851562, -8.713802337646484, -96.87261962890625, 117.78925323486328, -120.55237579345703, 94.86575317382812, -70.03584289550781, 64.70056915283203, -71.24844360351562, -132.35850524902344, -24.255630493164062, -176.8915252685547, -64.86980438232422, 96.31920623779297, 163.88099670410156, -102.29061889648438, -75.12067413330078, -50.338829040527344, -143.3312225341797, 108.84715270996094, 71.8154525756836, -72.19812774658203, 44.942291259765625, -170.98240661621094, 130.43389892578125, 24.882179260253906, 118.06417846679688, 90.95084381103516, -128.1725311279297, 13.128751754760742, 17.07387351989746, 86.50012969970703, -130.6295928955078, -43.497711181640625, -117.989501953125, 72.28633117675781, -30.2674560546875, 193.36634826660156, -12.82005500793457, -144.4437255859375, 104.04326629638672, -9.464655876159668, -14.97192096710205, 3.439486026763916, 107.4698486328125, -5.025943279266357, -105.8121337890625, 55.35224533081055, 129.9029541015625, -364.9237976074219, 54.3065299987793, -64.89852905273438, -13.399340629577637, 198.5692138671875, 156.42404174804688, 251.72442626953125, 181.50643920898438, -178.3128204345703, -113.18026733398438, 46.834190368652344, 101.7764892578125, 44.698638916015625, 100.52954864501953, 300.70196533203125, 129.1614990234375, 86.85806274414062, -88.03903198242188, 192.16860961914062, -101.32874298095703, 152.0892791748047, -219.78521728515625, -176.17808532714844, 15.051345825195312, 106.38569641113281, -104.59107971191406, -84.3028335571289, 27.879417419433594, -118.9611587524414, -21.493925094604492, -170.75018310546875, 105.39095306396484, -42.59680938720703, 114.50650787353516, -70.66529846191406, 16.33782196044922, 24.254352569580078, 40.05965805053711, 1.7980709075927734, 3.985826015472412, 28.398889541625977, -161.58706665039062, 32.46105194091797, -53.06275939941406, 38.592529296875, -76.97750091552734, -237.82357788085938, 58.006919860839844, 41.19953918457031, 49.448944091796875, -137.21681213378906, 18.195919036865234, 33.99142074584961, 16.202878952026367, -82.7942123413086, 117.40013122558594, 89.94615936279297, -102.84358215332031, 39.85797119140625, 158.80238342285156, -105.57241821289062, -189.86122131347656, 20.565690994262695, -17.354782104492188, -95.95430755615234, 35.577613830566406, -9.20702075958252, 29.761409759521484, 82.7530288696289, -130.5035400390625, -150.15025329589844, 70.91300201416016, 59.754608154296875, -26.683837890625, -20.154809951782227, 178.58876037597656, 161.24191284179688, 72.51637268066406, -25.48092269897461, 186.54046630859375, -79.38402557373047, -5.163500785827637, -90.803466796875, 98.39619445800781, 55.688743591308594, 157.8765411376953, -29.329782485961914, 44.10026550292969, -76.45648193359375, -175.1329345703125, -102.68124389648438, 109.45822143554688, -98.96443176269531, -135.631591796875, 38.897430419921875, 55.2990837097168, 28.149578094482422, 157.9579315185547, -53.54597091674805, 34.72496795654297, 102.32889556884766, -49.81188201904297, 8.321793556213379, -132.27520751953125, -77.9300537109375, -5.739438533782959, -156.06912231445312, -101.65653228759766, -140.55555725097656, -141.4843292236328, 29.34646224975586, -12.129326820373535, -201.46942138671875, -123.52958679199219, 63.850013732910156, 138.57577514648438, -159.119384765625, -7.476179122924805, -22.055665969848633, -90.92460632324219, -22.073246002197266, -113.40419006347656, 50.016380310058594, -165.7067108154297, -31.410327911376953, 37.2093505859375, 89.1348876953125, -77.89032745361328, 30.78862953186035, -108.72322082519531, -56.71392059326172, -3.9325926303863525, 40.075416564941406, 63.173248291015625, -52.14203643798828, 191.26470947265625, -35.29004669189453, 45.96254348754883, 82.03137969970703, 110.19412231445312, 49.72507095336914, 67.33496856689453, -0.9080740809440613, -8.081339836120605, 67.17034912109375, 59.401710510253906, 89.334716796875, -132.8759765625, 178.08529663085938, 114.2303695678711, 86.66622924804688, 121.09042358398438, 4.727505683898926, 148.24114990234375, 115.8365478515625, 147.50100708007812, 0.2465347945690155, -61.8525276184082, -135.08709716796875, 107.26789093017578, -84.88512420654297, -75.84325408935547, 106.82951354980469, 23.07181739807129, -0.09094317257404327, -83.42569732666016, 34.16294860839844, -179.67674255371094, -72.43183898925781, 79.45462036132812, 124.26525115966797, -49.167964935302734, 0.6206589341163635, 1.43045973777771, -134.87388610839844, 95.64299774169922, 51.132347106933594, 78.2736587524414, 60.57903289794922, -60.18953323364258, 43.44786834716797, 187.8395233154297, 5.430160999298096, -64.68180847167969, -141.32080078125, 181.24948120117188, -232.36534118652344, -28.638994216918945, 62.72701644897461], "dtype": "float32"}, {"__nd__": [-108.8140869140625, 169.97259521484375, 86.2524185180664, 151.80709838867188, 43.16222381591797, -95.53478240966797, 172.52227783203125, 8.087709426879883, -62.903438568115234, -20.869169235229492, -53.795169830322266, 13.896549224853516, -148.26260375976562, 31.41843605041504, -46.96072769165039, 4.501060485839844, -22.981273651123047, 59.16488265991211, -66.42566680908203, 166.2427978515625, 141.0765838623047, 1.9349204301834106, -49.113624572753906, -116.8537826538086, -24.02410125732422, 140.33370971679688, -108.50668334960938, 100.89045715332031, 6.375960826873779, 36.69369888305664, 5.127028942108154, -49.162967681884766, -19.190780639648438, 29.227018356323242, 53.531219482421875, 100.31004333496094, 129.04043579101562, -36.5483283996582, 61.52763366699219, -6.449954986572266, -106.68504333496094, 32.274253845214844, 30.4377384185791, -110.616943359375, 111.88530731201172, 115.35443115234375, 2.9448888301849365, -35.09677505493164, 57.935123443603516, -49.265594482421875, -240.53298950195312, 2.290024757385254, -18.8651180267334, -79.23117065429688, -12.229745864868164, -60.507957458496094, -17.489282608032227, 106.05655670166016, 61.37907028198242, -69.40279388427734, -63.15238952636719, 162.33982849121094, -23.50956916809082, -3.1185877323150635, 18.280475616455078, -15.221500396728516, 181.1841583251953, 232.42539978027344, -33.771942138671875, 147.8451385498047, 131.58135986328125, -125.18034362792969, -60.3829460144043, 248.95916748046875, 93.42080688476562, -32.716983795166016, -76.76163482666016, -44.80086135864258, -17.214445114135742, 118.45713806152344, 109.58040618896484, 68.28890228271484, 35.247554779052734, -107.2555160522461, -107.31415557861328, 157.4251708984375, -82.37310791015625, -121.97411346435547, 51.11103057861328, 27.761951446533203, -45.67251205444336, 101.01338958740234, 165.7204132080078, 31.777587890625, -185.15460205078125, -67.82890319824219, -90.65162658691406, 66.67848205566406, -131.2911376953125, 155.48452758789062, -17.133684158325195, -148.26065063476562, -4.557313919067383, 112.11238098144531, -0.4453979432582855, 5.627188682556152, -80.4355239868164, 108.70185089111328, -94.4318618774414, 84.61080932617188, -48.05681228637695, 61.915802001953125, 77.51419067382812, 53.73678207397461, -144.25482177734375, -177.6819610595703, -73.01814270019531, -78.66371154785156, 65.75379180908203, 0.8504925966262817, 46.169273376464844, -57.807254791259766, -95.90025329589844, 15.089332580566406, 91.17851257324219, -0.36312592029571533, -110.51207733154297, 70.27893829345703, -22.544010162353516, -105.31514739990234, 24.555530548095703, -20.277061462402344, 22.336071014404297, 214.11642456054688, 217.5380401611328, -2.881648302078247, 92.09378814697266, -91.2256851196289, -214.74859619140625, 14.051595687866211, 71.30755615234375, -109.63452911376953, 39.91804885864258, -14.299349784851074, -144.80364990234375, -57.000572204589844, -53.67019271850586, -65.51049041748047, 96.71378326416016, 49.34400177001953, 54.205299377441406, -64.58956909179688, -280.03546142578125, -133.33773803710938, 116.10556030273438, 88.94315338134766, -20.2482967376709, -201.19281005859375, -14.114163398742676, -20.491703033447266, -121.69708251953125, -51.61055374145508, 86.7593002319336, -1.0783356428146362, -31.633493423461914, -22.437345504760742, 88.67788696289062, -171.61862182617188, -73.92633056640625, -98.7654037475586, -164.47447204589844, 99.2717514038086, 231.28375244140625, -10.79609489440918, -2.0074543952941895, 133.4824676513672, -62.58358383178711, -19.429983139038086, -99.50377655029297, -6.769329071044922, 288.2931823730469, 41.5201416015625, 7.926990032196045, -3.4822423458099365, -39.851806640625, 147.91622924804688, 16.888471603393555, 41.14837646484375, 23.417728424072266, 62.09818649291992, 24.384746551513672, -150.30230712890625, 47.75110626220703, -21.32291603088379, 53.05957794189453, -48.06599426269531, -119.90286254882812, 8.572869300842285, -47.14007568359375, -56.15061950683594, 101.9677505493164, 38.527931213378906, -117.30266571044922, 68.16648864746094, -24.29114532470703, 89.60785675048828, 67.0279541015625, 100.93684387207031, -53.48475646972656, -20.526365280151367, -0.6130005121231079, -222.76760864257812, -106.67870330810547, -300.2406005859375, 22.184009552001953, -48.15937805175781, 49.83628845214844, -49.145870208740234, -170.1663360595703, 104.84727478027344, -139.98928833007812, 107.63104248046875, -131.74781799316406, -8.38839340209961, 95.79835510253906, 56.78730773925781, -45.621524810791016, 60.81364822387695, -33.122039794921875, -100.5759506225586, 16.701913833618164, -38.0939826965332, -236.9631805419922, 48.857582092285156, 25.612634658813477, -77.32880401611328, 50.58141326904297, 28.477970123291016, 137.17724609375, -112.64833068847656, -89.32038879394531, 28.868946075439453, 90.76740264892578, -45.30557632446289, 51.55210494995117, 12.499354362487793, 14.01379108428955, 58.25908279418945, -177.9130401611328, -113.28919982910156, 53.05046844482422, 65.88944244384766, 248.0496368408203, 3.0105159282684326, 115.49890899658203, 31.293169021606445, 150.2119598388672, 41.31827926635742, -130.15524291992188, -48.077022552490234, 8.514822006225586, 48.98770523071289, -138.39060974121094, 109.56283569335938, 44.42097854614258, 7.258269786834717, -187.44558715820312, -38.929481506347656, -122.23623657226562, 154.06918334960938, -1.6444417238235474, 82.37670135498047, 117.8553695678711, -106.78821563720703, -51.00474548339844, -110.32543182373047, 175.33816528320312, -52.0986328125, -98.0596694946289, 39.45302200317383, 45.370121002197266, 32.07572555541992, -5.352710723876953, 50.53064727783203, 71.15687561035156, 137.58009338378906, -205.1595001220703, -123.02471160888672, 30.79729652404785, -78.1437759399414, 14.417426109313965, -130.14163208007812, 43.61002731323242, 72.52232360839844, -98.71380615234375, 63.898311614990234, 108.18955993652344, 93.63573455810547, 24.476165771484375, 58.29571533203125, 54.42500305175781, -41.98841857910156, 2.813969612121582, 57.177764892578125, 125.46659851074219, -93.69813537597656, -181.31765747070312, 57.52637481689453, 132.41392517089844, -192.9333953857422, -39.18232727050781, 89.69487762451172, 166.02044677734375, -42.578285217285156, 152.39199829101562, 75.90327453613281, 110.88428497314453, -134.36106872558594, -284.73028564453125, -25.33024787902832, -104.44022369384766, 97.2557601928711, -48.96419906616211, 49.864234924316406, -101.71089172363281, 115.61373138427734, 3.834613084793091, -102.95162200927734, 147.2310333251953, 103.78649139404297, -48.13338851928711, -27.402687072753906, 72.46475219726562, 215.32118225097656, 5.888692855834961, 22.54764747619629, -89.99676513671875, -37.48249435424805, 50.43777847290039, -47.442230224609375, 35.97471618652344, -9.250489234924316, 13.789712905883789, -79.93962097167969, -81.66407775878906, 119.49856567382812, 11.13071346282959, -45.77758026123047, 98.09830474853516, 76.64273071289062, -18.981021881103516, 52.550933837890625, 33.03577423095703, -91.18173217773438, -26.838895797729492, 56.672096252441406, -91.82111358642578, -74.2446060180664, -43.94341278076172, 66.39069366455078, -56.5477294921875, 13.86144733428955, 70.96842956542969, 84.14013671875, -105.93598175048828, 13.685218811035156, -20.529943466186523, 7.847963333129883, 77.05839538574219, 137.68067932128906, 30.741342544555664, -58.900970458984375, -168.0133819580078, -100.99480438232422, 44.27955627441406, 9.316988945007324, 2.918717384338379, -185.8399200439453, 212.9515380859375, -75.80812072753906, -175.2727813720703, 29.151493072509766, -211.9896240234375, 79.50593566894531, -25.2862491607666, -153.434814453125, -181.00909423828125, -24.693212509155273, -61.59931945800781, -56.44823455810547, 9.953634262084961, -144.22647094726562, -14.951093673706055, 59.92744445800781, -3.9619948863983154, -3.4276504516601562, -94.126953125, 26.445552825927734, -84.764892578125, -184.69578552246094, 171.77792358398438, -150.03659057617188, -54.2012939453125, -15.581745147705078, -146.95333862304688, 0.23212142288684845, -7.2708563804626465, 22.84575080871582, 135.22982788085938, -189.18096923828125, 68.93524169921875, 62.938880920410156, -27.288389205932617, -19.118555068969727, -177.93272399902344, 44.90446472167969, -50.94208526611328, -10.563133239746094, -28.955585479736328, -12.891870498657227, -75.79938507080078, 72.701416015625, 30.09067726135254, -31.268808364868164, 222.97592163085938, 162.0902557373047, 32.80376052856445, -120.65605163574219, -33.5538444519043, -40.28428268432617, -56.544960021972656, 57.604854583740234, 29.519676208496094, 23.108570098876953, -72.15715026855469, 215.64764404296875, -19.577016830444336, 97.0185546875, 110.9897232055664, -11.721456527709961, -38.49686050415039, -4.315727710723877, -362.5564880371094, 27.61297607421875, 226.56495666503906, -7.547367572784424, -14.562278747558594, -108.5627212524414, -7.324496746063232, -87.01854705810547, -50.74173355102539, 5.775784492492676, -53.88814926147461, 59.03507614135742, 5.446613788604736, -208.81809997558594, -19.824350357055664, 124.04552459716797, -49.31892776489258, 75.27742004394531, 201.593017578125, -73.660400390625, -85.17871856689453, 85.17636108398438, 206.26893615722656, -161.89498901367188, 62.86574172973633, -51.97728729248047, -12.604342460632324, -132.69798278808594, -175.03826904296875, 0.3898167014122009, 96.57503509521484, 41.72821044921875, 27.52326202392578, -120.15309143066406, -211.5528564453125, 44.78286361694336, -95.98548126220703, 21.944913864135742, 148.37274169921875, 27.991586685180664, -95.07160949707031, -61.421573638916016, 20.680540084838867, 36.97489547729492, -143.97840881347656, -62.62491226196289, -54.241092681884766, 15.02979564666748, 129.3648681640625, 37.99323272705078, 102.66413879394531, 3.103675365447998, -12.663374900817871, -36.627906799316406, -72.55890655517578, 108.14073944091797, -7.961569786071777, 75.63126373291016, 15.12600040435791, 134.90386962890625, -64.839111328125, 153.4960479736328, -103.69371032714844, -55.581031799316406, -142.80679321289062, -86.89606475830078, -42.995025634765625, 163.48690795898438, -63.70855712890625, -76.22454833984375, -13.278983116149902, 44.722686767578125, -56.08180618286133, 28.46721076965332, -3.4810452461242676, -135.11630249023438, -24.065855026245117, -98.06148529052734, 63.74445724487305, 57.72625732421875, -28.175151824951172, 119.9314956665039, 10.784049034118652, 33.722938537597656, 16.737525939941406, -128.6256561279297, 0.20155994594097137, 137.8057403564453, -122.50086212158203, 39.98957443237305, 8.520874977111816, 143.36485290527344, 168.60093688964844, -31.676820755004883, -83.20938110351562, 154.6309051513672, 48.130069732666016, -59.91591262817383, -3.6835081577301025, -125.65740203857422, 161.90228271484375, -75.5757064819336, 57.059669494628906, -137.76296997070312, 84.27222442626953, -1.1717439889907837, -96.16090393066406, 49.48272705078125, 18.349674224853516, -29.375795364379883, 50.80086135864258, -18.77056121826172, -16.325027465820312, 50.10697937011719, 5.247755527496338, -59.32021713256836, -47.670379638671875, -56.70756530761719, 131.3732147216797, 271.998046875, 181.57144165039062, -126.70140075683594, -236.3686065673828, -41.93076705932617, -20.292903900146484, 52.02296829223633, -7.573666572570801, 52.03895568847656, 138.6595001220703, -16.184188842773438, 9.251967430114746, -53.241485595703125, -158.72286987304688, -101.71349334716797, 24.097803115844727, 87.07977294921875, -20.925893783569336, 32.75753402709961, -151.34580993652344, 68.74243927001953, 81.61595153808594, -126.66352081298828, -52.202415466308594, 49.5564079284668, 30.348953247070312, 76.57515716552734, -108.05504608154297, 108.65550994873047, -140.20181274414062, 115.58897399902344, -82.97661590576172, 58.97330093383789, -166.64732360839844, 66.43656158447266, 59.87864685058594, 32.466407775878906, -8.336420059204102, 31.754558563232422, 212.35423278808594, -76.87178039550781, 70.79996490478516, 56.971187591552734, -57.98907470703125, 32.908660888671875, 9.392297744750977, -9.94161319732666, 13.136953353881836, 81.47660827636719, -84.0300521850586, -22.682472229003906, -73.99514770507812, -44.39124298095703, 10.464533805847168, 43.27710723876953, 116.13475036621094, -76.24444580078125, -117.88301086425781, -24.415632247924805, 122.28414154052734, -156.6441650390625, -26.853158950805664, -122.83255004882812, 164.9317169189453, 88.4007568359375, -110.9895248413086, 68.46826934814453, -56.694950103759766, 52.15728759765625, -88.27120971679688, -37.76171875, -8.933550834655762, 153.8374481201172, 45.88634490966797, -69.56474304199219, -100.60806274414062, -52.69306182861328, -177.93263244628906, -185.63409423828125, -8.076168060302734, 201.44090270996094, 26.05331039428711, 192.98995971679688, -5.465341567993164, -82.09728240966797, 40.71619415283203, 88.05563354492188, 121.7383041381836, 5.185690402984619, 14.007777214050293, -89.82067108154297, -210.6648712158203, 57.62580108642578, 217.57891845703125, 91.56249237060547, 62.55692672729492, -16.09400177001953, 2.7757036685943604, -164.80975341796875, -48.53874588012695, -67.4486083984375, -42.89704895019531, 66.65645599365234, 61.21236038208008, 2.770674228668213, 128.1315155029297, -148.6875457763672, 117.34078979492188, 24.209745407104492, 49.812904357910156, -11.714157104492188, 66.56416320800781, -63.70375061035156, 69.60691833496094, -5.799211502075195, -266.9346008300781, 67.94598388671875, 74.13325500488281, -75.31446075439453, -162.3390350341797, 11.469205856323242, 194.73085021972656, 36.209197998046875, 83.9332275390625, -83.67330169677734, 161.9988250732422, -37.06661605834961, 37.34144973754883, 124.56370544433594, 203.34976196289062, 25.908903121948242, 78.47761535644531, -120.34176635742188, 111.94775390625, -154.5767364501953, 19.852346420288086, -166.74533081054688, 178.0615234375, 213.99424743652344, 60.39271545410156, 7.4940690994262695, -148.8043670654297, 16.7945556640625, -57.641483306884766, 187.64015197753906, -25.676532745361328, 28.8843936920166, 146.92947387695312, 97.65423583984375, -46.869571685791016, -109.20577239990234, 57.443565368652344, -36.48568344116211, -243.6532745361328, 65.85468292236328, 44.57447814941406, -38.74223709106445, -48.059444427490234, 131.42701721191406, -23.394073486328125, -37.60834503173828, 42.95660400390625, 106.26292419433594, -53.03058624267578, 55.96814727783203, 87.8766860961914, -163.3479461669922, -9.467222213745117, -49.664527893066406, 58.78071212768555, -73.05772399902344, 62.58818054199219, -61.887020111083984, -63.98146057128906, 3.034034013748169, 49.48746871948242, -21.956302642822266, -18.293197631835938, -57.964561462402344, 6.17865514755249, 117.97264862060547, -22.017379760742188, 82.95402526855469, 38.763397216796875, -74.46306610107422, 114.34410858154297, -134.53602600097656, 78.75667572021484, 99.91529083251953, 95.6093978881836, -7.27681827545166, 113.8871078491211, 177.9120635986328, -90.55271911621094, -118.48314666748047, -23.059240341186523, 12.575867652893066, 64.36766052246094, 8.227825164794922, -85.15304565429688, -133.68402099609375, -92.04424285888672, -69.66944122314453, -24.575794219970703, -59.71613693237305, 109.40096282958984, 159.49176025390625, -70.98955535888672, -65.97757720947266, 52.711151123046875, 66.81716918945312, 130.57958984375, 13.291138648986816, -15.955906867980957, 73.57937622070312, 48.89653778076172, -34.128841400146484, -87.02217102050781, -95.0344467163086, 146.2035369873047, -11.969108581542969, 250.62208557128906, -72.52592468261719, -39.784507751464844, 80.29712677001953, 105.58513641357422, 143.50335693359375, -60.78319549560547, 50.74885940551758, -99.17353057861328, 282.68597412109375, 231.4772186279297, 21.2041072845459, 56.97567367553711, -25.487085342407227, 68.1598892211914, -131.2677001953125, -50.24861145019531, -123.69075775146484, 35.8165283203125, 190.32870483398438, 7.027347564697266, -95.15316772460938, 14.439847946166992, 3.1188971996307373, 27.56156349182129, -229.56382751464844, -30.410301208496094, -45.62639236450195, -25.617361068725586, 190.2216339111328, -87.9879379272461, -164.45391845703125, 46.55089569091797, 44.611045837402344, -143.32362365722656, -107.80594635009766, -112.08695220947266, 172.15191650390625, -0.4765366315841675, 3.8902416229248047, -130.0264892578125, -58.166282653808594, 48.428348541259766, 142.45220947265625, 40.626590728759766, -5.423812389373779, 64.22357177734375, -53.87400436401367, -104.96336364746094, -35.33537292480469, 85.66874694824219, 128.08741760253906, -52.517879486083984, -132.74588012695312, 164.18121337890625, 37.799232482910156, -157.08843994140625, -120.14692687988281, 59.8893928527832, 11.004183769226074, -104.67752075195312, -84.89915466308594, 107.8497314453125, 107.88330841064453, 90.44007110595703, 106.08173370361328, -8.483438491821289, -5.190036296844482, -119.06104278564453, 92.84978485107422, 87.42109680175781, -64.3913345336914, -58.627899169921875, 128.54415893554688, 2.88995623588562, 65.42512512207031, 73.50914764404297, -4.7972259521484375, -28.83212661743164, 66.93162536621094, 158.887451171875, -84.42640686035156, 97.9608383178711, -123.51934051513672, -28.50423240661621, -165.96266174316406, 0.6875019073486328, 96.99234008789062, -12.435193061828613, 54.9514045715332, -80.55628204345703, 47.24973678588867, 39.42391586303711, -20.323612213134766, 1.5897396802902222, -143.83433532714844, -83.08512878417969, -76.15335845947266, 94.87040710449219, -2.884103536605835, -8.681113243103027, 66.24256896972656, 227.21658325195312, 136.70166015625, 110.92815399169922, -86.82415771484375, 78.0496826171875, -113.96111297607422, 93.65607452392578, -209.0806884765625, 21.72597312927246, -207.36715698242188, 29.797237396240234, -47.393436431884766, 73.22669982910156, 26.896873474121094, -110.44824981689453, -110.26467895507812, -98.3983154296875, 95.33618927001953, -18.538558959960938, 28.560056686401367, -70.16266632080078, -31.705909729003906, 104.54440307617188, -73.63614654541016, 152.89073181152344, 106.11958312988281, -25.578603744506836, -64.16197204589844, 66.61506652832031, -224.90870666503906, -100.19450378417969, 76.71045684814453, 142.73138427734375, -75.80451965332031, -90.37317657470703, 34.41144561767578, -34.95325469970703, 76.60248565673828, -111.10740661621094, 16.862014770507812, -68.90911102294922, 25.4303035736084, 108.06248474121094, 127.03050231933594, 44.428165435791016, 41.0447998046875, 146.9644012451172, -129.19210815429688, -10.053349494934082, 159.13790893554688, -36.36156463623047, 4.1598381996154785, -104.33885192871094, 73.70723724365234, -20.45740509033203, 111.43328857421875, -1.6622077226638794, -42.77302169799805, -98.28800201416016, -119.8248062133789, 24.93601417541504, -134.23922729492188, -6.997241973876953, -30.39967155456543, 7.117280960083008, 91.67552947998047, -119.41764068603516, -172.72384643554688, 9.352686882019043, -66.45838928222656, 28.450021743774414, -21.95949935913086, 36.42499923706055, -60.00910568237305, -44.88136291503906, -8.060847282409668, -7.008904933929443, 273.0484619140625, 20.12492561340332, -135.10797119140625, 71.60553741455078, -7.313737869262695, 58.06682586669922, 109.42530822753906, -80.6627426147461, 89.35018920898438, -91.63560485839844, 102.93392181396484, -149.1067657470703, 34.04923629760742, 2.3136205673217773, -29.77071189880371, -46.84893035888672, 8.726860046386719, -32.346229553222656, 183.92442321777344, 27.02931785583496, -39.506229400634766, -79.71833038330078, -189.73117065429688, 124.05357360839844, 119.45165252685547, 36.601531982421875, -0.6527462601661682, 57.97478485107422, 134.46214294433594, 158.51242065429688, -40.84865188598633, 5.966626167297363, -153.35118103027344, -15.880632400512695, 157.072509765625, 105.55718231201172, 24.939983367919922, 34.9600715637207, -59.493934631347656, -78.1143569946289, 199.0279998779297, -110.85257720947266, -9.330008506774902, 45.78363800048828, 37.964393615722656, -101.32958984375, 20.315204620361328, -4.974247455596924, 99.52587127685547, -147.04747009277344, -136.88278198242188, 121.32977294921875, -116.50489807128906, -50.008766174316406, -31.76679229736328, -152.26730346679688, 23.168737411499023, 15.880197525024414, -94.80608367919922, -32.890262603759766, 3.2803292274475098, -70.29562377929688, 127.08365631103516, -137.220703125], "dtype": "float32"}, {"__nd__": [0.30416351556777954, 0.0750979334115982, -1.3854275941848755, -1.2941640615463257, 0.04827878624200821, 0.3392394185066223, -2.5072548389434814, -0.5020976662635803, 2.266758441925049, -0.058361589908599854, -0.09517870843410492, -0.24179627001285553, 0.27291417121887207, 0.5223969221115112, 0.23199248313903809, 0.359825074672699, -1.0668518543243408, 0.7766429781913757, 0.8498809933662415, 1.0554255247116089, 1.0039854049682617, -0.27130237221717834, 1.1551402807235718, -0.7374710440635681, -0.3555510342121124, -0.11293294280767441, -1.6117737293243408, 1.1397367715835571, 0.795892059803009, -0.6946825385093689, 1.3502384424209595, 1.4222209453582764, -0.4805343747138977, -0.2859744429588318, -0.9388850927352905, 1.3815466165542603, -0.7175635099411011, 0.7135124802589417, -1.0265673398971558, 0.7817745804786682, 2.0655040740966797, -0.692293107509613, 1.0103683471679688, -0.9855865836143494, 0.12212460488080978, -0.3192991614341736, 0.20379526913166046, 0.7713497877120972, 0.6664051413536072, -0.7308366298675537, -0.6242380142211914, -0.8609181046485901, 0.6324027180671692, 0.6018337607383728, -0.5276059508323669, 0.462974488735199, 0.6029248237609863, 0.2977505922317505, -0.2681763768196106, 0.5984889268875122, 0.920909583568573, 0.4371848404407501, -0.354393869638443, 1.2647000551223755, 0.006939272861927748, 1.0365686416625977, -1.741943597793579, -0.8762789368629456, 0.3663250207901001, -1.3665618896484375, 1.343804121017456, -0.5211236476898193, -0.4120752811431885, 0.49971312284469604, 0.44958266615867615, -0.5223721265792847, -0.2020881474018097, 0.26349198818206787, 0.7179819345474243, 0.31263551115989685, 1.0558358430862427, 0.5300825834274292, -1.8767635822296143, 0.8305442929267883, 0.5633664131164551, -0.30220094323158264, -0.8618117570877075, -1.3002504110336304, -0.5907126069068909, -0.038016047328710556, 0.6916263103485107, -0.8750253915786743, -0.223235622048378, 1.580979824066162, -0.7593662738800049, -0.4461868107318878, 1.7236868143081665, -1.9904783964157104, 0.6515202522277832, -1.261665940284729, 0.15663713216781616, -1.0661967992782593, -0.3612931966781616, -0.7848984003067017, 0.11753921955823898, 0.8252999186515808, 1.1704394817352295, -0.08927472680807114, -0.8639283180236816, -1.7820099592208862, -0.07986579090356827, -1.3720322847366333, 1.0334757566452026, -0.046632830053567886, -0.9115752577781677, -1.2850497961044312, -1.513033390045166, 1.1638036966323853, 0.5103420615196228, -0.1846620887517929, -0.7715886235237122, 0.3137779235839844, 0.38709455728530884, -0.6093416213989258, -0.31503376364707947, -0.21498475968837738, -0.6993982791900635, -0.28193846344947815, 0.7789603471755981, -0.5523940920829773, 0.9170028567314148, 1.5417051315307617, -1.9885016679763794, 1.0725404024124146, -0.2705323100090027, -1.8657573461532593, -0.7225688099861145, 0.5301907658576965, 0.22470855712890625, 0.24941426515579224, -0.2956913411617279, 0.36169159412384033, -0.0004891957505606115, 0.28270283341407776, -1.2673230171203613, 2.3097360134124756, 0.6891237497329712, -0.969358503818512, -0.8631457686424255, -0.6286442875862122, 1.899955153465271, -0.34737902879714966, 0.30294400453567505, 2.027343988418579, 0.40868282318115234, -0.621088445186615, -0.13167151808738708, -0.3433016538619995, -1.1745953559875488, 1.2027146816253662, 1.2578351497650146, -1.5704333782196045, -0.05925951898097992, 0.20981425046920776, 0.6355790495872498, -1.697167158126831, -0.48461440205574036, -1.6637171506881714, -0.3473149538040161, -0.16737845540046692, 0.9606992602348328, 0.16747474670410156, 0.03118157759308815, -0.8783383369445801, -0.27664244174957275, 0.8248947262763977, -0.4779691994190216, -1.1954981088638306, 0.01291546318680048, -0.35148900747299194, 0.6044718623161316, 0.38125091791152954, -0.6914685964584351, -0.7210161685943604, -1.924457311630249, 0.6140177845954895, 0.9672759771347046, -0.871347963809967, 0.39053377509117126, 0.061898864805698395, -0.11866781860589981, 1.1071747541427612, -0.7147923707962036, 1.090257167816162, 0.15540514886379242, 2.541367530822754, -0.5102960467338562, -0.6067284941673279, -0.9845845103263855, -0.18655379116535187, 1.0919491052627563, -1.0349562168121338, -0.47157523036003113, -0.650859534740448, -0.14340224862098694, -0.36504843831062317, -0.7368165254592896, -1.1571334600448608, 1.4959334135055542, -0.37736523151397705, -0.30348077416419983, 1.461778998374939, 1.3388636112213135, -0.5462506413459778, 1.370146632194519, 0.3766254782676697, 0.5181877017021179, 0.7681740522384644, -1.0487194061279297, 0.3699917793273926, 0.07397911697626114, 0.676141619682312, -0.4142468571662903, -0.693109929561615, -0.5993158221244812, 0.33372634649276733, 0.9003465175628662, -1.534645676612854, -0.03564640134572983, 0.8974202871322632, -1.4866315126419067, 0.3239486813545227, -1.3883893489837646, -1.215603232383728, 0.34553855657577515, 0.7457476258277893, 1.2753825187683105, 0.4647214412689209, 0.3721260726451874, -0.46726664900779724, 0.31780633330345154, 0.8047104477882385, -0.12683016061782837, -0.32255032658576965, 1.317352294921875, 0.077407605946064, -1.4274190664291382, 0.27833905816078186, -0.5120036005973816, 0.6586127281188965, -1.1767442226409912, -1.7924156188964844, 0.07510366290807724, 0.48934081196784973, -2.166651964187622, 2.122403621673584, -0.07476427406072617, -0.9701271653175354, 0.6177467107772827, -0.60518878698349, -0.4493440091609955, -0.4735950827598572, 1.088716745376587, -0.7558091878890991, -0.3473445773124695, -0.7406924366950989, -0.6673215627670288, -0.7327039241790771, -0.6309723258018494, -1.2315484285354614, -0.15725524723529816, 0.20794005692005157, -0.29265856742858887, -1.2873022556304932, -1.4913477897644043, -0.7285147905349731, -0.920249879360199, -0.1914093941450119, 0.516957700252533, -0.3304693102836609, 1.3679296970367432, -1.0285553932189941, -0.42385298013687134, 1.3619773387908936, 1.5707508325576782, -0.42339691519737244, -1.1550167798995972, 1.5942801237106323, 0.1489178091287613, -0.926552414894104, -0.2773842215538025, 2.4302713871002197, -0.23742510378360748, 0.6636624932289124, -0.7995296120643616, 0.24891437590122223, 0.9217565059661865, 0.37382593750953674, 0.27609115839004517, -0.24562352895736694, 0.3821926712989807, -2.3298580646514893, -0.4821208417415619, 0.5923528075218201, 0.38024991750717163, -0.22840209305286407, 1.1603666543960571, -0.2693209946155548, 2.6953108310699463, 1.8461321592330933, 2.3383822441101074, 1.6619642972946167, 0.23607495427131653, -1.9678319692611694, 1.548236608505249, 0.593121349811554, -0.5365357995033264, -0.394298791885376, -0.9382224082946777, 0.6935080885887146, -1.5006732940673828, 0.8295698165893555, 0.36175984144210815, 0.6560433506965637, -0.7321495413780212, 0.36824604868888855, -0.3337757885456085, 0.24463289976119995, -0.9728031158447266, -0.3037661612033844, -0.3022421598434448, 0.6318398118019104, -0.9996948838233948, -1.9478505849838257, -0.4998226761817932, -0.7540019750595093, -0.44415104389190674, 0.5035504698753357, -0.7636038064956665, -0.005935669410973787, -0.7156797051429749, -0.3493334949016571, 0.19532185792922974, -0.40277621150016785, 1.0030970573425293, 0.5583010315895081, 0.43686962127685547, -1.0706843137741089, -0.16122935712337494, 0.8328915238380432, 1.7018247842788696, -1.6766514778137207, 0.11425270885229111, -0.25459617376327515, -0.39896664023399353, 1.1475558280944824, -0.0736861601471901, -1.388756513595581, 1.0688828229904175, -0.49981221556663513, -0.4421711564064026, -0.09358855336904526, -0.027598973363637924, 0.4284523129463196, 0.3018309772014618, 0.07491032779216766, 0.7737057209014893, 0.11762989312410355, 0.5608199834823608, 0.05714916065335274, -0.10991500318050385, -0.5364193916320801, 1.4232205152511597, -0.36560896039009094, 0.8549385666847229, 0.995694100856781, 0.4440120756626129, 0.6859151721000671, 1.562368631362915, 0.48148563504219055, 0.2282896488904953, 0.9686564803123474, 0.014971652999520302, -0.11907842755317688, 1.5149608850479126, 1.556241750717163, 0.7639626264572144, -0.11177817732095718, -0.8864186406135559, -0.27743497490882874, -0.32329970598220825, -0.36256250739097595, 0.8154304623603821, 0.2080870419740677, 0.4918444752693176, -0.6074821949005127, -1.0555901527404785, -0.7505338788032532, -0.4707713723182678, -0.11203186213970184, 0.18838383257389069, 0.6551573872566223, 1.6951040029525757, -0.7332025170326233, 0.3870474100112915, 2.107684373855591, -0.2206607162952423, -0.38135403394699097, 0.41999101638793945, -1.369123935699463, -1.5819973945617676, -0.8751477003097534, 0.25966838002204895, -1.1810569763183594, -0.9700459837913513, 0.1346856951713562, 1.445020079612732, -0.30910369753837585, -0.5168319940567017, 0.17788808047771454, -0.8210172057151794, -0.2978779077529907, 1.5294393301010132, 0.4735330641269684, -0.8887434601783752, -0.987830638885498, 0.73084956407547, -1.4487195014953613, 0.9090198278427124, 0.3178615868091583, 0.19599169492721558, -0.7866107821464539, 1.8468170166015625, -1.1782331466674805, 0.8374302387237549, 0.44337958097457886, 1.1955170631408691, -0.9342402815818787, -0.1122576966881752, -0.938606858253479, -0.2436797171831131, 0.3723523020744324, 0.7889324426651001, -0.4418555200099945, 2.196084499359131, -1.7477445602416992, -0.6051176190376282, -1.4186899662017822, 0.1293329894542694, -0.8936713337898254, 0.24634547531604767, -1.3965500593185425, -0.740972638130188, 1.567121982574463, 0.5120899677276611, -0.049571093171834946, 0.11840219050645828, 1.2325416803359985, 0.43326324224472046, -0.7791796326637268, -0.8205273747444153, 1.0422539710998535, -0.10734617710113525, 0.3130968511104584, -0.3875889778137207, 1.5674446821212769, -1.0702784061431885, 0.561543881893158, -0.6704161763191223, 1.0827271938323975, -0.09337473660707474, 0.18958677351474762, 0.9044829607009888, -0.09377972781658173, 0.983303427696228, -2.162501573562622, -0.6895270943641663, 0.6116495728492737, -0.2319449484348297, 0.3076338768005371, 1.8233351707458496, -1.6181707382202148, 0.03488495200872421, -1.9389219284057617, 0.47548237442970276, -0.6099759936332703, 0.5861839056015015, 0.962917685508728, 0.9282138347625732, -0.6475750803947449, 2.3244736194610596, -0.3135349750518799, 0.05638429895043373, -0.5119931697845459, -0.06526804715394974, 0.4504839777946472, 0.9751411080360413, 0.12204006314277649, -1.0562251806259155, 0.3261603116989136, -0.31518152356147766, 0.8882081508636475, -1.0694619417190552, 0.2833873927593231, -0.7938026189804077, 0.2303384244441986, -0.5055776238441467, 2.0459420680999756, 0.4781682789325714, -1.7535501718521118, 1.4035542011260986, -0.3876796364784241, 0.35721641778945923, -0.8375353217124939, 0.4193442463874817, 0.028127003461122513, -0.9580669403076172, 0.31640687584877014, 0.31726643443107605, 0.33392488956451416, -1.134662389755249, -1.1989425420761108, -0.3499538004398346, 0.5656489133834839, -1.499813199043274, -0.020844539627432823, -1.1914327144622803, 0.9820456504821777, 0.36298173666000366, -0.2784019112586975, 1.329768419265747, 0.23738063871860504, -1.7692254781723022, -1.5206654071807861, -0.38686618208885193, 1.7995381355285645, 0.049949899315834045, -0.6431681513786316, -0.7459158897399902, 0.24692057073116302, -0.9602437615394592, -0.43521174788475037, -0.126423180103302, -0.0057174102403223515, -0.03757603466510773, 1.4015380144119263, 1.2796058654785156, 0.9681007862091064, -1.2201834917068481, 0.7626587152481079, 0.21415013074874878, 0.3489561080932617, 0.36470451951026917, -0.03406357392668724, 1.8220261335372925, 0.2943093776702881, 0.60971999168396, 0.6876741051673889, 0.30571234226226807, 0.4250172972679138, -0.10445285588502884, -0.21975308656692505, -0.46305057406425476, -0.25642427802085876, -0.5331010818481445, -0.1412985920906067, -0.09579093754291534, -1.6305564641952515, 0.2860289514064789, 0.45730113983154297, -0.7927346229553223, 1.7947593927383423, -0.4047487676143646, 1.2137125730514526, -0.5511031746864319, -0.6356696486473083, -1.0860708951950073, -0.2243313491344452, -0.024482361972332, -1.5812835693359375, 1.1571763753890991, 0.6743115186691284, 0.6272966861724854, 0.059319980442523956, 0.5911664366722107, -0.08367415517568588, 2.5358400344848633, 0.328421950340271, 0.454734206199646, 0.9790194034576416, -0.17652782797813416, -0.23109452426433563, -0.17713946104049683, -0.923119306564331, -0.34389907121658325, 0.5570854544639587, 0.22561153769493103, 0.3965618312358856, 1.8905326128005981, 0.12343630939722061, -0.04314609616994858, -1.028270959854126, -1.0829616785049438, -0.4313638210296631, 0.2542431354522705, 0.15255960822105408, 0.170895516872406, 1.1303560733795166, 0.5563820600509644, -0.5636115670204163, 0.6913211345672607, -0.6467303037643433, -0.11733291298151016, -0.5054906010627747, -0.6066391468048096, 0.4676547050476074, -0.2612307667732239, -1.4733543395996094, -0.5896873474121094, 0.27601367235183716, -0.09158758819103241, 1.0742225646972656, -1.148196816444397, 0.8127534985542297, -0.593717634677887, -1.890881061553955, 1.5321180820465088, 0.8016890287399292, -0.09751876443624496, 0.07054546475410461, -1.0694972276687622, -0.7507438063621521, 1.7817641496658325, -1.7326984405517578, 0.4473508298397064, -1.1430221796035767, -0.37620460987091064, 0.3099404275417328, 1.1025503873825073, -0.35259783267974854, -1.4114466905593872, 0.7871652245521545, -0.7581013441085815, 1.0943446159362793, -0.9589308500289917, -0.3944593071937561, 1.0758004188537598, -0.0076768659055233, -2.764528751373291, 1.0680043697357178, 0.4569930136203766, -0.21203503012657166, -1.6573320627212524, -1.1320616006851196, -0.012045187875628471, -0.44091081619262695, -1.9465413093566895, -0.29910627007484436, 0.2877950370311737, 0.9176485538482666, -0.20613889396190643, -1.367498517036438, -0.4110315144062042, -1.5078098773956299, 0.3521508276462555, 1.2576426267623901, -0.5552185773849487, -1.7471137046813965, 0.19989214837551117, -0.30520734190940857, 0.3527620732784271, -1.3213205337524414, -0.1452062726020813, -0.8654552698135376, -1.0079458951950073, 0.7590906023979187, 0.0867685005068779, 1.1661335229873657, -0.5761693716049194, -0.03765242174267769, 0.45310840010643005, 0.6628629565238953, -0.4169750511646271, 1.4397908449172974, -1.130421757698059, 1.8668122291564941, 2.2880871295928955, 1.4597458839416504, -0.1264638751745224, -1.0611498355865479, -1.2581404447555542, 1.5338174104690552, 0.9180271625518799, 0.7378945350646973, -0.6621946692466736, 1.2502228021621704, 2.7538862228393555, -1.9764456748962402, 0.04978083819150925, -0.6114930510520935, -0.6222914457321167, 0.32346484065055847, 0.8982823491096497, 0.8842577338218689, -0.46026065945625305, 1.739255666732788, -0.15134061872959137, -0.07343925535678864, 1.1132022142410278, -0.571897029876709, 0.13062845170497894, -1.27953040599823, -0.14576661586761475, 2.01375412940979, 1.441703200340271, -0.029244380071759224, -0.542481541633606, 0.18216870725154877, 0.8953056335449219, 3.611408233642578, -1.1691523790359497, 0.2903006672859192, 0.2136511653661728, -1.413758635520935, 0.24824360013008118, 0.026231776922941208, 0.3722721040248871, 0.05361129343509674, -1.4453426599502563, -0.07625439018011093, 1.06722891330719, 0.2572423219680786, 2.0817697048187256, -0.8501023650169373, 0.72324138879776, -0.319781631231308, 1.1936793327331543, -0.2570798397064209, 1.2948094606399536, 0.4265584349632263, 0.777031660079956, -1.9773674011230469, 0.3557334244251251, 0.26436302065849304, 0.07298286259174347, -0.19605258107185364, 0.9058314561843872, -1.507800817489624, -0.6623824238777161, 0.6994845271110535, 0.8752632141113281, 0.9251810908317566, -1.3154957294464111, -0.7375224232673645, 0.15117444097995758, -0.5139407515525818, -1.3792946338653564, 0.06172572821378708, -0.29436472058296204, 0.07904422283172607, 1.0614569187164307, -0.9265371561050415, -0.6728485226631165, -0.6633070707321167, 0.7303860187530518, -0.03752751648426056, -0.09492689371109009, -0.7489765286445618, -1.312881350517273, 1.1123521327972412, 0.3963484764099121, 1.6157031059265137, -0.8135213255882263, -0.7533894181251526, 0.7599914073944092, -0.6721627712249756, 0.2448766529560089, -0.9346510767936707, -0.9538426399230957, 0.6070495843887329, -0.3517095744609833, 0.6346858143806458, 1.1409178972244263, 0.07216436415910721, -2.432680368423462, -2.52138614654541, 0.3826131224632263, 0.14283156394958496, 0.0064773038029670715, 0.6150259971618652, -0.653519868850708, -0.2681771516799927, -2.019700765609741, 1.767945647239685, 0.4283481240272522, 1.4850045442581177, 0.4480540156364441, -0.5415833592414856, 1.6895065307617188, -0.8549907207489014, 0.6509658694267273, 0.01956847868859768, -0.23705603182315826, -1.08976411819458, -1.0348844528198242, 0.1102050319314003, 1.4116036891937256, 1.0861144065856934, -0.5562897324562073, -1.7689279317855835, 1.1120569705963135, -0.3126002252101898, 0.0602668933570385, -2.4552223682403564, 1.039743185043335, 1.4820219278335571, 1.5378718376159668, -0.8587256073951721, -0.2995428144931793, 1.1927621364593506, -1.320755958557129, -0.31120696663856506, 0.9534106254577637, 0.641426682472229, -1.1546618938446045, 0.6540908813476562, -0.2948972284793854, 0.2304753065109253, -1.0497432947158813, -1.2257258892059326, 0.8750049471855164, -1.019768476486206, -0.34917688369750977, 0.7205513715744019, -0.005503013264387846, -1.1449346542358398, -1.617453694343567, 0.9067962765693665, 1.4114183187484741, 1.6682966947555542, -0.9699386954307556, 0.34463199973106384, 0.04493985325098038, 1.0888381004333496, -0.3556835353374481, 0.10982121527194977, -0.9512015581130981, 0.21769985556602478, 0.8108341693878174, -0.39356985688209534, -1.523355484008789, -1.9135572910308838, 0.8620961904525757, 1.0575926303863525, -0.2018735706806183, -0.17950579524040222, -3.2810332775115967, -0.09393052011728287, 0.8103249073028564, 1.1552101373672485, 0.6051948070526123, -0.3154369592666626, 1.31230890750885, 0.6831758618354797, -1.1501963138580322, -0.7120446562767029, -0.8079351186752319, 0.9472216963768005, -0.1080891415476799, 0.3853108286857605, -0.7288313508033752, -2.227546453475952, -1.7367693185806274, 0.6066814661026001, 0.7743222713470459, -1.0209579467773438, 0.29018741846084595, -0.5814225673675537, -1.810768723487854, 0.5322250127792358, -2.779839277267456, -0.8778704404830933, 1.6731905937194824, -1.9775513410568237, -0.5833765864372253, -0.10787470638751984, 0.19165299832820892, -1.0866589546203613, -0.43249598145484924, 1.0179502964019775, -0.33891162276268005, 0.8380998969078064, -0.718975305557251, -1.2173203229904175, 0.478079229593277, -0.11882231384515762, 1.6691234111785889, 0.4597591757774353, 0.381197988986969, -1.6241999864578247, 0.7089783549308777, 0.2506310045719147, 0.9573764204978943, 1.1184645891189575, 0.5094665288925171, 0.028979191556572914, 0.21687930822372437, -0.9203407168388367, 0.16594114899635315, -1.3711128234863281, -2.1985297203063965, -0.1583983302116394, -1.4217485189437866, -0.4269390404224396, -1.659852147102356, -0.20033809542655945, -0.09424372017383575, 0.5607037544250488, -1.565055012702942, -1.581573486328125, 1.4263453483581543, 0.6672201156616211, -1.7233773469924927, -1.440190076828003, 0.813632607460022, -1.1254178285598755, 0.8134656548500061, 0.8236014246940613, 1.5268999338150024, -0.3011069893836975, -0.9176192283630371, -0.24758107960224152, 0.7798491716384888, -1.2938621044158936, -0.5101773738861084, 0.31489959359169006, -0.27791959047317505, -0.22653193771839142, 0.030913015827536583, 0.3435293436050415, 0.13984286785125732, 1.737608551979065, -1.2814574241638184, 1.2697768211364746, -0.10355639457702637, 0.03369777277112007, -0.6195697784423828, 1.6435593366622925, -1.7031902074813843, -0.5420171618461609, -1.040040135383606, 0.011846533976495266, -0.987238347530365, 1.227157473564148, 0.8197654485702515, 0.4727245569229126, 0.9122774004936218, -0.8685067892074585, 1.150180459022522, -0.9653229713439941, 0.5571538209915161, -0.3913258910179138, -1.4972829818725586, -2.28336501121521, -0.8180189728736877, -1.6454331874847412, -0.5963929891586304, 1.2192764282226562, -0.7083064913749695, -1.4322078227996826, -1.340240478515625, -0.30358192324638367, 0.6720651984214783, -0.2913598120212555, 0.41760969161987305, 0.17524448037147522, -0.3331190049648285, 1.2858694791793823, -1.1294450759887695, 1.5095857381820679, -0.8972689509391785, 0.03184002265334129, 0.08480961620807648, 0.5780737996101379, 2.627596139907837, 1.9333531856536865, -0.5629820227622986, 0.4644429087638855, -0.030475089326500893, 0.5078567862510681, 0.043594200164079666, -0.9415494799613953, -0.4436807632446289, 0.47498080134391785, 1.3990477323532104, -0.3597622215747833, -0.38150978088378906, -0.8092904090881348, 0.3087564706802368, -1.2755416631698608, -1.4080032110214233, 0.9719602465629578, 0.1543218046426773, -1.0858378410339355, -1.6813031435012817, 0.8054561018943787, -0.04223605617880821, 1.0093276500701904, -0.7736709117889404, -0.9776308536529541, -0.035585176199674606, -0.6796084642410278, 0.5623839497566223, 0.18141764402389526, 2.8919026851654053, 1.7346924543380737, -0.2658531963825226, -0.18958444893360138, -2.453418731689453, 0.40156859159469604, 0.9332396984100342, -1.1873899698257446, -0.8489787578582764, -0.22913366556167603, -0.9724827408790588], "dtype": "float32"}], "e": {"__nd__": [8823.9716796875, -12930.1728515625, 9539.69140625, -20051.220703125, 5211.298828125, 19735.8671875, 20172.697265625, 374.03729248046875, 7628.94091796875, -2234.7861328125, 4740.86865234375, -464.6209716796875, 17921.380859375, 786.9027099609375, -2717.57080078125, 460.6722106933594, 428.66583251953125, -4950.35546875, 464.3541259765625, -7040.390625, 17414.25, 82.29904174804688, 8297.5087890625, -15531.53125, 1226.8646240234375, -16607.51171875, 5448.29052734375, -16012.3125, 238.169189453125, -7122.94921875, -488.621826171875, 5919.82080078125, -2338.3623046875, 1251.0692138671875, -4318.1767578125, 8236.2607421875, 19609.73828125, -3582.56884765625, 4775.81396484375, -122.76736450195312, 7952.63671875, -1351.6837158203125, -889.2298583984375, -8543.86328125, 1586.7227783203125, 8537.0703125, 429.6431884765625, 4456.58935546875, 540.6331787109375, 5063.23779296875, 55467.2734375, 156.6983184814453, -277.8060607910156, 6682.57861328125, -519.8294677734375, 12311.240234375, -4570.60009765625, 7925.7734375, -3785.170654296875, -772.568115234375, 3297.755859375, -33404.19140625, 772.79052734375, 98.31463623046875, -2553.195556640625, -1205.9434814453125, 21230.970703125, 9013.6396484375, -6143.1357421875, 6714.33935546875, -10315.333984375, -24298.697265625, -2370.71044921875, 11529.734375, -6661.9609375, 4540.716796875, 79.41778564453125, 1502.3411865234375, -782.8323364257812, -10849.5732421875, -14476.3525390625, -7277.36328125, -1712.392333984375, 11775.66796875, -8790.6767578125, 2215.14990234375, 9518.9833984375, -1259.064208984375, -3970.865966796875, 2572.677734375, -1035.9007568359375, -9739.0830078125, -8193.0205078125, 1116.04296875, 21845.119140625, 1290.415283203125, -12428.6025390625, 133.70382690429688, 1459.0684814453125, 1917.208984375, -7.245786190032959, -1548.220458984375, 1132.6552734375, -13645.1669921875, 59.482364654541016, -693.6702270507812, 9433.453125, 20554.44140625, -1519.1268310546875, 963.1466674804688, 2679.334716796875, -5402.0673828125, -966.9451904296875, 2449.1953125, 2948.480712890625, -24369.318359375, -4573.44482421875, 19488.2578125, 16385.673828125, -86.73155975341797, 739.8499145507812, -4490.73828125, 9584.5009765625, -1748.1614990234375, 13631.5703125, 39.887908935546875, -25578.384765625, 8616.080078125, 131.55499267578125, -12952.2314453125, 3509.3876953125, 267.32122802734375, -3927.386962890625, 22956.72265625, 16518.4453125, 537.3186645507812, 12798.765625, 5650.75732421875, 27416.482421875, -1265.8974609375, 7957.01611328125, -18680.705078125, 5509.84814453125, -322.8592224121094, 4110.111328125, -3491.277587890625, 952.0728759765625, 194.28102111816406, -1566.737548828125, 149.00233459472656, -7383.30517578125, 6647.16064453125, 3986.719482421875, -10011.544921875, -7474.61376953125, 13964.3017578125, -3259.151123046875, 26688.29296875, -38.31288146972656, -886.2850341796875, 8449.2626953125, -12033.99609375, -20675.421875, 7.686310291290283, 47.57246398925781, 1414.662353515625, -5019.07861328125, 23074.873046875, -255.36224365234375, -13112.994140625, 21455.962890625, 683.3323974609375, 22523.849609375, -1523.757080078125, 90.27474975585938, 12948.451171875, 11647.6650390625, -2043.7305908203125, 13523.603515625, -613.2338256835938, 31858.984375, -151.32814025878906, -373.1371765136719, -53.741722106933594, -280.26153564453125, -4298.9541015625, -470.9601135253906, -2192.36669921875, 2903.574462890625, 1494.90185546875, -983.6962890625, -16145.79296875, -1584.2864990234375, 391.39202880859375, 735.174560546875, 1427.79541015625, -9935.705078125, 518.2744750976562, -10104.4248046875, -214.65809631347656, 22820.279296875, -4089.28662109375, -835.274658203125, 3373.7333984375, 5284.1279296875, 10565.044921875, 150.5183868408203, -14002.517578125, -6858.2275390625, -2053.5673828125, 50.64995574951172, -5961.849609375, -1463.87890625, 20997.58984375, 2124.028564453125, 7027.38330078125, 4158.7099609375, 3199.939453125, -18549.900390625, 25204.04296875, -17892.341796875, -11657.59375, -9266.4912109375, 257.9679260253906, 8548.83984375, 9290.400390625, 5911.21484375, -12577.990234375, 1200.55517578125, 3284.3203125, -1823.7015380859375, -499.1353454589844, 2458.723876953125, 798.2823486328125, 49.672706604003906, 9124.3701171875, 7869.61474609375, 1407.9962158203125, -8998.1552734375, -13887.3623046875, 7710.0810546875, -5543.7314453125, 1675.88134765625, 3853.40869140625, -6096.251953125, 1264.3250732421875, -357.07867431640625, 1532.7052001953125, -22016.4921875, 12906.4931640625, 3752.3154296875, 821.6743774414062, 29521.75390625, 144.11129760742188, 3082.323486328125, 2025.194580078125, -2234.283203125, -5065.72412109375, 9744.837890625, 3188.245361328125, -245.8723602294922, -592.193359375, 730.4971313476562, 10137.6162109375, -2232.908203125, -366.83544921875, 179.59014892578125, 878.25244140625, 7410.48779296875, 18650.341796875, -30.95943260192871, 12375.7236328125, 6407.5927734375, 2495.603759765625, -7088.8251953125, 5937.4521484375, 8231.7197265625, -7511.68212890625, -10008.2685546875, -2959.23876953125, 970.7609252929688, 856.5922241210938, 111.49181365966797, -2047.2645263671875, 649.5447387695312, -27542.615234375, -30188.62109375, -15238.1728515625, 3129.14794921875, -6922.330078125, -1554.0361328125, -2340.117431640625, 14208.7763671875, -6131.67822265625, -14827.697265625, -2974.154541015625, -18191.375, -2614.955078125, 2535.437255859375, -3923.283203125, 3253.079345703125, -1888.1705322265625, 285.71368408203125, -1900.33056640625, 6253.9013671875, -2268.590087890625, -6768.71875, 2828.153564453125, -16049.0439453125, 11882.669921875, -3625.1337890625, 11289.7392578125, 14274.564453125, -4097.775390625, 5347.74462890625, -2790.366943359375, 14122.720703125, -8699.2314453125, -8015.75634765625, 339.63641357421875, -3122.232421875, -12976.79296875, 5164.091796875, -462.59375, 16699.337890625, 3397.38720703125, -298.74359130859375, 3150.517578125, 9082.3837890625, -10743.837890625, -2383.568359375, 897.764892578125, 5643.85205078125, 22534.201171875, 1174.7406005859375, 3597.258056640625, -1538.80859375, 5504.1875, 3522.34619140625, -8258.3671875, -687.5894775390625, 377.664794921875, -1053.90673828125, -3811.689697265625, -1996.589111328125, -6259.57177734375, -1994.862060546875, 2408.623779296875, -2911.4619140625, 4087.21533203125, 410.6601257324219, 352.7150573730469, -1437.8519287109375, 3861.994140625, -494.8898620605469, 3286.3701171875, 9352.0859375, 12739.5185546875, -2193.683349609375, -397.63934326171875, -7475.349609375, 1155.4921875, 4138.26416015625, -1062.8043212890625, -10605.4072265625, 554.6846923828125, 947.5948486328125, -1575.906005859375, -11162.0625, 5703.66943359375, 7497.572265625, -6302.1982421875, 2740.129150390625, -1235.5435791015625, 1546.626953125, -636.6797485351562, 120.72914123535156, 13312.9951171875, -8054.9208984375, -8055.146484375, -28772.677734375, 1438.986083984375, 20940.099609375, 1833.5364990234375, 2873.3046875, -25028.021484375, 21643.40625, 1097.36962890625, 10195.234375, -2990.01171875, -499.5268859863281, 17594.5, 1720.9315185546875, -7763.1513671875, 451.11138916015625, 90.28987121582031, 14327.443359375, -3077.436767578125, 1138.5087890625, -14997.26171875, 300.9930725097656, -31575.681640625, -5572.7529296875, -1186.302001953125, -35523.296875, -9.204233169555664, -1130.3209228515625, -1860.5277099609375, -13349.0322265625, -13319.2158203125, -5190.54296875, -6692.6318359375, -582.447265625, 1614.1925048828125, -21582.826171875, 10741.6064453125, -8234.158203125, -256.5048522949219, 1599.9383544921875, 637.1095581054688, -3622.367919921875, -14655.41796875, 3253.47509765625, 6080.205078125, -15888.1787109375, -7380.4091796875, 1039.163330078125, -4313.16845703125, -3245.05322265625, 1841.63525390625, -7866.595703125, -6750.80419921875, 1695.77685546875, -2630.8115234375, -9359.3974609375, -14398.5693359375, 302.6236572265625, 10990.0244140625, 317.42913818359375, -56.96062088012695, -2076.16455078125, 756.3701782226562, 1238.5679931640625, -1621.796630859375, -16378.552734375, 91.76986694335938, 1623.3785400390625, 2407.906982421875, -769.6536865234375, -2018.3916015625, -6064.2353515625, 195.7567596435547, 3221.47412109375, -3723.43408203125, -627.713623046875, 14765.1748046875, -1974.3272705078125, -1919.309814453125, -2810.66162109375, 10158.734375, -27083.255859375, 3735.61083984375, 11929.115234375, 7419.8623046875, 10733.283203125, -9955.3330078125, -1136.1160888671875, 82.3572769165039, 29.70551300048828, -17592.296875, 39717.44140625, 11.987791061401367, 11359.05859375, -3726.037109375, 70.34797668457031, -5178.25830078125, -9254.4736328125, -7819.4541015625, -117.5483169555664, 1023.875732421875, -26455.041015625, -275.2269592285156, 6802.05126953125, 26.947242736816406, 667.2003173828125, -5366.85595703125, -14180.705078125, 3707.789794921875, -852.5247802734375, -713.2965698242188, 13874.72265625, 6166.931640625, -1807.0491943359375, -193.14410400390625, -1311.5225830078125, -4408.1767578125, -4128.88818359375, -7635.08984375, 9.592714309692383, -3295.388671875, 1929.822265625, 5723.5595703125, -10355.078125, 7370.18212890625, -12720.326171875, -10334.0380859375, -5716.861328125, -11461.2421875, 6151.95751953125, -165.349365234375, 3671.753662109375, 8678.8173828125, 912.4305419921875, 1816.0518798828125, -220.09146118164062, 2334.546630859375, 447.9504699707031, 1314.58984375, 452.65283203125, -9636.140625, -2895.19580078125, 423.5537109375, -1097.4066162109375, -14529.26953125, 229.46409606933594, -2348.19384765625, 2.277651309967041, -13516.375, 20.409873962402344, -759.2308349609375, -1804.1649169921875, -367.8744812011719, -636.0839233398438, 20795.099609375, -1340.7833251953125, 1254.686767578125, -5683.4375, -8530.4921875, 5772.6748046875, 10180.1435546875, 391.1626892089844, 9817.8505859375, -1684.8358154296875, 3407.136474609375, -1362.4031982421875, -10898.345703125, -3253.379150390625, -139.99801635742188, -3987.876708984375, 768.101318359375, -1398.0892333984375, 99.79368591308594, 543.80078125, 703.71875, 423.3301696777344, 2626.3701171875, 809.7155151367188, 1943.3475341796875, 7158.6201171875, -6393.42041015625, -15464.2685546875, 22075.515625, 8930.8037109375, -335.5263366699219, 2809.91748046875, 6348.38916015625, -2843.979248046875, -2393.511962890625, -805.47314453125, -2201.4951171875, -939.0040893554688, 2399.516845703125, 50.42746353149414, 10780.7705078125, 9966.3115234375, -19374.2578125, 152.583740234375, -8351.541015625, -984.9024658203125, -1332.353759765625, -35.87034606933594, -6611.7265625, 2436.541259765625, 7730.16162109375, -3024.5693359375, 488.35821533203125, -2381.187744140625, -5433.94970703125, -719.7522583007812, -13151.7529296875, -1812.123291015625, 10772.6123046875, -10523.6875, -9383.9189453125, -4047.05517578125, 7891.91650390625, 11043.259765625, -3640.08837890625, 599.1658935546875, 816.1802978515625, 37799.39453125, -508.9644775390625, 498.86553955078125, 1589.2664794921875, -11532.7158203125, -8308.466796875, 494.0543212890625, 1054.3988037109375, -68.70816040039062, 6442.5966796875, -9497.072265625, 1128.1923828125, 7002.16162109375, -2268.45166015625, -366.48260498046875, -1583.6959228515625, 7394.53662109375, -4499.00830078125, -23279.16796875, 62.27383041381836, -7497.65966796875, 6104.310546875, -1745.4530029296875, -21326.861328125, -6779.74072265625, -11571.53125, -5730.1279296875, -771.052001953125, -257.2253723144531, -1560.5089111328125, -10775.3818359375, -2510.941650390625, -840.9115600585938, -19694.603515625, 4037.274169921875, -15134.6650390625, -2934.673095703125, 110.28606414794922, 3988.91259765625, 489.984375, 295.3616943359375, -190.50131225585938, 1130.797119140625, -25673.037109375, -88.60824584960938, -520.22412109375, 1614.82373046875, 61.30915069580078, -6431.25244140625, 460.08514404296875, -384.5304870605469, 11984.671875, 18580.349609375, 1351.5792236328125, -25708.60546875, 12453.986328125, 5788.36962890625, 1264.55908203125, -244.6144256591797, -2250.65576171875, -1519.8111572265625, -6902.5888671875, -5240.2373046875, 9108.5390625, -4340.39013671875, -159.50706481933594, -3451.8828125, 1725.288818359375, -17954.185546875, -2893.600341796875, 919.1441650390625, -215.9233856201172, 2362.171630859375, -10342.9208984375, 6795.41162109375, 285.9753112792969, 2831.674560546875, -1444.9754638671875, 3784.435302734375, 5687.32763671875, -1616.635498046875, -2321.548828125, 11559.4072265625, 3746.376220703125, -4283.7431640625, -8275.5634765625, -2097.021240234375, 3282.305908203125, -2769.10107421875, -7650.68212890625, 5490.01513671875, -1254.9327392578125, 5288.51806640625, -6073.51953125, -5935.904296875, 3128.343505859375, -1916.3341064453125, 5261.548828125, -2956.23876953125, 22760.943359375, -8407.0224609375, 87.5360336303711, -10353.9287109375, 814.23193359375, -860.8600463867188, 4733.50048828125, 3473.293212890625, 73.89299774169922, 13332.265625, 5301.625, -5914.150390625, -14136.6494140625, -4572.5283203125, -487.45330810546875, 24274.349609375, 3647.371826171875, -5220.765625, -82.10527801513672, 207.00375366210938, 10300.912109375, 2186.60302734375, 2374.06982421875, -2605.61962890625, -10628.93359375, 11052.8896484375, -9355.7080078125, 5119.7109375, 6596.412109375, -219.77159118652344, 4119.84130859375, -4086.14306640625, 3801.837890625, 5460.69091796875, -7438.31787109375, 6194.6376953125, 454.2849426269531, 969.7711181640625, -2530.139404296875, 215.1802215576172, 5721.05810546875, -655.9498901367188, -14080.416015625, -3528.609619140625, -13539.154296875, 1908.7662353515625, 5206.85302734375, 12069.2578125, -9704.6083984375, 5625.50830078125, 686.1463623046875, 3929.236328125, -783.8971557617188, -8740.7998046875, -9502.5869140625, 6723.36474609375, 30027.71484375, -139.53778076171875, -358.73162841796875, 13105.6875, -260.9153137207031, 2080.1416015625, 22163.306640625, -4306.2861328125, -1640.6441650390625, -1771.3758544921875, 4508.44287109375, -8768.560546875, 2064.48779296875, -132.959716796875, 9796.572265625, -3849.58154296875, 8658.8310546875, -46.77239227294922, -957.4548950195312, 252.9700927734375, 6276.65869140625, -6507.45458984375, 1544.3597412109375, -7453.03955078125, 12077.802734375, 15040.654296875, 1010.1251831054688, 3333.6923828125, 5426.2890625, -4898.2509765625, -11257.349609375, -10287.5771484375, -18706.361328125, 529.6592407226562, -4915.5595703125, -11682.2294921875, -34078.734375, 21957.240234375, -1483.279541015625, 3686.786865234375, 1817.400146484375, -9021.0927734375, 3183.439208984375, 8890.2431640625, 8022.9404296875, 3450.470458984375, 31191.27734375, -719.0687866210938, 7146.88037109375, -727.919921875, -446.9251403808594, 3001.409423828125, -16485.14453125, 2195.010498046875, -2052.32373046875, 4381.22998046875, 24811.037109375, -2189.271240234375, -19418.572265625, 4234.8828125, -5716.4287109375, -1880.1224365234375, -1841.5238037109375, -9695.8359375, -22486.94140625, 19.40749740600586, -459.3188781738281, -9398.1845703125, 1761.1868896484375, 9363.2587890625, -1825.591064453125, -5868.55126953125, -564.0806884765625, -608.9037475585938, 805.3716430664062, -360.14501953125, -3798.5068359375, -430.9154357910156, -13552.482421875, -2906.98828125, -17245.2265625, -59915.25, 2053.65185546875, 10196.2197265625, 1611.5579833984375, 11891.2197265625, 1721.6634521484375, -26349.84375, -15408.654296875, -19231.345703125, -12210.15234375, 4234.736328125, 10796.84375, -378.3873291015625, -522.1455688476562, -35803.4140625, 11990.7041015625, 7594.08935546875, 5670.00830078125, -11266.6435546875, -13025.3974609375, 436.2503356933594, -14379.5693359375, -12949.890625, -71.04949188232422, -3066.720703125, -7000.7666015625, -13393.349609375, -2353.075927734375, -11654.685546875, 2654.203369140625, 4866.294921875, -17490.015625, -29.393476486206055, 11106.6396484375, 878.0078125, 895.5587158203125, -1955.5772705078125, 1893.4150390625, 71.66131591796875, -82.02734375, 45.43703079223633, 23241.185546875, -2698.841552734375, 4041.439453125, 3658.50927734375, 221.13320922851562, 2066.24658203125, 3840.5498046875, 9360.6357421875, 6759.64501953125, -15221.0166015625, -1580.9320068359375, 2652.5869140625, -1845.480224609375, -7754.52001953125, -24545.26171875, 1953.4488525390625, 21325.1640625, 1188.135498046875, -7526.30908203125, -7729.05078125, -5106.21337890625, -2271.0634765625, 1911.995361328125, 9442.451171875, 3392.0849609375, 171.64227294921875, 851.1060180664062, -5805.6640625, 4137.76220703125, -15697.1513671875, -5222.6806640625, 9136.091796875, -2833.048828125, 513.3333740234375, -11458.765625, 10739.71875, -16309.990234375, 2551.388427734375, 14309.404296875, -11330.6865234375, 391.9773864746094, 8204.6318359375, 3384.373779296875, -1945.076416015625, 12094.40234375, 3257.03271484375, 742.1791381835938, 5269.36181640625, -4454.80908203125, -11095.1767578125, 13905.3564453125, -4395.28125, -5567.27294921875, 5715.6201171875, -7144.45263671875, -282.21771240234375, 25135.802734375, 1946.505126953125, 144.76513671875, -10677.158203125, -3671.72265625, -170.2113800048828, -14739.517578125, 129.67578125, 247.23074340820312, 15338.44140625, 12182.244140625, -3504.998779296875, 18992.779296875, -205.9638671875, 370.37109375, -1435.61767578125, -11325.1826171875, -7625.85791015625, -23935.330078125, -1489.180908203125, 498.08197021484375, -626.6644287109375, 1997.131591796875, -803.1056518554688, 6804.41552734375, -2243.653076171875, 1334.7711181640625, 220.70916748046875, 10159.564453125, 1792.335693359375, 10521.3212890625, 2203.818359375, 793.5277099609375, -3293.793701171875, -429.10589599609375, -3233.30126953125, 5643.109375, 4776.7265625, 19687.32421875, 5262.65673828125, 1564.6981201171875, 190.20709228515625, -3280.382080078125, -2329.8994140625, 588.9087524414062, 28.243328094482422, -1484.8460693359375, 1814.6715087890625, -2346.705810546875, -7121.529296875, 25211.29296875, 22094.744140625, 13646.9404296875, 3171.5537109375, -78.5768814086914, 274.0456237792969, 19933.330078125, 18361.57421875, -6026.15869140625, 1.027300238609314, 9485.6328125, 2146.66748046875, 16848.4765625, -8960.6162109375, -1892.3388671875, 3735.076171875, -1373.90869140625, 5.695964336395264, -16603.076171875, -3786.896484375, 1675.2996826171875, -3317.87451171875, 3017.251953125, -12591.7890625, -997.847900390625, -3.8609819412231445, 141.39012145996094, 19832.828125, -13092.5595703125, 6204.4384765625, -9119.0830078125, -3026.5908203125, 1913.76318359375, -6615.95556640625, 4351.81494140625, 83.77861022949219, 6132.63037109375, 4649.01123046875, 593.37060546875, 16333.41796875, -3639.77734375, -8608.41796875], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(fma, json.loads(_TESTS), 'triton', *[1e-05, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")